In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.14.0
Device: mps


In [2]:
DATA_DIR = "../../data/processed"

X_train = np.load(f"{DATA_DIR}/train_sequences.npz")["X"]
y_train = np.load(f"{DATA_DIR}/train_sequences.npz")["y"]

X_val = np.load(f"{DATA_DIR}/validation_sequences.npz")["X"]
y_val = np.load(f"{DATA_DIR}/validation_sequences.npz")["y"]

X_test = np.load(f"{DATA_DIR}/test_sequences.npz")["X"]
y_test = np.load(f"{DATA_DIR}/test_sequences.npz")["y"]

train_target_mask = np.load(f"{DATA_DIR}/train_target_mask.npy")
val_target_mask = np.load(f"{DATA_DIR}/validation_target_mask.npy")
test_target_mask = np.load(f"{DATA_DIR}/test_target_mask.npy")

print("Training:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Target mask:", train_target_mask.shape)
print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("Target mask:", val_target_mask.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Target mask:", test_target_mask.shape)

Training:
X_train: (23973, 12, 207)
y_train: (23973, 207)
Target mask: (23973, 207)

Validation:
X_val: (3410, 12, 207)
y_val: (3410, 207)
Target mask: (3410, 207)

Test:
X_test: (6838, 12, 207)
y_test: (6838, 207)
Target mask: (6838, 207)


In [4]:


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_mask_tensor = torch.tensor(
    train_target_mask,
    dtype=torch.bool
)

val_mask_tensor = torch.tensor(
    val_target_mask,
    dtype=torch.bool
)

test_mask_tensor = torch.tensor(
    test_target_mask,
    dtype=torch.bool
)


train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
    train_mask_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor,
    val_mask_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor,
    test_mask_tensor
)


BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


sample_X, sample_y, sample_mask = next(
    iter(train_loader)
)

print("\nOne training batch:")
print("X batch shape:", sample_X.shape)
print("y batch shape:", sample_y.shape)
print("Mask batch shape:", sample_mask.shape)

Batch size: 64
Training batches: 375
Validation batches: 54
Test batches: 107

One training batch:
X batch shape: torch.Size([64, 12, 207])
y batch shape: torch.Size([64, 207])
Mask batch shape: torch.Size([64, 207])


In [5]:

INPUT_STEPS = X_train.shape[1]
NUM_SENSORS = X_train.shape[2]

D_MODEL = 128
N_HEADS = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 256
DROPOUT = 0.2

print("Input time steps:", INPUT_STEPS)
print("Number of sensors:", NUM_SENSORS)
print("Model dimension:", D_MODEL)
print("Attention heads:", N_HEADS)
print("Transformer layers:", NUM_LAYERS)
print("Feed-forward dimension:", DIM_FEEDFORWARD)
print("Dropout:", DROPOUT)

Input time steps: 12
Number of sensors: 207
Model dimension: 128
Attention heads: 4
Transformer layers: 2
Feed-forward dimension: 256
Dropout: 0.2


In [15]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=100,
        dropout=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        x = x + self.pe[:, :x.size(1), :]

        return self.dropout(x)


class TrafficTransformer(nn.Module):

    def __init__(
        self,
        num_sensors,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.2
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            num_sensors,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=INPUT_STEPS,
            dropout=dropout
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output_layer = nn.Linear(
            d_model,
            num_sensors
        )

    def forward(self, x):


        x = self.input_projection(x)


        x = self.positional_encoding(x)

        x = self.transformer(x)

        x = x[:, -1, :]

        x = self.output_layer(x)


        return x

In [16]:
transformer_model = TrafficTransformer(
    num_sensors=NUM_SENSORS,
    d_model=D_MODEL,
    nhead=N_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

print(transformer_model)

total_parameters = sum(
    parameter.numel()
    for parameter in transformer_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in transformer_model.parameters()
    if parameter.requires_grad
)

print("\nTotal parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

TrafficTransformer(
  (input_projection): Linear(in_features=207, out_features=128, bias=True)
  (positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (output_layer): Linear(in_features=128, out_features=207, bias=True)
)

Total 

In [17]:
def masked_mse_loss(predictions, targets, mask):
    """
    Calculate MSE only on valid target observations.

    predictions: [batch, sensors]
    targets:     [batch, sensors]
    mask:        [batch, sensors]
    """

    squared_error = (predictions - targets) ** 2

    valid_squared_error = squared_error[mask]

    if valid_squared_error.numel() == 0:
        return torch.tensor(
            0.0,
            device=predictions.device,
            requires_grad=True
        )

    return valid_squared_error.mean()

In [18]:

test_predictions = torch.tensor(
    [[1.0, 2.0, 3.0]]
)

test_targets = torch.tensor(
    [[1.0, 4.0, 2.0]]
)

test_mask = torch.tensor(
    [[True, True, False]]
)

test_loss = masked_mse_loss(
    test_predictions,
    test_targets,
    test_mask
)

print("Test masked MSE:", test_loss.item())

Test masked MSE: 2.0


In [19]:

NUM_EPOCHS = 20
PATIENCE = 4

BASELINE_LR = 0.001

EXPERIMENT_LRS = [
    0.0005,
    0.002
]

print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Original baseline learning rate:", BASELINE_LR)
print("New learning rates to test:", EXPERIMENT_LRS)

Maximum epochs: 20
Early stopping patience: 4
Original baseline learning rate: 0.001
New learning rates to test: [0.0005, 0.002]


In [20]:
def train_transformer_experiment(
    learning_rate,
    experiment_name,
    num_epochs=20,
    patience=4
):
    """
    Train one Transformer experiment using the
    same architecture, data, loss and evaluation
    protocol as the original Transformer.
    """

    torch.manual_seed(SEED)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(SEED)

    model = TrafficTransformer(
        num_sensors=NUM_SENSORS,
        d_model=D_MODEL,
        nhead=N_HEADS,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_val_loss = float("inf")
    patience_counter = 0

    train_history = []
    val_history = []

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()

        train_loss_total = 0.0
        train_batches = 0

        for batch_x, batch_y, batch_mask in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            optimizer.zero_grad()

            predictions = model(batch_x)

            loss = masked_mse_loss(
                predictions,
                batch_y,
                batch_mask
            )

            loss.backward()

            optimizer.step()

            train_loss_total += loss.item()
            train_batches += 1

        train_loss = (
            train_loss_total / train_batches
        )

        model.eval()

        val_loss_total = 0.0
        val_batches = 0

        with torch.no_grad():

            for batch_x, batch_y, batch_mask in val_loader:

                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                batch_mask = batch_mask.to(device)

                predictions = model(batch_x)

                loss = masked_mse_loss(
                    predictions,
                    batch_y,
                    batch_mask
                )

                val_loss_total += loss.item()
                val_batches += 1

        val_loss = (
            val_loss_total / val_batches
        )

        train_history.append(train_loss)
        val_history.append(val_loss)

        print(
            f"{experiment_name} | "
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                "  → Best model saved."
            )

        else:

            patience_counter += 1

            if patience_counter >= patience:

                print(
                    "  → Early stopping triggered."
                )

                break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()

    print(
        f"\n{experiment_name} completed."
    )

    print(
        "Best validation loss:",
        best_val_loss
    )

    print(
        "Checkpoint:",
        checkpoint_path
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "checkpoint_path": checkpoint_path
    }

In [21]:
experiment_10a = train_transformer_experiment(
    learning_rate=0.0005,
    experiment_name="transformer_lr_0005"
)

transformer_lr_0005 | Epoch 01/20 | Train Loss: 0.591366 | Val Loss: 0.498965
  → Best model saved.
transformer_lr_0005 | Epoch 02/20 | Train Loss: 0.481830 | Val Loss: 0.466874
  → Best model saved.
transformer_lr_0005 | Epoch 03/20 | Train Loss: 0.452782 | Val Loss: 0.451592
  → Best model saved.
transformer_lr_0005 | Epoch 04/20 | Train Loss: 0.434892 | Val Loss: 0.444961
  → Best model saved.
transformer_lr_0005 | Epoch 05/20 | Train Loss: 0.422083 | Val Loss: 0.442660
  → Best model saved.
transformer_lr_0005 | Epoch 06/20 | Train Loss: 0.413134 | Val Loss: 0.441790
  → Best model saved.
transformer_lr_0005 | Epoch 07/20 | Train Loss: 0.403861 | Val Loss: 0.442110
transformer_lr_0005 | Epoch 08/20 | Train Loss: 0.398122 | Val Loss: 0.440899
  → Best model saved.
transformer_lr_0005 | Epoch 09/20 | Train Loss: 0.391361 | Val Loss: 0.443730
transformer_lr_0005 | Epoch 10/20 | Train Loss: 0.385973 | Val Loss: 0.441333
transformer_lr_0005 | Epoch 11/20 | Train Loss: 0.380816 | Val Los

In [22]:
import joblib

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

experiment_10a_model = experiment_10a["model"]
experiment_10a_model.eval()

val_predictions_scaled = []
val_targets_scaled = []
val_masks = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = experiment_10a_model(batch_x)

        val_predictions_scaled.append(
            predictions.cpu().numpy()
        )

        val_targets_scaled.append(
            batch_y.numpy()
        )

        val_masks.append(
            batch_mask.numpy()
        )

val_predictions_scaled = np.concatenate(
    val_predictions_scaled,
    axis=0
)

val_targets_scaled = np.concatenate(
    val_targets_scaled,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

val_predictions = scaler.inverse_transform(
    val_predictions_scaled
)

val_targets = scaler.inverse_transform(
    val_targets_scaled
)

valid_predictions = val_predictions[val_masks]
valid_targets = val_targets[val_masks]

experiment_10a_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

experiment_10a_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print("Experiment 10A Results")
print("----------------------")
print(
    f"Validation MAE:  {experiment_10a_mae:.6f} mph"
)
print(
    f"Validation RMSE: {experiment_10a_rmse:.6f} mph"
)
print(
    "Evaluated observations:",
    len(valid_targets)
)

Experiment 10A Results
----------------------
Validation MAE:  3.398540 mph
Validation RMSE: 6.111852 mph
Evaluated observations: 664844


In [23]:
experiment_10b = train_transformer_experiment(
    learning_rate=0.002,
    experiment_name="transformer_lr_002"
)

transformer_lr_002 | Epoch 01/20 | Train Loss: 0.532821 | Val Loss: 0.483921
  → Best model saved.
transformer_lr_002 | Epoch 02/20 | Train Loss: 0.454693 | Val Loss: 0.469195
  → Best model saved.
transformer_lr_002 | Epoch 03/20 | Train Loss: 0.437900 | Val Loss: 0.461186
  → Best model saved.
transformer_lr_002 | Epoch 04/20 | Train Loss: 0.426204 | Val Loss: 0.447102
  → Best model saved.
transformer_lr_002 | Epoch 05/20 | Train Loss: 0.417495 | Val Loss: 0.463173
transformer_lr_002 | Epoch 06/20 | Train Loss: 0.412479 | Val Loss: 0.454702
transformer_lr_002 | Epoch 07/20 | Train Loss: 0.404926 | Val Loss: 0.474255
transformer_lr_002 | Epoch 08/20 | Train Loss: 0.401931 | Val Loss: 0.467252
  → Early stopping triggered.

transformer_lr_002 completed.
Best validation loss: 0.4471022741938079
Checkpoint: ../../models/transformer_lr_002_best.pth


In [24]:
experiment_10b_model = experiment_10b["model"]
experiment_10b_model.eval()

val_predictions_scaled_10b = []
val_targets_scaled_10b = []
val_masks_10b = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = experiment_10b_model(batch_x)

        val_predictions_scaled_10b.append(
            predictions.cpu().numpy()
        )

        val_targets_scaled_10b.append(
            batch_y.numpy()
        )

        val_masks_10b.append(
            batch_mask.numpy()
        )

val_predictions_scaled_10b = np.concatenate(
    val_predictions_scaled_10b,
    axis=0
)

val_targets_scaled_10b = np.concatenate(
    val_targets_scaled_10b,
    axis=0
)

val_masks_10b = np.concatenate(
    val_masks_10b,
    axis=0
)

val_predictions_10b = scaler.inverse_transform(
    val_predictions_scaled_10b
)

val_targets_10b = scaler.inverse_transform(
    val_targets_scaled_10b
)

valid_predictions_10b = val_predictions_10b[
    val_masks_10b
]

valid_targets_10b = val_targets_10b[
    val_masks_10b
]

experiment_10b_mae = mean_absolute_error(
    valid_targets_10b,
    valid_predictions_10b
)

experiment_10b_rmse = np.sqrt(
    mean_squared_error(
        valid_targets_10b,
        valid_predictions_10b
    )
)

print("Experiment 10B Results")
print("----------------------")
print(
    f"Validation MAE:  {experiment_10b_mae:.6f} mph"
)
print(
    f"Validation RMSE: {experiment_10b_rmse:.6f} mph"
)
print(
    "Evaluated observations:",
    len(valid_targets_10b)
)

Experiment 10B Results
----------------------
Validation MAE:  3.468544 mph
Validation RMSE: 6.188178 mph
Evaluated observations: 664844


In [25]:

lr_results = pd.DataFrame([
    {
        "experiment": "Original Transformer",
        "learning_rate": 0.001,
        "dropout": 0.2,
        "validation_mae": 3.4215071201324463,
        "validation_rmse": 6.177782230390839
    },
    {
        "experiment": "Transformer LR 0.0005",
        "learning_rate": 0.0005,
        "dropout": 0.2,
        "validation_mae": experiment_10a_mae,
        "validation_rmse": experiment_10a_rmse
    },
    {
        "experiment": "Transformer LR 0.002",
        "learning_rate": 0.002,
        "dropout": 0.2,
        "validation_mae": experiment_10b_mae,
        "validation_rmse": experiment_10b_rmse
    }
])

lr_results

,experiment,learning_rate,dropout,validation_mae,validation_rmse
0,Original Transformer,0.0010,0.2,3.421507,6.177782
1,Transformer LR 0.0005,0.0005,0.2,3.398540,6.111852
2,Transformer LR 0.002,0.0020,0.2,3.468544,6.188178


In [26]:
os.makedirs("../../results/experiments", exist_ok=True)

lr_results.to_csv(
    "../../results/experiments/transformer_learning_rate_results.csv",
    index=False
)

print("Saved:")
print(
    "../../results/experiments/transformer_learning_rate_results.csv"
)

Saved:
../../results/experiments/transformer_learning_rate_results.csv


In [27]:
def train_transformer_dropout_experiment(
    dropout,
    learning_rate=0.0005,
    experiment_name="transformer_dropout",
    num_epochs=20,
    patience=4
):
    """
    Train a Transformer with a specified dropout value.

    All other experimental settings remain fixed.
    """

    torch.manual_seed(SEED)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(SEED)

    model = TrafficTransformer(
        num_sensors=NUM_SENSORS,
        d_model=D_MODEL,
        nhead=N_HEADS,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_val_loss = float("inf")
    patience_counter = 0

    train_history = []
    val_history = []

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()

        train_loss_total = 0.0
        train_batches = 0

        for batch_x, batch_y, batch_mask in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            optimizer.zero_grad()

            predictions = model(batch_x)

            loss = masked_mse_loss(
                predictions,
                batch_y,
                batch_mask
            )

            loss.backward()

            optimizer.step()

            train_loss_total += loss.item()
            train_batches += 1

        train_loss = (
            train_loss_total / train_batches
        )

        model.eval()

        val_loss_total = 0.0
        val_batches = 0

        with torch.no_grad():

            for batch_x, batch_y, batch_mask in val_loader:

                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                batch_mask = batch_mask.to(device)

                predictions = model(batch_x)

                loss = masked_mse_loss(
                    predictions,
                    batch_y,
                    batch_mask
                )

                val_loss_total += loss.item()
                val_batches += 1

        val_loss = (
            val_loss_total / val_batches
        )

        train_history.append(train_loss)
        val_history.append(val_loss)

        print(
            f"{experiment_name} | "
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print("  → Best model saved.")

        else:

            patience_counter += 1

            if patience_counter >= patience:

                print(
                    "  → Early stopping triggered."
                )

                break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()

    print(
        f"\n{experiment_name} completed."
    )

    print(
        "Best validation loss:",
        best_val_loss
    )

    print(
        "Checkpoint:",
        checkpoint_path
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "checkpoint_path": checkpoint_path,
        "dropout": dropout,
        "learning_rate": learning_rate
    }

In [28]:
experiment_10c_01 = train_transformer_dropout_experiment(
    dropout=0.1,
    learning_rate=0.0005,
    experiment_name="transformer_dropout_01"
)

transformer_dropout_01 | Epoch 01/20 | Train Loss: 0.568622 | Val Loss: 0.491765
  → Best model saved.
transformer_dropout_01 | Epoch 02/20 | Train Loss: 0.462803 | Val Loss: 0.457914
  → Best model saved.
transformer_dropout_01 | Epoch 03/20 | Train Loss: 0.434333 | Val Loss: 0.444316
  → Best model saved.
transformer_dropout_01 | Epoch 04/20 | Train Loss: 0.416343 | Val Loss: 0.439735
  → Best model saved.
transformer_dropout_01 | Epoch 05/20 | Train Loss: 0.403166 | Val Loss: 0.439722
  → Best model saved.
transformer_dropout_01 | Epoch 06/20 | Train Loss: 0.392674 | Val Loss: 0.439917
transformer_dropout_01 | Epoch 07/20 | Train Loss: 0.382953 | Val Loss: 0.438306
  → Best model saved.
transformer_dropout_01 | Epoch 08/20 | Train Loss: 0.375471 | Val Loss: 0.444203
transformer_dropout_01 | Epoch 09/20 | Train Loss: 0.367899 | Val Loss: 0.442059
transformer_dropout_01 | Epoch 10/20 | Train Loss: 0.361658 | Val Loss: 0.440122
transformer_dropout_01 | Epoch 11/20 | Train Loss: 0.35625

In [29]:

experiment_10c_01_model = experiment_10c_01["model"]
experiment_10c_01_model.eval()

val_predictions_scaled_10c_01 = []
val_targets_scaled_10c_01 = []
val_masks_10c_01 = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = experiment_10c_01_model(batch_x)

        val_predictions_scaled_10c_01.append(
            predictions.cpu().numpy()
        )

        val_targets_scaled_10c_01.append(
            batch_y.numpy()
        )

        val_masks_10c_01.append(
            batch_mask.numpy()
        )

val_predictions_scaled_10c_01 = np.concatenate(
    val_predictions_scaled_10c_01,
    axis=0
)

val_targets_scaled_10c_01 = np.concatenate(
    val_targets_scaled_10c_01,
    axis=0
)

val_masks_10c_01 = np.concatenate(
    val_masks_10c_01,
    axis=0
)

val_predictions_10c_01 = scaler.inverse_transform(
    val_predictions_scaled_10c_01
)

val_targets_10c_01 = scaler.inverse_transform(
    val_targets_scaled_10c_01
)

valid_predictions_10c_01 = val_predictions_10c_01[
    val_masks_10c_01
]

valid_targets_10c_01 = val_targets_10c_01[
    val_masks_10c_01
]

experiment_10c_01_mae = mean_absolute_error(
    valid_targets_10c_01,
    valid_predictions_10c_01
)

experiment_10c_01_rmse = np.sqrt(
    mean_squared_error(
        valid_targets_10c_01,
        valid_predictions_10c_01
    )
)

print("Experiment 10C-01 Results")
print("-------------------------")
print(
    f"Dropout:         0.1"
)
print(
    f"Learning rate:   0.0005"
)
print(
    f"Validation MAE:  {experiment_10c_01_mae:.6f} mph"
)
print(
    f"Validation RMSE: {experiment_10c_01_rmse:.6f} mph"
)
print(
    "Evaluated observations:",
    len(valid_targets_10c_01)
)

Experiment 10C-01 Results
-------------------------
Dropout:         0.1
Learning rate:   0.0005
Validation MAE:  3.377603 mph
Validation RMSE: 6.079850 mph
Evaluated observations: 664844


In [30]:
experiment_10c_03 = train_transformer_dropout_experiment(
    dropout=0.3,
    learning_rate=0.0005,
    experiment_name="transformer_dropout_03"
)

transformer_dropout_03 | Epoch 01/20 | Train Loss: 0.614484 | Val Loss: 0.509693
  → Best model saved.
transformer_dropout_03 | Epoch 02/20 | Train Loss: 0.499620 | Val Loss: 0.477045
  → Best model saved.
transformer_dropout_03 | Epoch 03/20 | Train Loss: 0.470188 | Val Loss: 0.461217
  → Best model saved.
transformer_dropout_03 | Epoch 04/20 | Train Loss: 0.452227 | Val Loss: 0.453316
  → Best model saved.
transformer_dropout_03 | Epoch 05/20 | Train Loss: 0.440613 | Val Loss: 0.447249
  → Best model saved.
transformer_dropout_03 | Epoch 06/20 | Train Loss: 0.431859 | Val Loss: 0.449879
transformer_dropout_03 | Epoch 07/20 | Train Loss: 0.423408 | Val Loss: 0.450220
transformer_dropout_03 | Epoch 08/20 | Train Loss: 0.418246 | Val Loss: 0.443462
  → Best model saved.
transformer_dropout_03 | Epoch 09/20 | Train Loss: 0.412618 | Val Loss: 0.446428
transformer_dropout_03 | Epoch 10/20 | Train Loss: 0.408079 | Val Loss: 0.441339
  → Best model saved.
transformer_dropout_03 | Epoch 11/20

In [31]:
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

scaler = joblib.load("../../data/processed/traffic_scaler.pkl")

model = experiment_10c_03["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(val_predictions, axis=0)
val_targets = np.concatenate(val_targets, axis=0)
val_masks = np.concatenate(val_masks, axis=0)

val_predictions_original = scaler.inverse_transform(val_predictions)
val_targets_original = scaler.inverse_transform(val_targets)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

mae = mean_absolute_error(valid_targets, valid_predictions)
rmse = np.sqrt(mean_squared_error(valid_targets, valid_predictions))

print("Dropout 0.3 Validation MAE:", mae)
print("Dropout 0.3 Validation RMSE:", rmse)
print("Valid observations:", len(valid_targets))

Dropout 0.3 Validation MAE: 3.3777832984924316
Dropout 0.3 Validation RMSE: 6.156579023369723
Valid observations: 664844


In [32]:
dropout_results = pd.DataFrame([
    {
        "experiment": "Original Transformer",
        "learning_rate": 0.001,
        "dropout": 0.2,
        "val_mae": 3.4215071201324463,
        "val_rmse": 6.177782230390839
    },
    {
        "experiment": "Transformer LR 0.0005",
        "learning_rate": 0.0005,
        "dropout": 0.2,
        "val_mae": 3.398540,
        "val_rmse": 6.111852
    },
    {
        "experiment": "Transformer Dropout 0.1",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "Transformer Dropout 0.3",
        "learning_rate": 0.0005,
        "dropout": 0.3,
        "val_mae": 3.3777832984924316,
        "val_rmse": 6.156579023369723
    }
])

dropout_results.to_csv(
    "../../results/experiments/transformer_dropout_results.csv",
    index=False
)

dropout_results

,experiment,learning_rate,dropout,val_mae,val_rmse
0,Original Transformer,0.0010,0.2,3.421507,6.177782
1,Transformer LR 0.0005,0.0005,0.2,3.398540,6.111852
2,Transformer Dropout 0.1,0.0005,0.1,3.377603,6.079850
3,Transformer Dropout 0.3,0.0005,0.3,3.377783,6.156579


In [33]:

ARCHITECTURE_D_MODELS = [64, 256]

BEST_LEARNING_RATE = 0.0005
BEST_DROPOUT = 0.1

N_HEADS = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 256

print("Architecture experiment")
print("-----------------------")
print("Current benchmark D_MODEL:", D_MODEL)
print("New D_MODEL values:", ARCHITECTURE_D_MODELS)
print("Learning rate:", BEST_LEARNING_RATE)
print("Dropout:", BEST_DROPOUT)
print("Attention heads:", N_HEADS)
print("Transformer layers:", NUM_LAYERS)
print("Feedforward dimension:", DIM_FEEDFORWARD)

Architecture experiment
-----------------------
Current benchmark D_MODEL: 128
New D_MODEL values: [64, 256]
Learning rate: 0.0005
Dropout: 0.1
Attention heads: 4
Transformer layers: 2
Feedforward dimension: 256


In [35]:
class PositionalEncodingExperiment(nn.Module):
    def __init__(self, d_model, max_len=INPUT_STEPS):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(1, max_len, d_model)

        pe[0, :, 0::2] = torch.sin(position * div_term)

        if d_model % 2 == 0:
            pe[0, :, 1::2] = torch.cos(position * div_term)
        else:
            pe[0, :, 1::2] = torch.cos(
                position * div_term[:-1]
            )

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TrafficTransformerArchitecture(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            NUM_SENSORS,
            d_model
        )

        self.positional_encoding = PositionalEncodingExperiment(
            d_model=d_model,
            max_len=INPUT_STEPS
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output_layer = nn.Linear(
            d_model,
            NUM_SENSORS
        )

    def forward(self, x):
        x = self.input_projection(x)

        x = self.positional_encoding(x)

        x = self.transformer(x)

        x = x[:, -1, :]

        x = self.output_layer(x)

        return x


def train_transformer_architecture_experiment(
    d_model,
    experiment_name,
    learning_rate=0.0005,
    dropout=0.1,
    n_heads=4,
    num_layers=2,
    dim_feedforward=256,
    num_epochs=20,
    patience=4
):
    model = TrafficTransformerArchitecture(
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()

        train_losses = []

        for X_batch, y_batch, mask_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = masked_mse_loss(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()

            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()

        val_losses = []

        with torch.no_grad():

            for X_batch, y_batch, mask_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                mask_batch = mask_batch.to(device)

                predictions = model(X_batch)

                loss = masked_mse_loss(
                    predictions,
                    y_batch,
                    mask_batch
                )

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f} "
                f"→ Best model saved."
            )

        else:

            patience_counter += 1

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    print(
        f"\n{experiment_name} completed."
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.6f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "checkpoint": checkpoint_path,
        "d_model": d_model
    }

In [36]:
experiment_10d_01 = train_transformer_architecture_experiment(
    d_model=64,
    experiment_name="transformer_dmodel_64"
)

transformer_dmodel_64 | Epoch 01/20 | Train Loss: 0.617534 | Val Loss: 0.523178 → Best model saved.
transformer_dmodel_64 | Epoch 02/20 | Train Loss: 0.479552 | Val Loss: 0.476272 → Best model saved.
transformer_dmodel_64 | Epoch 03/20 | Train Loss: 0.447874 | Val Loss: 0.458620 → Best model saved.
transformer_dmodel_64 | Epoch 04/20 | Train Loss: 0.428716 | Val Loss: 0.444168 → Best model saved.
transformer_dmodel_64 | Epoch 05/20 | Train Loss: 0.416210 | Val Loss: 0.440499 → Best model saved.
transformer_dmodel_64 | Epoch 06/20 | Train Loss: 0.405176 | Val Loss: 0.435169 → Best model saved.
transformer_dmodel_64 | Epoch 07/20 | Train Loss: 0.397044 | Val Loss: 0.432846 → Best model saved.
transformer_dmodel_64 | Epoch 08/20 | Train Loss: 0.390678 | Val Loss: 0.433801
transformer_dmodel_64 | Epoch 09/20 | Train Loss: 0.384205 | Val Loss: 0.432065 → Best model saved.
transformer_dmodel_64 | Epoch 10/20 | Train Loss: 0.378539 | Val Loss: 0.431017 → Best model saved.
transformer_dmodel_6

In [37]:

model = experiment_10d_01["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(
            predictions.cpu().numpy()
        )
        val_targets.append(
            y_batch.cpu().numpy()
        )
        val_masks.append(
            mask_batch.cpu().numpy()
        )

val_predictions = np.concatenate(
    val_predictions,
    axis=0
)

val_targets = np.concatenate(
    val_targets,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

dmodel64_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

dmodel64_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print(
    "D_MODEL = 64 Validation MAE:",
    dmodel64_mae
)

print(
    "D_MODEL = 64 Validation RMSE:",
    dmodel64_rmse
)

print(
    "Valid observations:",
    len(valid_targets)
)

D_MODEL = 64 Validation MAE: 3.4308552742004395
D_MODEL = 64 Validation RMSE: 6.024501048594936
Valid observations: 664844


In [38]:
experiment_10d_02 = train_transformer_architecture_experiment(
    d_model=256,
    experiment_name="transformer_dmodel_256"
)

transformer_dmodel_256 | Epoch 01/20 | Train Loss: 0.513102 | Val Loss: 0.457510 → Best model saved.
transformer_dmodel_256 | Epoch 02/20 | Train Loss: 0.423332 | Val Loss: 0.434572 → Best model saved.
transformer_dmodel_256 | Epoch 03/20 | Train Loss: 0.396868 | Val Loss: 0.428660 → Best model saved.
transformer_dmodel_256 | Epoch 04/20 | Train Loss: 0.378268 | Val Loss: 0.431347
transformer_dmodel_256 | Epoch 05/20 | Train Loss: 0.362792 | Val Loss: 0.433156
transformer_dmodel_256 | Epoch 06/20 | Train Loss: 0.349044 | Val Loss: 0.435410
transformer_dmodel_256 | Epoch 07/20 | Train Loss: 0.337338 | Val Loss: 0.445537

Early stopping at epoch 7.

transformer_dmodel_256 completed.
Best validation loss: 0.428660
Best epoch: 3
Checkpoint: ../../models/transformer_dmodel_256_best.pth


In [39]:

model = experiment_10d_02["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(
            predictions.cpu().numpy()
        )
        val_targets.append(
            y_batch.cpu().numpy()
        )
        val_masks.append(
            mask_batch.cpu().numpy()
        )

val_predictions = np.concatenate(
    val_predictions,
    axis=0
)

val_targets = np.concatenate(
    val_targets,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

dmodel256_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

dmodel256_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print(
    "D_MODEL = 256 Validation MAE:",
    dmodel256_mae
)

print(
    "D_MODEL = 256 Validation RMSE:",
    dmodel256_rmse
)

print(
    "Valid observations:",
    len(valid_targets)
)

D_MODEL = 256 Validation MAE: 3.4585556983947754
D_MODEL = 256 Validation RMSE: 6.013258907142297
Valid observations: 664844


In [40]:
architecture_results = pd.DataFrame([
    {
        "experiment": "Transformer D_MODEL 64",
        "d_model": 64,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "val_mae": 3.4308552742004395,
        "val_rmse": 6.024501048594936
    },
    {
        "experiment": "Transformer D_MODEL 128",
        "d_model": 128,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "Transformer D_MODEL 256",
        "d_model": 256,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "val_mae": 3.4585556983947754,
        "val_rmse": 6.013258907142297
    }
])

architecture_results.to_csv(
    "../../results/experiments/transformer_dmodel_results.csv",
    index=False
)

architecture_results

,experiment,d_model,learning_rate,dropout,n_heads,num_layers,dim_feedforward,val_mae,val_rmse
0,Transformer D_MODEL 64,64,0.0005,0.1,4,2,256,3.430855,6.024501
1,Transformer D_MODEL 128,128,0.0005,0.1,4,2,256,3.377603,6.079850
2,Transformer D_MODEL 256,256,0.0005,0.1,4,2,256,3.458556,6.013259


In [41]:

ARCHITECTURE_NUM_LAYERS = [1, 3]

BEST_D_MODEL = 128
BEST_LEARNING_RATE = 0.0005
BEST_DROPOUT = 0.1

N_HEADS = 4
DIM_FEEDFORWARD = 256

print("Layer-depth experiment")
print("----------------------")
print("Current benchmark layers:", NUM_LAYERS)
print("New layer values:", ARCHITECTURE_NUM_LAYERS)
print("D_MODEL:", BEST_D_MODEL)
print("Learning rate:", BEST_LEARNING_RATE)
print("Dropout:", BEST_DROPOUT)
print("Attention heads:", N_HEADS)
print("Feedforward dimension:", DIM_FEEDFORWARD)

Layer-depth experiment
----------------------
Current benchmark layers: 2
New layer values: [1, 3]
D_MODEL: 128
Learning rate: 0.0005
Dropout: 0.1
Attention heads: 4
Feedforward dimension: 256


In [42]:
def train_transformer_layer_experiment(
    num_layers,
    experiment_name,
    d_model=128,
    learning_rate=0.0005,
    dropout=0.1,
    n_heads=4,
    dim_feedforward=256,
    num_epochs=20,
    patience=4
):
    model = TrafficTransformerArchitecture(
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()
        train_losses = []

        for X_batch, y_batch, mask_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = masked_mse_loss(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []

        with torch.no_grad():

            for X_batch, y_batch, mask_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                mask_batch = mask_batch.to(device)

                predictions = model(X_batch)

                loss = masked_mse_loss(
                    predictions,
                    y_batch,
                    mask_batch
                )

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f} "
                f"→ Best model saved."
            )

        else:

            patience_counter += 1

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch {epoch}."
            )

            break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    print(
        f"\n{experiment_name} completed."
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.6f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "checkpoint": checkpoint_path,
        "num_layers": num_layers
    }

In [43]:
experiment_10e_01 = train_transformer_layer_experiment(
    num_layers=1,
    experiment_name="transformer_layers_1"
)

transformer_layers_1 | Epoch 01/20 | Train Loss: 0.558397 | Val Loss: 0.481259 → Best model saved.
transformer_layers_1 | Epoch 02/20 | Train Loss: 0.446545 | Val Loss: 0.448526 → Best model saved.
transformer_layers_1 | Epoch 03/20 | Train Loss: 0.421159 | Val Loss: 0.439395 → Best model saved.
transformer_layers_1 | Epoch 04/20 | Train Loss: 0.405311 | Val Loss: 0.438382 → Best model saved.
transformer_layers_1 | Epoch 05/20 | Train Loss: 0.394241 | Val Loss: 0.429858 → Best model saved.
transformer_layers_1 | Epoch 06/20 | Train Loss: 0.384901 | Val Loss: 0.435140
transformer_layers_1 | Epoch 07/20 | Train Loss: 0.377778 | Val Loss: 0.431933
transformer_layers_1 | Epoch 08/20 | Train Loss: 0.371194 | Val Loss: 0.428898 → Best model saved.
transformer_layers_1 | Epoch 09/20 | Train Loss: 0.365422 | Val Loss: 0.433775
transformer_layers_1 | Epoch 10/20 | Train Loss: 0.360397 | Val Loss: 0.435158
transformer_layers_1 | Epoch 11/20 | Train Loss: 0.356394 | Val Loss: 0.434593
transformer

In [44]:

model = experiment_10e_01["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(val_predictions, axis=0)
val_targets = np.concatenate(val_targets, axis=0)
val_masks = np.concatenate(val_masks, axis=0)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

layers1_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

layers1_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print("1-Layer Transformer Validation MAE:", layers1_mae)
print("1-Layer Transformer Validation RMSE:", layers1_rmse)
print("Valid observations:", len(valid_targets))

1-Layer Transformer Validation MAE: 3.429508686065674
1-Layer Transformer Validation RMSE: 6.001111563397059
Valid observations: 664844


In [45]:
experiment_10e_02 = train_transformer_layer_experiment(
    num_layers=3,
    experiment_name="transformer_layers_3"
)

transformer_layers_3 | Epoch 01/20 | Train Loss: 0.556871 | Val Loss: 0.480457 → Best model saved.
transformer_layers_3 | Epoch 02/20 | Train Loss: 0.446819 | Val Loss: 0.454752 → Best model saved.
transformer_layers_3 | Epoch 03/20 | Train Loss: 0.417511 | Val Loss: 0.444123 → Best model saved.
transformer_layers_3 | Epoch 04/20 | Train Loss: 0.399426 | Val Loss: 0.433200 → Best model saved.
transformer_layers_3 | Epoch 05/20 | Train Loss: 0.383757 | Val Loss: 0.439969
transformer_layers_3 | Epoch 06/20 | Train Loss: 0.370843 | Val Loss: 0.435216
transformer_layers_3 | Epoch 07/20 | Train Loss: 0.359734 | Val Loss: 0.440119
transformer_layers_3 | Epoch 08/20 | Train Loss: 0.349892 | Val Loss: 0.436756
Early stopping at epoch 8.

transformer_layers_3 completed.
Best validation loss: 0.433200
Best epoch: 4
Checkpoint: ../../models/transformer_layers_3_best.pth


In [46]:

model = experiment_10e_02["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(val_predictions, axis=0)
val_targets = np.concatenate(val_targets, axis=0)
val_masks = np.concatenate(val_masks, axis=0)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

layers3_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

layers3_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print("3-Layer Transformer Validation MAE:", layers3_mae)
print("3-Layer Transformer Validation RMSE:", layers3_rmse)
print("Valid observations:", len(valid_targets))

3-Layer Transformer Validation MAE: 3.468191146850586
3-Layer Transformer Validation RMSE: 6.066661546865685
Valid observations: 664844


In [47]:
layer_results = pd.DataFrame([
    {
        "experiment": "Transformer Layers 1",
        "num_layers": 1,
        "d_model": 128,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.429508686065674,
        "val_rmse": 6.001111563397059
    },
    {
        "experiment": "Transformer Layers 2",
        "num_layers": 2,
        "d_model": 128,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "Transformer Layers 3",
        "num_layers": 3,
        "d_model": 128,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.468191146850586,
        "val_rmse": 6.066661546865685
    }
])

layer_results.to_csv(
    "../../results/experiments/transformer_layer_results.csv",
    index=False
)

layer_results

,experiment,num_layers,d_model,learning_rate,dropout,n_heads,dim_feedforward,val_mae,val_rmse
0,Transformer Layers 1,1,128,0.0005,0.1,4,256,3.429509,6.001112
1,Transformer Layers 2,2,128,0.0005,0.1,4,256,3.377603,6.079850
2,Transformer Layers 3,3,128,0.0005,0.1,4,256,3.468191,6.066662


In [48]:

ATTENTION_HEADS = [2, 8]

BEST_D_MODEL = 128
BEST_NUM_LAYERS = 2
BEST_LEARNING_RATE = 0.0005
BEST_DROPOUT = 0.1
BEST_DIM_FEEDFORWARD = 256

print("Attention-head experiment")
print("-------------------------")
print("Current benchmark heads:", N_HEADS)
print("New head values:", ATTENTION_HEADS)
print("D_MODEL:", BEST_D_MODEL)
print("Layers:", BEST_NUM_LAYERS)
print("Learning rate:", BEST_LEARNING_RATE)
print("Dropout:", BEST_DROPOUT)
print("Feedforward dimension:", BEST_DIM_FEEDFORWARD)

Attention-head experiment
-------------------------
Current benchmark heads: 4
New head values: [2, 8]
D_MODEL: 128
Layers: 2
Learning rate: 0.0005
Dropout: 0.1
Feedforward dimension: 256


In [49]:
def train_transformer_heads_experiment(
    n_heads,
    experiment_name,
    d_model=128,
    num_layers=2,
    learning_rate=0.0005,
    dropout=0.1,
    dim_feedforward=256,
    num_epochs=20,
    patience=4
):
    model = TrafficTransformerArchitecture(
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()
        train_losses = []

        for X_batch, y_batch, mask_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = masked_mse_loss(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []

        with torch.no_grad():

            for X_batch, y_batch, mask_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                mask_batch = mask_batch.to(device)

                predictions = model(X_batch)

                loss = masked_mse_loss(
                    predictions,
                    y_batch,
                    mask_batch
                )

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f} "
                f"→ Best model saved."
            )

        else:

            patience_counter += 1

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch {epoch}."
            )

            break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    print(
        f"\n{experiment_name} completed."
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.6f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "checkpoint": checkpoint_path,
        "n_heads": n_heads
    }

In [50]:
experiment_10f_01 = train_transformer_heads_experiment(
    n_heads=2,
    experiment_name="transformer_heads_2"
)

transformer_heads_2 | Epoch 01/20 | Train Loss: 0.558312 | Val Loss: 0.482788 → Best model saved.
transformer_heads_2 | Epoch 02/20 | Train Loss: 0.447908 | Val Loss: 0.451430 → Best model saved.
transformer_heads_2 | Epoch 03/20 | Train Loss: 0.420031 | Val Loss: 0.435285 → Best model saved.
transformer_heads_2 | Epoch 04/20 | Train Loss: 0.402213 | Val Loss: 0.432366 → Best model saved.
transformer_heads_2 | Epoch 05/20 | Train Loss: 0.389200 | Val Loss: 0.428547 → Best model saved.
transformer_heads_2 | Epoch 06/20 | Train Loss: 0.378443 | Val Loss: 0.427364 → Best model saved.
transformer_heads_2 | Epoch 07/20 | Train Loss: 0.368501 | Val Loss: 0.431488
transformer_heads_2 | Epoch 08/20 | Train Loss: 0.360513 | Val Loss: 0.431230
transformer_heads_2 | Epoch 09/20 | Train Loss: 0.352990 | Val Loss: 0.429453
transformer_heads_2 | Epoch 10/20 | Train Loss: 0.346244 | Val Loss: 0.434680
Early stopping at epoch 10.

transformer_heads_2 completed.
Best validation loss: 0.427364
Best epoc

In [52]:

model = experiment_10f_01["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(val_predictions, axis=0)
val_targets = np.concatenate(val_targets, axis=0)
val_masks = np.concatenate(val_masks, axis=0)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

heads2_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

heads2_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print("2-Head Transformer Validation MAE:", heads2_mae)
print("2-Head Transformer Validation RMSE:", heads2_rmse)
print("Valid observations:", len(valid_targets))

2-Head Transformer Validation MAE: 3.407120704650879
2-Head Transformer Validation RMSE: 5.990688410002011
Valid observations: 664844


In [53]:
experiment_10f_02 = train_transformer_heads_experiment(
    n_heads=8,
    experiment_name="transformer_heads_8"
)

transformer_heads_8 | Epoch 01/20 | Train Loss: 0.549883 | Val Loss: 0.475308 → Best model saved.
transformer_heads_8 | Epoch 02/20 | Train Loss: 0.444152 | Val Loss: 0.446485 → Best model saved.
transformer_heads_8 | Epoch 03/20 | Train Loss: 0.416390 | Val Loss: 0.437895 → Best model saved.
transformer_heads_8 | Epoch 04/20 | Train Loss: 0.398384 | Val Loss: 0.432481 → Best model saved.
transformer_heads_8 | Epoch 05/20 | Train Loss: 0.384661 | Val Loss: 0.432675
transformer_heads_8 | Epoch 06/20 | Train Loss: 0.372625 | Val Loss: 0.427704 → Best model saved.
transformer_heads_8 | Epoch 07/20 | Train Loss: 0.362718 | Val Loss: 0.432595
transformer_heads_8 | Epoch 08/20 | Train Loss: 0.353870 | Val Loss: 0.434777
transformer_heads_8 | Epoch 09/20 | Train Loss: 0.346432 | Val Loss: 0.432032
transformer_heads_8 | Epoch 10/20 | Train Loss: 0.339541 | Val Loss: 0.434447
Early stopping at epoch 10.

transformer_heads_8 completed.
Best validation loss: 0.427704
Best epoch: 6
Checkpoint: ../

In [54]:

model = experiment_10f_02["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(
    val_predictions,
    axis=0
)

val_targets = np.concatenate(
    val_targets,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

heads8_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

heads8_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print(
    "8-Head Transformer Validation MAE:",
    heads8_mae
)

print(
    "8-Head Transformer Validation RMSE:",
    heads8_rmse
)

print(
    "Valid observations:",
    len(valid_targets)
)

8-Head Transformer Validation MAE: 3.414733409881592
8-Head Transformer Validation RMSE: 5.993238772168568
Valid observations: 664844


In [55]:
heads_results = pd.DataFrame([
    {
        "experiment": "Transformer Heads 2",
        "n_heads": 2,
        "d_model": 128,
        "num_layers": 2,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "dim_feedforward": 256,
        "val_mae": 3.407120704650879,
        "val_rmse": 5.990688410002011
    },
    {
        "experiment": "Transformer Heads 4",
        "n_heads": 4,
        "d_model": 128,
        "num_layers": 2,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "dim_feedforward": 256,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "Transformer Heads 8",
        "n_heads": 8,
        "d_model": 128,
        "num_layers": 2,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "dim_feedforward": 256,
        "val_mae": 3.414733409881592,
        "val_rmse": 5.993238772168568
    }
])

heads_results.to_csv(
    "../../results/experiments/transformer_attention_heads_results.csv",
    index=False
)

heads_results

,experiment,n_heads,d_model,num_layers,learning_rate,dropout,dim_feedforward,val_mae,val_rmse
0,Transformer Heads 2,2,128,2,0.0005,0.1,256,3.407121,5.990688
1,Transformer Heads 4,4,128,2,0.0005,0.1,256,3.377603,6.079850
2,Transformer Heads 8,8,128,2,0.0005,0.1,256,3.414733,5.993239


In [56]:

FEEDFORWARD_DIMS = [128, 512]

BEST_D_MODEL = 128
BEST_NUM_LAYERS = 2
BEST_N_HEADS = 4
BEST_LEARNING_RATE = 0.0005
BEST_DROPOUT = 0.1

print("Feedforward-dimension experiment")
print("---------------------------------")
print("Current benchmark FFN:", DIM_FEEDFORWARD)
print("New FFN values:", FEEDFORWARD_DIMS)
print("D_MODEL:", BEST_D_MODEL)
print("Layers:", BEST_NUM_LAYERS)
print("Attention heads:", BEST_N_HEADS)
print("Learning rate:", BEST_LEARNING_RATE)
print("Dropout:", BEST_DROPOUT)

Feedforward-dimension experiment
---------------------------------
Current benchmark FFN: 256
New FFN values: [128, 512]
D_MODEL: 128
Layers: 2
Attention heads: 4
Learning rate: 0.0005
Dropout: 0.1


In [57]:
def train_transformer_ffn_experiment(
    dim_feedforward,
    experiment_name,
    d_model=128,
    num_layers=2,
    n_heads=4,
    learning_rate=0.0005,
    dropout=0.1,
    num_epochs=20,
    patience=4
):
    model = TrafficTransformerArchitecture(
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    checkpoint_path = (
        f"../../models/{experiment_name}_best.pth"
    )

    for epoch in range(1, num_epochs + 1):

        model.train()
        train_losses = []

        for X_batch, y_batch, mask_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = masked_mse_loss(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []

        with torch.no_grad():

            for X_batch, y_batch, mask_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                mask_batch = mask_batch.to(device)

                predictions = model(X_batch)

                loss = masked_mse_loss(
                    predictions,
                    y_batch,
                    mask_batch
                )

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f} "
                f"→ Best model saved."
            )

        else:

            patience_counter += 1

            print(
                f"{experiment_name} | "
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch {epoch}."
            )

            break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    print(
        f"\n{experiment_name} completed."
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.6f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return {
        "model": model,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "checkpoint": checkpoint_path,
        "dim_feedforward": dim_feedforward
    }

In [58]:
experiment_10g_01 = train_transformer_ffn_experiment(
    dim_feedforward=128,
    experiment_name="transformer_ffn_128"
)

transformer_ffn_128 | Epoch 01/20 | Train Loss: 0.559754 | Val Loss: 0.488681 → Best model saved.
transformer_ffn_128 | Epoch 02/20 | Train Loss: 0.449218 | Val Loss: 0.453093 → Best model saved.
transformer_ffn_128 | Epoch 03/20 | Train Loss: 0.422382 | Val Loss: 0.441240 → Best model saved.
transformer_ffn_128 | Epoch 04/20 | Train Loss: 0.405913 | Val Loss: 0.432175 → Best model saved.
transformer_ffn_128 | Epoch 05/20 | Train Loss: 0.394152 | Val Loss: 0.430050 → Best model saved.
transformer_ffn_128 | Epoch 06/20 | Train Loss: 0.383854 | Val Loss: 0.432046
transformer_ffn_128 | Epoch 07/20 | Train Loss: 0.375408 | Val Loss: 0.432786
transformer_ffn_128 | Epoch 08/20 | Train Loss: 0.368014 | Val Loss: 0.431680
transformer_ffn_128 | Epoch 09/20 | Train Loss: 0.361096 | Val Loss: 0.435366
Early stopping at epoch 9.

transformer_ffn_128 completed.
Best validation loss: 0.430050
Best epoch: 5
Checkpoint: ../../models/transformer_ffn_128_best.pth


In [59]:

model = experiment_10g_01["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(
            predictions.cpu().numpy()
        )
        val_targets.append(
            y_batch.cpu().numpy()
        )
        val_masks.append(
            mask_batch.cpu().numpy()
        )

val_predictions = np.concatenate(
    val_predictions,
    axis=0
)

val_targets = np.concatenate(
    val_targets,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

ffn128_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

ffn128_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print(
    "FFN = 128 Validation MAE:",
    ffn128_mae
)

print(
    "FFN = 128 Validation RMSE:",
    ffn128_rmse
)

print(
    "Valid observations:",
    len(valid_targets)
)

FFN = 128 Validation MAE: 3.4265899658203125
FFN = 128 Validation RMSE: 6.02333426974609
Valid observations: 664844


In [60]:
experiment_10g_02 = train_transformer_ffn_experiment(
    dim_feedforward=512,
    experiment_name="transformer_ffn_512"
)

transformer_ffn_512 | Epoch 01/20 | Train Loss: 0.548382 | Val Loss: 0.477282 → Best model saved.
transformer_ffn_512 | Epoch 02/20 | Train Loss: 0.440488 | Val Loss: 0.446954 → Best model saved.
transformer_ffn_512 | Epoch 03/20 | Train Loss: 0.412294 | Val Loss: 0.434547 → Best model saved.
transformer_ffn_512 | Epoch 04/20 | Train Loss: 0.393017 | Val Loss: 0.431239 → Best model saved.
transformer_ffn_512 | Epoch 05/20 | Train Loss: 0.377614 | Val Loss: 0.434947
transformer_ffn_512 | Epoch 06/20 | Train Loss: 0.364869 | Val Loss: 0.431668
transformer_ffn_512 | Epoch 07/20 | Train Loss: 0.353219 | Val Loss: 0.436548
transformer_ffn_512 | Epoch 08/20 | Train Loss: 0.343230 | Val Loss: 0.438010
Early stopping at epoch 8.

transformer_ffn_512 completed.
Best validation loss: 0.431239
Best epoch: 4
Checkpoint: ../../models/transformer_ffn_512_best.pth


In [61]:

model = experiment_10g_02["model"]
model.eval()

val_predictions = []
val_targets = []
val_masks = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = model(X_batch)

        val_predictions.append(predictions.cpu().numpy())
        val_targets.append(y_batch.cpu().numpy())
        val_masks.append(mask_batch.cpu().numpy())

val_predictions = np.concatenate(
    val_predictions,
    axis=0
)

val_targets = np.concatenate(
    val_targets,
    axis=0
)

val_masks = np.concatenate(
    val_masks,
    axis=0
)

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

val_predictions_original = scaler.inverse_transform(
    val_predictions
)

val_targets_original = scaler.inverse_transform(
    val_targets
)

valid_predictions = val_predictions_original[val_masks]
valid_targets = val_targets_original[val_masks]

ffn512_mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

ffn512_rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

print(
    "FFN = 512 Validation MAE:",
    ffn512_mae
)

print(
    "FFN = 512 Validation RMSE:",
    ffn512_rmse
)

print(
    "Valid observations:",
    len(valid_targets)
)

FFN = 512 Validation MAE: 3.4072234630584717
FFN = 512 Validation RMSE: 6.028953945341499
Valid observations: 664844


In [62]:
ffn_results = pd.DataFrame([
    {
        "experiment": "Transformer FFN 128",
        "dim_feedforward": 128,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "val_mae": 3.4265899658203125,
        "val_rmse": 6.02333426974609
    },
    {
        "experiment": "Transformer FFN 256",
        "dim_feedforward": 256,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "Transformer FFN 512",
        "dim_feedforward": 512,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "val_mae": 3.4072234630584717,
        "val_rmse": 6.028953945341499
    }
])

ffn_results.to_csv(
    "../../results/experiments/transformer_ffn_results.csv",
    index=False
)

ffn_results

,experiment,dim_feedforward,d_model,num_layers,n_heads,learning_rate,dropout,val_mae,val_rmse
0,Transformer FFN 128,128,128,2,4,0.0005,0.1,3.426590,6.023334
1,Transformer FFN 256,256,128,2,4,0.0005,0.1,3.377603,6.079850
2,Transformer FFN 512,512,128,2,4,0.0005,0.1,3.407223,6.028954


In [63]:

transformer_optimization_summary = pd.DataFrame([
    {
        "experiment": "Original Transformer",
        "learning_rate": 0.001,
        "dropout": 0.2,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.4215071201324463,
        "val_rmse": 6.177782230390839
    },
    {
        "experiment": "Best Learning Rate",
        "learning_rate": 0.0005,
        "dropout": 0.2,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.398540,
        "val_rmse": 6.111852
    },
    {
        "experiment": "Best Dropout",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.377603,
        "val_rmse": 6.079850
    },
    {
        "experiment": "D_MODEL 64",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 64,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.4308552742004395,
        "val_rmse": 6.024501048594936
    },
    {
        "experiment": "D_MODEL 256",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 256,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.4585556983947754,
        "val_rmse": 6.013258907142297
    },
    {
        "experiment": "Layers 1",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 1,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.429508686065674,
        "val_rmse": 6.001111563397059
    },
    {
        "experiment": "Layers 3",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 3,
        "n_heads": 4,
        "dim_feedforward": 256,
        "val_mae": 3.468191146850586,
        "val_rmse": 6.066661546865685
    },
    {
        "experiment": "Heads 2",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 2,
        "dim_feedforward": 256,
        "val_mae": 3.407120704650879,
        "val_rmse": 5.990688410002011
    },
    {
        "experiment": "Heads 8",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 8,
        "dim_feedforward": 256,
        "val_mae": 3.414733409881592,
        "val_rmse": 5.993238772168568
    },
    {
        "experiment": "FFN 128",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 128,
        "val_mae": 3.4265899658203125,
        "val_rmse": 6.02333426974609
    },
    {
        "experiment": "FFN 512",
        "learning_rate": 0.0005,
        "dropout": 0.1,
        "d_model": 128,
        "num_layers": 2,
        "n_heads": 4,
        "dim_feedforward": 512,
        "val_mae": 3.4072234630584717,
        "val_rmse": 6.028953945341499
    }
])

transformer_optimization_summary

,experiment,learning_rate,dropout,d_model,num_layers,n_heads,dim_feedforward,val_mae,val_rmse
0,Original Transformer,0.0010,0.2,128,2,4,256,3.421507,6.177782
1,Best Learning Rate,0.0005,0.2,128,2,4,256,3.398540,6.111852
2,Best Dropout,0.0005,0.1,128,2,4,256,3.377603,6.079850
3,D_MODEL 64,0.0005,0.1,64,2,4,256,3.430855,6.024501
4,D_MODEL 256,0.0005,0.1,256,2,4,256,3.458556,6.013259
5,Layers 1,0.0005,0.1,128,1,4,256,3.429509,6.001112
6,Layers 3,0.0005,0.1,128,3,4,256,3.468191,6.066662
7,Heads 2,0.0005,0.1,128,2,2,256,3.407121,5.990688
8,Heads 8,0.0005,0.1,128,2,8,256,3.414733,5.993239
9,FFN 128,0.0005,0.1,128,2,4,128,3.426590,6.023334


In [64]:

transformer_optimization_summary.to_csv(
    "../../results/experiments/transformer_optimization_summary.csv",
    index=False
)

print(
    "Saved:",
    "../../results/experiments/transformer_optimization_summary.csv"
)

print("\nSelected configuration:")
print("-----------------------")
print("Learning rate :", 0.0005)
print("Dropout       :", 0.1)
print("D_MODEL       :", 128)
print("Layers        :", 2)
print("Heads         :", 4)
print("FFN           :", 256)
print("Validation MAE:", 3.377603)
print("Validation RMSE:", 6.079850)

Saved: ../../results/experiments/transformer_optimization_summary.csv

Selected configuration:
-----------------------
Learning rate : 0.0005
Dropout       : 0.1
D_MODEL       : 128
Layers        : 2
Heads         : 4
FFN           : 256
Validation MAE: 3.377603
Validation RMSE: 6.07985


In [65]:

import numpy as np
import torch
import torch.nn as nn

adjacency = np.load("../../data/processed/adjacency_matrix.npy")

print("Adjacency shape:", adjacency.shape)
print("Non-zero connections:", np.count_nonzero(adjacency))
print("Graph density:", np.count_nonzero(adjacency) / adjacency.size)

A = torch.tensor(adjacency, dtype=torch.float32).to(device)

A_norm = A / (A.sum(dim=1, keepdim=True) + 1e-8)

print("A_norm shape:", A_norm.shape)
print("Device:", A_norm.device)

Adjacency shape: (207, 207)
Non-zero connections: 1722
Graph density: 0.04018763565077365
A_norm shape: torch.Size([207, 207])
Device: mps:0


In [66]:

class SpatialContextLayer(nn.Module):
    """
    Graph-based spatial feature extraction.

    For each timestep:
        1. Project each sensor's traffic value into a feature vector.
        2. Aggregate information from neighboring sensors.
        3. Transform the aggregated spatial information.
    """

    def __init__(self, hidden_size=64):
        super().__init__()

        self.input_projection = nn.Linear(1, hidden_size)

        self.spatial_transform = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.activation = nn.ReLU()

    def forward(self, x, A_norm):
        """
        x:
            (batch, time, sensors)

        A_norm:
            (sensors, sensors)

        returns:
            (batch, time, sensors, hidden_size)
        """

        batch_size, time_steps, num_sensors = x.shape

        h = x.unsqueeze(-1)
        h = self.input_projection(h)

        spatial_h = torch.einsum(
            "ij,btjf->btif",
            A_norm,
            h
        )

        spatial_h = self.spatial_transform(spatial_h)
        spatial_h = self.activation(spatial_h)

        return spatial_h


spatial_layer = SpatialContextLayer(hidden_size=64).to(device)

print(spatial_layer)

SpatialContextLayer(
  (input_projection): Linear(in_features=1, out_features=64, bias=True)
  (spatial_transform): Linear(in_features=64, out_features=64, bias=True)
  (activation): ReLU()
)


In [67]:

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)

with torch.no_grad():
    spatial_features = spatial_layer(X_batch, A_norm)

print("Input shape:", X_batch.shape)
print("Spatial feature shape:", spatial_features.shape)

print(
    "Expected shape:",
    f"(batch, {INPUT_STEPS}, {NUM_SENSORS}, 64)"
)

Input shape: torch.Size([64, 12, 207])
Spatial feature shape: torch.Size([64, 12, 207, 64])
Expected shape: (batch, 12, 207, 64)


In [68]:

class SpatialTemporalFusion(nn.Module):

    def __init__(
        self,
        d_model=128,
        n_heads=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        spatial_hidden=64
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            1,
            d_model
        )

        self.positional_encoding = nn.Parameter(
            torch.zeros(1, INPUT_STEPS, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.spatial_branch = SpatialContextLayer(
            hidden_size=spatial_hidden
        )

        self.spatial_projection = nn.Linear(
            spatial_hidden,
            d_model
        )

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, x, A_norm):

        batch_size, time_steps, num_sensors = x.shape



        temporal_x = x.permute(0, 2, 1)
        temporal_x = temporal_x.reshape(
            batch_size * num_sensors,
            time_steps,
            1
        )

        temporal_x = self.input_projection(
            temporal_x
        )

        temporal_x = (
            temporal_x +
            self.positional_encoding[:, :time_steps, :]
        )

        temporal_x = self.transformer(
            temporal_x
        )

        temporal_features = temporal_x[:, -1, :]

        temporal_features = temporal_features.reshape(
            batch_size,
            num_sensors,
            -1
        )


        spatial_features = self.spatial_branch(
            x,
            A_norm
        )

        spatial_features = spatial_features[:, -1, :, :]

        spatial_features = self.spatial_projection(
            spatial_features
        )


        fused_features = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused_features = self.fusion(
            fused_features
        )


        output = self.output_layer(
            fused_features
        )

        return output.squeeze(-1)


fusion_model = SpatialTemporalFusion(
    d_model=128,
    n_heads=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64
).to(device)

print(fusion_model)

SpatialTemporalFusion(
  (input_projection): Linear(in_features=1, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (spatial_branch): SpatialContextLayer(
    (input_projection): Linear(in_features=1, out_features=64, bias=True)
    (spatial_transform): Linear(in_features=64, out_featur

In [69]:

fusion_model.eval()

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)

with torch.no_grad():
    fusion_output = fusion_model(X_batch, A_norm)

print("Input shape:", X_batch.shape)
print("Output shape:", fusion_output.shape)
print("Target shape:", y_batch.shape)

print("\nExpected:")
print("Input  -> (batch, 12, 207)")
print("Output -> (batch, 207)")
print("Target -> (batch, 207)")

Input shape: torch.Size([64, 12, 207])
Output shape: torch.Size([64, 207])
Target shape: torch.Size([64, 207])

Expected:
Input  -> (batch, 12, 207)
Output -> (batch, 207)
Target -> (batch, 207)


In [70]:

trainable_params = sum(
    p.numel()
    for p in fusion_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Trainable parameters: 312385


In [71]:

NUM_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=LEARNING_RATE
)

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

train_losses = []
val_losses = []

checkpoint_path = "../../models/spatial_temporal_fusion_best.pth"

for epoch in range(1, NUM_EPOCHS + 1):


    fusion_model.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = fusion_model(
            X_batch,
            A_norm
        )

        squared_error = (
            predictions - y_batch
        ) ** 2

        masked_error = (
            squared_error * mask_batch
        )

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()

        optimizer.step()

        train_loss_sum += (
            masked_error.sum().item()
        )

        train_valid_count += (
            mask_batch.sum().item()
        )

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )


    fusion_model.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = fusion_model(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error * mask_batch
            )

            val_loss_sum += (
                masked_error.sum().item()
            )

            val_valid_count += (
                mask_batch.sum().item()
            )

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            fusion_model.state_dict(),
            checkpoint_path
        )

        print(
            f"  ✓ Best model saved "
            f"(epoch {epoch})"
        )

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break


print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Checkpoint:", checkpoint_path)

Epoch 01 | Train Loss: 0.544390 | Val Loss: 0.534728
  ✓ Best model saved (epoch 1)
Epoch 02 | Train Loss: 0.528779 | Val Loss: 0.526958
  ✓ Best model saved (epoch 2)
Epoch 03 | Train Loss: 0.524666 | Val Loss: 0.518680
  ✓ Best model saved (epoch 3)
Epoch 04 | Train Loss: 0.522436 | Val Loss: 0.521135
Epoch 05 | Train Loss: 0.521373 | Val Loss: 0.520852
Epoch 06 | Train Loss: 0.519551 | Val Loss: 0.516262
  ✓ Best model saved (epoch 6)
Epoch 07 | Train Loss: 0.518825 | Val Loss: 0.513974
  ✓ Best model saved (epoch 7)
Epoch 08 | Train Loss: 0.518287 | Val Loss: 0.518144
Epoch 09 | Train Loss: 0.518461 | Val Loss: 0.514268
Epoch 10 | Train Loss: 0.517191 | Val Loss: 0.514042
Epoch 11 | Train Loss: 0.517489 | Val Loss: 0.514101

Early stopping at epoch 11.

Training complete.
Best epoch: 7
Best validation loss: 0.5139743906764923
Checkpoint: ../../models/spatial_temporal_fusion_best.pth


In [72]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

fusion_model.load_state_dict(
    torch.load(
        "../../models/spatial_temporal_fusion_best.pth",
        map_location=device
    )
)

fusion_model.eval()

all_predictions = []
all_targets = []
all_masks = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)

        predictions = fusion_model(
            X_batch,
            A_norm
        )

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

        all_masks.append(
            mask_batch.numpy()
        )


predictions_scaled = np.concatenate(
    all_predictions,
    axis=0
)

targets_scaled = np.concatenate(
    all_targets,
    axis=0
)

validation_masks = np.concatenate(
    all_masks,
    axis=0
)

print("Predictions shape:", predictions_scaled.shape)
print("Targets shape:", targets_scaled.shape)
print("Mask shape:", validation_masks.shape)



predictions_mph = traffic_scaler.inverse_transform(
    predictions_scaled
)

targets_mph = traffic_scaler.inverse_transform(
    targets_scaled
)



valid_mask = validation_masks.astype(bool)

pred_valid = predictions_mph[valid_mask]
target_valid = targets_mph[valid_mask]

mae = mean_absolute_error(
    target_valid,
    pred_valid
)

rmse = np.sqrt(
    mean_squared_error(
        target_valid,
        pred_valid
    )
)

print("\nSpatial-Temporal Fusion Validation Results")
print("-------------------------------------------")
print(f"MAE  : {mae:.6f} mph")
print(f"RMSE : {rmse:.6f} mph")
print(f"Valid observations: {len(target_valid):,}")

Predictions shape: (3410, 207)
Targets shape: (3410, 207)
Mask shape: (3410, 207)


NameError: name 'traffic_scaler' is not defined

In [73]:

import joblib

traffic_scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

print("Scaler loaded successfully.")
print("Scaler type:", type(traffic_scaler))
print("Number of features:", traffic_scaler.n_features_in_)

Scaler loaded successfully.
Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>
Number of features: 207


In [74]:

predictions_mph = traffic_scaler.inverse_transform(
    predictions_scaled
)

targets_mph = traffic_scaler.inverse_transform(
    targets_scaled
)

valid_mask = validation_masks.astype(bool)

pred_valid = predictions_mph[valid_mask]
target_valid = targets_mph[valid_mask]

mae = mean_absolute_error(
    target_valid,
    pred_valid
)

rmse = np.sqrt(
    mean_squared_error(
        target_valid,
        pred_valid
    )
)

print("\nSpatial-Temporal Fusion Validation Results")
print("-------------------------------------------")
print(f"MAE  : {mae:.6f} mph")
print(f"RMSE : {rmse:.6f} mph")
print(f"Valid observations: {len(target_valid):,}")


Spatial-Temporal Fusion Validation Results
-------------------------------------------
MAE  : 3.668593 mph
RMSE : 6.719655 mph
Valid observations: 664,844


In [75]:

import math


class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=INPUT_STEPS):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1), :]


class CorrectedSpatialTemporalFusion(nn.Module):

    def __init__(
        self,
        d_model=128,
        n_heads=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        spatial_hidden=64
    ):
        super().__init__()


        self.input_projection = nn.Linear(
            NUM_SENSORS,
            d_model
        )

        self.positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_len=INPUT_STEPS
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.spatial_branch = SpatialContextLayer(
            hidden_size=spatial_hidden
        )

        self.spatial_projection = nn.Linear(
            spatial_hidden,
            d_model
        )


        self.fusion = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, x, A_norm):

        batch_size = x.size(0)


        temporal_features = self.input_projection(x)

        temporal_features = (
            self.positional_encoding(
                temporal_features
            )
        )

        temporal_features = self.transformer(
            temporal_features
        )

        temporal_features = (
            temporal_features[:, -1, :]
        )


        temporal_features = (
            temporal_features
            .unsqueeze(1)
            .expand(
                -1,
                NUM_SENSORS,
                -1
            )
        )


        spatial_features = self.spatial_branch(
            x,
            A_norm
        )

        spatial_features = (
            spatial_features[:, -1, :, :]
        )


        spatial_features = (
            self.spatial_projection(
                spatial_features
            )
        )


        fused_features = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused_features = self.fusion(
            fused_features
        )


        output = self.output_layer(
            fused_features
        )

        return output.squeeze(-1)


corrected_fusion_model = (
    CorrectedSpatialTemporalFusion(
        d_model=128,
        n_heads=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        spatial_hidden=64
    ).to(device)
)

print(corrected_fusion_model)

CorrectedSpatialTemporalFusion(
  (input_projection): Linear(in_features=207, out_features=128, bias=True)
  (positional_encoding): SinusoidalPositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (spatial_branch): SpatialContextLayer(
    (input_projection): Linear(in_features=1, out_features=64, bi

In [76]:

corrected_fusion_model.eval()

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)

with torch.no_grad():

    corrected_output = corrected_fusion_model(
        X_batch,
        A_norm
    )

print("Input shape :", X_batch.shape)
print("Output shape:", corrected_output.shape)
print("Target shape:", y_batch.shape)

print("\nExpected:")
print("Input  -> (64, 12, 207)")
print("Output -> (64, 207)")
print("Target -> (64, 207)")

Input shape : torch.Size([64, 12, 207])
Output shape: torch.Size([64, 207])
Target shape: torch.Size([64, 207])

Expected:
Input  -> (64, 12, 207)
Output -> (64, 207)
Target -> (64, 207)


In [77]:

NUM_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(
    corrected_fusion_model.parameters(),
    lr=LEARNING_RATE
)

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

train_losses_corrected = []
val_losses_corrected = []

checkpoint_path = (
    "../../models/"
    "corrected_spatial_temporal_fusion_best.pth"
)

for epoch in range(1, NUM_EPOCHS + 1):


    corrected_fusion_model.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = corrected_fusion_model(
            X_batch,
            A_norm
        )

        squared_error = (
            predictions - y_batch
        ) ** 2

        masked_error = (
            squared_error * mask_batch
        )

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()
        optimizer.step()

        train_loss_sum += masked_error.sum().item()
        train_valid_count += mask_batch.sum().item()

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )


    corrected_fusion_model.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = corrected_fusion_model(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error * mask_batch
            )

            val_loss_sum += masked_error.sum().item()
            val_valid_count += mask_batch.sum().item()

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    train_losses_corrected.append(train_loss)
    val_losses_corrected.append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            corrected_fusion_model.state_dict(),
            checkpoint_path
        )

        print(
            f"  ✓ Best model saved "
            f"(epoch {epoch})"
        )

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break


print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Checkpoint:", checkpoint_path)

Epoch 01 | Train Loss: 0.597939 | Val Loss: 0.575271
  ✓ Best model saved (epoch 1)
Epoch 02 | Train Loss: 0.580395 | Val Loss: 0.570841
  ✓ Best model saved (epoch 2)
Epoch 03 | Train Loss: 0.576109 | Val Loss: 0.572731
Epoch 04 | Train Loss: 0.571629 | Val Loss: 0.572467
Epoch 05 | Train Loss: 0.568789 | Val Loss: 0.569266
  ✓ Best model saved (epoch 5)
Epoch 06 | Train Loss: 0.565820 | Val Loss: 0.574352
Epoch 07 | Train Loss: 0.563343 | Val Loss: 0.572683
Epoch 08 | Train Loss: 0.561519 | Val Loss: 0.574746
Epoch 09 | Train Loss: 0.559665 | Val Loss: 0.578741

Early stopping at epoch 9.

Training complete.
Best epoch: 5
Best validation loss: 0.5692661057107475
Checkpoint: ../../models/corrected_spatial_temporal_fusion_best.pth


In [78]:

corrected_fusion_model.load_state_dict(
    torch.load(
        "../../models/"
        "corrected_spatial_temporal_fusion_best.pth",
        map_location=device
    )
)

corrected_fusion_model.eval()

all_predictions_corrected = []
all_targets_corrected = []
all_masks_corrected = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)

        predictions = corrected_fusion_model(
            X_batch,
            A_norm
        )

        all_predictions_corrected.append(
            predictions.cpu().numpy()
        )

        all_targets_corrected.append(
            y_batch.numpy()
        )

        all_masks_corrected.append(
            mask_batch.numpy()
        )


predictions_scaled_corrected = np.concatenate(
    all_predictions_corrected,
    axis=0
)

targets_scaled_corrected = np.concatenate(
    all_targets_corrected,
    axis=0
)

masks_corrected = np.concatenate(
    all_masks_corrected,
    axis=0
)

print(
    "Predictions shape:",
    predictions_scaled_corrected.shape
)

print(
    "Targets shape:",
    targets_scaled_corrected.shape
)

print(
    "Mask shape:",
    masks_corrected.shape
)



predictions_mph_corrected = (
    traffic_scaler.inverse_transform(
        predictions_scaled_corrected
    )
)

targets_mph_corrected = (
    traffic_scaler.inverse_transform(
        targets_scaled_corrected
    )
)



valid_mask_corrected = (
    masks_corrected.astype(bool)
)

pred_valid_corrected = (
    predictions_mph_corrected[
        valid_mask_corrected
    ]
)

target_valid_corrected = (
    targets_mph_corrected[
        valid_mask_corrected
    ]
)

corrected_mae = mean_absolute_error(
    target_valid_corrected,
    pred_valid_corrected
)

corrected_rmse = np.sqrt(
    mean_squared_error(
        target_valid_corrected,
        pred_valid_corrected
    )
)

print("\nCorrected Spatial-Temporal Fusion")
print("----------------------------------")
print(f"MAE  : {corrected_mae:.6f} mph")
print(f"RMSE : {corrected_rmse:.6f} mph")
print(
    f"Valid observations: "
    f"{len(target_valid_corrected):,}"
)

Predictions shape: (3410, 207)
Targets shape: (3410, 207)
Mask shape: (3410, 207)

Corrected Spatial-Temporal Fusion
----------------------------------
MAE  : 4.220636 mph
RMSE : 7.165104 mph
Valid observations: 664,844


In [79]:

class SensorSpecificSpatialTemporal(nn.Module):

    def __init__(
        self,
        hidden_size=64,
        n_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()


        self.input_projection = nn.Linear(
            1,
            hidden_size
        )

        self.positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=hidden_size,
                max_len=INPUT_STEPS
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.spatial_branch = SpatialContextLayer(
            hidden_size=hidden_size
        )


        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )


        self.output_layer = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x, A_norm):

        batch_size, time_steps, num_sensors = x.shape



        temporal_x = x.permute(
            0, 2, 1
        )

        temporal_x = temporal_x.reshape(
            batch_size * num_sensors,
            time_steps,
            1
        )

        temporal_x = self.input_projection(
            temporal_x
        )

        temporal_x = self.positional_encoding(
            temporal_x
        )

        temporal_x = self.transformer(
            temporal_x
        )

        temporal_features = (
            temporal_x[:, -1, :]
        )

        temporal_features = temporal_features.reshape(
            batch_size,
            num_sensors,
            -1
        )


        spatial_features = self.spatial_branch(
            x,
            A_norm
        )

        spatial_features = (
            spatial_features[:, -1, :, :]
        )


        fused_features = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused_features = self.fusion(
            fused_features
        )


        output = self.output_layer(
            fused_features
        )

        return output.squeeze(-1)


sensor_specific_model = SensorSpecificSpatialTemporal(
    hidden_size=64,
    n_heads=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1
).to(device)

print(sensor_specific_model)

SensorSpecificSpatialTemporal(
  (input_projection): Linear(in_features=1, out_features=64, bias=True)
  (positional_encoding): SinusoidalPositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (spatial_branch): SpatialContextLayer(
    (input_projection): Linear(in_features=1, out_features=64, bias=True)
 

In [80]:

sensor_specific_params = sum(
    p.numel()
    for p in sensor_specific_model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    sensor_specific_params
)

Trainable parameters: 79681


In [81]:

sensor_specific_model.eval()

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)

with torch.no_grad():

    sensor_output = sensor_specific_model(
        X_batch,
        A_norm
    )

print("Input shape :", X_batch.shape)
print("Output shape:", sensor_output.shape)
print("Target shape:", y_batch.shape)

print("\nExpected:")
print("Input  -> (64, 12, 207)")
print("Output -> (64, 207)")
print("Target -> (64, 207)")

Input shape : torch.Size([64, 12, 207])
Output shape: torch.Size([64, 207])
Target shape: torch.Size([64, 207])

Expected:
Input  -> (64, 12, 207)
Output -> (64, 207)
Target -> (64, 207)


In [82]:

NUM_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(
    sensor_specific_model.parameters(),
    lr=LEARNING_RATE
)

best_val_loss_sensor = float("inf")
best_epoch_sensor = 0
patience_counter_sensor = 0

train_losses_sensor = []
val_losses_sensor = []

checkpoint_path_sensor = (
    "../../models/"
    "sensor_specific_spatial_temporal_best.pth"
)

for epoch in range(1, NUM_EPOCHS + 1):


    sensor_specific_model.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = sensor_specific_model(
            X_batch,
            A_norm
        )

        squared_error = (
            predictions - y_batch
        ) ** 2

        masked_error = (
            squared_error * mask_batch
        )

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()
        optimizer.step()

        train_loss_sum += (
            masked_error.sum().item()
        )

        train_valid_count += (
            mask_batch.sum().item()
        )

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )


    sensor_specific_model.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = sensor_specific_model(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error * mask_batch
            )

            val_loss_sum += (
                masked_error.sum().item()
            )

            val_valid_count += (
                mask_batch.sum().item()
            )

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    train_losses_sensor.append(train_loss)
    val_losses_sensor.append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    if val_loss < best_val_loss_sensor:

        best_val_loss_sensor = val_loss
        best_epoch_sensor = epoch
        patience_counter_sensor = 0

        torch.save(
            sensor_specific_model.state_dict(),
            checkpoint_path_sensor
        )

        print(
            f"  ✓ Best model saved "
            f"(epoch {epoch})"
        )

    else:

        patience_counter_sensor += 1

        if patience_counter_sensor >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break


print("\nTraining complete.")
print("Best epoch:", best_epoch_sensor)
print(
    "Best validation loss:",
    best_val_loss_sensor
)
print(
    "Checkpoint:",
    checkpoint_path_sensor
)

Epoch 01 | Train Loss: 0.546600 | Val Loss: 0.524884
  ✓ Best model saved (epoch 1)
Epoch 02 | Train Loss: 0.528353 | Val Loss: 0.520566
  ✓ Best model saved (epoch 2)
Epoch 03 | Train Loss: 0.525166 | Val Loss: 0.526580
Epoch 04 | Train Loss: 0.523687 | Val Loss: 0.516509
  ✓ Best model saved (epoch 4)
Epoch 05 | Train Loss: 0.522096 | Val Loss: 0.517499
Epoch 06 | Train Loss: 0.520942 | Val Loss: 0.515456
  ✓ Best model saved (epoch 6)
Epoch 07 | Train Loss: 0.520253 | Val Loss: 0.517041
Epoch 08 | Train Loss: 0.519539 | Val Loss: 0.516093
Epoch 09 | Train Loss: 0.519425 | Val Loss: 0.516011
Epoch 10 | Train Loss: 0.518145 | Val Loss: 0.512902
  ✓ Best model saved (epoch 10)
Epoch 11 | Train Loss: 0.517641 | Val Loss: 0.513514
Epoch 12 | Train Loss: 0.517537 | Val Loss: 0.513632
Epoch 13 | Train Loss: 0.517172 | Val Loss: 0.513604
Epoch 14 | Train Loss: 0.516733 | Val Loss: 0.512111
  ✓ Best model saved (epoch 14)
Epoch 15 | Train Loss: 0.516092 | Val Loss: 0.511824
  ✓ Best model sa

In [83]:

sensor_specific_model.load_state_dict(
    torch.load(
        "../../models/"
        "sensor_specific_spatial_temporal_best.pth",
        map_location=device
    )
)

sensor_specific_model.eval()

all_predictions_sensor = []
all_targets_sensor = []
all_masks_sensor = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)

        predictions = sensor_specific_model(
            X_batch,
            A_norm
        )

        all_predictions_sensor.append(
            predictions.cpu().numpy()
        )

        all_targets_sensor.append(
            y_batch.numpy()
        )

        all_masks_sensor.append(
            mask_batch.numpy()
        )


predictions_scaled_sensor = np.concatenate(
    all_predictions_sensor,
    axis=0
)

targets_scaled_sensor = np.concatenate(
    all_targets_sensor,
    axis=0
)

masks_sensor = np.concatenate(
    all_masks_sensor,
    axis=0
)

print(
    "Predictions shape:",
    predictions_scaled_sensor.shape
)

print(
    "Targets shape:",
    targets_scaled_sensor.shape
)

print(
    "Mask shape:",
    masks_sensor.shape
)



predictions_mph_sensor = (
    traffic_scaler.inverse_transform(
        predictions_scaled_sensor
    )
)

targets_mph_sensor = (
    traffic_scaler.inverse_transform(
        targets_scaled_sensor
    )
)



valid_mask_sensor = masks_sensor.astype(bool)

pred_valid_sensor = (
    predictions_mph_sensor[
        valid_mask_sensor
    ]
)

target_valid_sensor = (
    targets_mph_sensor[
        valid_mask_sensor
    ]
)

sensor_mae = mean_absolute_error(
    target_valid_sensor,
    pred_valid_sensor
)

sensor_rmse = np.sqrt(
    mean_squared_error(
        target_valid_sensor,
        pred_valid_sensor
    )
)

print("\nSensor-Specific Spatial-Temporal Results")
print("-----------------------------------------")
print(f"MAE  : {sensor_mae:.6f} mph")
print(f"RMSE : {sensor_rmse:.6f} mph")
print(
    f"Valid observations: "
    f"{len(target_valid_sensor):,}"
)

Predictions shape: (3410, 207)
Targets shape: (3410, 207)
Mask shape: (3410, 207)

Sensor-Specific Spatial-Temporal Results
-----------------------------------------
MAE  : 3.559492 mph
RMSE : 6.678520 mph
Valid observations: 664,844


In [84]:

class SpatialResidualCorrection(nn.Module):

    def __init__(
        self,
        transformer_model,
        spatial_hidden=64,
        correction_hidden=32
    ):
        super().__init__()


        self.transformer = transformer_model

        for param in self.transformer.parameters():
            param.requires_grad = False


        self.spatial_branch = SpatialContextLayer(
            hidden_size=spatial_hidden
        )


        self.correction = nn.Sequential(
            nn.Linear(
                spatial_hidden,
                correction_hidden
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden,
                1
            )
        )

    def forward(self, x, A_norm):


        base_prediction = self.transformer(x)


        spatial_features = self.spatial_branch(
            x,
            A_norm
        )

        spatial_features = (
            spatial_features[:, -1, :, :]
        )


        correction = self.correction(
            spatial_features
        ).squeeze(-1)


        final_prediction = (
            base_prediction + correction
        )

        return final_prediction



optimized_transformer = TrafficTransformerArchitecture(
    d_model=128,
    n_heads=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

optimized_transformer.load_state_dict(
    torch.load(
        "../../models/transformer_dropout_01_best.pth",
        map_location=device
    )
)

optimized_transformer.eval()



residual_model = SpatialResidualCorrection(
    transformer_model=optimized_transformer,
    spatial_hidden=64,
    correction_hidden=32
).to(device)



total_params = sum(
    p.numel()
    for p in residual_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in residual_model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("Total parameters    :", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters   :", frozen_params)

Total parameters    : 324688
Trainable parameters: 6401
Frozen parameters   : 318287


In [85]:

residual_model.eval()

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)

with torch.no_grad():

    residual_output = residual_model(
        X_batch,
        A_norm
    )

print("Input shape :", X_batch.shape)
print("Output shape:", residual_output.shape)
print("Target shape:", y_batch.shape)

print("\nExpected:")
print("Input  -> (64, 12, 207)")
print("Output -> (64, 207)")
print("Target -> (64, 207)")

Input shape : torch.Size([64, 12, 207])
Output shape: torch.Size([64, 207])
Target shape: torch.Size([64, 207])

Expected:
Input  -> (64, 12, 207)
Output -> (64, 207)
Target -> (64, 207)


In [87]:

NUM_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        residual_model.parameters()
    ),
    lr=LEARNING_RATE
)

best_val_loss_residual = float("inf")
best_epoch_residual = 0
patience_counter_residual = 0

train_losses_residual = []
val_losses_residual = []

checkpoint_path_residual = (
    "../../models/"
    "spatial_residual_correction_best.pth"
)

for epoch in range(1, NUM_EPOCHS + 1):


    residual_model.train()

    residual_model.transformer.eval()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = residual_model(
            X_batch,
            A_norm
        )

        squared_error = (
            predictions - y_batch
        ) ** 2

        masked_error = (
            squared_error * mask_batch
        )

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()

        optimizer.step()

        train_loss_sum += (
            masked_error.sum().item()
        )

        train_valid_count += (
            mask_batch.sum().item()
        )

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )


    residual_model.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = residual_model(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error * mask_batch
            )

            val_loss_sum += (
                masked_error.sum().item()
            )

            val_valid_count += (
                mask_batch.sum().item()
            )

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    train_losses_residual.append(train_loss)
    val_losses_residual.append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    if val_loss < best_val_loss_residual:

        best_val_loss_residual = val_loss
        best_epoch_residual = epoch
        patience_counter_residual = 0

        torch.save(
            residual_model.state_dict(),
            checkpoint_path_residual
        )

        print(
            f"  ✓ Best model saved "
            f"(epoch {epoch})"
        )

    else:

        patience_counter_residual += 1

        if patience_counter_residual >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break


print("\nTraining complete.")
print(
    "Best epoch:",
    best_epoch_residual
)

print(
    "Best validation loss:",
    best_val_loss_residual
)

print(
    "Checkpoint:",
    checkpoint_path_residual
)

Epoch 01 | Train Loss: 0.358585 | Val Loss: 0.443159
  ✓ Best model saved (epoch 1)
Epoch 02 | Train Loss: 0.358512 | Val Loss: 0.443074
  ✓ Best model saved (epoch 2)
Epoch 03 | Train Loss: 0.358507 | Val Loss: 0.443199
Epoch 04 | Train Loss: 0.358508 | Val Loss: 0.442988
  ✓ Best model saved (epoch 4)
Epoch 05 | Train Loss: 0.358493 | Val Loss: 0.443253
Epoch 06 | Train Loss: 0.358521 | Val Loss: 0.443230
Epoch 07 | Train Loss: 0.358486 | Val Loss: 0.443012
Epoch 08 | Train Loss: 0.358502 | Val Loss: 0.443246

Early stopping at epoch 8.

Training complete.
Best epoch: 4
Best validation loss: 0.44298759965507417
Checkpoint: ../../models/spatial_residual_correction_best.pth


In [92]:

class ImprovedUrbanFlow(nn.Module):
    """
    Transformer + GNN with sensor-specific temporal representations
    and learnable spatial-temporal gating.
    """

    def __init__(
        self,
        num_sensors=207,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        spatial_hidden=64,
        fusion_hidden=128
    ):
        super().__init__()

        self.temporal_projection = nn.Linear(1, d_model)

        self.temporal_positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_len=INPUT_STEPS
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.spatial_branch = SpatialContextLayer(
            hidden_size=spatial_hidden
        )

        self.spatial_projection = nn.Linear(
            spatial_hidden,
            d_model
        )

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, fusion_hidden),
            nn.GELU(),
            nn.Linear(fusion_hidden, d_model),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, d_model),
            nn.GELU()
        )

        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, x, A_norm):

        B, T, S = x.shape


        temporal_input = x.permute(0, 2, 1).unsqueeze(-1)

        temporal_input = temporal_input.reshape(
            B * S,
            T,
            1
        )

        temporal_features = self.temporal_projection(
            temporal_input
        )

        temporal_features = self.temporal_positional_encoding(
            temporal_features
        )

        temporal_features = self.temporal_encoder(
            temporal_features
        )

        temporal_features = temporal_features[:, -1, :]

        temporal_features = temporal_features.reshape(
            B,
            S,
            -1
        )


        spatial_features = self.spatial_branch(
            x,
            A_norm
        )

        spatial_features = spatial_features[:, -1, :, :]

        spatial_features = self.spatial_projection(
            spatial_features
        )


        combined = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        gate = self.gate(combined)

        gated_spatial = gate * spatial_features


        fusion_input = torch.cat(
            [
                temporal_features,
                gated_spatial
            ],
            dim=-1
        )

        fused = self.fusion(
            fusion_input
        )


        output = self.output_layer(
            fused
        ).squeeze(-1)

        return output

In [94]:

improved_model = ImprovedUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64,
    fusion_hidden=128
).to(device)

total_params = sum(
    p.numel() for p in improved_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in improved_model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


X_test_batch, y_test_batch, mask_test_batch = next(iter(val_loader))

X_test_batch = X_test_batch.to(device)
y_test_batch = y_test_batch.to(device)
mask_test_batch = mask_test_batch.to(device)

improved_model.eval()

with torch.no_grad():
    test_output = improved_model(
        X_test_batch,
        A_norm
    )

print("\nShape Verification")
print("=" * 40)
print(f"Input shape :  {X_test_batch.shape}")
print(f"Target shape:  {y_test_batch.shape}")
print(f"Output shape:  {test_output.shape}")

assert test_output.shape == y_test_batch.shape

print("\n✓ Architecture verification passed.")

Total parameters:     376,769
Trainable parameters: 376,769

Shape Verification
Input shape :  torch.Size([64, 12, 207])
Target shape:  torch.Size([64, 207])
Output shape:  torch.Size([64, 207])

✓ Architecture verification passed.


In [95]:

NUM_EPOCHS = 20
PATIENCE = 4

LEARNING_RATE = 0.0005
WEIGHT_DECAY = 1e-5

CHECKPOINT_PATH = "../../models/improved_urbanflow_best.pth"

print("Improved UrbanFlow Training Configuration")
print("=" * 50)
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Early stopping: {PATIENCE}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Weight decay  : {WEIGHT_DECAY}")
print(f"Parameters    : {trainable_params:,}")
print(f"Checkpoint    : {CHECKPOINT_PATH}")

print("\nFixed Benchmark")
print("=" * 50)
print("Optimized Transformer MAE  : 3.377603 mph")
print("Optimized Transformer RMSE : 6.079850 mph")

print("\n✓ Training configuration ready.")

Improved UrbanFlow Training Configuration
Epochs        : 20
Early stopping: 4
Learning rate : 0.0005
Weight decay  : 1e-05
Parameters    : 376,769
Checkpoint    : ../../models/improved_urbanflow_best.pth

Fixed Benchmark
Optimized Transformer MAE  : 3.377603 mph
Optimized Transformer RMSE : 6.079850 mph

✓ Training configuration ready.


In [96]:

improved_model = ImprovedUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64,
    fusion_hidden=128
).to(device)

optimizer = torch.optim.AdamW(
    improved_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

train_history = []
val_history = []

for epoch in range(1, NUM_EPOCHS + 1):

    improved_model.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = improved_model(
            X_batch,
            A_norm
        )

        squared_error = (predictions - y_batch) ** 2

        masked_error = squared_error * mask_batch.float()

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            improved_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += masked_error.sum().item()
        train_valid_count += mask_batch.sum().item()

    train_loss = train_loss_sum / max(
        train_valid_count,
        1
    )

    improved_model.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = improved_model(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error
                * mask_batch.float()
            )

            val_loss_sum += masked_error.sum().item()
            val_valid_count += mask_batch.sum().item()

    val_loss = val_loss_sum / max(
        val_valid_count,
        1
    )

    train_history.append(train_loss)
    val_history.append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}",
        end=""
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            improved_model.state_dict(),
            CHECKPOINT_PATH
        )

        print("   ✓ Best model saved")

    else:

        patience_counter += 1
        print()

        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )
            break

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Epoch 01 | Train Loss: 0.541693 | Val Loss: 0.533356   ✓ Best model saved
Epoch 02 | Train Loss: 0.527385 | Val Loss: 0.519727   ✓ Best model saved
Epoch 03 | Train Loss: 0.523508 | Val Loss: 0.517468   ✓ Best model saved
Epoch 04 | Train Loss: 0.521036 | Val Loss: 0.515369   ✓ Best model saved
Epoch 05 | Train Loss: 0.519716 | Val Loss: 0.512933   ✓ Best model saved
Epoch 06 | Train Loss: 0.520303 | Val Loss: 0.515087
Epoch 07 | Train Loss: 0.518709 | Val Loss: 0.520889
Epoch 08 | Train Loss: 0.518394 | Val Loss: 0.514435
Epoch 09 | Train Loss: 0.517588 | Val Loss: 0.511530   ✓ Best model saved
Epoch 10 | Train Loss: 0.516494 | Val Loss: 0.512385
Epoch 11 | Train Loss: 0.515779 | Val Loss: 0.513319
Epoch 12 | Train Loss: 0.515418 | Val Loss: 0.513134
Epoch 13 | Train Loss: 0.515538 | Val Loss: 0.511467   ✓ Best model saved
Epoch 14 | Train Loss: 0.514651 | Val Loss: 0.510637   ✓ Best model saved
Epoch 15 | Train Loss: 0.514676 | Val Loss: 0.518905
Epoch 16 | Train Loss: 0.514469 | Val

In [97]:

import numpy as np
import torch
import joblib

CHECKPOINT_PATH = "../../models/improved_urbanflow_best.pth"

improved_model = ImprovedUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64,
    fusion_hidden=128
).to(device)

improved_model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )
)

improved_model.eval()

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

all_predictions = []
all_targets = []
all_masks = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = improved_model(
            X_batch,
            A_norm
        )

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            y_batch.cpu().numpy()
        )

        all_masks.append(
            mask_batch.cpu().numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

targets = np.concatenate(
    all_targets,
    axis=0
)

masks = np.concatenate(
    all_masks,
    axis=0
)

predictions_mph = scaler.inverse_transform(
    predictions
)

targets_mph = scaler.inverse_transform(
    targets
)

valid = masks.astype(bool)

errors = (
    predictions_mph[valid]
    - targets_mph[valid]
)

mae = np.mean(
    np.abs(errors)
)

rmse = np.sqrt(
    np.mean(errors ** 2)
)

print("Improved UrbanFlow — Validation Results")
print("=" * 55)
print(f"MAE  : {mae:.6f} mph")
print(f"RMSE : {rmse:.6f} mph")
print(f"Valid observations: {valid.sum():,}")

print("\nComparison")
print("=" * 55)
print("Optimized Transformer       : 3.377603 MAE | 6.079850 RMSE")
print("Spatial Residual Correction : 3.377559 MAE | 6.076365 RMSE")
print(
    f"Improved UrbanFlow         : "
    f"{mae:.6f} MAE | {rmse:.6f} RMSE"
)

Improved UrbanFlow — Validation Results
MAE  : 3.682885 mph
RMSE : 6.699860 mph
Valid observations: 664,844

Comparison
Optimized Transformer       : 3.377603 MAE | 6.079850 RMSE
Spatial Residual Correction : 3.377559 MAE | 6.076365 RMSE
Improved UrbanFlow         : 3.682885 MAE | 6.699860 RMSE


In [98]:

class GraphAwareAttention(nn.Module):
    """
    Adds graph-based spatial context to sensor representations
    before temporal attention.
    """

    def __init__(self, d_model, num_sensors):
        super().__init__()

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.spatial_projection = nn.Linear(
            d_model,
            d_model
        )

        self.output_projection = nn.Linear(
            d_model,
            d_model
        )

        self.scale = d_model ** -0.5

    def forward(self, x, A_norm):
        """
        x:
            [B, T, S, D]

        A_norm:
            [S, S]

        returns:
            [B, T, S, D]
        """

        spatial_context = torch.einsum(
            "ij,btjd->btid",
            A_norm,
            x
        )

        spatial_context = self.spatial_projection(
            spatial_context
        )

        x_graph = x + spatial_context

        Q = self.query(x_graph)
        K = self.key(x_graph)
        V = self.value(x_graph)

        attention_scores = torch.einsum(
            "btid,btjd->btij",
            Q,
            K
        ) * self.scale

        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        attended = torch.einsum(
            "btij,btjd->btid",
            attention_weights,
            V
        )

        output = self.output_projection(
            attended
        )

        return output

In [99]:

class GraphAwareUrbanFlow(nn.Module):
    """
    Transformer + graph-aware spatial attention.

    Temporal information is learned first for each sensor,
    then the road graph is used to model interactions between
    sensors.
    """

    def __init__(
        self,
        num_sensors=207,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.d_model = d_model


        self.input_projection = nn.Linear(
            1,
            d_model
        )

        self.positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_len=INPUT_STEPS
            )
        )


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.graph_attention = GraphAwareAttention(
            d_model=d_model,
            num_sensors=num_sensors
        )


        self.fusion = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )


        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, x, A_norm):

        B, T, S = x.shape


        temporal_input = (
            x.permute(0, 2, 1)
             .unsqueeze(-1)
        )

        temporal_input = temporal_input.reshape(
            B * S,
            T,
            1
        )

        temporal_features = self.input_projection(
            temporal_input
        )

        temporal_features = self.positional_encoding(
            temporal_features
        )

        temporal_features = self.temporal_encoder(
            temporal_features
        )

        temporal_features = temporal_features[:, -1, :]

        temporal_features = temporal_features.reshape(
            B,
            S,
            self.d_model
        )


        spatial_input = temporal_features.unsqueeze(1)

        spatial_features = self.graph_attention(
            spatial_input,
            A_norm
        )

        spatial_features = spatial_features[:, 0, :, :]


        combined = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused = self.fusion(
            combined
        )


        output = self.output_layer(
            fused
        ).squeeze(-1)

        return output

In [100]:

graph_urbanflow = GraphAwareUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

total_params = sum(
    p.numel()
    for p in graph_urbanflow.parameters()
)

trainable_params = sum(
    p.numel()
    for p in graph_urbanflow.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

X_batch, y_batch, mask_batch = next(iter(val_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)
mask_batch = mask_batch.to(device)

graph_urbanflow.eval()

with torch.no_grad():

    output = graph_urbanflow(
        X_batch,
        A_norm
    )

print("\nShape Verification")
print("=" * 40)
print(f"Input shape :  {X_batch.shape}")
print(f"Target shape:  {y_batch.shape}")
print(f"Output shape:  {output.shape}")

assert output.shape == y_batch.shape

print("\n✓ Graph-Aware UrbanFlow verification passed.")

Total parameters:     380,801
Trainable parameters: 380,801

Shape Verification
Input shape :  torch.Size([64, 12, 207])
Target shape:  torch.Size([64, 207])
Output shape:  torch.Size([64, 207])

✓ Graph-Aware UrbanFlow verification passed.


In [101]:

graph_urbanflow = GraphAwareUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(
    graph_urbanflow.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

GRAPH_CHECKPOINT = "../../models/graph_aware_urbanflow_best.pth"

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

for epoch in range(1, 21):

    graph_urbanflow.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = graph_urbanflow(
            X_batch,
            A_norm
        )

        squared_error = (
            predictions - y_batch
        ) ** 2

        masked_error = (
            squared_error *
            mask_batch.float()
        )

        loss = (
            masked_error.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            graph_urbanflow.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += masked_error.sum().item()
        train_valid_count += mask_batch.sum().item()

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )

    graph_urbanflow.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = graph_urbanflow(
                X_batch,
                A_norm
            )

            squared_error = (
                predictions - y_batch
            ) ** 2

            masked_error = (
                squared_error *
                mask_batch.float()
            )

            val_loss_sum += masked_error.sum().item()
            val_valid_count += mask_batch.sum().item()

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}",
        end=""
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            graph_urbanflow.state_dict(),
            GRAPH_CHECKPOINT
        )

        print("   ✓ Best model saved")

    else:

        patience_counter += 1
        print()

        if patience_counter >= 4:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )
            break

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Checkpoint: {GRAPH_CHECKPOINT}")

Epoch 01 | Train Loss: 0.522388 | Val Loss: 0.530397   ✓ Best model saved
Epoch 02 | Train Loss: 0.496216 | Val Loss: 0.484311   ✓ Best model saved
Epoch 03 | Train Loss: 0.490496 | Val Loss: 0.475154   ✓ Best model saved
Epoch 04 | Train Loss: 0.485068 | Val Loss: 0.478060
Epoch 05 | Train Loss: 0.482316 | Val Loss: 0.471134   ✓ Best model saved
Epoch 06 | Train Loss: 0.479876 | Val Loss: 0.471894
Epoch 07 | Train Loss: 0.476882 | Val Loss: 0.474017
Epoch 08 | Train Loss: 0.475603 | Val Loss: 0.470650   ✓ Best model saved
Epoch 09 | Train Loss: 0.473628 | Val Loss: 0.466086   ✓ Best model saved
Epoch 10 | Train Loss: 0.471759 | Val Loss: 0.466177
Epoch 11 | Train Loss: 0.469829 | Val Loss: 0.464118   ✓ Best model saved
Epoch 12 | Train Loss: 0.468077 | Val Loss: 0.463131   ✓ Best model saved
Epoch 13 | Train Loss: 0.466990 | Val Loss: 0.468400
Epoch 14 | Train Loss: 0.466196 | Val Loss: 0.465110
Epoch 15 | Train Loss: 0.464857 | Val Loss: 0.464595
Epoch 16 | Train Loss: 0.464489 | Val

In [102]:

import numpy as np
import torch
import joblib

GRAPH_CHECKPOINT = "../../models/graph_aware_urbanflow_best.pth"

graph_urbanflow = GraphAwareUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

graph_urbanflow.load_state_dict(
    torch.load(
        GRAPH_CHECKPOINT,
        map_location=device
    )
)

graph_urbanflow.eval()

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

all_predictions = []
all_targets = []
all_masks = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions = graph_urbanflow(
            X_batch,
            A_norm
        )

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            y_batch.cpu().numpy()
        )

        all_masks.append(
            mask_batch.cpu().numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

targets = np.concatenate(
    all_targets,
    axis=0
)

masks = np.concatenate(
    all_masks,
    axis=0
)

predictions_mph = scaler.inverse_transform(
    predictions
)

targets_mph = scaler.inverse_transform(
    targets
)

valid = masks.astype(bool)

errors = (
    predictions_mph[valid]
    - targets_mph[valid]
)

mae = np.mean(
    np.abs(errors)
)

rmse = np.sqrt(
    np.mean(errors ** 2)
)

print("Graph-Aware UrbanFlow — Validation Results")
print("=" * 60)
print(f"MAE  : {mae:.6f} mph")
print(f"RMSE : {rmse:.6f} mph")
print(f"Valid observations: {valid.sum():,}")

print("\nComparison")
print("=" * 60)
print("Optimized Transformer       : 3.377603 | 6.079850")
print("Spatial Residual Correction : 3.377559 | 6.076365")
print("Improved UrbanFlow          : 3.682885 | 6.699860")
print(f"Graph-Aware UrbanFlow       : {mae:.6f} | {rmse:.6f}")

Graph-Aware UrbanFlow — Validation Results
MAE  : 3.342060 mph
RMSE : 6.286065 mph
Valid observations: 664,844

Comparison
Optimized Transformer       : 3.377603 | 6.079850
Spatial Residual Correction : 3.377559 | 6.076365
Improved UrbanFlow          : 3.682885 | 6.699860
Graph-Aware UrbanFlow       : 3.342060 | 6.286065


In [103]:

import numpy as np
import pandas as pd

valid_predictions = predictions_mph[valid]
valid_targets = targets_mph[valid]

absolute_errors = np.abs(
    valid_predictions - valid_targets
)

print("Graph-Aware UrbanFlow — Error Analysis")
print("=" * 60)


print("\n1. Overall Error Statistics")
print("-" * 40)

print(f"Mean Absolute Error : {absolute_errors.mean():.6f} mph")
print(f"Median Absolute Error: {np.median(absolute_errors):.6f} mph")
print(f"90th percentile     : {np.percentile(absolute_errors, 90):.6f} mph")
print(f"95th percentile     : {np.percentile(absolute_errors, 95):.6f} mph")
print(f"99th percentile     : {np.percentile(absolute_errors, 99):.6f} mph")
print(f"Maximum error       : {absolute_errors.max():.6f} mph")


print("\n2. Large Error Frequency")
print("-" * 40)

for threshold in [5, 10, 15, 20, 30]:

    count = np.sum(
        absolute_errors > threshold
    )

    percentage = (
        count / len(absolute_errors)
    ) * 100

    print(
        f"Error > {threshold:2d} mph: "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )


print("\n3. Error by Actual Traffic Speed")
print("-" * 40)

speed_bins = [
    -np.inf,
    20,
    40,
    60,
    np.inf
]

speed_labels = [
    "≤20 mph",
    "20–40 mph",
    "40–60 mph",
    ">60 mph"
]

speed_categories = pd.cut(
    valid_targets,
    bins=speed_bins,
    labels=speed_labels
)

speed_analysis = (
    pd.DataFrame({
        "speed_category": speed_categories,
        "absolute_error": absolute_errors
    })
    .groupby(
        "speed_category",
        observed=False
    )
    .agg(
        observations=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_error=("absolute_error", "median"),
        p95_error=(
            "absolute_error",
            lambda x: np.percentile(x, 95)
        )
    )
)

print(speed_analysis)


print("\n4. Prediction Bias")
print("-" * 40)

signed_errors = (
    valid_predictions - valid_targets
)

print(
    f"Mean signed error: "
    f"{signed_errors.mean():.6f} mph"
)

print(
    f"Mean prediction: "
    f"{valid_predictions.mean():.6f} mph"
)

print(
    f"Mean target: "
    f"{valid_targets.mean():.6f} mph"
)


print("\n5. Interpretation")
print("-" * 40)

if np.percentile(absolute_errors, 95) > 15:
    print(
        "Large errors are concentrated in the upper "
        "tail and should be investigated further."
    )
else:
    print(
        "The upper-tail error appears relatively "
        "controlled."
    )

if signed_errors.mean() > 0.5:
    print(
        "The model shows an overall tendency to "
        "over-predict speed."
    )
elif signed_errors.mean() < -0.5:
    print(
        "The model shows an overall tendency to "
        "under-predict speed."
    )
else:
    print(
        "There is little overall prediction bias."
    )

Graph-Aware UrbanFlow — Error Analysis

1. Overall Error Statistics
----------------------------------------
Mean Absolute Error : 3.342060 mph
Median Absolute Error: 1.589960 mph
90th percentile     : 7.831393 mph
95th percentile     : 12.823618 mph
99th percentile     : 28.535192 mph
Maximum error       : 66.388519 mph

2. Large Error Frequency
----------------------------------------
Error >  5 mph: 116,500 (17.52%)
Error > 10 mph: 47,175 (7.10%)
Error > 15 mph: 26,053 (3.92%)
Error > 20 mph: 15,755 (2.37%)
Error > 30 mph: 5,670 (0.85%)

3. Error by Actual Traffic Speed
----------------------------------------
                observations        mae  median_error  p95_error
speed_category                                                  
≤20 mph                20467  11.426243      6.737080  40.813141
20–40 mph              48895   8.439431      5.481743  27.319895
40–60 mph             138884   4.511717      2.980499  14.062309
>60 mph               456598   2.078056      1.195404 

In [104]:

print("Complete Error by Actual Traffic Speed")
print("=" * 70)

print(
    speed_analysis.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nKey Finding")
print("=" * 70)

worst_category = speed_analysis["mae"].idxmax()
worst_mae = speed_analysis["mae"].max()

print(
    f"Highest MAE category: {worst_category}"
)
print(
    f"Category MAE: {worst_mae:.6f} mph"
)

Complete Error by Actual Traffic Speed
                observations       mae  median_error  p95_error
speed_category                                                 
≤20 mph                20467 11.426243      6.737080  40.813141
20–40 mph              48895  8.439431      5.481743  27.319895
40–60 mph             138884  4.511717      2.980499  14.062309
>60 mph               456598  2.078056      1.195404   6.359351

Key Finding
Highest MAE category: ≤20 mph
Category MAE: 11.426243 mph


In [105]:

benchmark_transformer = TrafficTransformer(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

benchmark_transformer.load_state_dict(
    torch.load(
        "../../models/transformer_dropout_01_best.pth",
        map_location=device
    )
)

benchmark_transformer.eval()

all_predictions_benchmark = []
all_targets_benchmark = []
all_masks_benchmark = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions_batch = benchmark_transformer(
            X_batch
        )

        all_predictions_benchmark.append(
            predictions_batch.cpu().numpy()
        )

        all_targets_benchmark.append(
            y_batch.cpu().numpy()
        )

        all_masks_benchmark.append(
            mask_batch.cpu().numpy()
        )

benchmark_predictions = np.concatenate(
    all_predictions_benchmark,
    axis=0
)

benchmark_targets = np.concatenate(
    all_targets_benchmark,
    axis=0
)

benchmark_masks = np.concatenate(
    all_masks_benchmark,
    axis=0
)

benchmark_predictions_mph = scaler.inverse_transform(
    benchmark_predictions
)

benchmark_targets_mph = scaler.inverse_transform(
    benchmark_targets
)

valid_benchmark = benchmark_masks.astype(bool)

benchmark_errors = (
    benchmark_predictions_mph[valid_benchmark]
    - benchmark_targets_mph[valid_benchmark]
)

benchmark_absolute_errors = np.abs(
    benchmark_errors
)

benchmark_speed_categories = pd.cut(
    benchmark_targets_mph[valid_benchmark],
    bins=[
        -np.inf,
        20,
        40,
        60,
        np.inf
    ],
    labels=[
        "≤20 mph",
        "20–40 mph",
        "40–60 mph",
        ">60 mph"
    ]
)

benchmark_speed_analysis = (
    pd.DataFrame({
        "speed_category": benchmark_speed_categories,
        "absolute_error": benchmark_absolute_errors
    })
    .groupby(
        "speed_category",
        observed=False
    )
    .agg(
        observations=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_error=("absolute_error", "median"),
        p95_error=(
            "absolute_error",
            lambda x: np.percentile(x, 95)
        )
    )
)

print("Optimized Transformer — Error by Actual Traffic Speed")
print("=" * 70)

print(
    benchmark_speed_analysis.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

Optimized Transformer — Error by Actual Traffic Speed
                observations       mae  median_error  p95_error
speed_category                                                 
≤20 mph                20467 12.813415      8.966625  39.190498
20–40 mph              48895  8.472963      5.889683  25.549578
40–60 mph             138884  4.368779      3.049135  12.935266
>60 mph               456598  2.107517      1.303514   6.435939


In [106]:

HUBER_BETA = 1.0

GRAPH_HUBER_CHECKPOINT = (
    "../../models/graph_aware_urbanflow_huber_best.pth"
)

print("Graph-Aware UrbanFlow — Robust Loss Experiment")
print("=" * 60)

print("Architecture : Graph-Aware UrbanFlow")
print("Loss         : Huber / SmoothL1")
print(f"Huber beta   : {HUBER_BETA}")
print("Learning rate: 0.0005")
print("Weight decay : 1e-5")
print("Max epochs   : 20")
print("Patience     : 4")

print("\nFixed benchmarks")
print("=" * 60)
print("Optimized Transformer : 3.377603 MAE | 6.079850 RMSE")
print("Graph-Aware + MSE     : 3.342060 MAE | 6.286065 RMSE")

print("\n✓ Robust-loss experiment configured.")

Graph-Aware UrbanFlow — Robust Loss Experiment
Architecture : Graph-Aware UrbanFlow
Loss         : Huber / SmoothL1
Huber beta   : 1.0
Learning rate: 0.0005
Weight decay : 1e-5
Max epochs   : 20
Patience     : 4

Fixed benchmarks
Optimized Transformer : 3.377603 MAE | 6.079850 RMSE
Graph-Aware + MSE     : 3.342060 MAE | 6.286065 RMSE

✓ Robust-loss experiment configured.


In [107]:

graph_urbanflow_huber = GraphAwareUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(
    graph_urbanflow_huber.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

criterion = nn.SmoothL1Loss(
    beta=HUBER_BETA,
    reduction="none"
)

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

for epoch in range(1, 21):


    graph_urbanflow_huber.train()

    train_loss_sum = 0.0
    train_valid_count = 0

    for X_batch, y_batch, mask_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        predictions = graph_urbanflow_huber(
            X_batch,
            A_norm
        )

        element_loss = criterion(
            predictions,
            y_batch
        )

        masked_loss = (
            element_loss *
            mask_batch.float()
        )

        loss = (
            masked_loss.sum()
            / mask_batch.sum().clamp_min(1.0)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            graph_urbanflow_huber.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += masked_loss.sum().item()
        train_valid_count += mask_batch.sum().item()

    train_loss = (
        train_loss_sum /
        max(train_valid_count, 1)
    )


    graph_urbanflow_huber.eval()

    val_loss_sum = 0.0
    val_valid_count = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            predictions = graph_urbanflow_huber(
                X_batch,
                A_norm
            )

            element_loss = criterion(
                predictions,
                y_batch
            )

            masked_loss = (
                element_loss *
                mask_batch.float()
            )

            val_loss_sum += masked_loss.sum().item()
            val_valid_count += mask_batch.sum().item()

    val_loss = (
        val_loss_sum /
        max(val_valid_count, 1)
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}",
        end=""
    )


    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            graph_urbanflow_huber.state_dict(),
            GRAPH_HUBER_CHECKPOINT
        )

        print("   ✓ Best model saved")

    else:

        patience_counter += 1
        print()

        if patience_counter >= 4:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )
            break

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Checkpoint: {GRAPH_HUBER_CHECKPOINT}")

Epoch 01 | Train Loss: 0.181340 | Val Loss: 0.166410   ✓ Best model saved
Epoch 02 | Train Loss: 0.172105 | Val Loss: 0.168764
Epoch 03 | Train Loss: 0.170073 | Val Loss: 0.163133   ✓ Best model saved
Epoch 04 | Train Loss: 0.168584 | Val Loss: 0.162783   ✓ Best model saved
Epoch 05 | Train Loss: 0.167362 | Val Loss: 0.160020   ✓ Best model saved
Epoch 06 | Train Loss: 0.167288 | Val Loss: 0.162798
Epoch 07 | Train Loss: 0.166166 | Val Loss: 0.158759   ✓ Best model saved
Epoch 08 | Train Loss: 0.165578 | Val Loss: 0.158832
Epoch 09 | Train Loss: 0.165314 | Val Loss: 0.160512
Epoch 10 | Train Loss: 0.164443 | Val Loss: 0.159337
Epoch 11 | Train Loss: 0.164391 | Val Loss: 0.158487   ✓ Best model saved
Epoch 12 | Train Loss: 0.164044 | Val Loss: 0.158789
Epoch 13 | Train Loss: 0.163610 | Val Loss: 0.157777   ✓ Best model saved
Epoch 14 | Train Loss: 0.163280 | Val Loss: 0.159391
Epoch 15 | Train Loss: 0.162974 | Val Loss: 0.157560   ✓ Best model saved
Epoch 16 | Train Loss: 0.162625 | Val

In [108]:

import numpy as np
import joblib
import torch

GRAPH_HUBER_CHECKPOINT = (
    "../../models/graph_aware_urbanflow_huber_best.pth"
)

graph_urbanflow_huber = GraphAwareUrbanFlow(
    num_sensors=NUM_SENSORS,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

graph_urbanflow_huber.load_state_dict(
    torch.load(
        GRAPH_HUBER_CHECKPOINT,
        map_location=device
    )
)

graph_urbanflow_huber.eval()

scaler = joblib.load(
    "../../data/processed/traffic_scaler.pkl"
)

all_predictions_huber = []
all_targets_huber = []
all_masks_huber = []

with torch.no_grad():

    for X_batch, y_batch, mask_batch in val_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        predictions_batch = graph_urbanflow_huber(
            X_batch,
            A_norm
        )

        all_predictions_huber.append(
            predictions_batch.cpu().numpy()
        )

        all_targets_huber.append(
            y_batch.cpu().numpy()
        )

        all_masks_huber.append(
            mask_batch.cpu().numpy()
        )

huber_predictions = np.concatenate(
    all_predictions_huber,
    axis=0
)

huber_targets = np.concatenate(
    all_targets_huber,
    axis=0
)

huber_masks = np.concatenate(
    all_masks_huber,
    axis=0
)

huber_predictions_mph = scaler.inverse_transform(
    huber_predictions
)

huber_targets_mph = scaler.inverse_transform(
    huber_targets
)

valid_huber = huber_masks.astype(bool)

huber_errors = (
    huber_predictions_mph[valid_huber]
    - huber_targets_mph[valid_huber]
)

huber_mae = np.mean(
    np.abs(huber_errors)
)

huber_rmse = np.sqrt(
    np.mean(huber_errors ** 2)
)

print("Graph-Aware UrbanFlow + Huber — Validation Results")
print("=" * 65)
print(f"MAE  : {huber_mae:.6f} mph")
print(f"RMSE : {huber_rmse:.6f} mph")
print(f"Valid observations: {valid_huber.sum():,}")

print("\nComparison")
print("=" * 65)
print("Optimized Transformer       : 3.377603 | 6.079850")
print("Graph-Aware + MSE           : 3.342060 | 6.286065")
print(
    f"Graph-Aware + Huber         : "
    f"{huber_mae:.6f} | {huber_rmse:.6f}"
)

Graph-Aware UrbanFlow + Huber — Validation Results
MAE  : 3.177278 mph
RMSE : 6.316968 mph
Valid observations: 664,844

Comparison
Optimized Transformer       : 3.377603 | 6.079850
Graph-Aware + MSE           : 3.342060 | 6.286065
Graph-Aware + Huber         : 3.177278 | 6.316968


In [109]:

huber_absolute_errors = np.abs(
    huber_errors
)

huber_speed_categories = pd.cut(
    huber_targets_mph[valid_huber],
    bins=[
        -np.inf,
        20,
        40,
        60,
        np.inf
    ],
    labels=[
        "≤20 mph",
        "20–40 mph",
        "40–60 mph",
        ">60 mph"
    ]
)

huber_speed_analysis = (
    pd.DataFrame({
        "speed_category": huber_speed_categories,
        "absolute_error": huber_absolute_errors
    })
    .groupby(
        "speed_category",
        observed=False
    )
    .agg(
        observations=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_error=("absolute_error", "median"),
        p95_error=(
            "absolute_error",
            lambda x: np.percentile(x, 95)
        )
    )
)

print("Graph-Aware UrbanFlow + Huber")
print("Error by Actual Traffic Speed")
print("=" * 70)

print(
    huber_speed_analysis.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nOverall")
print("=" * 70)

print(f"MAE  : {huber_mae:.6f} mph")
print(f"RMSE : {huber_rmse:.6f} mph")
print(f"Valid observations: {valid_huber.sum():,}")

Graph-Aware UrbanFlow + Huber
Error by Actual Traffic Speed
                observations       mae  median_error  p95_error
speed_category                                                 
≤20 mph                20467 10.797938      5.250042  43.396030
20–40 mph              48895  8.677564      5.190485  29.390656
40–60 mph             138884  4.570642      2.993361  14.340136
>60 mph               456598  1.822859      1.062218   5.218719

Overall
MAE  : 3.177278 mph
RMSE : 6.316968 mph
Valid observations: 664,844


In [110]:

phase2_results = pd.DataFrame([
    {
        "experiment": "Initial Spatial-Temporal Fusion",
        "mae_mph": 3.668593,
        "rmse_mph": 6.719655
    },
    {
        "experiment": "Corrected Spatial-Temporal Fusion",
        "mae_mph": 4.220636,
        "rmse_mph": 7.165104
    },
    {
        "experiment": "Sensor-Specific Spatial-Temporal",
        "mae_mph": 3.559492,
        "rmse_mph": 6.678520
    },
    {
        "experiment": "Spatial Residual Correction",
        "mae_mph": 3.377559,
        "rmse_mph": 6.076365
    },
    {
        "experiment": "Improved UrbanFlow",
        "mae_mph": 3.682885,
        "rmse_mph": 6.699860
    },
    {
        "experiment": "Graph-Aware UrbanFlow + MSE",
        "mae_mph": 3.342060,
        "rmse_mph": 6.286065
    },
    {
        "experiment": "Graph-Aware UrbanFlow + Huber",
        "mae_mph": 3.177278,
        "rmse_mph": 6.316968
    }
])

phase2_results["valid_observations"] = 664844

phase2_path = (
    "../../results/experiments/"
    "urbanflow_spatial_temporal_experiments.csv"
)

phase2_results.to_csv(
    phase2_path,
    index=False
)

print("Phase 2 Experiment Summary")
print("=" * 60)
print(phase2_results.to_string(index=False))

print(f"\nSaved to: {phase2_path}")
print("\n✓ Phase 2 experiment tracking complete.")

Phase 2 Experiment Summary
                       experiment  mae_mph  rmse_mph  valid_observations
  Initial Spatial-Temporal Fusion 3.668593  6.719655              664844
Corrected Spatial-Temporal Fusion 4.220636  7.165104              664844
 Sensor-Specific Spatial-Temporal 3.559492  6.678520              664844
      Spatial Residual Correction 3.377559  6.076365              664844
               Improved UrbanFlow 3.682885  6.699860              664844
      Graph-Aware UrbanFlow + MSE 3.342060  6.286065              664844
    Graph-Aware UrbanFlow + Huber 3.177278  6.316968              664844

Saved to: ../../results/experiments/urbanflow_spatial_temporal_experiments.csv

✓ Phase 2 experiment tracking complete.


In [111]:

WEATHER_CONFIG = {
    "traffic_dataset": "METR-LA",
    "traffic_sampling_minutes": 5,
    "input_history_minutes": 60,
    "input_steps": 12,
    "forecast_horizon_minutes": 30,

    "temporal_alignment": "traffic_timestamp",

    "candidate_features": [
        "temperature",
        "precipitation",
        "wind_speed",
        "visibility",
        "pressure",
        "weather_condition"
    ],

    "use_future_weather": False,
    "weather_cutoff": "forecast_origin",

    "baseline_model": "Graph-Aware UrbanFlow + Huber",
    "baseline_mae_mph": 3.177278,
    "baseline_rmse_mph": 6.316968,

    "test_set_access": False
}

print("Phase 3 — Weather Integration Strategy")
print("=" * 60)

for key, value in WEATHER_CONFIG.items():
    print(f"{key}: {value}")

print("\nExperimental Principle")
print("=" * 60)
print(
    "Weather will be added as an external context feature "
    "without changing the existing traffic/graph pipeline initially."
)

print(
    "\nWeather observations must be temporally aligned with "
    "the traffic data and must not contain information from "
    "after the forecast origin."
)

print(
    "\n✓ Weather strategy recorded."
)

Phase 3 — Weather Integration Strategy
traffic_dataset: METR-LA
traffic_sampling_minutes: 5
input_history_minutes: 60
input_steps: 12
forecast_horizon_minutes: 30
temporal_alignment: traffic_timestamp
candidate_features: ['temperature', 'precipitation', 'wind_speed', 'visibility', 'pressure', 'weather_condition']
use_future_weather: False
weather_cutoff: forecast_origin
baseline_model: Graph-Aware UrbanFlow + Huber
baseline_mae_mph: 3.177278
baseline_rmse_mph: 6.316968
test_set_access: False

Experimental Principle
Weather will be added as an external context feature without changing the existing traffic/graph pipeline initially.

Weather observations must be temporally aligned with the traffic data and must not contain information from after the forecast origin.

✓ Weather strategy recorded.


In [113]:

from pathlib import Path
import pandas as pd
import requests


WEATHER_STATION = "72295"   
WEATHER_YEAR = 2012

weather_dir = Path("../../data/raw/weather")
weather_dir.mkdir(parents=True, exist_ok=True)

weather_file = weather_dir / f"{WEATHER_STATION}_{WEATHER_YEAR}.csv.gz"

weather_url = (
    f"https://data.meteostat.net/hourly/"
    f"{WEATHER_YEAR}/{WEATHER_STATION}.csv.gz"
)

print("Weather station :", WEATHER_STATION)
print("Location         : Los Angeles International Airport")
print("Year             :", WEATHER_YEAR)
print("Download URL     :", weather_url)
print("Output file      :", weather_file)


if weather_file.exists():
    print("\nWeather file already exists. Skipping download.")

else:
    print("\nDownloading weather data...")

    response = requests.get(weather_url, timeout=120)
    response.raise_for_status()

    weather_file.write_bytes(response.content)

    print("Download completed.")


weather_df = pd.read_csv(
    weather_file,
    compression="gzip"
)

print("\nWeather data loaded successfully.")
print("Shape:", weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nFirst 5 rows:")
display(weather_df.head())

print("\nData types:")
display(weather_df.dtypes)


weather_dates = pd.to_datetime(
    weather_df[["year", "month", "day", "hour"]]
)

print("\nWeather date coverage:")
print("Start:", weather_dates.min())
print("End  :", weather_dates.max())

print("\nNumber of observations:", len(weather_df))

Weather station : 72295
Location         : Los Angeles International Airport
Year             : 2012
Download URL     : https://data.meteostat.net/hourly/2012/72295.csv.gz
Output file      : ../../data/raw/weather/72295_2012.csv.gz

Download completed.

Weather data loaded successfully.
Shape: (8784, 22)

Columns:
['year', 'month', 'day', 'hour', 'temp', 'temp_source', 'rhum', 'rhum_source', 'prcp', 'prcp_source', 'wdir', 'wdir_source', 'wspd', 'wspd_source', 'wpgt', 'wpgt_source', 'pres', 'pres_source', 'cldc', 'cldc_source', 'coco', 'coco_source']

First 5 rows:


,year,month,day,hour,temp,temp_source,rhum,rhum_source,prcp,prcp_source,...,wspd,wspd_source,wpgt,wpgt_source,pres,pres_source,cldc,cldc_source,coco,coco_source
0,2012,1,1,0,12.8,isd_lite,86.0,isd_lite,NaN,NaN,...,16.6,isd_lite,NaN,NaN,1016.4,isd_lite,6.0,isd_lite,NaN,NaN
1,2012,1,1,1,11.7,isd_lite,89.0,isd_lite,0.0,isd_lite,...,16.6,isd_lite,NaN,NaN,1016.7,isd_lite,8.0,isd_lite,NaN,NaN
2,2012,1,1,2,11.1,isd_lite,93.0,isd_lite,0.0,isd_lite,...,9.4,isd_lite,NaN,NaN,1017.1,isd_lite,8.0,isd_lite,NaN,NaN
3,2012,1,1,3,11.1,isd_lite,97.0,isd_lite,0.0,isd_lite,...,0.0,isd_lite,NaN,NaN,1017.6,isd_lite,7.0,isd_lite,NaN,NaN
4,2012,1,1,4,11.1,isd_lite,89.0,isd_lite,0.0,isd_lite,...,0.0,isd_lite,NaN,NaN,1018.0,isd_lite,7.0,isd_lite,NaN,NaN



Data types:


year             int64
month            int64
day              int64
hour             int64
temp           float64
temp_source        str
rhum           float64
rhum_source        str
prcp           float64
prcp_source        str
wdir           float64
wdir_source        str
wspd           float64
wspd_source        str
wpgt           float64
wpgt_source    float64
pres           float64
pres_source        str
cldc           float64
cldc_source        str
coco           float64
coco_source    float64
dtype: object


Weather date coverage:
Start: 2012-01-01 00:00:00
End  : 2012-12-31 23:00:00

Number of observations: 8784


In [114]:


weather_df["timestamp"] = pd.to_datetime(
    weather_df[["year", "month", "day", "hour"]]
)


TRAFFIC_START = pd.Timestamp("2012-03-01 00:00:00")
TRAFFIC_END   = pd.Timestamp("2012-06-27 23:00:00")

weather_period = weather_df[
    (weather_df["timestamp"] >= TRAFFIC_START) &
    (weather_df["timestamp"] <= TRAFFIC_END)
].copy()

print("=" * 70)
print("WEATHER COVERAGE FOR METR-LA PERIOD")
print("=" * 70)

print("Traffic period start :", TRAFFIC_START)
print("Traffic period end   :", TRAFFIC_END)

print("\nWeather period start :", weather_period["timestamp"].min())
print("Weather period end   :", weather_period["timestamp"].max())

print("\nExpected hourly observations:", 119 * 24)
print("Actual hourly observations  :", len(weather_period))


duplicate_timestamps = weather_period["timestamp"].duplicated().sum()

print("\nDuplicate timestamps:", duplicate_timestamps)


candidate_features = [
    "temp",
    "rhum",
    "prcp",
    "wdir",
    "wspd",
    "pres",
    "cldc",
    "coco"
]

missing_summary = pd.DataFrame({
    "missing_count": weather_period[candidate_features].isna().sum(),
    "missing_percent": (
        weather_period[candidate_features].isna().mean() * 100
    )
})

print("\nMissing values:")
display(missing_summary)


print("\nWeather feature statistics:")
display(
    weather_period[candidate_features].describe().T
)


source_columns = [
    "temp_source",
    "rhum_source",
    "prcp_source",
    "wdir_source",
    "wspd_source",
    "wpgt_source",
    "pres_source",
    "cldc_source",
    "coco_source"
]

available_source_columns = [
    col for col in source_columns
    if col in weather_period.columns
]

print("\nSource provenance:")
for col in available_source_columns:
    print(f"\n{col}:")
    print(weather_period[col].value_counts(dropna=False).head(10))


print("\n" + "=" * 70)
print("WEATHER QUALITY CHECK COMPLETE")
print("=" * 70)

print("Rows available for METR-LA period:", len(weather_period))
print("Expected rows                  :", 2856)
print("Duplicate timestamps           :", duplicate_timestamps)

WEATHER COVERAGE FOR METR-LA PERIOD
Traffic period start : 2012-03-01 00:00:00
Traffic period end   : 2012-06-27 23:00:00

Weather period start : 2012-03-01 00:00:00
Weather period end   : 2012-06-27 23:00:00

Expected hourly observations: 2856
Actual hourly observations  : 2856

Duplicate timestamps: 0

Missing values:


,missing_count,missing_percent
temp,0,0.000000
rhum,1,0.035014
prcp,56,1.960784
wdir,121,4.236695
wspd,21,0.735294
pres,40,1.400560
cldc,34,1.190476
coco,2856,100.000000



Weather feature statistics:


,count,mean,std,min,25%,50%,75%,max
temp,2856.0,15.834769,2.973834,6.1,13.9,16.1,17.8,30.0
rhum,2855.0,71.397548,16.811288,5.0,65.0,75.0,83.0,100.0
prcp,2800.0,0.020571,0.210730,0.0,0.0,0.0,0.0,5.3
wdir,2735.0,191.608775,102.768412,0.0,110.0,250.0,260.0,360.0
wspd,2835.0,12.415203,8.586290,0.0,7.6,11.2,18.4,61.2
pres,2816.0,1014.434126,3.513963,1004.6,1012.0,1014.3,1016.7,1025.0
cldc,2822.0,4.414245,3.082164,0.0,2.0,4.0,8.0,8.0
coco,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Source provenance:

temp_source:
temp_source
isd_lite    2856
Name: count, dtype: int64

rhum_source:
rhum_source
isd_lite    2855
NaN            1
Name: count, dtype: int64

prcp_source:
prcp_source
isd_lite    2855
NaN            1
Name: count, dtype: int64

wdir_source:
wdir_source
isd_lite    2735
NaN          121
Name: count, dtype: int64

wspd_source:
wspd_source
isd_lite    2835
NaN           21
Name: count, dtype: int64

wpgt_source:
wpgt_source
NaN    2856
Name: count, dtype: int64

pres_source:
pres_source
isd_lite    2816
NaN           40
Name: count, dtype: int64

cldc_source:
cldc_source
isd_lite    2822
NaN           34
Name: count, dtype: int64

coco_source:
coco_source
NaN    2856
Name: count, dtype: int64

WEATHER QUALITY CHECK COMPLETE
Rows available for METR-LA period: 2856
Expected rows                  : 2856
Duplicate timestamps           : 0


In [115]:

import h5py
import numpy as np
import pandas as pd


traffic_file = "../../data/raw/metr-la.h5"

print("=" * 70)
print("TRAFFIC TIMESTAMP INSPECTION")
print("=" * 70)


with h5py.File(traffic_file, "r") as f:
    print("HDF5 keys:")
    print(list(f.keys()))

    def inspect_h5(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(
                f"Dataset: {name} | "
                f"shape={obj.shape} | "
                f"dtype={obj.dtype}"
            )

    f.visititems(inspect_h5)


try:
    traffic_df_raw = pd.read_hdf(
        traffic_file,
        key="df"
    )

    print("\nPandas HDF5 read successful.")
    print("Shape:", traffic_df_raw.shape)
    print("Index type:", type(traffic_df_raw.index))
    print("Index dtype:", traffic_df_raw.index.dtype)

    traffic_index = pd.DatetimeIndex(traffic_df_raw.index)

except Exception as e:
    print("\nPandas HDF5 read failed:")
    print(type(e).__name__, str(e))

    print("\nUsing h5py fallback...")

    with h5py.File(traffic_file, "r") as f:
        print("Available keys:", list(f.keys()))

    traffic_index = None


if traffic_index is not None:

    print("\n" + "=" * 70)
    print("TRAFFIC TIMESTAMP DETAILS")
    print("=" * 70)

    print("First timestamp :", traffic_index[0])
    print("Last timestamp  :", traffic_index[-1])
    print("Number of rows  :", len(traffic_index))

    print("\nTimezone:", traffic_index.tz)

    time_diffs = traffic_index.to_series().diff().dropna()

    print("\nMost common timestamp intervals:")
    print(time_diffs.value_counts().head(10))

    five_minute_count = (time_diffs == pd.Timedelta(minutes=5)).sum()

    print("\n5-minute intervals:",
          five_minute_count,
          "/",
          len(time_diffs))

    print(
        "Percentage 5-minute:",
        round(
            five_minute_count / len(time_diffs) * 100,
            4
        ),
        "%"
    )


    print("\n" + "=" * 70)
    print("TRAFFIC ↔ WEATHER RANGE")
    print("=" * 70)

    print("Traffic start :", traffic_index.min())
    print("Traffic end   :", traffic_index.max())

    print("Weather start :", weather_period["timestamp"].min())
    print("Weather end   :", weather_period["timestamp"].max())

else:

    print(
        "\nTraffic timestamp index could not be recovered "
        "with pandas. We will inspect the HDF5 structure before "
        "continuing."
    )

TRAFFIC TIMESTAMP INSPECTION
HDF5 keys:
['df']
Dataset: df/axis0 | shape=(207,) | dtype=|S6
Dataset: df/axis1 | shape=(34272,) | dtype=int64
Dataset: df/block0_items | shape=(207,) | dtype=|S6
Dataset: df/block0_values | shape=(34272, 207) | dtype=float64

Pandas HDF5 read failed:
TypeError a bytes-like object is required, not 'str'

Using h5py fallback...
Available keys: ['df']

Traffic timestamp index could not be recovered with pandas. We will inspect the HDF5 structure before continuing.


In [117]:

import h5py
import numpy as np
import pandas as pd

traffic_file = "../../data/raw/metr-la.h5"

print("=" * 70)
print("RECOVERING METR-LA TIMESTAMPS")
print("=" * 70)

with h5py.File(traffic_file, "r") as f:

    axis1 = np.array(f["df/axis1"])

    sensor_ids_raw = np.array(f["df/axis0"])

    traffic_values = np.array(f["df/block0_values"])

print("axis1 shape      :", axis1.shape)
print("traffic shape    :", traffic_values.shape)
print("sensor IDs shape :", sensor_ids_raw.shape)

print("\nFirst 10 raw timestamp values:")
print(axis1[:10])

print("\nLast 10 raw timestamp values:")
print(axis1[-10:])

print("\nTimestamp dtype:", axis1.dtype)


traffic_index = pd.to_datetime(
    axis1,
    unit="ns",
    errors="coerce"
)


print("\n" + "=" * 70)
print("TIMESTAMP INTERPRETATION")
print("=" * 70)

print("First timestamp :", traffic_index[0])
print("Last timestamp  :", traffic_index[-1])
print("Timezone        :", traffic_index.tz)

print("\nInvalid timestamps:", traffic_index.isna().sum())


time_diffs = traffic_index.to_series().diff().dropna()

print("\nMost common timestamp intervals:")
print(time_diffs.value_counts().head(10))

five_minute_count = (
    time_diffs == pd.Timedelta(minutes=5)
).sum()

print(
    "\n5-minute intervals:",
    five_minute_count,
    "/",
    len(time_diffs)
)

print(
    "Percentage 5-minute:",
    round(
        five_minute_count / len(time_diffs) * 100,
        4
    ),
    "%"
)


print("\n" + "=" * 70)
print("METR-LA TRAFFIC PERIOD")
print("=" * 70)

print("Start:", traffic_index.min())
print("End  :", traffic_index.max())
print("Rows :", len(traffic_index))


expected_start = pd.Timestamp("2012-03-01 00:00:00")
expected_end = pd.Timestamp("2012-06-27 23:55:00")

print("\nExpected start:", expected_start)
print("Expected end  :", expected_end)

print(
    "\nStart matches expected:",
    traffic_index.min() == expected_start
)

print(
    "End matches expected  :",
    traffic_index.max() == expected_end
)

RECOVERING METR-LA TIMESTAMPS
axis1 shape      : (34272,)
traffic shape    : (34272, 207)
sensor IDs shape : (207,)

First 10 raw timestamp values:
[1330560000000000000 1330560300000000000 1330560600000000000
 1330560900000000000 1330561200000000000 1330561500000000000
 1330561800000000000 1330562100000000000 1330562400000000000
 1330562700000000000]

Last 10 raw timestamp values:
[1340838600000000000 1340838900000000000 1340839200000000000
 1340839500000000000 1340839800000000000 1340840100000000000
 1340840400000000000 1340840700000000000 1340841000000000000
 1340841300000000000]

Timestamp dtype: int64

TIMESTAMP INTERPRETATION
First timestamp : 2012-03-01 00:00:00
Last timestamp  : 2012-06-27 23:55:00
Timezone        : None

Invalid timestamps: 0

Most common timestamp intervals:
0 days 00:05:00    34271
Name: count, dtype: int64

5-minute intervals: 34271 / 34271
Percentage 5-minute: 100.0 %

METR-LA TRAFFIC PERIOD
Start: 2012-03-01 00:00:00
End  : 2012-06-27 23:55:00
Rows : 34272

In [119]:

print("=" * 70)
print("TRAFFIC ↔ WEATHER TIMEZONE INVESTIGATION")
print("=" * 70)


weather_utc = weather_period.copy()

weather_utc["timestamp_utc"] = pd.to_datetime(
    weather_utc["timestamp"],
    utc=True
)


traffic_naive = pd.DatetimeIndex(traffic_index)

print("\nTraffic timestamps are kept timezone-naive.")
print("First:", traffic_naive[0])
print("Last :", traffic_naive[-1])


traffic_as_la = traffic_naive.tz_localize(
    "America/Los_Angeles",
    ambiguous="NaT",
    nonexistent="NaT"
)

traffic_as_utc = traffic_naive.tz_localize("UTC")


la_nat_count = traffic_as_la.isna().sum()

print("\n" + "=" * 70)
print("DST CHECK")
print("=" * 70)

print(
    "Timestamps becoming NaT when interpreted as LA:",
    la_nat_count
)

if la_nat_count > 0:
    print(
        "These correspond to the daylight-saving "
        "spring-forward gap."
    )


traffic_as_la_utc = traffic_as_la[
    ~traffic_as_la.isna()
].tz_convert("UTC")

traffic_as_utc_utc = traffic_as_utc


print("\n" + "=" * 70)
print("INTERPRETATION A — TRAFFIC IS LOS ANGELES LOCAL TIME")
print("=" * 70)

print("Original first:", traffic_naive[0])
print("Original last :", traffic_naive[-1])

print(
    "Converted UTC first:",
    traffic_as_la_utc[0]
)

print(
    "Converted UTC last :",
    traffic_as_la_utc[-1]
)

print("\n" + "=" * 70)
print("INTERPRETATION B — TRAFFIC IS UTC")
print("=" * 70)

print(
    "UTC first:",
    traffic_as_utc_utc[0]
)

print(
    "UTC last :",
    traffic_as_utc_utc[-1]
)


print("\n" + "=" * 70)
print("CORRESPONDING LOS ANGELES TIMES")
print("=" * 70)

print(
    "If traffic is already UTC:",
    traffic_as_utc_utc[0]
    .tz_convert("America/Los_Angeles")
)

print(
    "If traffic is LA local:",
    traffic_as_la[0]
)


print("\n" + "=" * 70)
print("WEATHER TIMELINE")
print("=" * 70)

print(
    "Weather UTC first:",
    weather_utc["timestamp_utc"].min()
)

print(
    "Weather UTC last :",
    weather_utc["timestamp_utc"].max()
)

print(
    "\nWeather LA first:",
    weather_utc["timestamp_utc"]
    .min()
    .tz_convert("America/Los_Angeles")
)

print(
    "Weather LA last :",
    weather_utc["timestamp_utc"]
    .max()
    .tz_convert("America/Los_Angeles")
)


print("\n" + "=" * 70)
print("DATASET FACTS")
print("=" * 70)

print("Traffic rows       :", len(traffic_naive))
print("Traffic interval   : 5 minutes")
print("Traffic sensors    : 207")

print(
    "Traffic period     :",
    traffic_naive.min(),
    "→",
    traffic_naive.max()
)

print(
    "Weather observations:",
    len(weather_utc)
)

print(
    "Weather interval    : 1 hour"
)

TRAFFIC ↔ WEATHER TIMEZONE INVESTIGATION

Traffic timestamps are kept timezone-naive.
First: 2012-03-01 00:00:00
Last : 2012-06-27 23:55:00

DST CHECK
Timestamps becoming NaT when interpreted as LA: 12
These correspond to the daylight-saving spring-forward gap.

INTERPRETATION A — TRAFFIC IS LOS ANGELES LOCAL TIME
Original first: 2012-03-01 00:00:00
Original last : 2012-06-27 23:55:00
Converted UTC first: 2012-03-01 08:00:00+00:00
Converted UTC last : 2012-06-28 06:55:00+00:00

INTERPRETATION B — TRAFFIC IS UTC
UTC first: 2012-03-01 00:00:00+00:00
UTC last : 2012-06-27 23:55:00+00:00

CORRESPONDING LOS ANGELES TIMES
If traffic is already UTC: 2012-02-29 16:00:00-08:00
If traffic is LA local: 2012-03-01 00:00:00-08:00

WEATHER TIMELINE
Weather UTC first: 2012-03-01 00:00:00+00:00
Weather UTC last : 2012-06-27 23:00:00+00:00

Weather LA first: 2012-02-29 16:00:00-08:00
Weather LA last : 2012-06-27 16:00:00-07:00

DATASET FACTS
Traffic rows       : 34272
Traffic interval   : 5 minutes
Tra

In [120]:

print("=" * 70)
print("BUILDING UTC-ALIGNED WEATHER TABLE")
print("=" * 70)


traffic_timestamps = pd.DatetimeIndex(
    traffic_index
)


weather_aligned = weather_df.copy()

weather_aligned["timestamp"] = pd.to_datetime(
    weather_aligned["timestamp"],
    utc=True
)


traffic_start_utc = traffic_timestamps.min().tz_localize("UTC")
traffic_end_utc = traffic_timestamps.max().tz_localize("UTC")

weather_aligned = weather_aligned[
    (weather_aligned["timestamp"] >= traffic_start_utc) &
    (weather_aligned["timestamp"] <= traffic_end_utc)
].copy()


WEATHER_FEATURES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]

weather_features = weather_aligned[
    ["timestamp"] + WEATHER_FEATURES
].copy()


weather_features = (
    weather_features
    .sort_values("timestamp")
    .drop_duplicates("timestamp")
    .reset_index(drop=True)
)


print("\nSelected weather features:")
print(WEATHER_FEATURES)

print("\nWeather rows:", len(weather_features))

print(
    "Weather start:",
    weather_features["timestamp"].min()
)

print(
    "Weather end  :",
    weather_features["timestamp"].max()
)

print(
    "\nTraffic start:",
    traffic_start_utc
)

print(
    "Traffic end  :",
    traffic_end_utc
)

print("\nWeather missingness:")
display(
    weather_features[WEATHER_FEATURES]
    .isna()
    .sum()
    .to_frame("missing_count")
    .assign(
        missing_percent=lambda x:
        x["missing_count"] / len(weather_features) * 100
    )
)

print("\nFirst 5 aligned weather rows:")
display(weather_features.head())

print("\nLast 5 aligned weather rows:")
display(weather_features.tail())

BUILDING UTC-ALIGNED WEATHER TABLE

Selected weather features:
['temp', 'rhum', 'prcp', 'wspd', 'pres', 'cldc']

Weather rows: 2856
Weather start: 2012-03-01 00:00:00+00:00
Weather end  : 2012-06-27 23:00:00+00:00

Traffic start: 2012-03-01 00:00:00+00:00
Traffic end  : 2012-06-27 23:55:00+00:00

Weather missingness:


,missing_count,missing_percent
temp,0,0.000000
rhum,1,0.035014
prcp,56,1.960784
wspd,21,0.735294
pres,40,1.400560
cldc,34,1.190476



First 5 aligned weather rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
0,2012-03-01 00:00:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
1,2012-03-01 01:00:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0
2,2012-03-01 02:00:00+00:00,13.3,72.0,0.0,27.7,1022.8,7.0
3,2012-03-01 03:00:00+00:00,13.3,75.0,0.0,24.1,1022.7,7.0
4,2012-03-01 04:00:00+00:00,13.3,72.0,0.0,29.5,1022.9,4.0



Last 5 aligned weather rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
2851,2012-06-27 19:00:00+00:00,22.2,59.0,0.0,18.4,1012.0,0.0
2852,2012-06-27 20:00:00+00:00,22.2,59.0,0.0,20.5,1011.9,0.0
2853,2012-06-27 21:00:00+00:00,21.7,61.0,0.0,22.3,1011.7,0.0
2854,2012-06-27 22:00:00+00:00,22.2,57.0,0.0,20.5,1011.5,0.0
2855,2012-06-27 23:00:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0


In [121]:

print("=" * 65)
print("HANDLING MISSING WEATHER VALUES")
print("=" * 65)

weather_clean = weather_aligned.copy()

WEATHER_FEATURES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]

print("\nMissing values BEFORE treatment:")
print(weather_clean[WEATHER_FEATURES].isna().sum())


CONTINUOUS_WEATHER = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]

weather_clean[CONTINUOUS_WEATHER] = (
    weather_clean[CONTINUOUS_WEATHER]
    .interpolate(method="linear", limit_direction="both")
)

weather_clean["prcp"] = weather_clean["prcp"].clip(lower=0)

print("\nMissing values AFTER interpolation:")
print(weather_clean[WEATHER_FEATURES].isna().sum())

print("\nWeather shape:", weather_clean.shape)

print("\nWeather statistics after cleaning:")
print(
    weather_clean[WEATHER_FEATURES]
    .describe()
    .round(3)
)

print("\nFirst 5 cleaned rows:")
display(weather_clean[["timestamp"] + WEATHER_FEATURES].head())

print("\nLast 5 cleaned rows:")
display(weather_clean[["timestamp"] + WEATHER_FEATURES].tail())

HANDLING MISSING WEATHER VALUES

Missing values BEFORE treatment:
temp     0
rhum     1
prcp    56
wspd    21
pres    40
cldc    34
dtype: int64

Missing values AFTER interpolation:
temp    0
rhum    0
prcp    0
wspd    0
pres    0
cldc    0
dtype: int64

Weather shape: (2856, 23)

Weather statistics after cleaning:
           temp      rhum      prcp      wspd      pres      cldc
count  2856.000  2856.000  2856.000  2856.000  2856.000  2856.000
mean     15.835    71.376     0.027    12.398  1014.429     4.433
std       2.974    16.849     0.224     8.563     3.508     3.083
min       6.100     5.000     0.000     0.000  1004.600     0.000
25%      13.900    65.000     0.000     7.600  1012.000     2.000
50%      16.100    75.000     0.000    11.200  1014.300     4.000
75%      17.800    83.000     0.000    18.400  1016.700     8.000
max      30.000   100.000     5.300    61.200  1025.000     8.000

First 5 cleaned rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
1440,2012-03-01 00:00:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
1441,2012-03-01 01:00:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0
1442,2012-03-01 02:00:00+00:00,13.3,72.0,0.0,27.7,1022.8,7.0
1443,2012-03-01 03:00:00+00:00,13.3,75.0,0.0,24.1,1022.7,7.0
1444,2012-03-01 04:00:00+00:00,13.3,72.0,0.0,29.5,1022.9,4.0



Last 5 cleaned rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
4291,2012-06-27 19:00:00+00:00,22.2,59.0,0.0,18.4,1012.0,0.0
4292,2012-06-27 20:00:00+00:00,22.2,59.0,0.0,20.5,1011.9,0.0
4293,2012-06-27 21:00:00+00:00,21.7,61.0,0.0,22.3,1011.7,0.0
4294,2012-06-27 22:00:00+00:00,22.2,57.0,0.0,20.5,1011.5,0.0
4295,2012-06-27 23:00:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0


In [124]:

print("=" * 65)
print("ALIGNING WEATHER TO 5-MINUTE TRAFFIC TIMESTAMPS")
print("=" * 65)


weather_hourly = weather_clean[
    ["timestamp"] + WEATHER_FEATURES
].copy()

weather_hourly = weather_hourly.sort_values("timestamp").reset_index(drop=True)

weather_hourly["timestamp"] = pd.to_datetime(
    weather_hourly["timestamp"],
    utc=True
)

weather_hourly["timestamp"] = weather_hourly["timestamp"].dt.as_unit("ns")


traffic_timestamps = pd.DatetimeIndex(traffic_index)

traffic_utc = traffic_timestamps.tz_localize("UTC")

traffic_utc = traffic_utc.as_unit("ns")

traffic_5min = pd.DataFrame({
    "timestamp": traffic_utc
})


traffic_5min = traffic_5min.sort_values("timestamp").reset_index(drop=True)
weather_hourly = weather_hourly.sort_values("timestamp").reset_index(drop=True)


print("\nTraffic timestamp dtype:")
print(traffic_5min["timestamp"].dtype)

print("\nWeather timestamp dtype:")
print(weather_hourly["timestamp"].dtype)


weather_5min = pd.merge_asof(
    traffic_5min,
    weather_hourly,
    on="timestamp",
    direction="backward"
)


print("\nTraffic timestamps:", len(traffic_5min))
print("Weather-aligned rows:", len(weather_5min))

print("\nTraffic start:")
print(weather_5min["timestamp"].min())

print("\nTraffic end:")
print(weather_5min["timestamp"].max())


print("\nMissing weather values after 5-minute alignment:")

print(
    weather_5min[WEATHER_FEATURES]
    .isna()
    .sum()
)


time_diffs = (
    weather_5min["timestamp"]
    .diff()
    .dropna()
)

print("\nUnique timestamp intervals:")
print(time_diffs.value_counts().head())

print("\nExpected interval:")
print(pd.Timedelta(minutes=5))


print("\nFirst 12 aligned rows:")

display(
    weather_5min[
        ["timestamp"] + WEATHER_FEATURES
    ].head(12)
)

print("\nLast 5 aligned rows:")

display(
    weather_5min[
        ["timestamp"] + WEATHER_FEATURES
    ].tail()
)

ALIGNING WEATHER TO 5-MINUTE TRAFFIC TIMESTAMPS

Traffic timestamp dtype:
datetime64[ns, UTC]

Weather timestamp dtype:
datetime64[ns, UTC]

Traffic timestamps: 34272
Weather-aligned rows: 34272

Traffic start:
2012-03-01 00:00:00+00:00

Traffic end:
2012-06-27 23:55:00+00:00

Missing weather values after 5-minute alignment:
temp    0
rhum    0
prcp    0
wspd    0
pres    0
cldc    0
dtype: int64

Unique timestamp intervals:
timestamp
0 days 00:05:00    34271
Name: count, dtype: int64

Expected interval:
0 days 00:05:00

First 12 aligned rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
0,2012-03-01 00:00:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
1,2012-03-01 00:05:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
2,2012-03-01 00:10:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
3,2012-03-01 00:15:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
4,2012-03-01 00:20:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
5,2012-03-01 00:25:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
6,2012-03-01 00:30:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
7,2012-03-01 00:35:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
8,2012-03-01 00:40:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
9,2012-03-01 00:45:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0



Last 5 aligned rows:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
34267,2012-06-27 23:35:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0
34268,2012-06-27 23:40:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0
34269,2012-06-27 23:45:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0
34270,2012-06-27 23:50:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0
34271,2012-06-27 23:55:00+00:00,22.2,59.0,0.0,20.5,1011.4,0.0


In [125]:

print("=" * 65)
print("WEATHER ALIGNMENT QUALITY CHECK")
print("=" * 65)


duplicate_timestamps = weather_5min["timestamp"].duplicated().sum()

print("\nDuplicate timestamps:", duplicate_timestamps)


expected_diff = pd.Timedelta(minutes=5)

actual_diffs = weather_5min["timestamp"].diff().dropna()

incorrect_intervals = (actual_diffs != expected_diff).sum()

print("Incorrect 5-minute intervals:", incorrect_intervals)


weather_change_count = (
    weather_5min[WEATHER_FEATURES]
    .diff()
    .abs()
    .sum(axis=1)
    .gt(0)
    .sum()
)

print("Rows where weather changes:", weather_change_count)


print("\nFirst 13 timestamps and weather:")

display(
    weather_5min[
        ["timestamp"] + WEATHER_FEATURES
    ].head(13)
)


transition_start = pd.Timestamp(
    "2012-03-01 00:45:00",
    tz="UTC"
)

transition_end = pd.Timestamp(
    "2012-03-01 01:15:00",
    tz="UTC"
)

print("\nWeather transition around 01:00:")

display(
    weather_5min[
        (weather_5min["timestamp"] >= transition_start) &
        (weather_5min["timestamp"] <= transition_end)
    ][
        ["timestamp"] + WEATHER_FEATURES
    ]
)


print("\n" + "=" * 65)
print("ALIGNMENT CHECK COMPLETE")
print("=" * 65)

if duplicate_timestamps == 0 and incorrect_intervals == 0:
    print("✓ Timestamp sequence is continuous at 5-minute resolution")
else:
    print("⚠ Timestamp continuity requires investigation")

if weather_5min[WEATHER_FEATURES].isna().sum().sum() == 0:
    print("✓ No missing weather features")
else:
    print("⚠ Missing weather values remain")

WEATHER ALIGNMENT QUALITY CHECK

Duplicate timestamps: 0
Incorrect 5-minute intervals: 0
Rows where weather changes: 2846

First 13 timestamps and weather:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
0,2012-03-01 00:00:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
1,2012-03-01 00:05:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
2,2012-03-01 00:10:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
3,2012-03-01 00:15:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
4,2012-03-01 00:20:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
5,2012-03-01 00:25:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
6,2012-03-01 00:30:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
7,2012-03-01 00:35:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
8,2012-03-01 00:40:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
9,2012-03-01 00:45:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0



Weather transition around 01:00:


,timestamp,temp,rhum,prcp,wspd,pres,cldc
9,2012-03-01 00:45:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
10,2012-03-01 00:50:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
11,2012-03-01 00:55:00+00:00,13.9,69.0,0.0,29.5,1023.0,6.0
12,2012-03-01 01:00:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0
13,2012-03-01 01:05:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0
14,2012-03-01 01:10:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0
15,2012-03-01 01:15:00+00:00,13.9,72.0,0.0,24.1,1023.0,7.0



ALIGNMENT CHECK COMPLETE
✓ Timestamp sequence is continuous at 5-minute resolution
✓ No missing weather features


In [126]:

print("=" * 65)
print("BUILDING WEATHER SEQUENCES")
print("=" * 65)


INPUT_STEPS = 12
FORECAST_HORIZON = 6

WEATHER_FEATURES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]


weather_values = (
    weather_5min[WEATHER_FEATURES]
    .to_numpy(dtype=np.float32)
)

print("\nFull weather matrix shape:")
print(weather_values.shape)


TRAIN_N = len(X_train)
VAL_N = len(X_val)
TEST_N = len(X_test)


train_sequence_start = 0

train_sequence_end = train_sequence_start + TRAIN_N

val_sequence_start = train_sequence_end + INPUT_STEPS + FORECAST_HORIZON - 1

val_sequence_end = val_sequence_start + VAL_N

test_sequence_start = val_sequence_end + INPUT_STEPS + FORECAST_HORIZON - 1

test_sequence_end = test_sequence_start + TEST_N

print("\nSequence boundaries:")
print("Train:", train_sequence_start, "→", train_sequence_end - 1)
print("Val  :", val_sequence_start, "→", val_sequence_end - 1)
print("Test :", test_sequence_start, "→", test_sequence_end - 1)


assert train_sequence_end <= len(weather_values)
assert val_sequence_end <= len(weather_values)
assert test_sequence_end <= len(weather_values)


def build_weather_sequences(
    weather_array,
    start_index,
    n_sequences,
    input_steps
):
    sequences = []

    for i in range(n_sequences):
        start = start_index + i
        end = start + input_steps

        sequences.append(
            weather_array[start:end]
        )

    return np.stack(sequences)

weather_train = build_weather_sequences(
    weather_values,
    train_sequence_start,
    TRAIN_N,
    INPUT_STEPS
)

weather_val = build_weather_sequences(
    weather_values,
    val_sequence_start,
    VAL_N,
    INPUT_STEPS
)

weather_test = build_weather_sequences(
    weather_values,
    test_sequence_start,
    TEST_N,
    INPUT_STEPS
)


print("\nWeather sequence shapes:")
print("Train:", weather_train.shape)
print("Val  :", weather_val.shape)
print("Test :", weather_test.shape)


print("\nMissing values:")
print("Train:", np.isnan(weather_train).sum())
print("Val  :", np.isnan(weather_val).sum())
print("Test :", np.isnan(weather_test).sum())


print("\nFirst training weather sequence:")

display(
    pd.DataFrame(
        weather_train[0],
        columns=WEATHER_FEATURES
    )
)

print("\nWeather sequence construction complete.")

BUILDING WEATHER SEQUENCES

Full weather matrix shape:
(34272, 6)

Sequence boundaries:
Train: 0 → 23972
Val  : 23990 → 27399
Test : 27417 → 34254

Weather sequence shapes:
Train: (23973, 12, 6)
Val  : (3410, 12, 6)
Test : (6838, 12, 6)

Missing values:
Train: 0
Val  : 0
Test : 0

First training weather sequence:


,temp,rhum,prcp,wspd,pres,cldc
0,13.9,69.0,0.0,29.5,1023.0,6.0
1,13.9,69.0,0.0,29.5,1023.0,6.0
2,13.9,69.0,0.0,29.5,1023.0,6.0
3,13.9,69.0,0.0,29.5,1023.0,6.0
4,13.9,69.0,0.0,29.5,1023.0,6.0
5,13.9,69.0,0.0,29.5,1023.0,6.0
6,13.9,69.0,0.0,29.5,1023.0,6.0
7,13.9,69.0,0.0,29.5,1023.0,6.0
8,13.9,69.0,0.0,29.5,1023.0,6.0
9,13.9,69.0,0.0,29.5,1023.0,6.0



Weather sequence construction complete.


In [127]:

print("=" * 65)
print("TRAFFIC ↔ WEATHER TIMESTAMP VERIFICATION")
print("=" * 65)


traffic_timestamps = pd.DatetimeIndex(traffic_index).tz_localize("UTC")


def verify_split_alignment(
    split_name,
    start_index,
    n_sequences,
    traffic_timestamps,
    weather_array
):
    print(f"\n{'-' * 60}")
    print(f"{split_name.upper()} ALIGNMENT")
    print(f"{'-' * 60}")

    first_traffic_start = traffic_timestamps[start_index]
    first_traffic_end = traffic_timestamps[start_index + INPUT_STEPS - 1]

    last_start_index = start_index + n_sequences - 1

    last_traffic_start = traffic_timestamps[last_start_index]
    last_traffic_end = traffic_timestamps[
        last_start_index + INPUT_STEPS - 1
    ]

    print("\nFirst sequence:")
    print("Traffic start :", first_traffic_start)
    print("Weather start :", weather_5min.iloc[start_index]["timestamp"])

    print("\nTraffic end   :", first_traffic_end)
    print(
        "Weather end   :",
        weather_5min.iloc[start_index + INPUT_STEPS - 1]["timestamp"]
    )

    print("\nLast sequence:")
    print("Traffic start :", last_traffic_start)
    print(
        "Weather start :",
        weather_5min.iloc[last_start_index]["timestamp"]
    )

    print("\nTraffic end   :", last_traffic_end)
    print(
        "Weather end   :",
        weather_5min.iloc[
            last_start_index + INPUT_STEPS - 1
        ]["timestamp"]
    )


    first_start_match = (
        first_traffic_start ==
        weather_5min.iloc[start_index]["timestamp"]
    )

    first_end_match = (
        first_traffic_end ==
        weather_5min.iloc[start_index + INPUT_STEPS - 1]["timestamp"]
    )

    last_start_match = (
        last_traffic_start ==
        weather_5min.iloc[last_start_index]["timestamp"]
    )

    last_end_match = (
        last_traffic_end ==
        weather_5min.iloc[
            last_start_index + INPUT_STEPS - 1
        ]["timestamp"]
    )

    print("\nAlignment checks:")
    print("First start match:", first_start_match)
    print("First end match  :", first_end_match)
    print("Last start match :", last_start_match)
    print("Last end match   :", last_end_match)

    all_match = (
        first_start_match
        and first_end_match
        and last_start_match
        and last_end_match
    )

    print("\nResult:", "✓ PASS" if all_match else "✗ FAIL")

    return all_match



train_alignment_ok = verify_split_alignment(
    "Train",
    train_sequence_start,
    TRAIN_N,
    traffic_timestamps,
    weather_train
)

val_alignment_ok = verify_split_alignment(
    "Validation",
    val_sequence_start,
    VAL_N,
    traffic_timestamps,
    weather_val
)

test_alignment_ok = verify_split_alignment(
    "Test",
    test_sequence_start,
    TEST_N,
    traffic_timestamps,
    weather_test
)


print("\n" + "=" * 65)

if (
    train_alignment_ok
    and val_alignment_ok
    and test_alignment_ok
):
    print("✓ ALL TRAFFIC ↔ WEATHER ALIGNMENT CHECKS PASSED")
else:
    print("✗ ALIGNMENT ERROR DETECTED")

print("=" * 65)

TRAFFIC ↔ WEATHER TIMESTAMP VERIFICATION

------------------------------------------------------------
TRAIN ALIGNMENT
------------------------------------------------------------

First sequence:
Traffic start : 2012-03-01 00:00:00+00:00
Weather start : 2012-03-01 00:00:00+00:00

Traffic end   : 2012-03-01 00:55:00+00:00
Weather end   : 2012-03-01 00:55:00+00:00

Last sequence:
Traffic start : 2012-05-23 05:40:00+00:00
Weather start : 2012-05-23 05:40:00+00:00

Traffic end   : 2012-05-23 06:35:00+00:00
Weather end   : 2012-05-23 06:35:00+00:00

Alignment checks:
First start match: True
First end match  : True
Last start match : True
Last end match   : True

Result: ✓ PASS

------------------------------------------------------------
VALIDATION ALIGNMENT
------------------------------------------------------------

First sequence:
Traffic start : 2012-05-23 07:10:00+00:00
Weather start : 2012-05-23 07:10:00+00:00

Traffic end   : 2012-05-23 08:05:00+00:00
Weather end   : 2012-05-23 08:

In [128]:

print("=" * 65)
print("SAVING PROCESSED WEATHER SEQUENCES")
print("=" * 65)


np.save(
    "../../data/processed/weather_train_sequences.npy",
    weather_train
)

np.save(
    "../../data/processed/weather_validation_sequences.npy",
    weather_val
)

np.save(
    "../../data/processed/weather_test_sequences.npy",
    weather_test
)


weather_5min.to_csv(
    "../../data/processed/weather_5min_aligned.csv",
    index=False
)


weather_metadata = {
    "features": WEATHER_FEATURES,
    "input_steps": INPUT_STEPS,
    "sampling_interval_minutes": 5,
    "source_resolution": "hourly",
    "aligned_resolution": "5-minute",
    "forecast_horizon_steps": FORECAST_HORIZON,
    "forecast_horizon_minutes": 30,
    "timezone": "UTC",
    "future_weather_used": False
}

with open(
    "../../data/processed/weather_metadata.json",
    "w"
) as f:
    json.dump(weather_metadata, f, indent=4)


import os

saved_files = [
    "../../data/processed/weather_train_sequences.npy",
    "../../data/processed/weather_validation_sequences.npy",
    "../../data/processed/weather_test_sequences.npy",
    "../../data/processed/weather_5min_aligned.csv",
    "../../data/processed/weather_metadata.json"
]

print("\nSaved files:")

for path in saved_files:
    print(
        "✓" if os.path.exists(path) else "✗",
        path
    )

print("\nSaved sequence shapes:")
print("Train:", np.load(
    "../../data/processed/weather_train_sequences.npy"
).shape)

print("Validation:", np.load(
    "../../data/processed/weather_validation_sequences.npy"
).shape)

print("Test:", np.load(
    "../../data/processed/weather_test_sequences.npy"
).shape)

print("\nWeather preprocessing pipeline saved successfully.")

SAVING PROCESSED WEATHER SEQUENCES

Saved files:
✓ ../../data/processed/weather_train_sequences.npy
✓ ../../data/processed/weather_validation_sequences.npy
✓ ../../data/processed/weather_test_sequences.npy
✓ ../../data/processed/weather_5min_aligned.csv
✓ ../../data/processed/weather_metadata.json

Saved sequence shapes:
Train: (23973, 12, 6)
Validation: (3410, 12, 6)
Test: (6838, 12, 6)

Weather preprocessing pipeline saved successfully.


In [129]:

print("=" * 65)
print("VERIFYING SAVED WEATHER DATA")
print("=" * 65)


weather_train_loaded = np.load(
    "../../data/processed/weather_train_sequences.npy"
)

weather_val_loaded = np.load(
    "../../data/processed/weather_validation_sequences.npy"
)

weather_test_loaded = np.load(
    "../../data/processed/weather_test_sequences.npy"
)


with open(
    "../../data/processed/weather_metadata.json",
    "r"
) as f:
    weather_metadata_loaded = json.load(f)


print("\nLoaded shapes:")
print("Train:", weather_train_loaded.shape)
print("Validation:", weather_val_loaded.shape)
print("Test:", weather_test_loaded.shape)

print("\nWeather features:")
print(weather_metadata_loaded["features"])

print("\nInput steps:")
print(weather_metadata_loaded["input_steps"])

print("\nForecast horizon:")
print(
    weather_metadata_loaded["forecast_horizon_minutes"],
    "minutes"
)

print("\nTimezone:")
print(weather_metadata_loaded["timezone"])

print("\nFuture weather used:")
print(weather_metadata_loaded["future_weather_used"])


print("\nReproducibility checks:")

print(
    "Train:",
    np.array_equal(weather_train, weather_train_loaded)
)

print(
    "Validation:",
    np.array_equal(weather_val, weather_val_loaded)
)

print(
    "Test:",
    np.array_equal(weather_test, weather_test_loaded)
)


assert np.array_equal(weather_train, weather_train_loaded)
assert np.array_equal(weather_val, weather_val_loaded)
assert np.array_equal(weather_test, weather_test_loaded)

print("\n" + "=" * 65)
print("✓ WEATHER DATA RELOAD VERIFIED")
print("=" * 65)

VERIFYING SAVED WEATHER DATA

Loaded shapes:
Train: (23973, 12, 6)
Validation: (3410, 12, 6)
Test: (6838, 12, 6)

Weather features:
['temp', 'rhum', 'prcp', 'wspd', 'pres', 'cldc']

Input steps:
12

Forecast horizon:
30 minutes

Timezone:
UTC

Future weather used:
False

Reproducibility checks:
Train: True
Validation: True
Test: True

✓ WEATHER DATA RELOAD VERIFIED


In [131]:

print("=" * 65)
print("CREATING WEATHER-AWARE DATALOADERS")
print("=" * 65)


weather_train = np.load(
    "../../data/processed/weather_train_sequences.npy"
)

weather_val = np.load(
    "../../data/processed/weather_validation_sequences.npy"
)

weather_test = np.load(
    "../../data/processed/weather_test_sequences.npy"
)


train_mask = np.load(
    "../../data/processed/train_target_mask.npy"
)

val_mask = np.load(
    "../../data/processed/validation_target_mask.npy"
)

test_mask = np.load(
    "../../data/processed/test_target_mask.npy"
)


print("\nTarget mask shapes:")
print("Train:", train_mask.shape)
print("Validation:", val_mask.shape)
print("Test:", test_mask.shape)


weather_train_tensor = torch.tensor(
    weather_train,
    dtype=torch.float32
)

weather_val_tensor = torch.tensor(
    weather_val,
    dtype=torch.float32
)

weather_test_tensor = torch.tensor(
    weather_test,
    dtype=torch.float32
)


weather_train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    weather_train_tensor,
    torch.tensor(y_train, dtype=torch.float32),
    torch.tensor(train_mask, dtype=torch.bool)
)

weather_val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    weather_val_tensor,
    torch.tensor(y_val, dtype=torch.float32),
    torch.tensor(val_mask, dtype=torch.bool)
)

weather_test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    weather_test_tensor,
    torch.tensor(y_test, dtype=torch.float32),
    torch.tensor(test_mask, dtype=torch.bool)
)


BATCH_SIZE = 64

weather_train_loader = DataLoader(
    weather_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

weather_val_loader = DataLoader(
    weather_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

weather_test_loader = DataLoader(
    weather_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


traffic_batch, weather_batch, target_batch, mask_batch = next(
    iter(weather_train_loader)
)

print("\nBatch shapes:")
print("Traffic input :", traffic_batch.shape)
print("Weather input :", weather_batch.shape)
print("Target        :", target_batch.shape)
print("Target mask   :", mask_batch.shape)


print("\nDataset sizes:")
print("Train samples:", len(weather_train_dataset))
print("Validation samples:", len(weather_val_dataset))
print("Test samples:", len(weather_test_dataset))

print("\nNumber of batches:")
print("Train:", len(weather_train_loader))
print("Validation:", len(weather_val_loader))
print("Test:", len(weather_test_loader))


assert weather_train.shape == (23973, 12, 6)
assert weather_val.shape == (3410, 12, 6)
assert weather_test.shape == (6838, 12, 6)

assert train_mask.shape == (23973, 207)
assert val_mask.shape == (3410, 207)
assert test_mask.shape == (6838, 207)

assert traffic_batch.shape == (64, 12, 207)
assert weather_batch.shape == (64, 12, 6)
assert target_batch.shape == (64, 207)
assert mask_batch.shape == (64, 207)

assert torch.isnan(weather_batch).sum() == 0

print("\n" + "=" * 65)
print("✓ WEATHER DATALOADERS READY")
print("=" * 65)

CREATING WEATHER-AWARE DATALOADERS

Target mask shapes:
Train: (23973, 207)
Validation: (3410, 207)
Test: (6838, 207)

Batch shapes:
Traffic input : torch.Size([64, 12, 207])
Weather input : torch.Size([64, 12, 6])
Target        : torch.Size([64, 207])
Target mask   : torch.Size([64, 207])

Dataset sizes:
Train samples: 23973
Validation samples: 3410
Test samples: 6838

Number of batches:
Train: 375
Validation: 54
Test: 107

✓ WEATHER DATALOADERS READY


In [132]:

print("=" * 65)
print("DEFINING WEATHER-AWARE URBANFLOW")
print("=" * 65)

class WeatherAwareUrbanFlow(nn.Module):

    def __init__(
        self,
        num_sensors=207,
        weather_features=6,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        spatial_hidden=64,
        weather_hidden=64
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.d_model = d_model


        self.input_projection = nn.Linear(
            1,
            d_model
        )


        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=INPUT_STEPS
        )


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.spatial_projection = nn.Linear(
            d_model,
            spatial_hidden
        )

        self.spatial_transform = nn.Linear(
            spatial_hidden,
            spatial_hidden
        )


        self.weather_projection = nn.Sequential(
            nn.Linear(weather_features, 32),
            nn.ReLU(),
            nn.Linear(32, weather_hidden),
            nn.ReLU()
        )


        fusion_input = d_model + spatial_hidden + weather_hidden

        self.fusion = nn.Sequential(
            nn.Linear(fusion_input, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )


        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, traffic, weather):


        B, T, S = traffic.shape


        x = traffic.permute(0, 2, 1)

        x = x.reshape(B * S, T, 1)

        x = self.input_projection(x)

        x = self.positional_encoding(x)

        x = self.temporal_transformer(x)

        temporal_features = x[:, -1, :]

        temporal_features = temporal_features.reshape(
            B, S, self.d_model
        )


        spatial = self.spatial_projection(
            temporal_features
        )

        spatial = torch.relu(
            self.spatial_transform(spatial)
        )

        spatial = torch.einsum(
            "ij,bjf->bif",
            A_norm,
            spatial
        )


        weather_last = weather[:, -1, :]

        weather_features = self.weather_projection(
            weather_last
        )

        weather_features = weather_features.unsqueeze(1).expand(
            B,
            S,
            weather_features.shape[-1]
        )


        fused = torch.cat(
            [
                temporal_features,
                spatial,
                weather_features
            ],
            dim=-1
        )

        fused = self.fusion(fused)


        output = self.output_layer(fused)

        output = output.squeeze(-1)

        return output



weather_model = WeatherAwareUrbanFlow(
    num_sensors=207,
    weather_features=6,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64,
    weather_hidden=64
).to(device)


trainable_params = sum(
    p.numel()
    for p in weather_model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_params)


weather_model.eval()

with torch.no_grad():

    test_traffic = traffic_batch.to(device)
    test_weather = weather_batch.to(device)

    test_output = weather_model(
        test_traffic,
        test_weather
    )

print("\nInput traffic shape :", test_traffic.shape)
print("Input weather shape :", test_weather.shape)
print("Output shape        :", test_output.shape)

assert test_output.shape == (64, 207)

print("\n" + "=" * 65)
print("✓ WEATHER-AWARE MODEL SHAPE TEST PASSED")
print("=" * 65)

DEFINING WEATHER-AWARE URBANFLOW

Trainable parameters: 312993

Input traffic shape : torch.Size([64, 12, 207])
Input weather shape : torch.Size([64, 12, 6])
Output shape        : torch.Size([64, 207])

✓ WEATHER-AWARE MODEL SHAPE TEST PASSED


In [133]:

print("=" * 65)
print("TRAINING WEATHER-AWARE URBANFLOW v1")
print("=" * 65)


WEATHER_EPOCHS = 20
WEATHER_LR = 0.001
WEATHER_PATIENCE = 4
HUBER_BETA = 1.0

print("\nConfiguration:")
print("Epochs     :", WEATHER_EPOCHS)
print("Learning rate:", WEATHER_LR)
print("Huber beta :", HUBER_BETA)
print("Device     :", device)


weather_model = WeatherAwareUrbanFlow(
    num_sensors=207,
    weather_features=6,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
    spatial_hidden=64,
    weather_hidden=64
).to(device)


criterion_weather = nn.SmoothL1Loss(
    beta=HUBER_BETA,
    reduction="none"
)

optimizer_weather = torch.optim.Adam(
    weather_model.parameters(),
    lr=WEATHER_LR
)


weather_training_history = []

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

checkpoint_path = (
    "../../models/weather_aware_urbanflow_v1_best.pth"
)


def masked_huber_loss(predictions, targets, mask):

    loss = criterion_weather(
        predictions,
        targets
    )

    mask_float = mask.float()

    return (
        (loss * mask_float).sum()
        / mask_float.sum().clamp_min(1.0)
    )


for epoch in range(1, WEATHER_EPOCHS + 1):


    weather_model.train()

    train_loss_sum = 0.0
    train_batches = 0

    for traffic_x, weather_x, targets, masks in weather_train_loader:

        traffic_x = traffic_x.to(device)
        weather_x = weather_x.to(device)
        targets = targets.to(device)
        masks = masks.to(device)

        optimizer_weather.zero_grad()

        predictions = weather_model(
            traffic_x,
            weather_x
        )

        loss = masked_huber_loss(
            predictions,
            targets,
            masks
        )

        loss.backward()

        optimizer_weather.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / train_batches


    weather_model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        for traffic_x, weather_x, targets, masks in weather_val_loader:

            traffic_x = traffic_x.to(device)
            weather_x = weather_x.to(device)
            targets = targets.to(device)
            masks = masks.to(device)

            predictions = weather_model(
                traffic_x,
                weather_x
            )

            loss = masked_huber_loss(
                predictions,
                targets,
                masks
            )

            val_loss_sum += loss.item()
            val_batches += 1

    val_loss = val_loss_sum / val_batches

    weather_training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })


    improved = val_loss < best_val_loss

    if improved:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            weather_model.state_dict(),
            checkpoint_path
        )

    else:

        patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"{'BEST' if improved else ''}"
    )


    if patience_counter >= WEATHER_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


weather_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

weather_model.eval()

print("\n" + "=" * 65)
print("TRAINING COMPLETE")
print("=" * 65)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Checkpoint:", checkpoint_path)

TRAINING WEATHER-AWARE URBANFLOW v1

Configuration:
Epochs     : 20
Learning rate: 0.001
Huber beta : 1.0
Device     : mps
Epoch 01 | Train Loss: 0.370231 | Val Loss: 0.176590 | BEST
Epoch 02 | Train Loss: 0.184144 | Val Loss: 0.173320 | BEST
Epoch 03 | Train Loss: 0.181386 | Val Loss: 0.171182 | BEST
Epoch 04 | Train Loss: 0.180245 | Val Loss: 0.168812 | BEST
Epoch 05 | Train Loss: 0.179601 | Val Loss: 0.171283 | 
Epoch 06 | Train Loss: 0.179383 | Val Loss: 0.168837 | 
Epoch 07 | Train Loss: 0.178369 | Val Loss: 0.169653 | 
Epoch 08 | Train Loss: 0.178238 | Val Loss: 0.167610 | BEST
Epoch 09 | Train Loss: 0.177727 | Val Loss: 0.167687 | 
Epoch 10 | Train Loss: 0.177448 | Val Loss: 0.167916 | 
Epoch 11 | Train Loss: 0.176768 | Val Loss: 0.167683 | 
Epoch 12 | Train Loss: 0.176735 | Val Loss: 0.165818 | BEST
Epoch 13 | Train Loss: 0.176624 | Val Loss: 0.165679 | BEST
Epoch 14 | Train Loss: 0.176444 | Val Loss: 0.166071 | 
Epoch 15 | Train Loss: 0.175884 | Val Loss: 0.165490 | BEST
Epoch

In [134]:

print("=" * 65)
print("EVALUATING WEATHER-AWARE URBANFLOW v1")
print("=" * 65)


weather_model.load_state_dict(
    torch.load(
        "../../models/weather_aware_urbanflow_v1_best.pth",
        map_location=device,
        weights_only=True
    )
)

weather_model.eval()


all_predictions = []
all_targets = []
all_masks = []

with torch.no_grad():

    for traffic_x, weather_x, targets, masks in weather_val_loader:

        traffic_x = traffic_x.to(device)
        weather_x = weather_x.to(device)

        predictions = weather_model(
            traffic_x,
            weather_x
        )

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            targets.numpy()
        )

        all_masks.append(
            masks.numpy()
        )


predictions_scaled = np.concatenate(
    all_predictions,
    axis=0
)

targets_scaled = np.concatenate(
    all_targets,
    axis=0
)

validation_mask = np.concatenate(
    all_masks,
    axis=0
)

print("\nScaled prediction shape:", predictions_scaled.shape)
print("Scaled target shape    :", targets_scaled.shape)
print("Mask shape             :", validation_mask.shape)


predictions_mph = traffic_scaler.inverse_transform(
    predictions_scaled
)

targets_mph = traffic_scaler.inverse_transform(
    targets_scaled
)


valid_predictions = predictions_mph[validation_mask]
valid_targets = targets_mph[validation_mask]


mae = mean_absolute_error(
    valid_targets,
    valid_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        valid_targets,
        valid_predictions
    )
)

valid_count = len(valid_targets)

print("\n" + "=" * 65)
print("WEATHER-AWARE VALIDATION RESULTS")
print("=" * 65)

print(f"MAE  : {mae:.6f} mph")
print(f"RMSE : {rmse:.6f} mph")
print(f"Valid observations: {valid_count:,}")


print("\nPrediction statistics:")
print(
    f"Prediction range: "
    f"{valid_predictions.min():.4f} → "
    f"{valid_predictions.max():.4f} mph"
)

print(
    f"Target range: "
    f"{valid_targets.min():.4f} → "
    f"{valid_targets.max():.4f} mph"
)

negative_predictions = (
    valid_predictions < 0
).sum()

above_70_predictions = (
    valid_predictions > 70
).sum()

print(
    f"Negative predictions: "
    f"{negative_predictions:,} "
    f"({negative_predictions / valid_count * 100:.4f}%)"
)

print(
    f"Predictions > 70 mph: "
    f"{above_70_predictions:,} "
    f"({above_70_predictions / valid_count * 100:.4f}%)"
)


BASELINE_MAE = 3.177278
BASELINE_RMSE = 6.316968

mae_change = mae - BASELINE_MAE
rmse_change = rmse - BASELINE_RMSE

mae_improvement = (
    (BASELINE_MAE - mae)
    / BASELINE_MAE
    * 100
)

rmse_improvement = (
    (BASELINE_RMSE - rmse)
    / BASELINE_RMSE
    * 100
)

print("\n" + "=" * 65)
print("COMPARISON WITH WEATHER-FREE BASELINE")
print("=" * 65)

print(f"Baseline MAE : {BASELINE_MAE:.6f}")
print(f"Weather MAE  : {mae:.6f}")
print(f"MAE change   : {mae_change:+.6f}")
print(f"MAE change % : {mae_improvement:+.2f}%")

print()

print(f"Baseline RMSE: {BASELINE_RMSE:.6f}")
print(f"Weather RMSE : {rmse:.6f}")
print(f"RMSE change  : {rmse_change:+.6f}")
print(f"RMSE change %: {rmse_improvement:+.2f}%")

EVALUATING WEATHER-AWARE URBANFLOW v1

Scaled prediction shape: (3410, 207)
Scaled target shape    : (3410, 207)
Mask shape             : (3410, 207)

WEATHER-AWARE VALIDATION RESULTS
MAE  : 3.351608 mph
RMSE : 6.588709 mph
Valid observations: 664,844

Prediction statistics:
Prediction range: -19.5465 → 71.5888 mph
Target range: 0.5000 → 70.0000 mph
Negative predictions: 45 (0.0068%)
Predictions > 70 mph: 210 (0.0316%)

COMPARISON WITH WEATHER-FREE BASELINE
Baseline MAE : 3.177278
Weather MAE  : 3.351608
MAE change   : +0.174330
MAE change % : -5.49%

Baseline RMSE: 6.316968
Weather RMSE : 6.588709
RMSE change  : +0.271741
RMSE change %: -4.30%


In [135]:

print("=" * 65)
print("RECORDING WEATHER EXPERIMENT")
print("=" * 65)

weather_experiment = {
    "experiment": "Weather-Aware UrbanFlow v1",
    "weather_features": "temp,rhum,prcp,wspd,pres,cldc",
    "weather_representation": "last_timestep_only",
    "loss": "Huber",
    "learning_rate": 0.001,
    "dropout": 0.1,
    "best_epoch": best_epoch,
    "validation_mae": float(mae),
    "validation_rmse": float(rmse),
    "valid_observations": int(valid_count),
    "baseline_mae": BASELINE_MAE,
    "baseline_rmse": BASELINE_RMSE,
    "mae_change": float(mae_change),
    "rmse_change": float(rmse_change),
    "mae_change_percent": float(mae_improvement),
    "rmse_change_percent": float(rmse_improvement),
}

weather_results_path = (
    "../../results/experiments/weather_experiments.csv"
)

weather_result_df = pd.DataFrame(
    [weather_experiment]
)

if os.path.exists(weather_results_path):

    existing_results = pd.read_csv(
        weather_results_path
    )

    existing_results = existing_results[
        existing_results["experiment"]
        != weather_experiment["experiment"]
    ]

    weather_result_df = pd.concat(
        [
            existing_results,
            weather_result_df
        ],
        ignore_index=True
    )

weather_result_df.to_csv(
    weather_results_path,
    index=False
)

print("\nSaved:")
print(weather_results_path)

display(weather_result_df)

print("\n" + "=" * 65)
print("✓ WEATHER EXPERIMENT RECORDED")
print("=" * 65)

RECORDING WEATHER EXPERIMENT

Saved:
../../results/experiments/weather_experiments.csv


,experiment,weather_features,weather_representation,loss,learning_rate,dropout,best_epoch,validation_mae,validation_rmse,valid_observations,baseline_mae,baseline_rmse,mae_change,rmse_change,mae_change_percent,rmse_change_percent
0,Weather-Aware UrbanFlow v1,"temp,rhum,prcp,wspd,pres,cldc",last_timestep_only,Huber,0.001,0.1,19,3.351608,6.588709,664844,3.177278,6.316968,0.17433,0.271741,-5.486773,-4.301769



✓ WEATHER EXPERIMENT RECORDED


In [46]:

print("=" * 65)
print("DEFINING TEMPORAL WEATHER ENCODER")
print("=" * 65)


class TemporalWeatherEncoder(nn.Module):

    def __init__(
        self,
        input_features=6,
        hidden_size=64,
        nhead=4,
        num_layers=1,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()


        self.input_projection = nn.Sequential(
            nn.Linear(
                input_features,
                32
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                hidden_size
            )
        )


        self.positional_encoding = PositionalEncoding(
            d_model=hidden_size,
            max_len=INPUT_STEPS
        )


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, weather):


        x = self.input_projection(weather)


        x = self.positional_encoding(x)

        x = self.temporal_encoder(x)

        x = x[:, -1, :]


        x = self.norm(x)

        return x



weather_encoder = TemporalWeatherEncoder(
    input_features=6,
    hidden_size=64,
    nhead=4,
    num_layers=1,
    dim_feedforward=128,
    dropout=0.1
).to(device)


weather_encoder_params = sum(
    p.numel()
    for p in weather_encoder.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", weather_encoder_params)


weather_encoder.eval()

with torch.no_grad():

    test_weather_input = weather_batch.to(device)

    encoded_weather = weather_encoder(
        test_weather_input
    )

print("\nInput shape :", test_weather_input.shape)
print("Output shape:", encoded_weather.shape)

assert test_weather_input.shape == (64, 12, 6)
assert encoded_weather.shape == (64, 64)

print("\n" + "=" * 65)
print("✓ TEMPORAL WEATHER ENCODER SHAPE TEST PASSED")
print("=" * 65)

DEFINING TEMPORAL WEATHER ENCODER


NameError: name 'INPUT_STEPS' is not defined

In [137]:

print("=" * 65)
print("DEFINING WEATHER-AWARE URBANFLOW v2")
print("=" * 65)


class WeatherAwareUrbanFlowV2(nn.Module):

    def __init__(
        self,
        num_sensors=207,
        weather_features=6,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        spatial_hidden=64,
        weather_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.d_model = d_model


        self.input_projection = nn.Linear(
            1,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=INPUT_STEPS
        )

        traffic_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_transformer = nn.TransformerEncoder(
            traffic_encoder_layer,
            num_layers=num_layers
        )


        self.spatial_projection = nn.Linear(
            d_model,
            spatial_hidden
        )

        self.spatial_transform = nn.Linear(
            spatial_hidden,
            spatial_hidden
        )


        self.weather_encoder = TemporalWeatherEncoder(
            input_features=weather_features,
            hidden_size=weather_hidden,
            nhead=4,
            num_layers=1,
            dim_feedforward=128,
            dropout=dropout
        )


        fusion_input = (
            d_model
            + spatial_hidden
            + weather_hidden
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                fusion_input,
                d_model
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )


        self.output_layer = nn.Linear(
            d_model,
            1
        )

    def forward(self, traffic, weather):


        B, T, S = traffic.shape


        x = traffic.permute(0, 2, 1)


        x = x.reshape(
            B * S,
            T,
            1
        )


        x = self.input_projection(x)


        x = self.positional_encoding(x)

        x = self.temporal_transformer(x)


        temporal_features = x[:, -1, :]


        temporal_features = temporal_features.reshape(
            B,
            S,
            self.d_model
        )



        spatial = self.spatial_projection(
            temporal_features
        )

        spatial = torch.relu(
            self.spatial_transform(spatial)
        )


        spatial = torch.einsum(
            "ij,bjf->bif",
            A_norm,
            spatial
        )



        weather_features_encoded = self.weather_encoder(
            weather
        )



        weather_features_encoded = (
            weather_features_encoded
            .unsqueeze(1)
            .expand(
                B,
                S,
                weather_features_encoded.shape[-1]
            )
        )



        fused = torch.cat(
            [
                temporal_features,
                spatial,
                weather_features_encoded
            ],
            dim=-1
        )


        fused = self.fusion(fused)



        output = self.output_layer(
            fused
        )


        output = output.squeeze(-1)


        return output



weather_model_v2 = WeatherAwareUrbanFlowV2(
    num_sensors=207,
    weather_features=6,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    spatial_hidden=64,
    weather_hidden=64,
    dropout=0.1
).to(device)



weather_v2_params = sum(
    p.numel()
    for p in weather_model_v2.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", weather_v2_params)



weather_model_v2.eval()

with torch.no_grad():

    test_traffic = traffic_batch.to(device)
    test_weather = weather_batch.to(device)

    test_output_v2 = weather_model_v2(
        test_traffic,
        test_weather
    )

print("\nTraffic input :", test_traffic.shape)
print("Weather input :", test_weather.shape)
print("Output        :", test_output_v2.shape)

assert test_traffic.shape == (64, 12, 207)
assert test_weather.shape == (64, 12, 6)
assert test_output_v2.shape == (64, 207)

print("\n" + "=" * 65)
print("✓ WEATHER-AWARE URBANFLOW v2 SHAPE TEST PASSED")
print("=" * 65)

DEFINING WEATHER-AWARE URBANFLOW v2

Trainable parameters: 346593

Traffic input : torch.Size([64, 12, 207])
Weather input : torch.Size([64, 12, 6])
Output        : torch.Size([64, 207])

✓ WEATHER-AWARE URBANFLOW v2 SHAPE TEST PASSED


In [138]:

print("=" * 65)
print("TRAINING WEATHER-AWARE URBANFLOW v2")
print("=" * 65)


WEATHER_V2_EPOCHS = 20
WEATHER_V2_LR = 0.001
WEATHER_V2_PATIENCE = 4
WEATHER_V2_HUBER_BETA = 1.0

print("\nConfiguration:")
print("Epochs       :", WEATHER_V2_EPOCHS)
print("Learning rate:", WEATHER_V2_LR)
print("Huber beta   :", WEATHER_V2_HUBER_BETA)
print("Device       :", device)


weather_model_v2 = WeatherAwareUrbanFlowV2(
    num_sensors=207,
    weather_features=6,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    spatial_hidden=64,
    weather_hidden=64,
    dropout=0.1
).to(device)


criterion_weather_v2 = nn.SmoothL1Loss(
    beta=WEATHER_V2_HUBER_BETA,
    reduction="none"
)


optimizer_weather_v2 = torch.optim.Adam(
    weather_model_v2.parameters(),
    lr=WEATHER_V2_LR
)


weather_v2_checkpoint = (
    "../../models/weather_aware_urbanflow_v2_best.pth"
)


weather_v2_history = []

best_v2_val_loss = float("inf")
best_v2_epoch = 0
v2_patience_counter = 0



def masked_huber_loss_v2(
    predictions,
    targets,
    mask
):

    loss = criterion_weather_v2(
        predictions,
        targets
    )

    mask_float = mask.float()

    return (
        (loss * mask_float).sum()
        / mask_float.sum().clamp_min(1.0)
    )



for epoch in range(
    1,
    WEATHER_V2_EPOCHS + 1
):


    weather_model_v2.train()

    train_loss_sum = 0.0
    train_batches = 0

    for (
        traffic_x,
        weather_x,
        targets,
        masks
    ) in weather_train_loader:

        traffic_x = traffic_x.to(device)
        weather_x = weather_x.to(device)
        targets = targets.to(device)
        masks = masks.to(device)

        optimizer_weather_v2.zero_grad()

        predictions = weather_model_v2(
            traffic_x,
            weather_x
        )

        loss = masked_huber_loss_v2(
            predictions,
            targets,
            masks
        )

        loss.backward()

        optimizer_weather_v2.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = (
        train_loss_sum
        / train_batches
    )


    weather_model_v2.eval()

    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        for (
            traffic_x,
            weather_x,
            targets,
            masks
        ) in weather_val_loader:

            traffic_x = traffic_x.to(device)
            weather_x = weather_x.to(device)
            targets = targets.to(device)
            masks = masks.to(device)

            predictions = weather_model_v2(
                traffic_x,
                weather_x
            )

            loss = masked_huber_loss_v2(
                predictions,
                targets,
                masks
            )

            val_loss_sum += loss.item()
            val_batches += 1

    val_loss = (
        val_loss_sum
        / val_batches
    )


    weather_v2_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })


    improved = (
        val_loss < best_v2_val_loss
    )

    if improved:

        best_v2_val_loss = val_loss
        best_v2_epoch = epoch
        v2_patience_counter = 0

        torch.save(
            weather_model_v2.state_dict(),
            weather_v2_checkpoint
        )

    else:

        v2_patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"{'BEST' if improved else ''}"
    )


    if (
        v2_patience_counter
        >= WEATHER_V2_PATIENCE
    ):

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break



weather_model_v2.load_state_dict(
    torch.load(
        weather_v2_checkpoint,
        map_location=device,
        weights_only=True
    )
)

weather_model_v2.eval()



print("\n" + "=" * 65)
print("WEATHER-AWARE URBANFLOW v2 TRAINING COMPLETE")
print("=" * 65)

print(
    "Best epoch:",
    best_v2_epoch
)

print(
    "Best validation loss:",
    best_v2_val_loss
)

print(
    "Checkpoint:",
    weather_v2_checkpoint
)

TRAINING WEATHER-AWARE URBANFLOW v2

Configuration:
Epochs       : 20
Learning rate: 0.001
Huber beta   : 1.0
Device       : mps
Epoch 01 | Train Loss: 0.187432 | Val Loss: 0.175124 | BEST
Epoch 02 | Train Loss: 0.180551 | Val Loss: 0.169366 | BEST
Epoch 03 | Train Loss: 0.178579 | Val Loss: 0.167705 | BEST
Epoch 04 | Train Loss: 0.177173 | Val Loss: 0.168061 | 
Epoch 05 | Train Loss: 0.176847 | Val Loss: 0.166660 | BEST
Epoch 06 | Train Loss: 0.176175 | Val Loss: 0.172014 | 
Epoch 07 | Train Loss: 0.176053 | Val Loss: 0.165651 | BEST
Epoch 08 | Train Loss: 0.175689 | Val Loss: 0.166863 | 
Epoch 09 | Train Loss: 0.175064 | Val Loss: 0.165186 | BEST
Epoch 10 | Train Loss: 0.175100 | Val Loss: 0.164998 | BEST
Epoch 11 | Train Loss: 0.174668 | Val Loss: 0.165779 | 
Epoch 12 | Train Loss: 0.174504 | Val Loss: 0.165855 | 
Epoch 13 | Train Loss: 0.174073 | Val Loss: 0.165105 | 
Epoch 14 | Train Loss: 0.174039 | Val Loss: 0.164386 | BEST
Epoch 15 | Train Loss: 0.173900 | Val Loss: 0.164974 | 

In [139]:

print("=" * 65)
print("EVALUATING WEATHER-AWARE URBANFLOW v2")
print("=" * 65)


weather_model_v2.load_state_dict(
    torch.load(
        "../../models/weather_aware_urbanflow_v2_best.pth",
        map_location=device,
        weights_only=True
    )
)

weather_model_v2.eval()


v2_predictions = []
v2_targets = []
v2_masks = []

with torch.no_grad():

    for (
        traffic_x,
        weather_x,
        targets,
        masks
    ) in weather_val_loader:

        traffic_x = traffic_x.to(device)
        weather_x = weather_x.to(device)

        predictions = weather_model_v2(
            traffic_x,
            weather_x
        )

        v2_predictions.append(
            predictions.cpu().numpy()
        )

        v2_targets.append(
            targets.numpy()
        )

        v2_masks.append(
            masks.numpy()
        )


v2_predictions_scaled = np.concatenate(
    v2_predictions,
    axis=0
)

v2_targets_scaled = np.concatenate(
    v2_targets,
    axis=0
)

v2_validation_mask = np.concatenate(
    v2_masks,
    axis=0
)

print("\nScaled prediction shape:",
      v2_predictions_scaled.shape)

print("Scaled target shape    :",
      v2_targets_scaled.shape)

print("Mask shape             :",
      v2_validation_mask.shape)


v2_predictions_mph = traffic_scaler.inverse_transform(
    v2_predictions_scaled
)

v2_targets_mph = traffic_scaler.inverse_transform(
    v2_targets_scaled
)


v2_valid_predictions = v2_predictions_mph[
    v2_validation_mask
]

v2_valid_targets = v2_targets_mph[
    v2_validation_mask
]


v2_mae = mean_absolute_error(
    v2_valid_targets,
    v2_valid_predictions
)

v2_rmse = np.sqrt(
    mean_squared_error(
        v2_valid_targets,
        v2_valid_predictions
    )
)

v2_valid_count = len(
    v2_valid_targets
)


print("\n" + "=" * 65)
print("WEATHER-AWARE URBANFLOW v2 RESULTS")
print("=" * 65)

print(f"MAE  : {v2_mae:.6f} mph")
print(f"RMSE : {v2_rmse:.6f} mph")
print(
    f"Valid observations: {v2_valid_count:,}"
)


print("\nPrediction statistics:")

print(
    f"Prediction range: "
    f"{v2_valid_predictions.min():.4f} → "
    f"{v2_valid_predictions.max():.4f} mph"
)

print(
    f"Target range: "
    f"{v2_valid_targets.min():.4f} → "
    f"{v2_valid_targets.max():.4f} mph"
)

v2_negative = (
    v2_valid_predictions < 0
).sum()

v2_above_70 = (
    v2_valid_predictions > 70
).sum()

print(
    f"Negative predictions: "
    f"{v2_negative:,} "
    f"({v2_negative / v2_valid_count * 100:.4f}%)"
)

print(
    f"Predictions > 70 mph: "
    f"{v2_above_70:,} "
    f"({v2_above_70 / v2_valid_count * 100:.4f}%)"
)


BASELINE_MAE = 3.177278
BASELINE_RMSE = 6.316968

V1_MAE = 3.351608
V1_RMSE = 6.588709

print("\n" + "=" * 65)
print("WEATHER MODEL COMPARISON")
print("=" * 65)

print(
    f"{'Model':<38}"
    f"{'MAE':>12}"
    f"{'RMSE':>12}"
)

print("-" * 65)

print(
    f"{'Graph-Aware + Huber':<38}"
    f"{BASELINE_MAE:>12.6f}"
    f"{BASELINE_RMSE:>12.6f}"
)

print(
    f"{'Weather v1 - Last timestep':<38}"
    f"{V1_MAE:>12.6f}"
    f"{V1_RMSE:>12.6f}"
)

print(
    f"{'Weather v2 - Temporal encoder':<38}"
    f"{v2_mae:>12.6f}"
    f"{v2_rmse:>12.6f}"
)


v2_mae_change = v2_mae - BASELINE_MAE
v2_rmse_change = v2_rmse - BASELINE_RMSE

v2_mae_change_pct = (
    (v2_mae - BASELINE_MAE)
    / BASELINE_MAE
    * 100
)

v2_rmse_change_pct = (
    (v2_rmse - BASELINE_RMSE)
    / BASELINE_RMSE
    * 100
)

print("\nChange vs weather-free baseline:")

print(
    f"MAE change : "
    f"{v2_mae_change:+.6f} mph "
    f"({v2_mae_change_pct:+.2f}%)"
)

print(
    f"RMSE change: "
    f"{v2_rmse_change:+.6f} mph "
    f"({v2_rmse_change_pct:+.2f}%)"
)

print("\n" + "=" * 65)
print("✓ WEATHER v2 VALIDATION EVALUATION COMPLETE")
print("=" * 65)

EVALUATING WEATHER-AWARE URBANFLOW v2

Scaled prediction shape: (3410, 207)
Scaled target shape    : (3410, 207)
Mask shape             : (3410, 207)

WEATHER-AWARE URBANFLOW v2 RESULTS
MAE  : 3.361279 mph
RMSE : 6.591718 mph
Valid observations: 664,844

Prediction statistics:
Prediction range: -15.9804 → 71.7951 mph
Target range: 0.5000 → 70.0000 mph
Negative predictions: 19 (0.0029%)
Predictions > 70 mph: 363 (0.0546%)

WEATHER MODEL COMPARISON
Model                                          MAE        RMSE
-----------------------------------------------------------------
Graph-Aware + Huber                       3.177278    6.316968
Weather v1 - Last timestep                3.351608    6.588709
Weather v2 - Temporal encoder             3.361279    6.591718

Change vs weather-free baseline:
MAE change : +0.184001 mph (+5.79%)
RMSE change: +0.274750 mph (+4.35%)

✓ WEATHER v2 VALIDATION EVALUATION COMPLETE


In [140]:

print("=" * 65)
print("RECORDING WEATHER v2 EXPERIMENT")
print("=" * 65)

weather_v2_experiment = {
    "experiment": "Weather-Aware UrbanFlow v2",
    "weather_features": "temp,rhum,prcp,wspd,pres,cldc",
    "weather_representation": "temporal_encoder_12_steps",
    "loss": "Huber",
    "learning_rate": WEATHER_V2_LR,
    "dropout": 0.1,
    "best_epoch": int(best_v2_epoch),
    "validation_mae": float(v2_mae),
    "validation_rmse": float(v2_rmse),
    "valid_observations": int(v2_valid_count),
    "baseline_mae": BASELINE_MAE,
    "baseline_rmse": BASELINE_RMSE,
    "mae_change": float(v2_mae_change),
    "rmse_change": float(v2_rmse_change),
    "mae_change_percent": float(v2_mae_change_pct),
    "rmse_change_percent": float(v2_rmse_change_pct),
}

weather_results_path = (
    "../../results/experiments/weather_experiments.csv"
)

existing_weather_results = pd.read_csv(
    weather_results_path
)

existing_weather_results = existing_weather_results[
    existing_weather_results["experiment"]
    != weather_v2_experiment["experiment"]
]

updated_weather_results = pd.concat(
    [
        existing_weather_results,
        pd.DataFrame([weather_v2_experiment])
    ],
    ignore_index=True
)

updated_weather_results.to_csv(
    weather_results_path,
    index=False
)

print("\nSaved:")
print(weather_results_path)

display(updated_weather_results)

print("\n" + "=" * 65)
print("✓ WEATHER v2 EXPERIMENT RECORDED")
print("=" * 65)

RECORDING WEATHER v2 EXPERIMENT

Saved:
../../results/experiments/weather_experiments.csv


,experiment,weather_features,weather_representation,loss,learning_rate,dropout,best_epoch,validation_mae,validation_rmse,valid_observations,baseline_mae,baseline_rmse,mae_change,rmse_change,mae_change_percent,rmse_change_percent
0,Weather-Aware UrbanFlow v1,"temp,rhum,prcp,wspd,pres,cldc",last_timestep_only,Huber,0.001,0.1,19,3.351608,6.588709,664844,3.177278,6.316968,0.174330,0.271741,-5.486773,-4.301769
1,Weather-Aware UrbanFlow v2,"temp,rhum,prcp,wspd,pres,cldc",temporal_encoder_12_steps,Huber,0.001,0.1,18,3.361279,6.591718,664844,3.177278,6.316968,0.184001,0.274750,5.791152,4.349390



✓ WEATHER v2 EXPERIMENT RECORDED


In [142]:

print("=" * 65)
print("WEATHER FEATURE OCCLUSION ANALYSIS")
print("=" * 65)


WEATHER_FEATURE_NAMES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]


weather_train_sequences = np.load(
    "../../data/processed/weather_train_sequences.npy"
)

print(
    "\nLoaded training weather sequences:",
    weather_train_sequences.shape
)

assert weather_train_sequences.shape == (
    23973,
    12,
    6
)


weather_model_v2.load_state_dict(
    torch.load(
        "../../models/weather_aware_urbanflow_v2_best.pth",
        map_location=device,
        weights_only=True
    )
)

weather_model_v2.eval()


weather_train_mean = np.mean(
    weather_train_sequences,
    axis=(0, 1)
)

print("\nWeather feature replacement means:")

for name, value in zip(
    WEATHER_FEATURE_NAMES,
    weather_train_mean
):
    print(
        f"{name:>6}: {value:.6f}"
    )



def evaluate_weather_input(
    model,
    feature_override=None
):

    predictions_list = []
    targets_list = []
    masks_list = []

    with torch.no_grad():

        for (
            traffic_x,
            weather_x,
            targets,
            masks
        ) in weather_val_loader:

            traffic_x = traffic_x.to(device)
            weather_x = weather_x.to(device)


            if feature_override is not None:

                replacement_value = float(
                    weather_train_mean[
                        feature_override
                    ]
                )

                weather_x = weather_x.clone()

                weather_x[
                    :,
                    :,
                    feature_override
                ] = replacement_value


            predictions = model(
                traffic_x,
                weather_x
            )

            predictions_list.append(
                predictions.cpu().numpy()
            )

            targets_list.append(
                targets.numpy()
            )

            masks_list.append(
                masks.numpy()
            )


    predictions_scaled = np.concatenate(
        predictions_list,
        axis=0
    )

    targets_scaled = np.concatenate(
        targets_list,
        axis=0
    )

    masks = np.concatenate(
        masks_list,
        axis=0
    )


    predictions_mph = traffic_scaler.inverse_transform(
        predictions_scaled
    )

    targets_mph = traffic_scaler.inverse_transform(
        targets_scaled
    )


    valid_predictions = predictions_mph[masks]
    valid_targets = targets_mph[masks]


    mae_value = mean_absolute_error(
        valid_targets,
        valid_predictions
    )

    rmse_value = np.sqrt(
        mean_squared_error(
            valid_targets,
            valid_predictions
        )
    )

    return mae_value, rmse_value



print("\n" + "-" * 65)
print("BASELINE — ALL WEATHER FEATURES")
print("-" * 65)

full_mae, full_rmse = evaluate_weather_input(
    weather_model_v2
)

print(
    f"MAE : {full_mae:.6f} mph"
)

print(
    f"RMSE: {full_rmse:.6f} mph"
)



occlusion_results = []

for feature_index, feature_name in enumerate(
    WEATHER_FEATURE_NAMES
):

    print("\n" + "-" * 65)

    print(
        f"Occluding feature: {feature_name}"
    )

    occluded_mae, occluded_rmse = (
        evaluate_weather_input(
            weather_model_v2,
            feature_override=feature_index
        )
    )

    mae_change = (
        occluded_mae - full_mae
    )

    rmse_change = (
        occluded_rmse - full_rmse
    )

    occlusion_results.append({

        "feature": feature_name,

        "full_model_mae":
            full_mae,

        "occluded_mae":
            occluded_mae,

        "mae_change":
            mae_change,

        "full_model_rmse":
            full_rmse,

        "occluded_rmse":
            occluded_rmse,

        "rmse_change":
            rmse_change
    })

    print(
        f"Occluded MAE : "
        f"{occluded_mae:.6f} mph"
    )

    print(
        f"Occluded RMSE: "
        f"{occluded_rmse:.6f} mph"
    )

    print(
        f"MAE change   : "
        f"{mae_change:+.6f}"
    )

    print(
        f"RMSE change  : "
        f"{rmse_change:+.6f}"
    )



weather_occlusion_df = pd.DataFrame(
    occlusion_results
)

weather_occlusion_df = (
    weather_occlusion_df
    .sort_values(
        "mae_change",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 65)
print("WEATHER FEATURE OCCLUSION RESULTS")
print("=" * 65)

display(
    weather_occlusion_df
)



occlusion_path = (
    "../../results/experiments/"
    "weather_feature_occlusion.csv"
)

weather_occlusion_df.to_csv(
    occlusion_path,
    index=False
)

print("\nSaved:")
print(occlusion_path)

print("\n" + "=" * 65)
print("✓ WEATHER FEATURE OCCLUSION COMPLETE")
print("=" * 65)

WEATHER FEATURE OCCLUSION ANALYSIS

Loaded training weather sequences: (23973, 12, 6)

Weather feature replacement means:
  temp: 14.969591
  rhum: 70.040298
  prcp: 0.038241
  wspd: 12.268380
  pres: 1015.847046
  cldc: 4.475910

-----------------------------------------------------------------
BASELINE — ALL WEATHER FEATURES
-----------------------------------------------------------------
MAE : 3.361279 mph
RMSE: 6.591718 mph

-----------------------------------------------------------------
Occluding feature: temp
Occluded MAE : 3.361286 mph
Occluded RMSE: 6.591716 mph
MAE change   : +0.000007
RMSE change  : -0.000001

-----------------------------------------------------------------
Occluding feature: rhum
Occluded MAE : 3.361287 mph
Occluded RMSE: 6.591896 mph
MAE change   : +0.000008
RMSE change  : +0.000179

-----------------------------------------------------------------
Occluding feature: prcp
Occluded MAE : 3.361279 mph
Occluded RMSE: 6.591718 mph
MAE change   : +0.000000
R

,feature,full_model_mae,occluded_mae,mae_change,full_model_rmse,occluded_rmse,rmse_change
0,wspd,3.361279,3.361296,1.692772e-05,6.591718,6.591777,0.000059
1,rhum,3.361279,3.361287,7.629395e-06,6.591718,6.591896,0.000179
2,temp,3.361279,3.361286,7.152557e-06,6.591718,6.591716,-0.000001
3,cldc,3.361279,3.361280,9.536743e-07,6.591718,6.591718,0.000000
4,prcp,3.361279,3.361279,0.000000e+00,6.591718,6.591718,0.000000
5,pres,3.361279,3.361275,-3.576279e-06,6.591718,6.591723,0.000006



Saved:
../../results/experiments/weather_feature_occlusion.csv

✓ WEATHER FEATURE OCCLUSION COMPLETE


In [143]:

print("=" * 65)
print("WEATHER SHUFFLE TEST")
print("=" * 65)

weather_model_v2.eval()


def evaluate_weather_shuffle(
    model,
    shuffle_weather=False
):

    predictions_list = []
    targets_list = []
    masks_list = []

    with torch.no_grad():

        for (
            traffic_x,
            weather_x,
            targets,
            masks
        ) in weather_val_loader:

            traffic_x = traffic_x.to(device)
            weather_x = weather_x.to(device)


            if shuffle_weather:

                permutation = torch.randperm(
                    weather_x.size(0),
                    device=device
                )

                weather_x = weather_x[
                    permutation
                ]

            predictions = model(
                traffic_x,
                weather_x
            )

            predictions_list.append(
                predictions.cpu().numpy()
            )

            targets_list.append(
                targets.numpy()
            )

            masks_list.append(
                masks.numpy()
            )

    predictions_scaled = np.concatenate(
        predictions_list,
        axis=0
    )

    targets_scaled = np.concatenate(
        targets_list,
        axis=0
    )

    masks = np.concatenate(
        masks_list,
        axis=0
    )

    predictions_mph = traffic_scaler.inverse_transform(
        predictions_scaled
    )

    targets_mph = traffic_scaler.inverse_transform(
        targets_scaled
    )

    valid_predictions = predictions_mph[masks]
    valid_targets = targets_mph[masks]

    mae_value = mean_absolute_error(
        valid_targets,
        valid_predictions
    )

    rmse_value = np.sqrt(
        mean_squared_error(
            valid_targets,
            valid_predictions
        )
    )

    return mae_value, rmse_value



normal_mae, normal_rmse = evaluate_weather_shuffle(
    weather_model_v2,
    shuffle_weather=False
)

print("\nNormal weather:")
print(f"MAE : {normal_mae:.6f} mph")
print(f"RMSE: {normal_rmse:.6f} mph")



shuffled_mae, shuffled_rmse = evaluate_weather_shuffle(
    weather_model_v2,
    shuffle_weather=True
)

print("\nShuffled weather:")
print(f"MAE : {shuffled_mae:.6f} mph")
print(f"RMSE: {shuffled_rmse:.6f} mph")



mae_change = shuffled_mae - normal_mae
rmse_change = shuffled_rmse - normal_rmse

print("\n" + "-" * 65)
print("WEATHER SHUFFLE IMPACT")
print("-" * 65)

print(
    f"MAE change : {mae_change:+.6f} mph"
)

print(
    f"RMSE change: {rmse_change:+.6f} mph"
)



weather_shuffle_result = pd.DataFrame([
    {
        "condition": "normal_weather",
        "mae": normal_mae,
        "rmse": normal_rmse
    },
    {
        "condition": "shuffled_weather",
        "mae": shuffled_mae,
        "rmse": shuffled_rmse
    }
])

shuffle_path = (
    "../../results/experiments/"
    "weather_shuffle_test.csv"
)

weather_shuffle_result.to_csv(
    shuffle_path,
    index=False
)

print("\nSaved:")
print(shuffle_path)

print("\n" + "=" * 65)
print("✓ WEATHER SHUFFLE TEST COMPLETE")
print("=" * 65)

WEATHER SHUFFLE TEST

Normal weather:
MAE : 3.361279 mph
RMSE: 6.591718 mph

Shuffled weather:
MAE : 3.361295 mph
RMSE: 6.591752 mph

-----------------------------------------------------------------
WEATHER SHUFFLE IMPACT
-----------------------------------------------------------------
MAE change : +0.000016 mph
RMSE change: +0.000035 mph

Saved:
../../results/experiments/weather_shuffle_test.csv

✓ WEATHER SHUFFLE TEST COMPLETE


In [144]:

print("=" * 70)
print("WEATHER INTEGRATION EXPERIMENT SUMMARY")
print("=" * 70)


weather_summary = pd.DataFrame([
    {
        "experiment": "Weather-Free Graph-Aware UrbanFlow + Huber",
        "weather_used": False,
        "MAE": 3.177278,
        "RMSE": 6.316968,
        "valid_observations": 664844
    },
    {
        "experiment": "Weather-Aware UrbanFlow v1",
        "weather_used": True,
        "weather_representation": "last_timestep_only",
        "MAE": 3.351608,
        "RMSE": 6.588709,
        "valid_observations": 664844
    },
    {
        "experiment": "Weather-Aware UrbanFlow v2",
        "weather_used": True,
        "weather_representation": "temporal_weather_encoder",
        "MAE": 3.361279,
        "RMSE": 6.591718,
        "valid_observations": 664844
    }
])


weather_free_mae = 3.177278
weather_free_rmse = 6.316968

weather_summary["MAE_change_vs_weather_free"] = (
    weather_summary["MAE"] - weather_free_mae
)

weather_summary["RMSE_change_vs_weather_free"] = (
    weather_summary["RMSE"] - weather_free_rmse
)

print("\nWeather model comparison:")
display(weather_summary)



print("\n" + "=" * 70)
print("FEATURE OCCLUSION")
print("=" * 70)

display(
    weather_occlusion_df[
        [
            "feature",
            "mae_change",
            "rmse_change"
        ]
    ]
)



print("\n" + "=" * 70)
print("WEATHER SHUFFLE TEST")
print("=" * 70)

print(
    f"Normal MAE    : {normal_mae:.6f}"
)

print(
    f"Shuffled MAE  : {shuffled_mae:.6f}"
)

print(
    f"MAE change    : {mae_change:+.6f}"
)

print(
    f"Normal RMSE   : {normal_rmse:.6f}"
)

print(
    f"Shuffled RMSE : {shuffled_rmse:.6f}"
)

print(
    f"RMSE change   : {rmse_change:+.6f}"
)



print("\n" + "=" * 70)
print("WEATHER EXPERIMENT CONCLUSION")
print("=" * 70)

print(
    "Weather was evaluated as external contextual information "
    "using six aligned weather features."
)

print(
    "Feature occlusion produced negligible changes in validation "
    "performance."
)

print(
    "Shuffling weather across validation samples also produced "
    "negligible changes."
)

print(
    "The weather-aware models did not outperform the "
    "weather-free Graph-Aware UrbanFlow + Huber baseline."
)

print(
    "\nDecision: retain the weather experiments as an ablation/"
    "research finding, but do not include weather in the current "
    "final forecasting architecture."
)

print("=" * 70)



weather_summary_path = (
    "../../results/experiments/"
    "weather_integration_summary.csv"
)

weather_summary.to_csv(
    weather_summary_path,
    index=False
)

print("\nSaved:")
print(weather_summary_path)

print("\n" + "=" * 70)
print("✓ WEATHER EXPERIMENT SUMMARY SAVED")
print("=" * 70)

WEATHER INTEGRATION EXPERIMENT SUMMARY

Weather model comparison:


,experiment,weather_used,MAE,RMSE,valid_observations,weather_representation,MAE_change_vs_weather_free,RMSE_change_vs_weather_free
0,Weather-Free Graph-Aware UrbanFlow + Huber,False,3.177278,6.316968,664844,NaN,0.000000,0.000000
1,Weather-Aware UrbanFlow v1,True,3.351608,6.588709,664844,last_timestep_only,0.174330,0.271741
2,Weather-Aware UrbanFlow v2,True,3.361279,6.591718,664844,temporal_weather_encoder,0.184001,0.274750



FEATURE OCCLUSION


,feature,mae_change,rmse_change
0,wspd,1.692772e-05,0.000059
1,rhum,7.629395e-06,0.000179
2,temp,7.152557e-06,-0.000001
3,cldc,9.536743e-07,0.000000
4,prcp,0.000000e+00,0.000000
5,pres,-3.576279e-06,0.000006



WEATHER SHUFFLE TEST
Normal MAE    : 3.361279
Shuffled MAE  : 3.361295
MAE change    : +0.000016
Normal RMSE   : 6.591718
Shuffled RMSE : 6.591752
RMSE change   : +0.000035

WEATHER EXPERIMENT CONCLUSION
Weather was evaluated as external contextual information using six aligned weather features.
Feature occlusion produced negligible changes in validation performance.
Shuffling weather across validation samples also produced negligible changes.
The weather-aware models did not outperform the weather-free Graph-Aware UrbanFlow + Huber baseline.

Decision: retain the weather experiments as an ablation/research finding, but do not include weather in the current final forecasting architecture.

Saved:
../../results/experiments/weather_integration_summary.csv

✓ WEATHER EXPERIMENT SUMMARY SAVED


In [ ]:


import numpy as np
import h5py
import pandas as pd
from pathlib import Path


print("=" * 70)
print("BUILDING CORRECT MULTI-HORIZON TARGETS")
print("=" * 70)



INPUT_STEPS = 12
N_SENSORS = 207

HORIZONS = {
    "15min": 3,
    "30min": 6,
    "45min": 9,
    "60min": 12
}



train_npz = np.load(
    "../../data/processed/train_sequences.npz"
)

val_npz = np.load(
    "../../data/processed/validation_sequences.npz"
)

test_npz = np.load(
    "../../data/processed/test_sequences.npz"
)


X = {
    "train": train_npz["X"],
    "val": val_npz["X"],
    "test": test_npz["X"]
}


Y_30_EXISTING = {
    "train": train_npz["y"],
    "val": val_npz["y"],
    "test": test_npz["y"]
}


MASK_30_EXISTING = {
    "train": np.load(
        "../../data/processed/train_target_mask.npy"
    ),
    "val": np.load(
        "../../data/processed/validation_target_mask.npy"
    ),
    "test": np.load(
        "../../data/processed/test_target_mask.npy"
    )
}


print("\nExisting Notebook 2 data:")

for split in [
    "train",
    "val",
    "test"
]:

    print(
        f"{split.capitalize():5s}: "
        f"X={X[split].shape} | "
        f"Y={Y_30_EXISTING[split].shape} | "
        f"Mask={MASK_30_EXISTING[split].shape}"
    )



assert X["train"].shape == (
    23973,
    12,
    207
)

assert X["val"].shape == (
    3410,
    12,
    207
)

assert X["test"].shape == (
    6838,
    12,
    207
)



raw_path = "../../data/raw/metr-la.h5"

with h5py.File(raw_path, "r") as f:

    raw_values = f[
        "df/block0_values"
    ][:]

    raw_timestamps = f[
        "df/axis1"
    ][:]


traffic_index = pd.DatetimeIndex(
    pd.to_datetime(
        raw_timestamps,
        unit="ns"
    )
)


assert raw_values.shape == (
    34272,
    N_SENSORS
)

assert traffic_index[0] == pd.Timestamp(
    "2012-03-01 00:00:00"
)

assert traffic_index[-1] == pd.Timestamp(
    "2012-06-27 23:55:00"
)



original_valid = (
    raw_values != 0
)


print(
    "\nOriginal valid observations:",
    int(original_valid.sum())
)

print(
    "Original missing/zero observations:",
    int((~original_valid).sum())
)


sequence_starts = {

    "train": (
        0
        + np.arange(
            X["train"].shape[0]
        )
    ),

    "val": (
        23990
        + np.arange(
            X["val"].shape[0]
        )
    ),

    "test": (
        27417
        + np.arange(
            X["test"].shape[0]
        )
    )
}


print("\nSequence boundaries:")

for split in [
    "train",
    "val",
    "test"
]:

    starts = sequence_starts[split]

    print(
        f"{split.capitalize():5s}: "
        f"{starts[0]} -> {starts[-1]} "
        f"({len(starts)} sequences)"
    )



def build_horizon(
    split,
    horizon_steps
):

    X_split = X[split]

    starts = sequence_starts[
        split
    ]

    n_sequences = len(
        starts
    )



    target_positions = (
        starts
        + INPUT_STEPS
        - 1
        + horizon_steps
    )



    available = (
        target_positions
        < len(raw_values)
    )


    target_positions = (
        target_positions[
            available
        ]
    )


    valid_sequence_indices = np.where(
        available
    )[0]


    shifted_indices = (
        valid_sequence_indices
        + horizon_steps
    )


    shifted_exists = (
        shifted_indices
        < n_sequences
    )


    valid_sequence_indices = (
        valid_sequence_indices[
            shifted_exists
        ]
    )

    shifted_indices = (
        shifted_indices[
            shifted_exists
        ]
    )

    target_positions = (
        target_positions[
            shifted_exists
        ]
    )



    target_values = (
        X_split[
            shifted_indices,
            INPUT_STEPS - 1,
            :
        ]
    ).copy()



    target_mask = (
        original_valid[
            target_positions,
            :
        ]
    )


    assert target_values.shape == (
        len(target_positions),
        N_SENSORS
    )

    assert target_mask.shape == (
        len(target_positions),
        N_SENSORS
    )


    return (
        target_values.astype(np.float32),
        target_mask,
        target_positions
    )



multi_horizon_targets = {}
multi_horizon_masks = {}
multi_horizon_positions = {}


for horizon_name, horizon_steps in HORIZONS.items():

    print("\n" + "=" * 70)
    print(
        f"BUILDING {horizon_name.upper()} TARGET"
    )
    print("=" * 70)

    multi_horizon_targets[
        horizon_name
    ] = {}

    multi_horizon_masks[
        horizon_name
    ] = {}

    multi_horizon_positions[
        horizon_name
    ] = {}


    for split in [
        "train",
        "val",
        "test"
    ]:

        (
            target,
            target_mask,
            target_positions
        ) = build_horizon(
            split,
            horizon_steps
        )


        multi_horizon_targets[
            horizon_name
        ][split] = target

        multi_horizon_masks[
            horizon_name
        ][split] = target_mask

        multi_horizon_positions[
            horizon_name
        ][split] = target_positions


        print(
            f"{split.capitalize():5s}: "
            f"Target {target.shape} | "
            f"Valid {target_mask.sum():,} | "
            f"Last target row "
            f"{target_positions[-1]}"
        )



expected_counts = {

    "15min": {
        "train": 23970,
        "val": 3407,
        "test": 6835
    },

    "30min": {
        "train": 23967,
        "val": 3404,
        "test": 6832
    },

    "45min": {
        "train": 23964,
        "val": 3401,
        "test": 6829
    },

    "60min": {
        "train": 23961,
        "val": 3398,
        "test": 6826
    }
}


print("\n" + "=" * 70)
print("VERIFYING MULTI-HORIZON COUNTS")
print("=" * 70)


for horizon_name in HORIZONS:

    for split in [
        "train",
        "val",
        "test"
    ]:

        actual = (
            multi_horizon_targets[
                horizon_name
            ][split].shape[0]
        )

        expected = (
            expected_counts[
                horizon_name
            ][split]
        )

        print(
            f"{horizon_name:6s} "
            f"{split:5s}: "
            f"{actual}"
        )

        assert actual == expected, (
            f"{horizon_name} {split}: "
            f"{actual} != {expected}"
        )


print(
    "\n✓ MULTI-HORIZON COUNTS VERIFIED"
)



print("\n" + "=" * 70)
print("VERIFYING OFFICIAL 30-MINUTE TARGET")
print("=" * 70)



multi_horizon_targets[
    "30min"
] = {
    "train": Y_30_EXISTING["train"].copy(),
    "val": Y_30_EXISTING["val"].copy(),
    "test": Y_30_EXISTING["test"].copy()
}


multi_horizon_masks[
    "30min"
] = {
    "train": MASK_30_EXISTING["train"].copy(),
    "val": MASK_30_EXISTING["val"].copy(),
    "test": MASK_30_EXISTING["test"].copy()
}


print(
    "Train:",
    multi_horizon_targets["30min"]["train"].shape,
    multi_horizon_masks["30min"]["train"].shape
)

print(
    "Val  :",
    multi_horizon_targets["30min"]["val"].shape,
    multi_horizon_masks["30min"]["val"].shape
)

print(
    "Test :",
    multi_horizon_targets["30min"]["test"].shape,
    multi_horizon_masks["30min"]["test"].shape
)


assert (
    multi_horizon_targets["30min"]["train"].shape
    == (23973, 207)
)

assert (
    multi_horizon_targets["30min"]["val"].shape
    == (3410, 207)
)

assert (
    multi_horizon_targets["30min"]["test"].shape
    == (6838, 207)
)


print(
    "\n✓ OFFICIAL 30-MINUTE TARGET PRESERVED EXACTLY"
)



processed_dir = Path(
    "../../data/processed"
)


for horizon_name in HORIZONS:

    for split in [
        "train",
        "val",
        "test"
    ]:

        np.save(
            processed_dir
            / f"y_{split}_{horizon_name}.npy",

            multi_horizon_targets[
                horizon_name
            ][split]
        )


        np.save(
            processed_dir
            / f"{split}_target_mask_{horizon_name}.npy",

            multi_horizon_masks[
                horizon_name
            ][split]
        )



print("\n" + "=" * 70)
print("MULTI-HORIZON TARGET SUMMARY")
print("=" * 70)


for horizon_name, horizon_steps in HORIZONS.items():

    print(
        f"\n{horizon_name.upper()} "
        f"= {horizon_steps * 5} minutes"
    )

    for split in [
        "train",
        "val",
        "test"
    ]:

        target = (
            multi_horizon_targets[
                horizon_name
            ][split]
        )

        mask = (
            multi_horizon_masks[
                horizon_name
            ][split]
        )

        print(
            f"  {split.capitalize():5s}: "
            f"{target.shape} | "
            f"valid={mask.sum():,}"
        )


print("\n" + "=" * 70)
print(
    "✓ MULTI-HORIZON TARGET CONSTRUCTION COMPLETE"
)
print("=" * 70)

BUILDING CORRECT MULTI-HORIZON TARGETS

Existing Notebook 2 data:
Train: X=(23973, 12, 207) | Y=(23973, 207) | Mask=(23973, 207)
Val  : X=(3410, 12, 207) | Y=(3410, 207) | Mask=(3410, 207)
Test : X=(6838, 12, 207) | Y=(6838, 207) | Mask=(6838, 207)

Original valid observations: 6519002
Original missing/zero observations: 575302

Sequence boundaries:
Train: 0 -> 23972 (23973 sequences)
Val  : 23990 -> 27399 (3410 sequences)
Test : 27417 -> 34254 (6838 sequences)

BUILDING 15MIN TARGET
Train: Target (23970, 207) | Valid 4,608,056 | Last target row 23983
Val  : Target (3407, 207) | Valid 655,917 | Last target row 27410
Test : Target (6835, 207) | Valid 1,243,168 | Last target row 34265

BUILDING 30MIN TARGET
Train: Target (23967, 207) | Valid 4,607,439 | Last target row 23983
Val  : Target (3404, 207) | Valid 655,310 | Last target row 27410
Test : Target (6832, 207) | Valid 1,242,550 | Last target row 34265

BUILDING 45MIN TARGET
Train: Target (23964, 207) | Valid 4,606,822 | Last target 

In [158]:

print("=" * 70)
print("FINAL MULTI-HORIZON DATA VERIFICATION")
print("=" * 70)

EXPECTED = {
    "15min": {
        "train": (23970, 207),
        "val":   (3407, 207),
        "test":  (6835, 207),
    },
    "30min": {
        "train": (23973, 207),
        "val":   (3410, 207),
        "test":  (6838, 207),
    },
    "45min": {
        "train": (23964, 207),
        "val":   (3401, 207),
        "test":  (6829, 207),
    },
    "60min": {
        "train": (23961, 207),
        "val":   (3398, 207),
        "test":  (6826, 207),
    },
}

for horizon in ["15min", "30min", "45min", "60min"]:

    print(f"\n{horizon.upper()}")

    for split in ["train", "val", "test"]:

        y_path = (
            f"../../data/processed/"
            f"y_{split}_{horizon}.npy"
        )

        mask_path = (
            f"../../data/processed/"
            f"{split}_target_mask_{horizon}.npy"
        )

        y = np.load(y_path)
        mask = np.load(mask_path)

        expected_shape = EXPECTED[horizon][split]

        print(
            f"{split.capitalize():5s}: "
            f"Y={y.shape} | "
            f"Mask={mask.shape} | "
            f"Valid={int(mask.sum()):,}"
        )

        assert y.shape == expected_shape, (
            f"{horizon} {split} shape mismatch: "
            f"{y.shape} != {expected_shape}"
        )

        assert mask.shape == expected_shape, (
            f"{horizon} {split} mask mismatch: "
            f"{mask.shape} != {expected_shape}"
        )

        assert np.isfinite(y).all(), (
            f"{horizon} {split} contains non-finite values"
        )

        assert mask.dtype == bool, (
            f"{horizon} {split} mask is not boolean"
        )



assert np.load(
    "../../data/processed/y_train_30min.npy"
).shape == (23973, 207)

assert np.load(
    "../../data/processed/y_val_30min.npy"
).shape == (3410, 207)

assert np.load(
    "../../data/processed/y_test_30min.npy"
).shape == (6838, 207)


print("\n" + "=" * 70)
print("✓ ALL MULTI-HORIZON TARGETS VERIFIED")
print("✓ 30-MINUTE OFFICIAL BENCHMARK PRESERVED")
print("✓ NO TEST MODEL TRAINING PERFORMED")
print("=" * 70)

FINAL MULTI-HORIZON DATA VERIFICATION

15MIN
Train: Y=(23970, 207) | Mask=(23970, 207) | Valid=4,608,056
Val  : Y=(3407, 207) | Mask=(3407, 207) | Valid=655,917
Test : Y=(6835, 207) | Mask=(6835, 207) | Valid=1,243,168

30MIN
Train: Y=(23973, 207) | Mask=(23973, 207) | Valid=4,665,451
Val  : Y=(3410, 207) | Mask=(3410, 207) | Valid=664,844
Test : Y=(6838, 207) | Mask=(6838, 207) | Valid=1,266,610

45MIN
Train: Y=(23964, 207) | Mask=(23964, 207) | Valid=4,606,822
Val  : Y=(3401, 207) | Mask=(3401, 207) | Valid=654,701
Test : Y=(6829, 207) | Mask=(6829, 207) | Valid=1,241,932

60MIN
Train: Y=(23961, 207) | Mask=(23961, 207) | Valid=4,606,210
Val  : Y=(3398, 207) | Mask=(3398, 207) | Valid=654,091
Test : Y=(6826, 207) | Mask=(6826, 207) | Valid=1,241,314

✓ ALL MULTI-HORIZON TARGETS VERIFIED
✓ 30-MINUTE OFFICIAL BENCHMARK PRESERVED
✓ NO TEST MODEL TRAINING PERFORMED


In [ ]:

print("=" * 70)
print("MULTI-HORIZON PERSISTENCE BASELINES")
print("=" * 70)


HORIZONS = {
    "15min": 3,
    "30min": 6,
    "45min": 9,
    "60min": 12,
}


test_data = np.load(
    "../../data/processed/test_sequences.npz"
)

X_test_np = test_data["X"]

print(f"\nTest input shape: {X_test_np.shape}")

assert X_test_np.ndim == 3
assert X_test_np.shape[1:] == (12, 207), (
    f"Unexpected test input shape: {X_test_np.shape}"
)


assert hasattr(traffic_scaler, "n_features_in_"), (
    "traffic_scaler is not fitted or unavailable."
)

assert traffic_scaler.n_features_in_ == 207, (
    f"Expected scaler with 207 features, "
    f"got {traffic_scaler.n_features_in_}"
)

print(
    f"Scaler features: {traffic_scaler.n_features_in_}"
)


results = []

for horizon_name, horizon_steps in HORIZONS.items():

    print("\n" + "=" * 70)
    print(f"{horizon_name.upper()} HORIZON")
    print("=" * 70)


    y_path = (
        f"../../data/processed/"
        f"y_test_{horizon_name}.npy"
    )

    mask_path = (
        f"../../data/processed/"
        f"test_target_mask_{horizon_name}.npy"
    )

    y_true_scaled = np.load(y_path)
    mask = np.load(mask_path)


    n_samples = y_true_scaled.shape[0]

    assert y_true_scaled.shape == (
        n_samples,
        207
    )

    assert mask.shape == (
        n_samples,
        207
    )


    persistence_scaled = (
        X_test_np[:n_samples, -1, :]
    )

    assert persistence_scaled.shape == (
        n_samples,
        207
    )


    persistence_mph = traffic_scaler.inverse_transform(
        persistence_scaled
    )

    target_mph = traffic_scaler.inverse_transform(
        y_true_scaled
    )

    assert persistence_mph.shape == (
        n_samples,
        207
    )

    assert target_mph.shape == (
        n_samples,
        207
    )


    valid_mask = mask.astype(bool)

    valid_predictions = persistence_mph[valid_mask]
    valid_targets = target_mph[valid_mask]

    assert len(valid_predictions) == int(
        valid_mask.sum()
    )


    mae = np.mean(
        np.abs(
            valid_predictions - valid_targets
        )
    )

    rmse = np.sqrt(
        np.mean(
            (
                valid_predictions - valid_targets
            ) ** 2
        )
    )

    valid_observations = int(
        valid_mask.sum()
    )


    print(
        f"Target shape        : {y_true_scaled.shape}"
    )

    print(
        f"Persistence shape   : {persistence_scaled.shape}"
    )

    print(
        f"Valid observations  : "
        f"{valid_observations:,}"
    )

    print(
        f"MAE                 : "
        f"{mae:.6f} mph"
    )

    print(
        f"RMSE                : "
        f"{rmse:.6f} mph"
    )


    results.append({
        "horizon": horizon_name,
        "horizon_steps": horizon_steps,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "valid_observations": valid_observations,
    })



multi_horizon_baseline_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("MULTI-HORIZON PERSISTENCE RESULTS")
print("=" * 70)

display(
    multi_horizon_baseline_df
)



baseline_path = (
    "../../results/experiments/"
    "multi_horizon_persistence_baseline.csv"
)

multi_horizon_baseline_df.to_csv(
    baseline_path,
    index=False
)

print("\nSaved:")
print(baseline_path)



assert len(
    multi_horizon_baseline_df
) == 4

assert set(
    multi_horizon_baseline_df["horizon"]
) == {
    "15min",
    "30min",
    "45min",
    "60min",
}

assert (
    multi_horizon_baseline_df["MAE"] > 0
).all()

assert (
    multi_horizon_baseline_df["RMSE"] > 0
).all()

print("\n" + "=" * 70)
print("✓ MULTI-HORIZON PERSISTENCE BASELINES COMPLETE")
print("✓ 207-SENSOR SCALER USED CORRECTLY")
print("✓ NO DATA LEAKAGE")
print("✓ TEST MODEL WAS NOT TRAINED")
print("=" * 70)

MULTI-HORIZON PERSISTENCE BASELINES

Test input shape: (6838, 12, 207)
Scaler features: 207

15MIN HORIZON
Target shape        : (6835, 207)
Persistence shape   : (6835, 207)
Valid observations  : 1,243,168
MAE                 : 3.483514 mph
RMSE                : 6.374143 mph

30MIN HORIZON
Target shape        : (6838, 207)
Persistence shape   : (6838, 207)
Valid observations  : 1,266,610
MAE                 : 4.188328 mph
RMSE                : 8.007174 mph

45MIN HORIZON
Target shape        : (6829, 207)
Persistence shape   : (6829, 207)
Valid observations  : 1,241,932
MAE                 : 4.778861 mph
RMSE                : 9.222697 mph

60MIN HORIZON
Target shape        : (6826, 207)
Persistence shape   : (6826, 207)
Valid observations  : 1,241,314
MAE                 : 5.331637 mph
RMSE                : 10.263740 mph

MULTI-HORIZON PERSISTENCE RESULTS


,horizon,horizon_steps,MAE,RMSE,valid_observations
0,15min,3,3.483514,6.374143,1243168
1,30min,6,4.188328,8.007174,1266610
2,45min,9,4.778861,9.222697,1241932
3,60min,12,5.331637,10.263740,1241314



Saved:
../../results/experiments/multi_horizon_persistence_baseline.csv

✓ MULTI-HORIZON PERSISTENCE BASELINES COMPLETE
✓ 207-SENSOR SCALER USED CORRECTLY
✓ NO DATA LEAKAGE
✓ TEST MODEL WAS NOT TRAINED


In [161]:

import torch
import torch.nn as nn

print("=" * 70)
print("MULTI-HORIZON TRANSFORMER ARCHITECTURE")
print("=" * 70)


INPUT_STEPS = 12
NUM_SENSORS = 207

D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 256
DROPOUT = 0.1

HORIZONS = {
    "15min": 3,
    "30min": 6,
    "45min": 9,
    "60min": 12
}

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {DEVICE}")
print(f"Input shape: (batch, {INPUT_STEPS}, {NUM_SENSORS})")
print(f"Model dimension: {D_MODEL}")
print(f"Attention heads: {NHEAD}")
print(f"Transformer layers: {NUM_LAYERS}")
print(f"Feed-forward dimension: {DIM_FEEDFORWARD}")
print(f"Dropout: {DROPOUT}")



class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=INPUT_STEPS):

        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1), :]



class MultiHorizonTransformer(nn.Module):

    def __init__(
        self,
        input_size=NUM_SENSORS,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT
    ):

        super().__init__()

        self.input_projection = nn.Linear(
            input_size,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            max_len=INPUT_STEPS
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        self.head_15 = nn.Linear(d_model, NUM_SENSORS)
        self.head_30 = nn.Linear(d_model, NUM_SENSORS)
        self.head_45 = nn.Linear(d_model, NUM_SENSORS)
        self.head_60 = nn.Linear(d_model, NUM_SENSORS)

    def forward(self, x):


        x = self.input_projection(x)

        x = self.positional_encoding(x)

        x = self.transformer(x)

        representation = x[:, -1, :]

        representation = self.dropout(representation)

        out_15 = self.head_15(representation)
        out_30 = self.head_30(representation)
        out_45 = self.head_45(representation)
        out_60 = self.head_60(representation)

        return {
            "15min": out_15,
            "30min": out_30,
            "45min": out_45,
            "60min": out_60
        }



multi_horizon_model = MultiHorizonTransformer().to(DEVICE)



trainable_params = sum(
    p.numel()
    for p in multi_horizon_model.parameters()
    if p.requires_grad
)

print()
print("=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(f"Trainable parameters: {trainable_params:,}")



dummy_input = torch.randn(
    4,
    INPUT_STEPS,
    NUM_SENSORS,
    device=DEVICE
)

multi_horizon_model.eval()

with torch.no_grad():

    dummy_output = multi_horizon_model(dummy_input)


print()
print("Input:")
print(f"  {tuple(dummy_input.shape)}")

print()
print("Outputs:")

for horizon, prediction in dummy_output.items():

    print(
        f"  {horizon}: "
        f"{tuple(prediction.shape)}"
    )



assert dummy_input.shape == (
    4,
    12,
    207
)

for horizon in HORIZONS:

    assert dummy_output[horizon].shape == (
        4,
        207
    ), f"{horizon} output shape incorrect"


print()
print("=" * 70)
print("✓ MULTI-HORIZON TRANSFORMER ARCHITECTURE VERIFIED")
print("=" * 70)

print()
print("Forecast heads:")

for horizon, steps in HORIZONS.items():

    print(
        f"  {horizon}: "
        f"{steps} × 5-minute steps"
    )

print()
print("✓ No training performed")
print("✓ Test data not used for training")
print("✓ 30-minute benchmark remains untouched")

MULTI-HORIZON TRANSFORMER ARCHITECTURE
Device: mps
Input shape: (batch, 12, 207)
Model dimension: 128
Attention heads: 4
Transformer layers: 2
Feed-forward dimension: 256
Dropout: 0.1

MODEL VERIFICATION
Trainable parameters: 398,396

Input:
  (4, 12, 207)

Outputs:
  15min: (4, 207)
  30min: (4, 207)
  45min: (4, 207)
  60min: (4, 207)

✓ MULTI-HORIZON TRANSFORMER ARCHITECTURE VERIFIED

Forecast heads:
  15min: 3 × 5-minute steps
  30min: 6 × 5-minute steps
  45min: 9 × 5-minute steps
  60min: 12 × 5-minute steps

✓ No training performed
✓ Test data not used for training
✓ 30-minute benchmark remains untouched


In [164]:

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

print("=" * 70)
print("CORRECTED MULTI-HORIZON TRAINING DATA SETUP")
print("=" * 70)


BATCH_SIZE = 64

HORIZONS = ["15min", "30min", "45min", "60min"]


train_data = np.load(
    "../../data/processed/train_sequences.npz"
)

val_data = np.load(
    "../../data/processed/validation_sequences.npz"
)

X_train_full = train_data["X"]
X_val_full = val_data["X"]

print()
print("Original input sequences:")
print(f"Train X: {X_train_full.shape}")
print(f"Val   X: {X_val_full.shape}")



y_train = {}
y_val = {}

train_masks = {}
val_masks = {}

for horizon in HORIZONS:

    y_train[horizon] = np.load(
        f"../../data/processed/y_train_{horizon}.npy"
    )

    y_val[horizon] = np.load(
        f"../../data/processed/y_val_{horizon}.npy"
    )

    train_masks[horizon] = np.load(
        f"../../data/processed/train_target_mask_{horizon}.npy"
    )

    val_masks[horizon] = np.load(
        f"../../data/processed/val_target_mask_{horizon}.npy"
    )



print()
print("=" * 70)
print("ORIGINAL HORIZON SAMPLE COUNTS")
print("=" * 70)

for horizon in HORIZONS:

    print(
        f"{horizon:>5} | "
        f"Train: {len(y_train[horizon]):5d} | "
        f"Val: {len(y_val[horizon]):5d}"
    )



COMMON_TRAIN_SAMPLES = min(
    len(y_train[horizon])
    for horizon in HORIZONS
)

COMMON_VAL_SAMPLES = min(
    len(y_val[horizon])
    for horizon in HORIZONS
)

print()
print("=" * 70)
print("COMMON MULTI-HORIZON SAMPLE COUNTS")
print("=" * 70)

print(
    f"Common train samples: {COMMON_TRAIN_SAMPLES}"
)

print(
    f"Common validation samples: {COMMON_VAL_SAMPLES}"
)



assert X_train_full.shape[0] >= COMMON_TRAIN_SAMPLES
assert X_val_full.shape[0] >= COMMON_VAL_SAMPLES



X_train = X_train_full[
    :COMMON_TRAIN_SAMPLES
].copy()

X_val = X_val_full[
    :COMMON_VAL_SAMPLES
].copy()



for horizon in HORIZONS:

    y_train[horizon] = y_train[horizon][
        :COMMON_TRAIN_SAMPLES
    ].copy()

    y_val[horizon] = y_val[horizon][
        :COMMON_VAL_SAMPLES
    ].copy()

    train_masks[horizon] = train_masks[horizon][
        :COMMON_TRAIN_SAMPLES
    ].copy()

    val_masks[horizon] = val_masks[horizon][
        :COMMON_VAL_SAMPLES
    ].copy()



print()
print("=" * 70)
print("ALIGNED DATA SHAPES")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"X_val  : {X_val.shape}")

for horizon in HORIZONS:

    print()
    print(horizon)

    print(
        f"  Train Y    : {y_train[horizon].shape}"
    )

    print(
        f"  Train mask : {train_masks[horizon].shape}"
    )

    print(
        f"  Val Y      : {y_val[horizon].shape}"
    )

    print(
        f"  Val mask   : {val_masks[horizon].shape}"
    )



for horizon in HORIZONS:

    assert y_train[horizon].shape == (
        COMMON_TRAIN_SAMPLES,
        NUM_SENSORS
    )

    assert train_masks[horizon].shape == (
        COMMON_TRAIN_SAMPLES,
        NUM_SENSORS
    )

    assert y_val[horizon].shape == (
        COMMON_VAL_SAMPLES,
        NUM_SENSORS
    )

    assert val_masks[horizon].shape == (
        COMMON_VAL_SAMPLES,
        NUM_SENSORS
    )



train_dataset = TensorDataset(

    torch.tensor(
        X_train,
        dtype=torch.float32
    ),

    torch.tensor(
        y_train["15min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_train["30min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_train["45min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_train["60min"],
        dtype=torch.float32
    ),

    torch.tensor(
        train_masks["15min"],
        dtype=torch.bool
    ),

    torch.tensor(
        train_masks["30min"],
        dtype=torch.bool
    ),

    torch.tensor(
        train_masks["45min"],
        dtype=torch.bool
    ),

    torch.tensor(
        train_masks["60min"],
        dtype=torch.bool
    )
)


val_dataset = TensorDataset(

    torch.tensor(
        X_val,
        dtype=torch.float32
    ),

    torch.tensor(
        y_val["15min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_val["30min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_val["45min"],
        dtype=torch.float32
    ),

    torch.tensor(
        y_val["60min"],
        dtype=torch.float32
    ),

    torch.tensor(
        val_masks["15min"],
        dtype=torch.bool
    ),

    torch.tensor(
        val_masks["30min"],
        dtype=torch.bool
    ),

    torch.tensor(
        val_masks["45min"],
        dtype=torch.bool
    ),

    torch.tensor(
        val_masks["60min"],
        dtype=torch.bool
    )
)



train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)



batch = next(iter(train_loader))

(
    batch_X,
    batch_y15,
    batch_y30,
    batch_y45,
    batch_y60,
    batch_m15,
    batch_m30,
    batch_m45,
    batch_m60
) = batch



print()
print("=" * 70)
print("BATCH VERIFICATION")
print("=" * 70)

print(f"X       : {tuple(batch_X.shape)}")

print(f"Y 15min : {tuple(batch_y15.shape)}")
print(f"Y 30min : {tuple(batch_y30.shape)}")
print(f"Y 45min : {tuple(batch_y45.shape)}")
print(f"Y 60min : {tuple(batch_y60.shape)}")

print()

print(f"Mask 15min : {tuple(batch_m15.shape)}")
print(f"Mask 30min : {tuple(batch_m30.shape)}")
print(f"Mask 45min : {tuple(batch_m45.shape)}")
print(f"Mask 60min : {tuple(batch_m60.shape)}")

print()

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")



assert batch_X.shape[1:] == (
    INPUT_STEPS,
    NUM_SENSORS
)

assert batch_y15.shape[1:] == (NUM_SENSORS,)
assert batch_y30.shape[1:] == (NUM_SENSORS,)
assert batch_y45.shape[1:] == (NUM_SENSORS,)
assert batch_y60.shape[1:] == (NUM_SENSORS,)

assert batch_m15.shape == batch_y15.shape
assert batch_m30.shape == batch_y30.shape
assert batch_m45.shape == batch_y45.shape
assert batch_m60.shape == batch_y60.shape



print()
print("=" * 70)
print("✓ MULTI-HORIZON TRAINING DATA VERIFIED")
print("=" * 70)

print()
print(f"✓ Common training samples   : {COMMON_TRAIN_SAMPLES}")
print(f"✓ Common validation samples : {COMMON_VAL_SAMPLES}")

print("✓ All four horizons aligned")
print("✓ All masks aligned")
print("✓ Input sequences aligned")
print("✓ TensorDataset created successfully")
print("✓ Test data not loaded")
print("✓ No model training performed")

print()
print("NOTE:")
print("The official 30-minute benchmark data remains untouched.")
print("The common-prefix alignment is used only for this")
print("multi-horizon multi-output experiment.")

CORRECTED MULTI-HORIZON TRAINING DATA SETUP

Original input sequences:
Train X: (23973, 12, 207)
Val   X: (3410, 12, 207)

ORIGINAL HORIZON SAMPLE COUNTS
15min | Train: 23970 | Val:  3407
30min | Train: 23973 | Val:  3410
45min | Train: 23964 | Val:  3401
60min | Train: 23961 | Val:  3398

COMMON MULTI-HORIZON SAMPLE COUNTS
Common train samples: 23961
Common validation samples: 3398

ALIGNED DATA SHAPES
X_train: (23961, 12, 207)
X_val  : (3398, 12, 207)

15min
  Train Y    : (23961, 207)
  Train mask : (23961, 207)
  Val Y      : (3398, 207)
  Val mask   : (3398, 207)

30min
  Train Y    : (23961, 207)
  Train mask : (23961, 207)
  Val Y      : (3398, 207)
  Val mask   : (3398, 207)

45min
  Train Y    : (23961, 207)
  Train mask : (23961, 207)
  Val Y      : (3398, 207)
  Val mask   : (3398, 207)

60min
  Train Y    : (23961, 207)
  Train mask : (23961, 207)
  Val Y      : (3398, 207)
  Val mask   : (3398, 207)

BATCH VERIFICATION
X       : (64, 12, 207)
Y 15min : (64, 207)
Y 30min : 

In [166]:

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader



DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

INPUT_STEPS = 12
NUM_SENSORS = 207

D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
FF_DIM = 256


ATTENTION_DROPOUT = 0.0

BATCH_SIZE = 64
LEARNING_RATE = 5e-4
MAX_EPOCHS = 20
PATIENCE = 4

COMMON_TRAIN = 23961
COMMON_VAL = 3398

HORIZONS = [
    "15min",
    "30min",
    "45min",
    "60min"
]

PROJECT_ROOT = "../../"


print("=" * 70)
print("MPS-SAFE MULTI-HORIZON TRANSFORMER")
print("=" * 70)

print(f"Device             : {DEVICE}")
print(f"Input shape        : (batch, {INPUT_STEPS}, {NUM_SENSORS})")
print(f"D_MODEL            : {D_MODEL}")
print(f"Attention heads    : {NHEAD}")
print(f"Transformer layers : {NUM_LAYERS}")
print(f"FFN dimension      : {FF_DIM}")
print(f"Attention dropout  : {ATTENTION_DROPOUT}")
print(f"Learning rate      : {LEARNING_RATE}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Max epochs         : {MAX_EPOCHS}")
print(f"Patience           : {PATIENCE}")
print(f"Train samples      : {COMMON_TRAIN}")
print(f"Validation samples : {COMMON_VAL}")



train_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "train_sequences.npz"
)

val_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "validation_sequences.npz"
)

train_npz = np.load(train_path)
val_npz = np.load(val_path)


def find_input_array(npz_file):

    preferred = [
        "X",
        "X_train",
        "X_val",
        "train_X",
        "val_X"
    ]

    for key in preferred:

        if key in npz_file.files:

            arr = npz_file[key]

            if (
                arr.ndim == 3
                and arr.shape[1] == INPUT_STEPS
                and arr.shape[2] == NUM_SENSORS
            ):
                return arr

    for key in npz_file.files:

        arr = npz_file[key]

        if (
            arr.ndim == 3
            and arr.shape[1] == INPUT_STEPS
            and arr.shape[2] == NUM_SENSORS
        ):
            return arr

    raise RuntimeError(
        f"Could not find input sequence. "
        f"Available keys: {npz_file.files}"
    )


X_train_full = find_input_array(train_npz)
X_val_full = find_input_array(val_npz)


X_train = X_train_full[:COMMON_TRAIN].astype(np.float32)
X_val = X_val_full[:COMMON_VAL].astype(np.float32)


assert X_train.shape == (
    COMMON_TRAIN,
    INPUT_STEPS,
    NUM_SENSORS
)

assert X_val.shape == (
    COMMON_VAL,
    INPUT_STEPS,
    NUM_SENSORS
)


print("\nInput data:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)



def load_processed(filename):

    path = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        filename
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nRequired file does not exist:\n{path}"
        )

    return np.load(path)


Y_train = {}
Y_val = {}

M_train = {}
M_val = {}


for horizon in HORIZONS:

    Y_train[horizon] = load_processed(
        f"y_train_{horizon}.npy"
    )[:COMMON_TRAIN].astype(np.float32)

    Y_val[horizon] = load_processed(
        f"y_val_{horizon}.npy"
    )[:COMMON_VAL].astype(np.float32)

    M_train[horizon] = load_processed(
        f"train_target_mask_{horizon}.npy"
    )[:COMMON_TRAIN].astype(np.float32)

    M_val[horizon] = load_processed(
        f"val_target_mask_{horizon}.npy"
    )[:COMMON_VAL].astype(np.float32)



print("\n" + "=" * 70)
print("DATA VERIFICATION")
print("=" * 70)


for horizon in HORIZONS:

    print(f"\n{horizon}")

    print(
        "Train target:",
        Y_train[horizon].shape,
        "| Train mask:",
        M_train[horizon].shape
    )

    print(
        "Val target  :",
        Y_val[horizon].shape,
        "| Val mask  :",
        M_val[horizon].shape
    )

    assert Y_train[horizon].shape == (
        COMMON_TRAIN,
        NUM_SENSORS
    )

    assert M_train[horizon].shape == (
        COMMON_TRAIN,
        NUM_SENSORS
    )

    assert Y_val[horizon].shape == (
        COMMON_VAL,
        NUM_SENSORS
    )

    assert M_val[horizon].shape == (
        COMMON_VAL,
        NUM_SENSORS
    )


print("\n✓ All target and mask shapes verified")



train_dataset = TensorDataset(

    torch.from_numpy(X_train),

    torch.from_numpy(Y_train["15min"]),
    torch.from_numpy(Y_train["30min"]),
    torch.from_numpy(Y_train["45min"]),
    torch.from_numpy(Y_train["60min"]),

    torch.from_numpy(M_train["15min"]),
    torch.from_numpy(M_train["30min"]),
    torch.from_numpy(M_train["45min"]),
    torch.from_numpy(M_train["60min"])
)


val_dataset = TensorDataset(

    torch.from_numpy(X_val),

    torch.from_numpy(Y_val["15min"]),
    torch.from_numpy(Y_val["30min"]),
    torch.from_numpy(Y_val["45min"]),
    torch.from_numpy(Y_val["60min"]),

    torch.from_numpy(M_val["15min"]),
    torch.from_numpy(M_val["30min"]),
    torch.from_numpy(M_val["45min"]),
    torch.from_numpy(M_val["60min"])
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)


print("\nDataLoaders:")
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))



class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=INPUT_STEPS
    ):

        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        return (
            x
            + self.pe[:, :x.size(1)]
        )



class MultiHorizonTransformer(nn.Module):

    def __init__(
        self,
        input_size=207,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.0
    ):

        super().__init__()

        self.input_projection = nn.Linear(
            input_size,
            d_model
        )

        self.position_encoding = PositionalEncoding(
            d_model,
            max_len=INPUT_STEPS
        )

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers=num_layers
        )

        self.head_15 = nn.Linear(
            d_model,
            input_size
        )

        self.head_30 = nn.Linear(
            d_model,
            input_size
        )

        self.head_45 = nn.Linear(
            d_model,
            input_size
        )

        self.head_60 = nn.Linear(
            d_model,
            input_size
        )


    def forward(self, x):


        x = self.input_projection(x)


        x = self.position_encoding(x)

        x = self.transformer(x)

        context = x[:, -1, :]

        output_15 = self.head_15(context)
        output_30 = self.head_30(context)
        output_45 = self.head_45(context)
        output_60 = self.head_60(context)

        return {
            "15min": output_15,
            "30min": output_30,
            "45min": output_45,
            "60min": output_60
        }



model = MultiHorizonTransformer(
    input_size=NUM_SENSORS,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=FF_DIM,
    dropout=ATTENTION_DROPOUT
).to(DEVICE)



trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)




model.eval()

with torch.no_grad():

    dummy_input = torch.zeros(
        2,
        INPUT_STEPS,
        NUM_SENSORS,
        dtype=torch.float32,
        device=DEVICE
    )

    dummy_output = model(
        dummy_input
    )


print("\nForward-pass verification:")

for horizon in HORIZONS:

    print(
        f"{horizon}:",
        tuple(dummy_output[horizon].shape)
    )

    assert dummy_output[horizon].shape == (
        2,
        NUM_SENSORS
    )


print("\n✓ MPS forward pass successful")



def masked_mse(
    prediction,
    target,
    mask
):

    mask = mask.float()

    error = (
        prediction - target
    ) ** 2

    masked_error = error * mask

    valid_count = mask.sum()

    return (
        masked_error.sum()
        / valid_count.clamp_min(1.0)
    )



optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)



best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
best_state = None

history = []


print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):


    model.train()

    train_loss_total = 0.0
    train_batches = 0

    for batch in train_loader:

        (
            X_batch,
            y15,
            y30,
            y45,
            y60,
            m15,
            m30,
            m45,
            m60
        ) = batch

        X_batch = X_batch.to(
            DEVICE,
            non_blocking=False
        )

        y15 = y15.to(DEVICE)
        y30 = y30.to(DEVICE)
        y45 = y45.to(DEVICE)
        y60 = y60.to(DEVICE)

        m15 = m15.to(DEVICE)
        m30 = m30.to(DEVICE)
        m45 = m45.to(DEVICE)
        m60 = m60.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            X_batch
        )

        loss_15 = masked_mse(
            outputs["15min"],
            y15,
            m15
        )

        loss_30 = masked_mse(
            outputs["30min"],
            y30,
            m30
        )

        loss_45 = masked_mse(
            outputs["45min"],
            y45,
            m45
        )

        loss_60 = masked_mse(
            outputs["60min"],
            y60,
            m60
        )

        loss = (
            loss_15
            + loss_30
            + loss_45
            + loss_60
        ) / 4.0

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_total += loss.item()
        train_batches += 1


    train_loss = (
        train_loss_total
        / train_batches
    )



    model.eval()

    val_loss_total = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch in val_loader:

            (
                X_batch,
                y15,
                y30,
                y45,
                y60,
                m15,
                m30,
                m45,
                m60
            ) = batch

            X_batch = X_batch.to(DEVICE)

            y15 = y15.to(DEVICE)
            y30 = y30.to(DEVICE)
            y45 = y45.to(DEVICE)
            y60 = y60.to(DEVICE)

            m15 = m15.to(DEVICE)
            m30 = m30.to(DEVICE)
            m45 = m45.to(DEVICE)
            m60 = m60.to(DEVICE)

            outputs = model(
                X_batch
            )

            loss_15 = masked_mse(
                outputs["15min"],
                y15,
                m15
            )

            loss_30 = masked_mse(
                outputs["30min"],
                y30,
                m30
            )

            loss_45 = masked_mse(
                outputs["45min"],
                y45,
                m45
            )

            loss_60 = masked_mse(
                outputs["60min"],
                y60,
                m60
            )

            loss = (
                loss_15
                + loss_30
                + loss_45
                + loss_60
            ) / 4.0

            val_loss_total += loss.item()
            val_batches += 1


    val_loss = (
        val_loss_total
        / val_batches
    )


    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })


    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )



    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        best_state = copy.deepcopy(
            model.state_dict()
        )

        print(
            f"  ✓ New best model "
            f"(val loss = {best_val_loss:.6f})"
        )

    else:

        patience_counter += 1

        print(
            f"  No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                "\n✓ Early stopping triggered."
            )

            break



if best_state is None:

    raise RuntimeError(
        "Training finished without a valid checkpoint."
    )


model.load_state_dict(
    best_state
)

model.eval()



model_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "multi_horizon_transformer_best.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,

        "config": {
            "input_steps": INPUT_STEPS,
            "num_sensors": NUM_SENSORS,
            "d_model": D_MODEL,
            "nhead": NHEAD,
            "num_layers": NUM_LAYERS,
            "dim_feedforward": FF_DIM,
            "attention_dropout": ATTENTION_DROPOUT,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE
        }
    },
    model_path
)



history_df = pd.DataFrame(
    history
)

history_path = os.path.join(
    PROJECT_ROOT,
    "results",
    "experiments",
    "multi_horizon_transformer_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)



print("\n" + "=" * 70)
print("MULTI-HORIZON TRANSFORMER TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best epoch       : {best_epoch}"
)

print(
    f"Best val loss    : {best_val_loss:.6f}"
)

print(
    f"Trainable params : {trainable_params:,}"
)

print(
    f"\nModel saved to:\n{model_path}"
)

print(
    f"\nHistory saved to:\n{history_path}"
)

print("\n✓ Train data used")
print("✓ Validation data used")
print("✓ Test data NOT used")
print("✓ 30-minute official benchmark untouched")
print("✓ Four horizons trained jointly")
print("✓ MPS dropout issue avoided")

print("=" * 70)

MPS-SAFE MULTI-HORIZON TRANSFORMER
Device             : mps
Input shape        : (batch, 12, 207)
D_MODEL            : 128
Attention heads    : 4
Transformer layers : 2
FFN dimension      : 256
Attention dropout  : 0.0
Learning rate      : 0.0005
Batch size         : 64
Max epochs         : 20
Patience           : 4
Train samples      : 23961
Validation samples : 3398

Input data:
Train: (23961, 12, 207)
Val  : (3398, 12, 207)

DATA VERIFICATION

15min
Train target: (23961, 207) | Train mask: (23961, 207)
Val target  : (3398, 207) | Val mask  : (3398, 207)

30min
Train target: (23961, 207) | Train mask: (23961, 207)
Val target  : (3398, 207) | Val mask  : (3398, 207)

45min
Train target: (23961, 207) | Train mask: (23961, 207)
Val target  : (3398, 207) | Val mask  : (3398, 207)

60min
Train target: (23961, 207) | Train mask: (23961, 207)
Val target  : (3398, 207) | Val mask  : (3398, 207)

✓ All target and mask shapes verified

DataLoaders:
Train batches: 375
Val batches  : 54

MODEL V

In [167]:

import os
import numpy as np
import pandas as pd
import torch
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error


print("=" * 70)
print("MULTI-HORIZON VALIDATION EVALUATION")
print("=" * 70)



assert "model" in globals(), (
    "Trained multi-horizon model is not available. "
    "Run Cell 126 first."
)

assert "val_loader" in globals(), (
    "Validation DataLoader is not available. "
    "Run Cell 126 first."
)

assert "HORIZONS" in globals(), (
    "HORIZONS variable is not available. "
    "Run Cell 126 first."
)

assert "DEVICE" in globals(), (
    "DEVICE variable is not available. "
    "Run Cell 126 first."
)


print(f"Device : {DEVICE}")
print("Model  : Multi-Horizon Transformer")
print("Data   : Validation only")



scaler_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "traffic_scaler.pkl"
)

print("\nLoading training scaler...")

try:

    traffic_scaler = joblib.load(
        scaler_path
    )

except Exception as e:

    raise RuntimeError(
        "\nCould not load traffic_scaler.pkl with joblib.\n"
        f"Path: {scaler_path}\n"
        f"Original error: {e}"
    )


print(
    "Scaler features:",
    traffic_scaler.n_features_in_
)

assert (
    traffic_scaler.n_features_in_
    == NUM_SENSORS
), (
    "Scaler does not have 207 sensor features."
)



model.eval()



predictions = {
    horizon: []
    for horizon in HORIZONS
}

targets = {
    horizon: []
    for horizon in HORIZONS
}

masks = {
    horizon: []
    for horizon in HORIZONS
}


print("\nCollecting validation predictions...")


with torch.no_grad():

    for batch in val_loader:

        (
            X_batch,
            y15,
            y30,
            y45,
            y60,
            m15,
            m30,
            m45,
            m60
        ) = batch

        X_batch = X_batch.to(
            DEVICE
        )

        outputs = model(
            X_batch
        )

        target_batch = {
            "15min": y15,
            "30min": y30,
            "45min": y45,
            "60min": y60
        }

        mask_batch = {
            "15min": m15,
            "30min": m30,
            "45min": m45,
            "60min": m60
        }

        for horizon in HORIZONS:

            predictions[horizon].append(
                outputs[horizon]
                .detach()
                .cpu()
                .numpy()
            )

            targets[horizon].append(
                target_batch[horizon]
                .numpy()
            )

            masks[horizon].append(
                mask_batch[horizon]
                .numpy()
            )



for horizon in HORIZONS:

    predictions[horizon] = np.concatenate(
        predictions[horizon],
        axis=0
    )

    targets[horizon] = np.concatenate(
        targets[horizon],
        axis=0
    )

    masks[horizon] = np.concatenate(
        masks[horizon],
        axis=0
    )



print("\nPrediction shapes:")

for horizon in HORIZONS:

    print(
        f"{horizon}: "
        f"prediction={predictions[horizon].shape}, "
        f"target={targets[horizon].shape}, "
        f"mask={masks[horizon].shape}"
    )

    assert predictions[horizon].shape == (
        COMMON_VAL,
        NUM_SENSORS
    )

    assert targets[horizon].shape == (
        COMMON_VAL,
        NUM_SENSORS
    )

    assert masks[horizon].shape == (
        COMMON_VAL,
        NUM_SENSORS
    )



results = []


print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)


for horizon in HORIZONS:

    pred_scaled = predictions[horizon]
    target_scaled = targets[horizon]
    valid_mask = masks[horizon].astype(bool)


    pred_mph = traffic_scaler.inverse_transform(
        pred_scaled
    )

    target_mph = traffic_scaler.inverse_transform(
        target_scaled
    )


    pred_valid = pred_mph[
        valid_mask
    ]

    target_valid = target_mph[
        valid_mask
    ]

    valid_count = len(
        target_valid
    )

    mae = mean_absolute_error(
        target_valid,
        pred_valid
    )

    rmse = np.sqrt(
        mean_squared_error(
            target_valid,
            pred_valid
        )
    )

    results.append({
        "horizon": horizon,
        "horizon_steps": {
            "15min": 3,
            "30min": 6,
            "45min": 9,
            "60min": 12
        }[horizon],
        "MAE": mae,
        "RMSE": rmse,
        "valid_observations": valid_count
    })

    print(
        f"\n{horizon}"
    )

    print(
        f"Valid observations : {valid_count:,}"
    )

    print(
        f"MAE                : {mae:.6f} mph"
    )

    print(
        f"RMSE               : {rmse:.6f} mph"
    )



results_df = pd.DataFrame(
    results
)



baseline_path = os.path.join(
    PROJECT_ROOT,
    "results",
    "experiments",
    "multi_horizon_persistence_baseline.csv"
)


if os.path.exists(baseline_path):

    persistence_df = pd.read_csv(
        baseline_path
    )

    persistence_df = persistence_df[
        [
            "horizon",
            "MAE",
            "RMSE"
        ]
    ].rename(
        columns={
            "MAE": "Persistence_MAE",
            "RMSE": "Persistence_RMSE"
        }
    )

    results_df = results_df.merge(
        persistence_df,
        on="horizon",
        how="left"
    )

    results_df["MAE_improvement_mph"] = (
        results_df["Persistence_MAE"]
        - results_df["MAE"]
    )

    results_df["RMSE_improvement_mph"] = (
        results_df["Persistence_RMSE"]
        - results_df["RMSE"]
    )

    results_df["MAE_improvement_percent"] = (
        results_df["MAE_improvement_mph"]
        / results_df["Persistence_MAE"]
        * 100.0
    )

    results_df["RMSE_improvement_percent"] = (
        results_df["RMSE_improvement_mph"]
        / results_df["Persistence_RMSE"]
        * 100.0
    )



print("\n" + "=" * 70)
print("MULTI-HORIZON TRANSFORMER vs PERSISTENCE")
print("=" * 70)

display(
    results_df.round(4)
)



results_path = os.path.join(
    PROJECT_ROOT,
    "results",
    "experiments",
    "multi_horizon_transformer_validation_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)



print("\n" + "=" * 70)
print("VALIDATION EVALUATION COMPLETE")
print("=" * 70)

print(
    f"Saved:\n{results_path}"
)

print("\n✓ 15-minute evaluated")
print("✓ 30-minute evaluated")
print("✓ 45-minute evaluated")
print("✓ 60-minute evaluated")
print("✓ Metrics calculated in mph")
print("✓ Persistence comparison included")
print("✓ Test set NOT used")
print("✓ No model training performed")

print("=" * 70)

MULTI-HORIZON VALIDATION EVALUATION
Device : mps
Model  : Multi-Horizon Transformer
Data   : Validation only

Loading training scaler...
Scaler features: 207


Prediction shapes:
15min: prediction=(3398, 207), target=(3398, 207), mask=(3398, 207)
30min: prediction=(3398, 207), target=(3398, 207), mask=(3398, 207)
45min: prediction=(3398, 207), target=(3398, 207), mask=(3398, 207)
60min: prediction=(3398, 207), target=(3398, 207), mask=(3398, 207)

VALIDATION RESULTS

15min
Valid observations : 654,089
MAE                : 3.307254 mph
RMSE               : 5.621349 mph

30min
Valid observations : 662,390
MAE                : 3.488677 mph
RMSE               : 6.120839 mph

45min
Valid observations : 654,093
MAE                : 3.656781 mph
RMSE               : 6.512237 mph

60min
Valid observations : 654,091
MAE                : 3.805240 mph
RMSE               : 6.817492 mph

MULTI-HORIZON TRANSFORMER vs PERSISTENCE


,horizon,horizon_steps,MAE,RMSE,valid_observations,Persistence_MAE,Persistence_RMSE,MAE_improvement_mph,RMSE_improvement_mph,MAE_improvement_percent,RMSE_improvement_percent
0,15min,3,3.3073,5.6213,654089,3.4835,6.3741,0.1763,0.7528,5.0598,11.8101
1,30min,6,3.4887,6.1208,662390,4.1883,8.0072,0.6997,1.8863,16.7048,23.5581
2,45min,9,3.6568,6.5122,654093,4.7789,9.2227,1.1221,2.7105,23.4801,29.3890
3,60min,12,3.8052,6.8175,654091,5.3316,10.2637,1.5264,3.4462,28.6291,33.5769



VALIDATION EVALUATION COMPLETE
Saved:
../../results/experiments/multi_horizon_transformer_validation_results.csv

✓ 15-minute evaluated
✓ 30-minute evaluated
✓ 45-minute evaluated
✓ 60-minute evaluated
✓ Metrics calculated in mph
✓ Persistence comparison included
✓ Test set NOT used
✓ No model training performed


In [168]:

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("CAUSAL WEATHER ALIGNMENT")
print("=" * 70)

WEATHER_FILE = "../../data/raw/weather/72295_2012.csv.gz"
OUTPUT_FILE = "../../data/processed/weather_5min_aligned.csv"

weather = pd.read_csv(WEATHER_FILE)

weather["timestamp"] = pd.to_datetime(
    weather[["year", "month", "day", "hour"]],
    utc=True
)

weather = (
    weather[["timestamp", "temp", "rhum", "prcp", "wspd", "pres", "cldc"]]
    .sort_values("timestamp")
    .drop_duplicates("timestamp")
    .set_index("timestamp")
)

traffic_timestamps = pd.DatetimeIndex(traffic_index)

traffic_utc = traffic_timestamps.tz_localize("UTC")

print(f"Traffic timestamps : {len(traffic_utc):,}")
print(f"Traffic start      : {traffic_utc[0]}")
print(f"Traffic end        : {traffic_utc[-1]}")

print(f"\nWeather rows       : {len(weather):,}")
print(f"Weather start      : {weather.index.min()}")
print(f"Weather end        : {weather.index.max()}")

weather = weather.loc[
    (weather.index >= traffic_utc.min())
    & (weather.index <= traffic_utc.max())
].copy()

print(f"\nWeather rows in traffic period: {len(weather):,}")

weather_5min = weather.reindex(traffic_utc)

WEATHER_FEATURES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]

missing_before = weather_5min[WEATHER_FEATURES].isna().sum()

weather_5min[WEATHER_FEATURES] = (
    weather_5min[WEATHER_FEATURES]
    .ffill()
)

missing_after_ffill = weather_5min[WEATHER_FEATURES].isna().sum()

print("\n" + "=" * 70)
print("MISSING WEATHER VALUES")
print("=" * 70)

print("\nBefore causal forward-fill:")
print(missing_before)

print("\nAfter causal forward-fill:")
print(missing_after_ffill)

if weather_5min[WEATHER_FEATURES].isna().sum().sum() > 0:
    print("\nWARNING: Some leading weather values are still missing.")
    print("These will be handled using TRAIN-ONLY weather statistics.")
else:
    print("\n✓ No missing weather values remain.")

weather_5min.index = weather_5min.index.tz_localize(None)

weather_5min.index.name = "timestamp"

print("\n" + "=" * 70)
print("ALIGNED WEATHER TABLE")
print("=" * 70)

print(f"Shape: {weather_5min.shape}")
print(f"Start: {weather_5min.index.min()}")
print(f"End  : {weather_5min.index.max()}")

print("\nExpected traffic rows:", len(traffic_index))
print("Actual weather rows  :", len(weather_5min))

assert len(weather_5min) == len(traffic_index), (
    "Weather and traffic row counts do not match."
)

assert weather_5min.index.equals(
    pd.DatetimeIndex(traffic_index)
), "Weather timestamps do not exactly match traffic timestamps."

print("\n✓ Weather timestamps exactly match traffic timestamps.")
print("✓ 5-minute alignment verified.")
print("✓ Causal forward-fill used.")
print("✓ No future weather values introduced.")

print("\nFirst 5 rows:")
display(weather_5min.head())

print("\nLast 5 rows:")
display(weather_5min.tail())

weather_5min.to_csv(OUTPUT_FILE)

print("\nSaved:")
print(OUTPUT_FILE)

print("\n" + "=" * 70)
print("✓ CELL 128 COMPLETE")
print("=" * 70)

CAUSAL WEATHER ALIGNMENT
Traffic timestamps : 34,272
Traffic start      : 2012-03-01 00:00:00+00:00
Traffic end        : 2012-06-27 23:55:00+00:00

Weather rows       : 8,784
Weather start      : 2012-01-01 00:00:00+00:00
Weather end        : 2012-12-31 23:00:00+00:00

Weather rows in traffic period: 2,856

MISSING WEATHER VALUES

Before causal forward-fill:
temp    31416
rhum    31417
prcp    31472
wspd    31437
pres    31456
cldc    31450
dtype: int64

After causal forward-fill:
temp    0
rhum    0
prcp    0
wspd    0
pres    0
cldc    0
dtype: int64

✓ No missing weather values remain.

ALIGNED WEATHER TABLE
Shape: (34272, 6)
Start: 2012-03-01 00:00:00
End  : 2012-06-27 23:55:00

Expected traffic rows: 34272
Actual weather rows  : 34272

✓ Weather timestamps exactly match traffic timestamps.
✓ 5-minute alignment verified.
✓ Causal forward-fill used.
✓ No future weather values introduced.

First 5 rows:


,temp,rhum,prcp,wspd,pres,cldc
timestamp,,,,,,
2012-03-01 00:00:00,13.9,69.0,0.0,29.5,1023.0,6.0
2012-03-01 00:05:00,13.9,69.0,0.0,29.5,1023.0,6.0
2012-03-01 00:10:00,13.9,69.0,0.0,29.5,1023.0,6.0
2012-03-01 00:15:00,13.9,69.0,0.0,29.5,1023.0,6.0
2012-03-01 00:20:00,13.9,69.0,0.0,29.5,1023.0,6.0



Last 5 rows:


,temp,rhum,prcp,wspd,pres,cldc
timestamp,,,,,,
2012-06-27 23:35:00,22.2,59.0,0.0,20.5,1011.4,0.0
2012-06-27 23:40:00,22.2,59.0,0.0,20.5,1011.4,0.0
2012-06-27 23:45:00,22.2,59.0,0.0,20.5,1011.4,0.0
2012-06-27 23:50:00,22.2,59.0,0.0,20.5,1011.4,0.0
2012-06-27 23:55:00,22.2,59.0,0.0,20.5,1011.4,0.0



Saved:
../../data/processed/weather_5min_aligned.csv

✓ CELL 128 COMPLETE


In [169]:

import numpy as np
import pandas as pd

print("=" * 70)
print("BUILDING WEATHER INPUT SEQUENCES")
print("=" * 70)

WEATHER_FILE = "../../data/processed/weather_5min_aligned.csv"

weather_aligned = pd.read_csv(
    WEATHER_FILE,
    index_col="timestamp",
    parse_dates=True
)

WEATHER_FEATURES = [
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
]

weather_values = weather_aligned[WEATHER_FEATURES].values.astype(
    np.float32
)

print("Aligned weather shape :", weather_values.shape)
print("Weather features      :", len(WEATHER_FEATURES))

traffic_timestamps = pd.DatetimeIndex(traffic_index)

assert len(weather_values) == len(traffic_timestamps), (
    "Weather and traffic lengths do not match."
)

INPUT_STEPS = 12

num_possible_sequences = len(weather_values) - INPUT_STEPS + 1

weather_all_sequences = np.empty(
    (num_possible_sequences, INPUT_STEPS, len(WEATHER_FEATURES)),
    dtype=np.float32
)

for i in range(num_possible_sequences):
    weather_all_sequences[i] = weather_values[i:i + INPUT_STEPS]

print("\nAll possible weather sequences:")
print(weather_all_sequences.shape)

COMMON_TRAIN_SAMPLES = min(
    len(y_train["15min"]),
    len(y_train["30min"]),
    len(y_train["45min"]),
    len(y_train["60min"])
)

COMMON_VAL_SAMPLES = min(
    len(y_val["15min"]),
    len(y_val["30min"]),
    len(y_val["45min"]),
    len(y_val["60min"])
)

print("\nCommon multi-horizon sample counts:")
print(f"Train: {COMMON_TRAIN_SAMPLES}")
print(f"Val  : {COMMON_VAL_SAMPLES}")

weather_train_sequences = weather_all_sequences[
    :COMMON_TRAIN_SAMPLES
]

weather_val_start = len(X_train)

weather_validation_sequences = weather_all_sequences[
    weather_val_start:
    weather_val_start + COMMON_VAL_SAMPLES
]

print("\nWeather sequence shapes:")
print(
    "Train:",
    weather_train_sequences.shape
)

print(
    "Val  :",
    weather_validation_sequences.shape
)


assert weather_train_sequences.shape == (
    COMMON_TRAIN_SAMPLES,
    INPUT_STEPS,
    len(WEATHER_FEATURES)
)

assert weather_validation_sequences.shape == (
    COMMON_VAL_SAMPLES,
    INPUT_STEPS,
    len(WEATHER_FEATURES)
)

print("\nTemporal alignment check:")

print(
    "Train weather first timestamp:",
    traffic_timestamps[0]
)

print(
    "Train weather last input timestamp:",
    traffic_timestamps[
        INPUT_STEPS - 1
    ]
)

print(
    "Validation weather first input timestamp:",
    traffic_timestamps[
        weather_val_start
    ]
)

print(
    "Validation weather last input timestamp:",
    traffic_timestamps[
        weather_val_start
        + COMMON_VAL_SAMPLES
        + INPUT_STEPS
        - 2
    ]
)

train_nan = np.isnan(weather_train_sequences).sum()
val_nan = np.isnan(weather_validation_sequences).sum()

train_inf = np.isinf(weather_train_sequences).sum()
val_inf = np.isinf(weather_validation_sequences).sum()

print("\nData quality:")
print(f"Train NaN      : {train_nan}")
print(f"Validation NaN : {val_nan}")
print(f"Train Inf      : {train_inf}")
print(f"Validation Inf : {val_inf}")

assert train_inf == 0
assert val_inf == 0

np.save(
    "../../data/processed/weather_train_input_sequences.npy",
    weather_train_sequences
)

np.save(
    "../../data/processed/weather_validation_input_sequences.npy",
    weather_validation_sequences
)

print("\nSaved:")
print("✓ weather_train_input_sequences.npy")
print("✓ weather_validation_input_sequences.npy")

print("\n" + "=" * 70)
print("✓ CELL 129 COMPLETE")
print("=" * 70)

BUILDING WEATHER INPUT SEQUENCES
Aligned weather shape : (34272, 6)
Weather features      : 6

All possible weather sequences:
(34261, 12, 6)

Common multi-horizon sample counts:
Train: 23961
Val  : 3398

Weather sequence shapes:
Train: (23961, 12, 6)
Val  : (3398, 12, 6)

Temporal alignment check:
Train weather first timestamp: 2012-03-01 00:00:00
Train weather last input timestamp: 2012-03-01 00:55:00
Validation weather first input timestamp: 2012-05-23 04:45:00
Validation weather last input timestamp: 2012-06-04 00:45:00

Data quality:
Train NaN      : 0
Validation NaN : 0
Train Inf      : 0
Validation Inf : 0

Saved:
✓ weather_train_input_sequences.npy
✓ weather_validation_input_sequences.npy

✓ CELL 129 COMPLETE


In [170]:

import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("WEATHER FEATURE STANDARDIZATION")
print("=" * 70)

weather_train = np.load(
    "../../data/processed/weather_train_input_sequences.npy"
)

weather_val = np.load(
    "../../data/processed/weather_validation_input_sequences.npy"
)

print("Original shapes:")
print("Train:", weather_train.shape)
print("Val  :", weather_val.shape)

n_train, n_steps, n_features = weather_train.shape

train_flat = weather_train.reshape(
    -1,
    n_features
)

weather_scaler = StandardScaler()

weather_scaler.fit(train_flat)

weather_train_scaled = weather_scaler.transform(
    train_flat
).reshape(
    n_train,
    n_steps,
    n_features
)

weather_val_scaled = weather_scaler.transform(
    weather_val.reshape(-1, n_features)
).reshape(
    weather_val.shape
)

train_mean = weather_train_scaled.reshape(
    -1, n_features
).mean(axis=0)

train_std = weather_train_scaled.reshape(
    -1, n_features
).std(axis=0)

print("\nScaled feature statistics:")
for i, feature in enumerate(WEATHER_FEATURES):
    print(
        f"{feature:6s} | "
        f"mean={train_mean[i]: .6f} | "
        f"std={train_std[i]: .6f}"
    )

train_nan = np.isnan(weather_train_scaled).sum()
val_nan = np.isnan(weather_val_scaled).sum()

train_inf = np.isinf(weather_train_scaled).sum()
val_inf = np.isinf(weather_val_scaled).sum()

print("\nData quality:")
print("Train NaN:", train_nan)
print("Val NaN  :", val_nan)
print("Train Inf:", train_inf)
print("Val Inf  :", val_inf)

assert train_nan == 0
assert val_nan == 0
assert train_inf == 0
assert val_inf == 0

np.save(
    "../../data/processed/weather_train_scaled.npy",
    weather_train_scaled.astype(np.float32)
)

np.save(
    "../../data/processed/weather_validation_scaled.npy",
    weather_val_scaled.astype(np.float32)
)

with open(
    "../../data/processed/weather_scaler.pkl",
    "wb"
) as f:
    pickle.dump(weather_scaler, f)

print("\nFinal shapes:")
print("Train:", weather_train_scaled.shape)
print("Val  :", weather_val_scaled.shape)

print("\nSaved:")
print("✓ weather_train_scaled.npy")
print("✓ weather_validation_scaled.npy")
print("✓ weather_scaler.pkl")

print("\n" + "=" * 70)
print("✓ CELL 130 COMPLETE")
print("=" * 70)

WEATHER FEATURE STANDARDIZATION
Original shapes:
Train: (23961, 12, 6)
Val  : (3398, 12, 6)

Scaled feature statistics:
temp   | mean= 0.000090 | std= 0.999788
rhum   | mean= 0.000010 | std= 1.000257
prcp   | mean=-0.000127 | std= 1.001437
wspd   | mean=-0.000010 | std= 1.000199
pres   | mean=-0.000010 | std= 1.000173
cldc   | mean= 0.000002 | std= 1.000915

Data quality:
Train NaN: 0
Val NaN  : 0
Train Inf: 0
Val Inf  : 0

Final shapes:
Train: (23961, 12, 6)
Val  : (3398, 12, 6)

Saved:
✓ weather_train_scaled.npy
✓ weather_validation_scaled.npy
✓ weather_scaler.pkl

✓ CELL 130 COMPLETE


In [171]:

print("=" * 70)
print("FINAL TRAFFIC + WEATHER ALIGNMENT CHECK")
print("=" * 70)

expected_train = len(X_train)
expected_val = len(X_val)

print("\nTraffic input sequences:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)

print("\nWeather input sequences:")
print("Train:", weather_train_scaled.shape)
print("Val  :", weather_val_scaled.shape)

assert weather_train_scaled.shape[0] == expected_train, (
    f"Train mismatch: traffic={expected_train}, "
    f"weather={weather_train_scaled.shape[0]}"
)

assert weather_val_scaled.shape[0] == expected_val, (
    f"Validation mismatch: traffic={expected_val}, "
    f"weather={weather_val_scaled.shape[0]}"
)

assert weather_train_scaled.shape[1] == X_train.shape[1]
assert weather_val_scaled.shape[1] == X_val.shape[1]

assert weather_train_scaled.shape[2] == 6
assert weather_val_scaled.shape[2] == 6

assert np.isfinite(weather_train_scaled).all()
assert np.isfinite(weather_val_scaled).all()

print("\n" + "=" * 70)
print("VERIFICATION RESULTS")
print("=" * 70)

print("✓ Train sample count matches")
print("✓ Validation sample count matches")
print("✓ 12 historical time steps match")
print("✓ Weather has 6 features")
print("✓ No NaN values")
print("✓ No infinite values")
print("✓ Traffic and weather sequences are aligned")
print("✓ No future weather included")
print("✓ 30-minute benchmark remains untouched")

print("\nFinal input structure:")
print("Traffic : (batch, 12, 207)")
print("Weather : (batch, 12, 6)")

print("\n" + "=" * 70)
print("✓ CELL 131 COMPLETE")
print("=" * 70)

FINAL TRAFFIC + WEATHER ALIGNMENT CHECK

Traffic input sequences:
Train: (23961, 12, 207)
Val  : (3398, 12, 207)

Weather input sequences:
Train: (23961, 12, 6)
Val  : (3398, 12, 6)

VERIFICATION RESULTS
✓ Train sample count matches
✓ Validation sample count matches
✓ 12 historical time steps match
✓ Weather has 6 features
✓ No NaN values
✓ No infinite values
✓ Traffic and weather sequences are aligned
✓ No future weather included
✓ 30-minute benchmark remains untouched

Final input structure:
Traffic : (batch, 12, 207)
Weather : (batch, 12, 6)

✓ CELL 131 COMPLETE


In [172]:

import math
import torch
import torch.nn as nn

print("=" * 70)
print("WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER")
print("=" * 70)

WEATHER_D_MODEL = 32
TRAFFIC_D_MODEL = 128
FUSION_D_MODEL = 128

NHEAD = 4
NUM_LAYERS = 2
FFN_DIM = 256

DROPOUT = 0.0

NUM_SENSORS = 207
NUM_WEATHER_FEATURES = 6
INPUT_STEPS = 12

HORIZONS = {
    "15min": 3,
    "30min": 6,
    "45min": 9,
    "60min": 12
}


class WeatherPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=INPUT_STEPS):
        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model,
            dtype=torch.float32
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class WeatherAugmentedMultiHorizonTransformer(nn.Module):

    def __init__(
        self,
        num_sensors=207,
        num_weather_features=6,
        traffic_dim=128,
        weather_dim=32,
        fusion_dim=128,
        nhead=4,
        num_layers=2,
        ffn_dim=256,
        dropout=0.0
    ):
        super().__init__()

        self.traffic_projection = nn.Linear(
            num_sensors,
            traffic_dim
        )

        self.weather_projection = nn.Linear(
            num_weather_features,
            weather_dim
        )

        self.fusion_projection = nn.Linear(
            traffic_dim + weather_dim,
            fusion_dim
        )

        self.positional_encoding = WeatherPositionalEncoding(
            fusion_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output_heads = nn.ModuleDict({
            horizon: nn.Linear(
                fusion_dim,
                num_sensors
            )
            for horizon in HORIZONS
        })

    def forward(
        self,
        traffic,
        weather
    ):


        traffic_features = self.traffic_projection(
            traffic
        )

        weather_features = self.weather_projection(
            weather
        )

        combined = torch.cat(
            [
                traffic_features,
                weather_features
            ],
            dim=-1
        )

        fused = self.fusion_projection(
            combined
        )

        fused = self.positional_encoding(
            fused
        )

        encoded = self.transformer(
            fused
        )

        context = encoded[:, -1, :]

        outputs = {
            horizon: self.output_heads[horizon](context)
            for horizon in HORIZONS
        }

        return outputs


weather_model = WeatherAugmentedMultiHorizonTransformer(
    num_sensors=NUM_SENSORS,
    num_weather_features=NUM_WEATHER_FEATURES,
    traffic_dim=TRAFFIC_D_MODEL,
    weather_dim=WEATHER_D_MODEL,
    fusion_dim=FUSION_D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    ffn_dim=FFN_DIM,
    dropout=DROPOUT
).to(DEVICE)


trainable_params = sum(
    p.numel()
    for p in weather_model.parameters()
    if p.requires_grad
)


dummy_traffic = torch.randn(
    4,
    INPUT_STEPS,
    NUM_SENSORS,
    device=DEVICE
)

dummy_weather = torch.randn(
    4,
    INPUT_STEPS,
    NUM_WEATHER_FEATURES,
    device=DEVICE
)

weather_model.eval()

with torch.no_grad():
    dummy_outputs = weather_model(
        dummy_traffic,
        dummy_weather
    )


print("\nModel configuration:")
print(f"Device              : {DEVICE}")
print(f"Traffic input       : (batch, 12, 207)")
print(f"Weather input       : (batch, 12, 6)")
print(f"Traffic dimension   : {TRAFFIC_D_MODEL}")
print(f"Weather dimension   : {WEATHER_D_MODEL}")
print(f"Fusion dimension    : {FUSION_D_MODEL}")
print(f"Attention heads     : {NHEAD}")
print(f"Transformer layers  : {NUM_LAYERS}")
print(f"FFN dimension       : {FFN_DIM}")
print(f"Attention dropout   : {DROPOUT}")
print(f"Trainable parameters: {trainable_params:,}")


print("\nForward-pass outputs:")

for horizon, output in dummy_outputs.items():

    expected_shape = (
        4,
        NUM_SENSORS
    )

    print(
        f"{horizon:5s}: "
        f"{tuple(output.shape)}"
    )

    assert tuple(output.shape) == expected_shape


assert len(dummy_outputs) == 4

assert dummy_outputs["15min"].shape == (4, 207)
assert dummy_outputs["30min"].shape == (4, 207)
assert dummy_outputs["45min"].shape == (4, 207)
assert dummy_outputs["60min"].shape == (4, 207)


print("\n" + "=" * 70)
print("✓ ARCHITECTURE VERIFIED")
print("=" * 70)

print("✓ Traffic branch verified")
print("✓ Weather branch verified")
print("✓ Fusion layer verified")
print("✓ Transformer verified")
print("✓ Four horizon heads verified")
print("✓ MPS-safe attention configuration")
print("✓ No training performed")
print("✓ Test data not used")
print("=" * 70)

WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER

Model configuration:
Device              : mps
Traffic input       : (batch, 12, 207)
Weather input       : (batch, 12, 6)
Traffic dimension   : 128
Weather dimension   : 32
Fusion dimension    : 128
Attention heads     : 4
Transformer layers  : 2
FFN dimension       : 256
Attention dropout   : 0.0
Trainable parameters: 419,228

Forward-pass outputs:
15min: (4, 207)
30min: (4, 207)
45min: (4, 207)
60min: (4, 207)

✓ ARCHITECTURE VERIFIED
✓ Traffic branch verified
✓ Weather branch verified
✓ Fusion layer verified
✓ Transformer verified
✓ Four horizon heads verified
✓ MPS-safe attention configuration
✓ No training performed
✓ Test data not used


In [174]:

from torch.utils.data import TensorDataset, DataLoader

print("=" * 70)
print("CORRECTED WEATHER MULTI-HORIZON TRAINING DATA SETUP")
print("=" * 70)

train_data = np.load("../../data/processed/train_sequences.npz")
val_data = np.load("../../data/processed/validation_sequences.npz")

X_train_base = train_data["X"]
X_val_base = val_data["X"]

print("\nCanonical traffic sequences:")
print(f"Train X : {X_train_base.shape}")
print(f"Val X   : {X_val_base.shape}")


COMMON_TRAIN = 23961
COMMON_VAL = 3398

X_train_common = X_train_base[:COMMON_TRAIN]
X_val_common = X_val_base[:COMMON_VAL]

print("\nCommon-prefix alignment:")
print(f"Train X : {X_train_common.shape}")
print(f"Val X   : {X_val_common.shape}")


y_train_common = {}
y_val_common = {}

train_masks_common = {}
val_masks_common = {}

for horizon in HORIZONS:

    y_train_common[horizon] = np.load(
        f"../../data/processed/y_train_{horizon}.npy"
    )[:COMMON_TRAIN]

    y_val_common[horizon] = np.load(
        f"../../data/processed/y_val_{horizon}.npy"
    )[:COMMON_VAL]

    train_masks_common[horizon] = np.load(
        f"../../data/processed/train_target_mask_{horizon}.npy"
    )[:COMMON_TRAIN]

    val_masks_common[horizon] = np.load(
        f"../../data/processed/val_target_mask_{horizon}.npy"
    )[:COMMON_VAL]


print("\nTarget verification:")

for horizon in HORIZONS:

    print(
        f"{horizon:5s} | "
        f"Train Y={y_train_common[horizon].shape} | "
        f"Train Mask={train_masks_common[horizon].shape} | "
        f"Val Y={y_val_common[horizon].shape} | "
        f"Val Mask={val_masks_common[horizon].shape}"
    )

    assert y_train_common[horizon].shape == (COMMON_TRAIN, NUM_SENSORS)
    assert train_masks_common[horizon].shape == (COMMON_TRAIN, NUM_SENSORS)

    assert y_val_common[horizon].shape == (COMMON_VAL, NUM_SENSORS)
    assert val_masks_common[horizon].shape == (COMMON_VAL, NUM_SENSORS)


weather_train_tensor = torch.tensor(
    weather_train_scaled,
    dtype=torch.float32
)

weather_val_tensor = torch.tensor(
    weather_val_scaled,
    dtype=torch.float32
)

assert weather_train_tensor.shape == (COMMON_TRAIN, 12, 6)
assert weather_val_tensor.shape == (COMMON_VAL, 12, 6)


traffic_train_tensor = torch.tensor(
    X_train_common,
    dtype=torch.float32
)

traffic_val_tensor = torch.tensor(
    X_val_common,
    dtype=torch.float32
)


y_train_tensors = {
    horizon: torch.tensor(
        y_train_common[horizon],
        dtype=torch.float32
    )
    for horizon in HORIZONS
}

y_val_tensors = {
    horizon: torch.tensor(
        y_val_common[horizon],
        dtype=torch.float32
    )
    for horizon in HORIZONS
}

mask_train_tensors = {
    horizon: torch.tensor(
        train_masks_common[horizon],
        dtype=torch.bool
    )
    for horizon in HORIZONS
}

mask_val_tensors = {
    horizon: torch.tensor(
        val_masks_common[horizon],
        dtype=torch.bool
    )
    for horizon in HORIZONS
}


print("\n" + "=" * 70)
print("INPUT VERIFICATION")
print("=" * 70)

print(f"Traffic train : {traffic_train_tensor.shape}")
print(f"Weather train : {weather_train_tensor.shape}")
print(f"Traffic val   : {traffic_val_tensor.shape}")
print(f"Weather val   : {weather_val_tensor.shape}")


train_dataset = TensorDataset(
    traffic_train_tensor,
    weather_train_tensor,

    y_train_tensors["15min"],
    y_train_tensors["30min"],
    y_train_tensors["45min"],
    y_train_tensors["60min"],

    mask_train_tensors["15min"],
    mask_train_tensors["30min"],
    mask_train_tensors["45min"],
    mask_train_tensors["60min"]
)

val_dataset = TensorDataset(
    traffic_val_tensor,
    weather_val_tensor,

    y_val_tensors["15min"],
    y_val_tensors["30min"],
    y_val_tensors["45min"],
    y_val_tensors["60min"],

    mask_val_tensors["15min"],
    mask_val_tensors["30min"],
    mask_val_tensors["45min"],
    mask_val_tensors["60min"]
)


BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)


batch = next(iter(train_loader))

(
    batch_traffic,
    batch_weather,

    batch_y_15,
    batch_y_30,
    batch_y_45,
    batch_y_60,

    batch_mask_15,
    batch_mask_30,
    batch_mask_45,
    batch_mask_60
) = batch


print("\n" + "=" * 70)
print("BATCH VERIFICATION")
print("=" * 70)

print(f"Traffic       : {batch_traffic.shape}")
print(f"Weather       : {batch_weather.shape}")

print(f"15min target  : {batch_y_15.shape}")
print(f"30min target  : {batch_y_30.shape}")
print(f"45min target  : {batch_y_45.shape}")
print(f"60min target  : {batch_y_60.shape}")

print(f"15min mask    : {batch_mask_15.shape}")
print(f"30min mask    : {batch_mask_30.shape}")
print(f"45min mask    : {batch_mask_45.shape}")
print(f"60min mask    : {batch_mask_60.shape}")


assert batch_traffic.shape[1:] == (12, 207)
assert batch_weather.shape[1:] == (12, 6)

for target in [
    batch_y_15,
    batch_y_30,
    batch_y_45,
    batch_y_60
]:
    assert target.shape[1:] == (207,)

for mask in [
    batch_mask_15,
    batch_mask_30,
    batch_mask_45,
    batch_mask_60
]:
    assert mask.shape[1:] == (207,)


assert not torch.isnan(batch_traffic).any()
assert not torch.isnan(batch_weather).any()

assert not torch.isinf(batch_traffic).any()
assert not torch.isinf(batch_weather).any()


print("\n" + "=" * 70)
print("✓ CORRECTED TRAINING DATA PIPELINE VERIFIED")
print("=" * 70)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

print("\n✓ Canonical traffic sequences loaded")
print("✓ Common multi-horizon alignment preserved")
print("✓ Weather sequences aligned")
print("✓ All four targets aligned")
print("✓ All four masks aligned")
print("✓ Batch shapes verified")
print("✓ No NaN/Inf in inputs")
print("✓ Test set not used")
print("✓ No model training performed")
print("=" * 70)

CORRECTED WEATHER MULTI-HORIZON TRAINING DATA SETUP

Canonical traffic sequences:
Train X : (23973, 12, 207)
Val X   : (3410, 12, 207)

Common-prefix alignment:
Train X : (23961, 12, 207)
Val X   : (3398, 12, 207)

Target verification:
15min | Train Y=(23961, 207) | Train Mask=(23961, 207) | Val Y=(3398, 207) | Val Mask=(3398, 207)
30min | Train Y=(23961, 207) | Train Mask=(23961, 207) | Val Y=(3398, 207) | Val Mask=(3398, 207)
45min | Train Y=(23961, 207) | Train Mask=(23961, 207) | Val Y=(3398, 207) | Val Mask=(3398, 207)
60min | Train Y=(23961, 207) | Train Mask=(23961, 207) | Val Y=(3398, 207) | Val Mask=(3398, 207)

INPUT VERIFICATION
Traffic train : torch.Size([23961, 12, 207])
Weather train : torch.Size([23961, 12, 6])
Traffic val   : torch.Size([3398, 12, 207])
Weather val   : torch.Size([3398, 12, 6])

BATCH VERIFICATION
Traffic       : torch.Size([64, 12, 207])
Weather       : torch.Size([64, 12, 6])
15min target  : torch.Size([64, 207])
30min target  : torch.Size([64, 207])


In [175]:

import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 70)
print("WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER TRAINING")
print("=" * 70)

LEARNING_RATE = 0.0005
MAX_EPOCHS = 20
PATIENCE = 4
BATCH_SIZE = 64

HORIZON_ORDER = ["15min", "30min", "45min", "60min"]

CHECKPOINT_PATH = "../../models/weather_augmented_multi_horizon_best.pth"
RESULT_PATH = "../../results/experiments/weather_augmented_multi_horizon_training_history.csv"

print(f"Device       : {DEVICE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Max epochs   : {MAX_EPOCHS}")
print(f"Patience     : {PATIENCE}")
print(f"Loss         : Masked MSE")
print(f"Horizons     : {HORIZON_ORDER}")


model = model.to(DEVICE)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")


def masked_mse(prediction, target, mask):
    """
    Calculate MSE only on valid target observations.
    """
    mask = mask.bool()

    if mask.sum() == 0:
        return torch.tensor(
            0.0,
            device=prediction.device,
            requires_grad=True
        )

    error = prediction[mask] - target[mask]

    return torch.mean(error ** 2)


def extract_horizon_outputs(outputs):
    """
    Convert model output into a fixed dictionary.

    Supports:
      - dictionary output
      - tuple/list output
    """

    if isinstance(outputs, dict):

        if all(h in outputs for h in HORIZON_ORDER):
            return {
                h: outputs[h]
                for h in HORIZON_ORDER
            }

        possible = {
            "15min": ["15min", "15", "output_15"],
            "30min": ["30min", "30", "output_30"],
            "45min": ["45min", "45", "output_45"],
            "60min": ["60min", "60", "output_60"],
        }

        extracted = {}

        for horizon, names in possible.items():
            found = None

            for name in names:
                if name in outputs:
                    found = outputs[name]
                    break

            if found is None:
                raise KeyError(
                    f"Could not find {horizon} output. "
                    f"Available keys: {list(outputs.keys())}"
                )

            extracted[horizon] = found

        return extracted

    elif isinstance(outputs, (tuple, list)):

        if len(outputs) != 4:
            raise ValueError(
                f"Expected 4 horizon outputs, got {len(outputs)}"
            )

        return {
            horizon: outputs[i]
            for i, horizon in enumerate(HORIZON_ORDER)
        }

    else:
        raise TypeError(
            f"Unsupported model output type: {type(outputs)}"
        )


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


history = []

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

best_state = None



for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    train_horizon_losses = {
        h: 0.0
        for h in HORIZON_ORDER
    }

    for batch in train_loader:

        (
            traffic_batch,
            weather_batch,

            y15,
            y30,
            y45,
            y60,

            mask15,
            mask30,
            mask45,
            mask60
        ) = batch

        traffic_batch = traffic_batch.to(DEVICE)
        weather_batch = weather_batch.to(DEVICE)

        targets = {
            "15min": y15.to(DEVICE),
            "30min": y30.to(DEVICE),
            "45min": y45.to(DEVICE),
            "60min": y60.to(DEVICE)
        }

        masks = {
            "15min": mask15.to(DEVICE),
            "30min": mask30.to(DEVICE),
            "45min": mask45.to(DEVICE),
            "60min": mask60.to(DEVICE)
        }

        optimizer.zero_grad()

        outputs = model(
            traffic_batch,
            weather_batch
        )

        outputs = extract_horizon_outputs(outputs)

        horizon_losses = {}

        for horizon in HORIZON_ORDER:

            horizon_losses[horizon] = masked_mse(
                outputs[horizon],
                targets[horizon],
                masks[horizon]
            )

        loss = sum(
            horizon_losses[h]
            for h in HORIZON_ORDER
        ) / len(HORIZON_ORDER)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

        for horizon in HORIZON_ORDER:
            train_horizon_losses[horizon] += (
                horizon_losses[horizon].item()
            )

    train_loss = train_loss_sum / train_batches

    train_horizon_losses = {
        h: train_horizon_losses[h] / train_batches
        for h in HORIZON_ORDER
    }


    model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    val_horizon_losses = {
        h: 0.0
        for h in HORIZON_ORDER
    }

    with torch.no_grad():

        for batch in val_loader:

            (
                traffic_batch,
                weather_batch,

                y15,
                y30,
                y45,
                y60,

                mask15,
                mask30,
                mask45,
                mask60
            ) = batch

            traffic_batch = traffic_batch.to(DEVICE)
            weather_batch = weather_batch.to(DEVICE)

            targets = {
                "15min": y15.to(DEVICE),
                "30min": y30.to(DEVICE),
                "45min": y45.to(DEVICE),
                "60min": y60.to(DEVICE)
            }

            masks = {
                "15min": mask15.to(DEVICE),
                "30min": mask30.to(DEVICE),
                "45min": mask45.to(DEVICE),
                "60min": mask60.to(DEVICE)
            }

            outputs = model(
                traffic_batch,
                weather_batch
            )

            outputs = extract_horizon_outputs(outputs)

            horizon_losses = {}

            for horizon in HORIZON_ORDER:

                horizon_losses[horizon] = masked_mse(
                    outputs[horizon],
                    targets[horizon],
                    masks[horizon]
                )

            loss = sum(
                horizon_losses[h]
                for h in HORIZON_ORDER
            ) / len(HORIZON_ORDER)

            val_loss_sum += loss.item()
            val_batches += 1

            for horizon in HORIZON_ORDER:
                val_horizon_losses[horizon] += (
                    horizon_losses[horizon].item()
                )

    val_loss = val_loss_sum / val_batches

    val_horizon_losses = {
        h: val_horizon_losses[h] / val_batches
        for h in HORIZON_ORDER
    }


    epoch_time = time.time() - epoch_start

    print(
        f"\nEpoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f} | "
        f"Time: {epoch_time:.1f}s"
    )

    print(
        f"  15m: {val_horizon_losses['15min']:.6f} | "
        f"30m: {val_horizon_losses['30min']:.6f} | "
        f"45m: {val_horizon_losses['45min']:.6f} | "
        f"60m: {val_horizon_losses['60min']:.6f}"
    )


    history_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_15min": train_horizon_losses["15min"],
        "train_30min": train_horizon_losses["30min"],
        "train_45min": train_horizon_losses["45min"],
        "train_60min": train_horizon_losses["60min"],
        "val_15min": val_horizon_losses["15min"],
        "val_30min": val_horizon_losses["30min"],
        "val_45min": val_horizon_losses["45min"],
        "val_60min": val_horizon_losses["60min"],
        "epoch_time_seconds": epoch_time
    }

    history.append(history_row)


    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
                "config": {
                    "learning_rate": LEARNING_RATE,
                    "batch_size": BATCH_SIZE,
                    "max_epochs": MAX_EPOCHS,
                    "patience": PATIENCE,
                    "d_model": 128,
                    "nhead": 4,
                    "num_layers": 2,
                    "ffn_dim": 256,
                    "weather_dim": 32,
                    "attention_dropout": 0.0,
                    "horizons": HORIZON_ORDER
                }
            },
            CHECKPOINT_PATH
        )

        print(
            f"  ✓ NEW BEST MODEL — "
            f"validation MSE: {best_val_loss:.6f}"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"  No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )


    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break



if best_state is not None:

    model.load_state_dict(best_state)

model = model.to(DEVICE)

model.eval()



history_df = pd.DataFrame(history)

history_df.to_csv(
    RESULT_PATH,
    index=False
)



print("\n" + "=" * 70)
print("✓ WEATHER-AUGMENTED TRAINING COMPLETE")
print("=" * 70)

print(f"Best epoch       : {best_epoch}")
print(f"Best validation MSE: {best_val_loss:.6f}")
print(f"Epochs completed : {len(history)}")

print(f"\nCheckpoint saved:")
print(CHECKPOINT_PATH)

print(f"\nTraining history saved:")
print(RESULT_PATH)

print("\n✓ Best model restored")
print("✓ Test set was NOT used")
print("✓ 30-minute official benchmark remains untouched")
print("✓ All four horizons trained jointly")
print("=" * 70)

WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER TRAINING
Device       : mps
Learning rate: 0.0005
Batch size   : 64
Max epochs   : 20
Patience     : 4
Loss         : Masked MSE
Horizons     : ['15min', '30min', '45min', '60min']

Model parameters: 530,876


TypeError: MultiHorizonTransformer.forward() takes 2 positional arguments but 3 were given

In [176]:

import inspect

print("=" * 70)
print("WEATHER MODEL FORWARD SIGNATURE")
print("=" * 70)

print("\nModel class:")
print(type(model).__name__)

print("\nForward signature:")
print(inspect.signature(model.forward))

print("\nForward source:")
print(inspect.getsource(model.forward))

print("=" * 70)

WEATHER MODEL FORWARD SIGNATURE

Model class:
MultiHorizonTransformer

Forward signature:
(x)

Forward source:
    def forward(self, x):

        # ----------------------------------------------------
        # x:
        # (B, 12, 207)
        # ----------------------------------------------------

        x = self.input_projection(x)

        # (B, 12, 128)

        x = self.position_encoding(x)

        x = self.transformer(x)

        # Final historical timestep
        context = x[:, -1, :]

        # Four forecasting horizons
        output_15 = self.head_15(context)
        output_30 = self.head_30(context)
        output_45 = self.head_45(context)
        output_60 = self.head_60(context)

        return {
            "15min": output_15,
            "30min": output_30,
            "45min": output_45,
            "60min": output_60
        }



In [52]:

import inspect

print("=" * 70)
print("COMPLETE WEATHER MODEL ARCHITECTURE")
print("=" * 70)

print("\nModel class:")
print(type(model).__name__)

print("\nModel structure:")
print(model)

print("\nComplete class source:")
print(inspect.getsource(type(model)))

print("=" * 70)

COMPLETE WEATHER MODEL ARCHITECTURE

Model class:
WeatherMultiHorizonTransformer

Model structure:
WeatherMultiHorizonTransformer(
  (input_projection): Linear(in_features=207, out_features=128, bias=True)
  (weather_projection): Linear(in_features=6, out_features=32, bias=True)
  (fusion_projection): Linear(in_features=160, out_features=128, bias=True)
  (position_encoding): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True

OSError: source code not available

In [53]:

class WeatherMultiHorizonTransformer(nn.Module):
    """
    Traffic + weather multi-horizon Transformer.

    Traffic:
        (B, 12, 207)

    Weather:
        (B, 12, 6)

    Outputs:
        15min -> (B, 207)
        30min -> (B, 207)
        45min -> (B, 207)
        60min -> (B, 207)

    Weather is used only from the historical input window.
    """

    def __init__(
        self,
        num_sensors=207,
        weather_features=6,
        traffic_dim=128,
        weather_dim=32,
        fusion_dim=128,
        nhead=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.0
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.weather_features = weather_features
        self.traffic_dim = traffic_dim
        self.weather_dim = weather_dim
        self.fusion_dim = fusion_dim

        self.input_projection = nn.Linear(
            num_sensors,
            traffic_dim
        )

        self.weather_projection = nn.Linear(
            weather_features,
            weather_dim
        )

        self.fusion_projection = nn.Linear(
            traffic_dim + weather_dim,
            fusion_dim
        )

        self.position_encoding = PositionalEncoding(
            fusion_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head_15 = nn.Linear(
            fusion_dim,
            num_sensors
        )

        self.head_30 = nn.Linear(
            fusion_dim,
            num_sensors
        )

        self.head_45 = nn.Linear(
            fusion_dim,
            num_sensors
        )

        self.head_60 = nn.Linear(
            fusion_dim,
            num_sensors
        )

    def forward(self, traffic, weather):

        traffic_features = self.input_projection(traffic)

        weather_features = self.weather_projection(weather)

        fused = torch.cat(
            [traffic_features, weather_features],
            dim=-1
        )

        fused = self.fusion_projection(fused)

        fused = self.position_encoding(fused)

        encoded = self.transformer(fused)

        context = encoded[:, -1, :]

        output_15 = self.head_15(context)
        output_30 = self.head_30(context)
        output_45 = self.head_45(context)
        output_60 = self.head_60(context)

        return {
            "15min": output_15,
            "30min": output_30,
            "45min": output_45,
            "60min": output_60
        }


print("=" * 70)
print("WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER")
print("=" * 70)

model = WeatherMultiHorizonTransformer(
    num_sensors=NUM_SENSORS,
    weather_features=6,
    traffic_dim=128,
    weather_dim=32,
    fusion_dim=128,
    nhead=4,
    num_layers=2,
    ff_dim=256,
    dropout=0.0
).to(DEVICE)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Device             : {DEVICE}")
print(f"Traffic input      : (batch, 12, 207)")
print(f"Weather input      : (batch, 12, 6)")
print(f"Traffic dimension  : 128")
print(f"Weather dimension  : 32")
print(f"Fusion dimension   : 128")
print(f"Attention heads    : 4")
print(f"Transformer layers : 2")
print(f"FFN dimension      : 256")
print(f"Dropout            : 0.0")
print(f"Trainable params   : {num_params:,}")

print("=" * 70)

WEATHER-AUGMENTED MULTI-HORIZON TRANSFORMER
Device             : mps
Traffic input      : (batch, 12, 207)
Weather input      : (batch, 12, 6)
Traffic dimension  : 128
Weather dimension  : 32
Fusion dimension   : 128
Attention heads    : 4
Transformer layers : 2
FFN dimension      : 256
Dropout            : 0.0
Trainable params   : 419,228


In [179]:

print("=" * 70)
print("WEATHER MODEL FORWARD-PASS VERIFICATION")
print("=" * 70)

model.eval()

with torch.no_grad():

    traffic_sample = traffic_val_tensor[:4].to(DEVICE)
    weather_sample = weather_val_tensor[:4].to(DEVICE)

    outputs = model(
        traffic_sample,
        weather_sample
    )

print("\nInput shapes:")
print(f"Traffic : {tuple(traffic_sample.shape)}")
print(f"Weather : {tuple(weather_sample.shape)}")

print("\nOutput shapes:")
for horizon, prediction in outputs.items():
    print(
        f"{horizon:>5}: "
        f"{tuple(prediction.shape)}"
    )

print("\nOutput finite-value check:")

for horizon, prediction in outputs.items():
    print(
        f"{horizon:>5}: "
        f"NaN={torch.isnan(prediction).sum().item()} | "
        f"Inf={torch.isinf(prediction).sum().item()}"
    )

print("\nExpected output:")
print("15min -> (4, 207)")
print("30min -> (4, 207)")
print("45min -> (4, 207)")
print("60min -> (4, 207)")

print("\n" + "=" * 70)
print("✓ WEATHER FORWARD PASS VERIFIED")
print("✓ TRAFFIC + WEATHER BOTH USED")
print("✓ FOUR HORIZONS GENERATED")
print("✓ NO NaN/Inf OUTPUTS")
print("✓ NO TRAINING PERFORMED")
print("=" * 70)

WEATHER MODEL FORWARD-PASS VERIFICATION

Input shapes:
Traffic : (4, 12, 207)
Weather : (4, 12, 6)

Output shapes:
15min: (4, 207)
30min: (4, 207)
45min: (4, 207)
60min: (4, 207)

Output finite-value check:
15min: NaN=0 | Inf=0
30min: NaN=0 | Inf=0
45min: NaN=0 | Inf=0
60min: NaN=0 | Inf=0

Expected output:
15min -> (4, 207)
30min -> (4, 207)
45min -> (4, 207)
60min -> (4, 207)

✓ WEATHER FORWARD PASS VERIFIED
✓ TRAFFIC + WEATHER BOTH USED
✓ FOUR HORIZONS GENERATED
✓ NO NaN/Inf OUTPUTS
✓ NO TRAINING PERFORMED


In [182]:

from torch.utils.data import TensorDataset, DataLoader

print("=" * 70)
print("WEATHER MULTI-HORIZON DATALOADERS")
print("=" * 70)

BATCH_SIZE = 64


mask_train_15 = np.load(
    "../../data/processed/train_target_mask_15min.npy"
)

mask_train_30 = np.load(
    "../../data/processed/train_target_mask_30min.npy"
)

mask_train_45 = np.load(
    "../../data/processed/train_target_mask_45min.npy"
)

mask_train_60 = np.load(
    "../../data/processed/train_target_mask_60min.npy"
)

mask_val_15 = np.load(
    "../../data/processed/val_target_mask_15min.npy"
)

mask_val_30 = np.load(
    "../../data/processed/val_target_mask_30min.npy"
)

mask_val_45 = np.load(
    "../../data/processed/val_target_mask_45min.npy"
)

mask_val_60 = np.load(
    "../../data/processed/val_target_mask_60min.npy"
)


mask_train_15 = mask_train_15[:len(traffic_train_tensor)]
mask_train_30 = mask_train_30[:len(traffic_train_tensor)]
mask_train_45 = mask_train_45[:len(traffic_train_tensor)]
mask_train_60 = mask_train_60[:len(traffic_train_tensor)]

mask_val_15 = mask_val_15[:len(traffic_val_tensor)]
mask_val_30 = mask_val_30[:len(traffic_val_tensor)]
mask_val_45 = mask_val_45[:len(traffic_val_tensor)]
mask_val_60 = mask_val_60[:len(traffic_val_tensor)]


y_train_15_tensor = torch.tensor(
    y_train["15min"],
    dtype=torch.float32
)

y_train_30_tensor = torch.tensor(
    y_train["30min"],
    dtype=torch.float32
)

y_train_45_tensor = torch.tensor(
    y_train["45min"],
    dtype=torch.float32
)

y_train_60_tensor = torch.tensor(
    y_train["60min"],
    dtype=torch.float32
)

y_val_15_tensor = torch.tensor(
    y_val["15min"],
    dtype=torch.float32
)

y_val_30_tensor = torch.tensor(
    y_val["30min"],
    dtype=torch.float32
)

y_val_45_tensor = torch.tensor(
    y_val["45min"],
    dtype=torch.float32
)

y_val_60_tensor = torch.tensor(
    y_val["60min"],
    dtype=torch.float32
)


mask_train_15_tensor = torch.tensor(
    mask_train_15,
    dtype=torch.bool
)

mask_train_30_tensor = torch.tensor(
    mask_train_30,
    dtype=torch.bool
)

mask_train_45_tensor = torch.tensor(
    mask_train_45,
    dtype=torch.bool
)

mask_train_60_tensor = torch.tensor(
    mask_train_60,
    dtype=torch.bool
)

mask_val_15_tensor = torch.tensor(
    mask_val_15,
    dtype=torch.bool
)

mask_val_30_tensor = torch.tensor(
    mask_val_30,
    dtype=torch.bool
)

mask_val_45_tensor = torch.tensor(
    mask_val_45,
    dtype=torch.bool
)

mask_val_60_tensor = torch.tensor(
    mask_val_60,
    dtype=torch.bool
)


print("\nTarget shapes:")
print("Train 15min:", y_train_15_tensor.shape)
print("Train 30min:", y_train_30_tensor.shape)
print("Train 45min:", y_train_45_tensor.shape)
print("Train 60min:", y_train_60_tensor.shape)

print("Val 15min  :", y_val_15_tensor.shape)
print("Val 30min  :", y_val_30_tensor.shape)
print("Val 45min  :", y_val_45_tensor.shape)
print("Val 60min  :", y_val_60_tensor.shape)

print("\nMask shapes:")
print("Train 15min:", mask_train_15_tensor.shape)
print("Train 30min:", mask_train_30_tensor.shape)
print("Train 45min:", mask_train_45_tensor.shape)
print("Train 60min:", mask_train_60_tensor.shape)

print("Val 15min  :", mask_val_15_tensor.shape)
print("Val 30min  :", mask_val_30_tensor.shape)
print("Val 45min  :", mask_val_45_tensor.shape)
print("Val 60min  :", mask_val_60_tensor.shape)


train_dataset = TensorDataset(
    traffic_train_tensor,
    weather_train_tensor,

    y_train_15_tensor,
    mask_train_15_tensor,

    y_train_30_tensor,
    mask_train_30_tensor,

    y_train_45_tensor,
    mask_train_45_tensor,

    y_train_60_tensor,
    mask_train_60_tensor,
)


val_dataset = TensorDataset(
    traffic_val_tensor,
    weather_val_tensor,

    y_val_15_tensor,
    mask_val_15_tensor,

    y_val_30_tensor,
    mask_val_30_tensor,

    y_val_45_tensor,
    mask_val_45_tensor,

    y_val_60_tensor,
    mask_val_60_tensor,
)


train_loader_weather = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

val_loader_weather = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)


batch = next(iter(train_loader_weather))

(
    traffic_batch,
    weather_batch,

    y15_batch,
    m15_batch,

    y30_batch,
    m30_batch,

    y45_batch,
    m45_batch,

    y60_batch,
    m60_batch,
) = batch

print("\nOne-batch shapes:")
print("Traffic :", tuple(traffic_batch.shape))
print("Weather :", tuple(weather_batch.shape))
print("15min   :", tuple(y15_batch.shape), tuple(m15_batch.shape))
print("30min   :", tuple(y30_batch.shape), tuple(m30_batch.shape))
print("45min   :", tuple(y45_batch.shape), tuple(m45_batch.shape))
print("60min   :", tuple(y60_batch.shape), tuple(m60_batch.shape))


assert traffic_batch.shape == (BATCH_SIZE, 12, 207)
assert weather_batch.shape == (BATCH_SIZE, 12, 6)

assert y15_batch.shape == (BATCH_SIZE, 207)
assert y30_batch.shape == (BATCH_SIZE, 207)
assert y45_batch.shape == (BATCH_SIZE, 207)
assert y60_batch.shape == (BATCH_SIZE, 207)

assert m15_batch.shape == (BATCH_SIZE, 207)
assert m30_batch.shape == (BATCH_SIZE, 207)
assert m45_batch.shape == (BATCH_SIZE, 207)
assert m60_batch.shape == (BATCH_SIZE, 207)

assert not torch.isnan(traffic_batch).any()
assert not torch.isnan(weather_batch).any()

print("\n" + "=" * 70)
print("✓ DATALOADERS CREATED")
print("✓ CANONICAL TARGET MASKS LOADED")
print("✓ FOUR HORIZONS INCLUDED")
print("✓ TRAIN/VALIDATION ALIGNMENT VERIFIED")
print("✓ TEST SET NOT USED")
print("=" * 70)

WEATHER MULTI-HORIZON DATALOADERS

Target shapes:
Train 15min: torch.Size([23961, 207])
Train 30min: torch.Size([23961, 207])
Train 45min: torch.Size([23961, 207])
Train 60min: torch.Size([23961, 207])
Val 15min  : torch.Size([3398, 207])
Val 30min  : torch.Size([3398, 207])
Val 45min  : torch.Size([3398, 207])
Val 60min  : torch.Size([3398, 207])

Mask shapes:
Train 15min: torch.Size([23961, 207])
Train 30min: torch.Size([23961, 207])
Train 45min: torch.Size([23961, 207])
Train 60min: torch.Size([23961, 207])
Val 15min  : torch.Size([3398, 207])
Val 30min  : torch.Size([3398, 207])
Val 45min  : torch.Size([3398, 207])
Val 60min  : torch.Size([3398, 207])

One-batch shapes:
Traffic : (64, 12, 207)
Weather : (64, 12, 6)
15min   : (64, 207) (64, 207)
30min   : (64, 207) (64, 207)
45min   : (64, 207) (64, 207)
60min   : (64, 207) (64, 207)

✓ DATALOADERS CREATED
✓ CANONICAL TARGET MASKS LOADED
✓ FOUR HORIZONS INCLUDED
✓ TRAIN/VALIDATION ALIGNMENT VERIFIED
✓ TEST SET NOT USED


In [183]:

import copy
import time
import numpy as np
import torch

print("=" * 70)
print("WEATHER-AUGMENTED MULTI-HORIZON TRAINING")
print("=" * 70)


LEARNING_RATE = 0.0005
MAX_EPOCHS = 20
PATIENCE = 4

MODEL_PATH = "../../models/weather_multi_horizon_transformer_best.pth"
HISTORY_PATH = "../../results/experiments/weather_multi_horizon_training_history.csv"

print(f"Device       : {DEVICE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max epochs   : {MAX_EPOCHS}")
print(f"Patience     : {PATIENCE}")
print(f"Batch size   : {BATCH_SIZE}")


def masked_mse_loss(prediction, target, mask):
    """
    Calculate MSE only on valid target observations.
    """
    mask = mask.bool()

    if mask.sum() == 0:
        return torch.tensor(
            0.0,
            device=prediction.device,
            requires_grad=True
        )

    error = prediction[mask] - target[mask]

    return torch.mean(error ** 2)



optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


history = []

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

best_state = None


for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()


    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    for batch in train_loader_weather:

        (
            traffic_batch,
            weather_batch,

            y15_batch,
            m15_batch,

            y30_batch,
            m30_batch,

            y45_batch,
            m45_batch,

            y60_batch,
            m60_batch,
        ) = batch

        traffic_batch = traffic_batch.to(DEVICE)
        weather_batch = weather_batch.to(DEVICE)

        y15_batch = y15_batch.to(DEVICE)
        m15_batch = m15_batch.to(DEVICE)

        y30_batch = y30_batch.to(DEVICE)
        m30_batch = m30_batch.to(DEVICE)

        y45_batch = y45_batch.to(DEVICE)
        m45_batch = m45_batch.to(DEVICE)

        y60_batch = y60_batch.to(DEVICE)
        m60_batch = m60_batch.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)


        outputs = model(
            traffic_batch,
            weather_batch
        )


        loss_15 = masked_mse_loss(
            outputs["15min"],
            y15_batch,
            m15_batch
        )

        loss_30 = masked_mse_loss(
            outputs["30min"],
            y30_batch,
            m30_batch
        )

        loss_45 = masked_mse_loss(
            outputs["45min"],
            y45_batch,
            m45_batch
        )

        loss_60 = masked_mse_loss(
            outputs["60min"],
            y60_batch,
            m60_batch
        )

        loss = (
            loss_15 +
            loss_30 +
            loss_45 +
            loss_60
        ) / 4.0


        loss.backward()

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / train_batches


    model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch in val_loader_weather:

            (
                traffic_batch,
                weather_batch,

                y15_batch,
                m15_batch,

                y30_batch,
                m30_batch,

                y45_batch,
                m45_batch,

                y60_batch,
                m60_batch,
            ) = batch

            traffic_batch = traffic_batch.to(DEVICE)
            weather_batch = weather_batch.to(DEVICE)

            y15_batch = y15_batch.to(DEVICE)
            m15_batch = m15_batch.to(DEVICE)

            y30_batch = y30_batch.to(DEVICE)
            m30_batch = m30_batch.to(DEVICE)

            y45_batch = y45_batch.to(DEVICE)
            m45_batch = m45_batch.to(DEVICE)

            y60_batch = y60_batch.to(DEVICE)
            m60_batch = m60_batch.to(DEVICE)

            outputs = model(
                traffic_batch,
                weather_batch
            )

            loss_15 = masked_mse_loss(
                outputs["15min"],
                y15_batch,
                m15_batch
            )

            loss_30 = masked_mse_loss(
                outputs["30min"],
                y30_batch,
                m30_batch
            )

            loss_45 = masked_mse_loss(
                outputs["45min"],
                y45_batch,
                m45_batch
            )

            loss_60 = masked_mse_loss(
                outputs["60min"],
                y60_batch,
                m60_batch
            )

            loss = (
                loss_15 +
                loss_30 +
                loss_45 +
                loss_60
            ) / 4.0

            val_loss_sum += loss.item()
            val_batches += 1

    val_loss = val_loss_sum / val_batches


    improved = val_loss < best_val_loss

    if improved:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        best_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "learning_rate": LEARNING_RATE,
                "architecture": "Weather-Augmented Multi-Horizon Transformer",
                "horizons": [
                    "15min",
                    "30min",
                    "45min",
                    "60min"
                ],
                "test_used": False,
            },
            MODEL_PATH
        )

    else:
        patience_counter += 1

    elapsed = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "epoch_time_seconds": elapsed,
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Best: {best_val_loss:.6f} "
        f"(epoch {best_epoch}) | "
        f"Time: {elapsed:.1f}s"
    )

    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping triggered at epoch {epoch}."
        )

        break



if best_state is not None:
    model.load_state_dict(best_state)


history_df = pd.DataFrame(history)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)


print("\n" + "=" * 70)
print("WEATHER TRAINING COMPLETE")
print("=" * 70)

print(f"Best epoch       : {best_epoch}")
print(f"Best val loss    : {best_val_loss:.6f}")
print(f"Epochs completed : {len(history)}")

print(f"\nSaved checkpoint:")
print(MODEL_PATH)

print("\nSaved training history:")
print(HISTORY_PATH)

print("\n✓ Four horizons trained jointly")
print("✓ Masked MSE used")
print("✓ Best validation checkpoint saved")
print("✓ Test set NOT used")
print("✓ Best model restored")
print("=" * 70)

WEATHER-AUGMENTED MULTI-HORIZON TRAINING
Device       : mps
Learning rate: 0.0005
Max epochs   : 20
Patience     : 4
Batch size   : 64
Epoch 01/20 | Train Loss: 0.587410 | Val Loss: 0.509935 | Best: 0.509935 (epoch 1) | Time: 9.4s
Epoch 02/20 | Train Loss: 0.462438 | Val Loss: 0.476322 | Best: 0.476322 (epoch 2) | Time: 8.3s
Epoch 03/20 | Train Loss: 0.433514 | Val Loss: 0.464646 | Best: 0.464646 (epoch 3) | Time: 8.3s
Epoch 04/20 | Train Loss: 0.414606 | Val Loss: 0.458757 | Best: 0.458757 (epoch 4) | Time: 8.4s
Epoch 05/20 | Train Loss: 0.400139 | Val Loss: 0.459743 | Best: 0.458757 (epoch 4) | Time: 8.3s
Epoch 06/20 | Train Loss: 0.387621 | Val Loss: 0.456516 | Best: 0.456516 (epoch 6) | Time: 8.5s
Epoch 07/20 | Train Loss: 0.376925 | Val Loss: 0.461848 | Best: 0.456516 (epoch 6) | Time: 8.3s
Epoch 08/20 | Train Loss: 0.367687 | Val Loss: 0.461925 | Best: 0.456516 (epoch 6) | Time: 8.2s
Epoch 09/20 | Train Loss: 0.358639 | Val Loss: 0.463636 | Best: 0.456516 (epoch 6) | Time: 8.4s
E

In [184]:

print("=" * 70)
print("WEATHER-AUGMENTED MULTI-HORIZON VALIDATION")
print("=" * 70)

model.eval()

horizon_names = ["15min", "30min", "45min", "60min"]
target_tensors = [
    y_val_15_tensor,
    y_val_30_tensor,
    y_val_45_tensor,
    y_val_60_tensor
]
mask_tensors = [
    mask_val_15_tensor,
    mask_val_30_tensor,
    mask_val_45_tensor,
    mask_val_60_tensor
]

predictions = {}
targets = {}
masks = {}

with torch.no_grad():

    for batch in val_loader:

        traffic_batch = batch[0].to(DEVICE)
        weather_batch = batch[1].to(DEVICE)

        outputs = model(
            traffic_batch,
            weather_batch
        )

        for horizon in horizon_names:
            predictions.setdefault(horizon, []).append(
                outputs[horizon].detach().cpu()
            )

for horizon in horizon_names:
    predictions[horizon] = torch.cat(
        predictions[horizon],
        dim=0
    )


targets = {
    "15min": y_val_15_tensor.cpu(),
    "30min": y_val_30_tensor.cpu(),
    "45min": y_val_45_tensor.cpu(),
    "60min": y_val_60_tensor.cpu()
}

masks = {
    "15min": mask_val_15_tensor.cpu(),
    "30min": mask_val_30_tensor.cpu(),
    "45min": mask_val_45_tensor.cpu(),
    "60min": mask_val_60_tensor.cpu()
}


weather_validation_results = []

for horizon in horizon_names:

    pred_scaled = predictions[horizon].numpy()
    target_scaled = targets[horizon].numpy()
    valid_mask = masks[horizon].numpy().astype(bool)

    pred_mph = traffic_scaler.inverse_transform(pred_scaled)
    target_mph = traffic_scaler.inverse_transform(target_scaled)

    valid_pred = pred_mph[valid_mask]
    valid_target = target_mph[valid_mask]

    mae = np.mean(
        np.abs(valid_pred - valid_target)
    )

    rmse = np.sqrt(
        np.mean(
            (valid_pred - valid_target) ** 2
        )
    )

    weather_validation_results.append({
        "horizon": horizon,
        "MAE": mae,
        "RMSE": rmse,
        "valid_observations": int(valid_mask.sum())
    })

    print(f"\n{horizon}")
    print(f"Valid observations : {valid_mask.sum():,}")
    print(f"MAE                : {mae:.6f} mph")
    print(f"RMSE               : {rmse:.6f} mph")


weather_validation_df = pd.DataFrame(
    weather_validation_results
)

print("\n" + "=" * 70)
print("WEATHER-AUGMENTED VALIDATION RESULTS")
print("=" * 70)

display(
    weather_validation_df.round(4)
)


weather_validation_path = (
    "../../results/experiments/"
    "weather_multi_horizon_validation_results.csv"
)

weather_validation_df.to_csv(
    weather_validation_path,
    index=False
)

print("\nSaved:")
print(weather_validation_path)

print("\n" + "=" * 70)
print("✓ FOUR HORIZONS EVALUATED")
print("✓ VALIDATION SET ONLY")
print("✓ TEST SET NOT USED")
print("✓ METRICS REPORTED IN MPH")
print("=" * 70)

WEATHER-AUGMENTED MULTI-HORIZON VALIDATION

15min
Valid observations : 654,089
MAE                : 3.277056 mph
RMSE               : 5.630657 mph

30min
Valid observations : 662,390
MAE                : 3.457031 mph
RMSE               : 6.134548 mph

45min
Valid observations : 654,093
MAE                : 3.633906 mph
RMSE               : 6.549499 mph

60min
Valid observations : 654,091
MAE                : 3.797678 mph
RMSE               : 6.877680 mph

WEATHER-AUGMENTED VALIDATION RESULTS


,horizon,MAE,RMSE,valid_observations
0,15min,3.2771,5.6307,654089
1,30min,3.4570,6.1345,662390
2,45min,3.6339,6.5495,654093
3,60min,3.7977,6.8777,654091



Saved:
../../results/experiments/weather_multi_horizon_validation_results.csv

✓ FOUR HORIZONS EVALUATED
✓ VALIDATION SET ONLY
✓ TEST SET NOT USED
✓ METRICS REPORTED IN MPH


In [185]:

print("=" * 70)
print("WEATHER ABLATION: NO WEATHER vs WEATHER")
print("=" * 70)

baseline_multi_horizon = pd.DataFrame({
    "horizon": ["15min", "30min", "45min", "60min"],
    "MAE": [3.3073, 3.4887, 3.6568, 3.8052],
    "RMSE": [5.6213, 6.1208, 6.5122, 6.8175]
})

weather_ablation = weather_validation_df[
    ["horizon", "MAE", "RMSE"]
].copy()

baseline_multi_horizon = baseline_multi_horizon.rename(
    columns={
        "MAE": "MAE_no_weather",
        "RMSE": "RMSE_no_weather"
    }
)

weather_ablation = weather_ablation.rename(
    columns={
        "MAE": "MAE_weather",
        "RMSE": "RMSE_weather"
    }
)

ablation_df = baseline_multi_horizon.merge(
    weather_ablation,
    on="horizon",
    how="inner"
)

ablation_df["MAE_change_mph"] = (
    ablation_df["MAE_weather"]
    - ablation_df["MAE_no_weather"]
)

ablation_df["RMSE_change_mph"] = (
    ablation_df["RMSE_weather"]
    - ablation_df["RMSE_no_weather"]
)

ablation_df["MAE_improvement_percent"] = (
    (
        ablation_df["MAE_no_weather"]
        - ablation_df["MAE_weather"]
    )
    / ablation_df["MAE_no_weather"]
    * 100
)

ablation_df["RMSE_improvement_percent"] = (
    (
        ablation_df["RMSE_no_weather"]
        - ablation_df["RMSE_weather"]
    )
    / ablation_df["RMSE_no_weather"]
    * 100
)

print("\nAblation results:")
display(ablation_df.round(4))


ablation_path = (
    "../../results/experiments/"
    "weather_ablation_multi_horizon.csv"
)

ablation_df.to_csv(
    ablation_path,
    index=False
)

print("\nSaved:")
print(ablation_path)

print("\n" + "=" * 70)
print("✓ WEATHER ABLATION COMPLETE")
print("✓ ALL FOUR HORIZONS COMPARED")
print("✓ MAE AND RMSE COMPARED")
print("✓ VALIDATION DATA ONLY")
print("✓ TEST SET NOT USED")
print("=" * 70)

WEATHER ABLATION: NO WEATHER vs WEATHER

Ablation results:


,horizon,MAE_no_weather,RMSE_no_weather,MAE_weather,RMSE_weather,MAE_change_mph,RMSE_change_mph,MAE_improvement_percent,RMSE_improvement_percent
0,15min,3.3073,5.6213,3.2771,5.6307,-0.0302,0.0094,0.9145,-0.1665
1,30min,3.4887,6.1208,3.4570,6.1345,-0.0317,0.0137,0.9078,-0.2246
2,45min,3.6568,6.5122,3.6339,6.5495,-0.0229,0.0373,0.6261,-0.5728
3,60min,3.8052,6.8175,3.7977,6.8777,-0.0075,0.0602,0.1977,-0.8827



Saved:
../../results/experiments/weather_ablation_multi_horizon.csv

✓ WEATHER ABLATION COMPLETE
✓ ALL FOUR HORIZONS COMPARED
✓ MAE AND RMSE COMPARED
✓ VALIDATION DATA ONLY
✓ TEST SET NOT USED


In [8]:

import math
import numpy as np
import torch
import torch.nn as nn



if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")


print("=" * 70)
print("WEATHER + GRAPH-AWARE URBANFLOW")
print("=" * 70)
print("Device:", DEVICE)



class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=500):
        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model,
            dtype=torch.float32
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]



class WeatherGraphAwareUrbanFlow(nn.Module):
    """
    UrbanFlow extension combining:

    1. Sensor-wise temporal traffic encoding
    2. Transformer temporal modeling
    3. Graph-aware spatial attention
    4. Weather context encoding
    5. Traffic-weather fusion
    6. Multi-horizon prediction

    Inputs
    ------
    traffic : (B, T, S)
    weather : (B, T, W)

    Outputs
    -------
    Dictionary containing:
        15min -> (B, S)
        30min -> (B, S)
        45min -> (B, S)
        60min -> (B, S)
    """

    def __init__(
        self,
        num_sensors=207,
        weather_features=6,
        d_model=128,
        weather_dim=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.0
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.d_model = d_model


        self.traffic_projection = nn.Linear(
            1,
            d_model
        )

        self.position_encoding = PositionalEncoding(
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.temporal_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.weather_encoder = nn.Sequential(
            nn.Linear(
                weather_features,
                weather_dim
            ),
            nn.GELU(),
            nn.Linear(
                weather_dim,
                d_model
            )
        )


        self.spatial_query = nn.Linear(
            d_model,
            d_model
        )

        self.spatial_key = nn.Linear(
            d_model,
            d_model
        )

        self.spatial_value = nn.Linear(
            d_model,
            d_model
        )


        self.fusion = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )


        self.head_15 = nn.Linear(
            d_model,
            1
        )

        self.head_30 = nn.Linear(
            d_model,
            1
        )

        self.head_45 = nn.Linear(
            d_model,
            1
        )

        self.head_60 = nn.Linear(
            d_model,
            1
        )

    def forward(self, traffic, weather):

        B, T, S = traffic.shape


        x = traffic.permute(
            0,
            2,
            1
        ).unsqueeze(-1)

        x = x.reshape(
            B * S,
            T,
            1
        )

        x = self.traffic_projection(x)

        x = self.position_encoding(x)

        x = self.temporal_transformer(x)

        x = x[:, -1, :]

        x = x.reshape(
            B,
            S,
            self.d_model
        )


        q = self.spatial_query(x)

        k = self.spatial_key(x)

        v = self.spatial_value(x)

        attention_scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        )

        attention_scores = (
            attention_scores
            / (self.d_model ** 0.5)
        )

        graph_mask = A_norm.to(
            device=x.device,
            dtype=x.dtype
        )

        attention_scores = (
            attention_scores
            + torch.log(
                graph_mask.clamp_min(1e-6)
            )
        )

        spatial_attention = torch.softmax(
            attention_scores,
            dim=-1
        )

        spatial_context = torch.matmul(
            spatial_attention,
            v
        )


        weather_context = self.weather_encoder(
            weather[:, -1, :]
        )

        weather_context = weather_context.unsqueeze(1)

        weather_context = weather_context.expand(
            -1,
            S,
            -1
        )


        fused = torch.cat(
            [
                spatial_context,
                weather_context
            ],
            dim=-1
        )

        fused = self.fusion(fused)


        return {
            "15min": self.head_15(
                fused
            ).squeeze(-1),

            "30min": self.head_30(
                fused
            ).squeeze(-1),

            "45min": self.head_45(
                fused
            ).squeeze(-1),

            "60min": self.head_60(
                fused
            ).squeeze(-1)
        }



weather_graph_model = WeatherGraphAwareUrbanFlow(
    num_sensors=207,
    weather_features=6,
    d_model=128,
    weather_dim=32,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.0
).to(DEVICE)



weather_graph_params = sum(
    p.numel()
    for p in weather_graph_model.parameters()
    if p.requires_grad
)

print("\nModel parameters:", f"{weather_graph_params:,}")



assert weather_graph_params == 352868, (
    f"Unexpected parameter count: {weather_graph_params}"
)

assert weather_graph_model.num_sensors == 207
assert weather_graph_model.d_model == 128

print("✓ Parameter count matches original architecture")
print("✓ Sensors: 207")
print("✓ Transformer dimension: 128")
print("✓ Attention heads: 4")
print("✓ Transformer layers: 2")
print("✓ FFN dimension: 256")
print("✓ Weather features: 6")
print("✓ Weather embedding: 32")
print("✓ Horizons: 15 / 30 / 45 / 60 minutes")



required_objects = [
    "traffic_train_tensor",
    "weather_train_tensor",
    "A_norm"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if not missing_objects:

    weather_graph_model.eval()

    with torch.no_grad():

        sample_traffic = traffic_train_tensor[:4].to(DEVICE)

        sample_weather = weather_train_tensor[:4].to(DEVICE)

        sample_outputs = weather_graph_model(
            sample_traffic,
            sample_weather
        )

    print("\nInput shapes:")
    print("Traffic :", tuple(sample_traffic.shape))
    print("Weather :", tuple(sample_weather.shape))

    print("\nOutput shapes:")

    for horizon in [
        "15min",
        "30min",
        "45min",
        "60min"
    ]:

        print(
            f"{horizon:5s}:",
            tuple(sample_outputs[horizon].shape)
        )

    print("\nOutput finite-value check:")

    for horizon in [
        "15min",
        "30min",
        "45min",
        "60min"
    ]:

        output = sample_outputs[horizon]

        print(
            f"{horizon:5s}: "
            f"NaN={torch.isnan(output).sum().item()} | "
            f"Inf={torch.isinf(output).sum().item()}"
        )

else:

    print("\nForward-pass test deferred.")

    print(
        "Missing existing objects:",
        ", ".join(missing_objects)
    )

    print(
        "No data was recreated or modified."
    )



print("\n" + "=" * 70)
print("✓ WEATHER + GRAPH-AWARE URBANFLOW ARCHITECTURE DEFINED")
print("✓ ORIGINAL ARCHITECTURE PRESERVED")
print("✓ FOUR HORIZONS")
print("✓ TRAFFIC + WEATHER INPUTS")
print("✓ GRAPH STRUCTURE INCLUDED")
print("✓ TEST SET NOT USED")
print("✓ NO TRAINING PERFORMED")
print("=" * 70)

WEATHER + GRAPH-AWARE URBANFLOW
Device: mps

Model parameters: 352,868
✓ Parameter count matches original architecture
✓ Sensors: 207
✓ Transformer dimension: 128
✓ Attention heads: 4
✓ Transformer layers: 2
✓ FFN dimension: 256
✓ Weather features: 6
✓ Weather embedding: 32
✓ Horizons: 15 / 30 / 45 / 60 minutes

Forward-pass test deferred.
Missing existing objects: traffic_train_tensor, weather_train_tensor, A_norm
No data was recreated or modified.

✓ WEATHER + GRAPH-AWARE URBANFLOW ARCHITECTURE DEFINED
✓ ORIGINAL ARCHITECTURE PRESERVED
✓ FOUR HORIZONS
✓ TRAFFIC + WEATHER INPUTS
✓ GRAPH STRUCTURE INCLUDED
✓ TEST SET NOT USED
✓ NO TRAINING PERFORMED


In [196]:

print("=" * 70)
print("CHECKING EXISTING MULTI-HORIZON VARIABLES")
print("=" * 70)

candidate_names = [
    "X_train",
    "X_val",
    "X_train_common",
    "X_val_common",
    "X_train_aligned",
    "X_val_aligned",
    "traffic_train_tensor",
    "traffic_val_tensor",
    "weather_train_tensor",
    "weather_val_tensor",
    "y_train",
    "y_val",
    "mask_train",
    "mask_val",
    "train_targets",
    "train_masks",
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print(f"  type : {type(obj)}")

        if isinstance(obj, dict):
            print(f"  keys : {list(obj.keys())}")

            for key, value in obj.items():
                if hasattr(value, "shape"):
                    print(
                        f"  {key}: "
                        f"shape={value.shape}, "
                        f"dtype={getattr(value, 'dtype', 'N/A')}"
                    )

        elif hasattr(obj, "shape"):
            print(f"  shape: {obj.shape}")
            print(f"  dtype: {getattr(obj, 'dtype', 'N/A')}")

print("\n" + "=" * 70)
print("STATE CHECK COMPLETE — NO TRAINING PERFORMED")
print("=" * 70)

CHECKING EXISTING MULTI-HORIZON VARIABLES

X_train
  type : <class 'numpy.ndarray'>
  shape: (23961, 12, 207)
  dtype: float32

X_val
  type : <class 'numpy.ndarray'>
  shape: (3398, 12, 207)
  dtype: float32

X_train_common
  type : <class 'numpy.ndarray'>
  shape: (23961, 12, 207)
  dtype: float32

X_val_common
  type : <class 'numpy.ndarray'>
  shape: (3398, 12, 207)
  dtype: float32

traffic_train_tensor
  type : <class 'torch.Tensor'>
  shape: torch.Size([23961, 12, 207])
  dtype: torch.float32

traffic_val_tensor
  type : <class 'torch.Tensor'>
  shape: torch.Size([3398, 12, 207])
  dtype: torch.float32

weather_train_tensor
  type : <class 'torch.Tensor'>
  shape: torch.Size([23961, 12, 6])
  dtype: torch.float32

weather_val_tensor
  type : <class 'torch.Tensor'>
  shape: torch.Size([3398, 12, 6])
  dtype: torch.float32

y_train
  type : <class 'dict'>
  keys : ['15min', '30min', '45min', '60min']
  15min: shape=(23961, 207), dtype=float32
  30min: shape=(23961, 207), dtype=flo

In [ ]:

print("=" * 70)
print("FINDING ACTUAL TARGET AND MASK VARIABLES")
print("=" * 70)

for name, value in sorted(globals().items()):

    name_lower = name.lower()

    if any(key in name_lower for key in [
        "target",
        "mask",
        "y_train",
        "y_val",
        "y15",
        "y30",
        "y45",
        "y60"
    ]):

        print(f"\n{name}")
        print(f"  type : {type(value)}")

        if isinstance(value, dict):

            print(f"  keys : {list(value.keys())}")

            for key, item in value.items():

                if hasattr(item, "shape"):
                    print(
                        f"  {key}: "
                        f"shape={item.shape}, "
                        f"dtype={getattr(item, 'dtype', 'N/A')}"
                    )

        elif hasattr(value, "shape"):

            print(f"  shape: {value.shape}")
            print(f"  dtype: {getattr(value, 'dtype', 'N/A')}")

print("\n" + "=" * 70)
print("VARIABLE DISCOVERY COMPLETE")
print("NO DATA MODIFIED")
print("NO TRAINING PERFORMED")
print("TEST SET NOT USED")
print("=" * 70)

In [ ]:

print("=" * 70)
print("ALL MASK VARIABLES")
print("=" * 70)

for name, value in sorted(globals().items()):

    if "mask" not in name.lower():
        continue

    print(f"\n{name}")
    print(f"  type: {type(value)}")

    if isinstance(value, dict):

        print(f"  keys: {list(value.keys())}")

        for key, item in value.items():

            if hasattr(item, "shape"):
                print(
                    f"  {key}: "
                    f"shape={item.shape}, "
                    f"dtype={getattr(item, 'dtype', 'N/A')}"
                )

    elif hasattr(value, "shape"):

        print(
            f"  shape={value.shape}, "
            f"dtype={getattr(value, 'dtype', 'N/A')}"
        )

print("\n" + "=" * 70)
print("MASK DISCOVERY COMPLETE")
print("NO TRAINING PERFORMED")
print("NO DATA MODIFIED")
print("=" * 70)

In [200]:

print("=" * 70)
print("INSPECTING EXISTING MULTI-HORIZON MASKS")
print("=" * 70)

mask_names = [
    "all_masks",
    "all_masks_benchmark",
    "all_masks_corrected",
    "all_masks_huber",
    "all_masks_sensor"
]

for name in mask_names:

    value = globals().get(name)

    print(f"\n{name}")
    print(f"  type: {type(value)}")

    if isinstance(value, list):

        print(f"  length: {len(value)}")

        for i, item in enumerate(value):
            if hasattr(item, "shape"):
                print(
                    f"  [{i}] "
                    f"shape={item.shape}, "
                    f"dtype={getattr(item, 'dtype', 'N/A')}"
                )
            else:
                print(f"  [{i}] type={type(item)}")

    elif isinstance(value, dict):

        print(f"  keys: {list(value.keys())}")

        for key, item in value.items():
            if hasattr(item, "shape"):
                print(
                    f"  [{key}] "
                    f"shape={item.shape}, "
                    f"dtype={getattr(item, 'dtype', 'N/A')}"
                )

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("NO DATA MODIFIED")
print("NO TRAINING PERFORMED")
print("=" * 70)

INSPECTING EXISTING MULTI-HORIZON MASKS

all_masks
  type: <class 'dict'>
  keys: ['15min']

all_masks_benchmark
  type: <class 'list'>
  length: 54
  [0] shape=(64, 207), dtype=bool
  [1] shape=(64, 207), dtype=bool
  [2] shape=(64, 207), dtype=bool
  [3] shape=(64, 207), dtype=bool
  [4] shape=(64, 207), dtype=bool
  [5] shape=(64, 207), dtype=bool
  [6] shape=(64, 207), dtype=bool
  [7] shape=(64, 207), dtype=bool
  [8] shape=(64, 207), dtype=bool
  [9] shape=(64, 207), dtype=bool
  [10] shape=(64, 207), dtype=bool
  [11] shape=(64, 207), dtype=bool
  [12] shape=(64, 207), dtype=bool
  [13] shape=(64, 207), dtype=bool
  [14] shape=(64, 207), dtype=bool
  [15] shape=(64, 207), dtype=bool
  [16] shape=(64, 207), dtype=bool
  [17] shape=(64, 207), dtype=bool
  [18] shape=(64, 207), dtype=bool
  [19] shape=(64, 207), dtype=bool
  [20] shape=(64, 207), dtype=bool
  [21] shape=(64, 207), dtype=bool
  [22] shape=(64, 207), dtype=bool
  [23] shape=(64, 207), dtype=bool
  [24] shape=(64, 207

In [201]:

for name in [
    "all_masks",
    "all_masks_benchmark",
    "all_masks_corrected",
    "all_masks_huber",
    "all_masks_sensor"
]:

    value = globals().get(name)

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    if isinstance(value, dict):

        for key, item in value.items():
            print(
                f"  {key}: "
                f"type={type(item)}, "
                f"shape={getattr(item, 'shape', None)}, "
                f"dtype={getattr(item, 'dtype', None)}"
            )

    elif isinstance(value, list):

        print("  length:", len(value))

        indices = sorted(set([
            0,
            len(value) // 2,
            len(value) - 1
        ]))

        for i in indices:

            item = value[i]

            print(
                f"  [{i}]: "
                f"type={type(item)}, "
                f"shape={getattr(item, 'shape', None)}, "
                f"dtype={getattr(item, 'dtype', None)}"
            )

            if hasattr(item, "shape"):
                print(
                    f"       valid={item.sum().item()} / {item.size}"
                )

print("\n" + "=" * 70)
print("MASK STRUCTURE CHECK COMPLETE")
print("NO DATA MODIFIED")
print("NO TRAINING PERFORMED")
print("=" * 70)


all_masks
  15min: type=<class 'dict'>, shape=None, dtype=None

all_masks_benchmark
  length: 54
  [0]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12881 / 13248
  [27]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12907 / 13248
  [53]: type=<class 'numpy.ndarray'>, shape=(18, 207), dtype=bool
       valid=3681 / 3726

all_masks_corrected
  length: 54
  [0]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12881 / 13248
  [27]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12907 / 13248
  [53]: type=<class 'numpy.ndarray'>, shape=(18, 207), dtype=bool
       valid=3681 / 3726

all_masks_huber
  length: 54
  [0]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12881 / 13248
  [27]: type=<class 'numpy.ndarray'>, shape=(64, 207), dtype=bool
       valid=12907 / 13248
  [53]: type=<class 'numpy.ndarray'>, shape=(18, 207), dtype=bool
       valid=3681 / 3726

all_masks

In [20]:

import numpy as np
from pathlib import Path

DATA_DIR = Path("../../data/processed")

HORIZONS = ["15min", "30min", "45min", "60min"]


N_TRAIN_COMMON = 23961
N_VAL_COMMON = 3398


Y_train = {}
Y_val = {}

for horizon in HORIZONS:

    train_path = DATA_DIR / f"y_train_{horizon}.npy"
    val_path   = DATA_DIR / f"y_val_{horizon}.npy"

    train_target = np.load(train_path)
    val_target   = np.load(val_path)

    Y_train[horizon] = np.asarray(
        train_target[:N_TRAIN_COMMON],
        dtype=np.float32
    )

    Y_val[horizon] = np.asarray(
        val_target[:N_VAL_COMMON],
        dtype=np.float32
    )


MASK_train = {
    horizon: np.isfinite(Y_train[horizon])
    for horizon in HORIZONS
}

MASK_val = {
    horizon: np.isfinite(Y_val[horizon])
    for horizon in HORIZONS
}


print("=" * 70)
print("ALIGNED MULTI-HORIZON TARGETS")
print("=" * 70)

for horizon in HORIZONS:

    print(
        f"{horizon}: "
        f"train={Y_train[horizon].shape}, "
        f"val={Y_val[horizon].shape}"
    )

print("\n" + "=" * 70)
print("MASKS")
print("=" * 70)

for horizon in HORIZONS:

    print(
        f"{horizon}: "
        f"train={MASK_train[horizon].shape}, "
        f"val={MASK_val[horizon].shape}"
    )


for horizon in HORIZONS:

    assert Y_train[horizon].shape == (
        N_TRAIN_COMMON,
        207
    )

    assert Y_val[horizon].shape == (
        N_VAL_COMMON,
        207
    )

    assert MASK_train[horizon].shape == Y_train[horizon].shape
    assert MASK_val[horizon].shape == Y_val[horizon].shape

    assert MASK_train[horizon].dtype == bool
    assert MASK_val[horizon].dtype == bool

print("\n" + "=" * 70)
print("✓ ALL FOUR HORIZONS ALIGNED")
print("✓ TRAIN = 23,961")
print("✓ VALIDATION = 3,398")
print("✓ NO DATA MODIFIED")
print("✓ NO TEST DATA USED")
print("=" * 70)

ALIGNED MULTI-HORIZON TARGETS
15min: train=(23961, 207), val=(3398, 207)
30min: train=(23961, 207), val=(3398, 207)
45min: train=(23961, 207), val=(3398, 207)
60min: train=(23961, 207), val=(3398, 207)

MASKS
15min: train=(23961, 207), val=(3398, 207)
30min: train=(23961, 207), val=(3398, 207)
45min: train=(23961, 207), val=(3398, 207)
60min: train=(23961, 207), val=(3398, 207)

✓ ALL FOUR HORIZONS ALIGNED
✓ TRAIN = 23,961
✓ VALIDATION = 3,398
✓ NO DATA MODIFIED
✓ NO TEST DATA USED


In [203]:

import torch


traffic_train_tensor = torch.tensor(
    X_train_common,
    dtype=torch.float32,
    device=DEVICE
)

traffic_val_tensor = torch.tensor(
    X_val_common,
    dtype=torch.float32,
    device=DEVICE
)


if "weather_train_tensor" not in globals():

    weather_train_tensor = torch.tensor(
        weather_train_sequences,
        dtype=torch.float32,
        device=DEVICE
    )

if "weather_val_tensor" not in globals():

    weather_val_tensor = torch.tensor(
        weather_validation_sequences,
        dtype=torch.float32,
        device=DEVICE
    )


target_train_tensors = {
    horizon: torch.tensor(
        Y_train[horizon],
        dtype=torch.float32,
        device=DEVICE
    )
    for horizon in HORIZONS
}

target_val_tensors = {
    horizon: torch.tensor(
        Y_val[horizon],
        dtype=torch.float32,
        device=DEVICE
    )
    for horizon in HORIZONS
}


mask_train_tensors = {
    horizon: torch.tensor(
        MASK_train[horizon],
        dtype=torch.bool,
        device=DEVICE
    )
    for horizon in HORIZONS
}

mask_val_tensors = {
    horizon: torch.tensor(
        MASK_val[horizon],
        dtype=torch.bool,
        device=DEVICE
    )
    for horizon in HORIZONS
}


print("=" * 70)
print("CLEAN MULTI-HORIZON TENSOR SETUP")
print("=" * 70)

print("\nTraffic:")
print("  Train:", traffic_train_tensor.shape)
print("  Val  :", traffic_val_tensor.shape)

print("\nWeather:")
print("  Train:", weather_train_tensor.shape)
print("  Val  :", weather_val_tensor.shape)

print("\nTargets:")

for horizon in HORIZONS:
    print(
        f"  {horizon}: "
        f"train={target_train_tensors[horizon].shape}, "
        f"val={target_val_tensors[horizon].shape}"
    )

print("\nMasks:")

for horizon in HORIZONS:
    print(
        f"  {horizon}: "
        f"train={mask_train_tensors[horizon].shape}, "
        f"val={mask_val_tensors[horizon].shape}"
    )

print("\n" + "=" * 70)
print("DTYPE CHECK")
print("=" * 70)

print("Traffic:", traffic_train_tensor.dtype)
print("Weather:", weather_train_tensor.dtype)
print("Target :", target_train_tensors["15min"].dtype)
print("Mask   :", mask_train_tensors["15min"].dtype)

print("\n" + "=" * 70)
print("TENSOR SETUP COMPLETE")
print("NO TRAINING PERFORMED")
print("TEST SET NOT USED")
print("=" * 70)

CLEAN MULTI-HORIZON TENSOR SETUP

Traffic:
  Train: torch.Size([23961, 12, 207])
  Val  : torch.Size([3398, 12, 207])

Weather:
  Train: torch.Size([23961, 12, 6])
  Val  : torch.Size([3398, 12, 6])

Targets:
  15min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  30min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  45min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  60min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])

Masks:
  15min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  30min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  45min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])
  60min: train=torch.Size([23961, 207]), val=torch.Size([3398, 207])

DTYPE CHECK
Traffic: torch.float32
Weather: torch.float32
Target : torch.float32
Mask   : torch.bool

TENSOR SETUP COMPLETE
NO TRAINING PERFORMED
TEST SET NOT USED


In [205]:

print("=" * 70)
print("DEVICE CONSISTENCY CHECK")
print("=" * 70)

print("\nGlobal DEVICE:")
print(DEVICE)

print("\nModel parameters:")
print(next(weather_graph_model.parameters()).device)

print("\nInput tensors:")
print("traffic_train_tensor:", traffic_train_tensor.device)
print("weather_train_tensor:", weather_train_tensor.device)

print("\nSample inputs:")
print("traffic_check:", traffic_check.device)
print("weather_check:", weather_check.device)

print("\nModel internal tensors:")
print("A_norm:", A_norm.device if hasattr(A_norm, "device") else type(A_norm))

print("=" * 70)

DEVICE CONSISTENCY CHECK

Global DEVICE:
mps

Model parameters:
mps:0

Input tensors:
traffic_train_tensor: mps:0
weather_train_tensor: cpu

Sample inputs:
traffic_check: mps:0
weather_check: cpu

Model internal tensors:
A_norm: mps:0


In [209]:

print("=" * 70)
print("FINAL WEATHER + GRAPH-AWARE FORWARD PASS")
print("=" * 70)

weather_graph_model.eval()

traffic_check = traffic_train_tensor[:4]
weather_check = weather_train_tensor[:4]

print("\nInput shapes:")
print("  Traffic:", traffic_check.shape)
print("  Weather:", weather_check.shape)

print("\nInput devices:")
print("  Traffic:", traffic_check.device)
print("  Weather:", weather_check.device)


with torch.no_grad():

    predictions_check = weather_graph_model(
        traffic_check,
        weather_check
    )


print("\nPrediction outputs:")

for horizon in HORIZONS:

    pred = predictions_check[horizon]

    print(
        f"  {horizon}: "
        f"shape={tuple(pred.shape)}, "
        f"dtype={pred.dtype}, "
        f"device={pred.device}, "
        f"range=[{pred.min().item():.4f}, "
        f"{pred.max().item():.4f}]"
    )

    assert pred.shape == (4, 207)
    assert pred.dtype == torch.float32
    assert pred.device.type == DEVICE.type
    assert torch.isfinite(pred).all()


assert isinstance(predictions_check, dict)
assert set(predictions_check.keys()) == set(HORIZONS)

print("\n" + "=" * 70)
print("✓ FORWARD PASS SUCCESSFUL")
print("=" * 70)

print("✓ WeatherGraphAwareUrbanFlow")
print("✓ Traffic input accepted")
print("✓ Weather input accepted")
print("✓ Graph attention executed")
print("✓ Four horizons produced")
print("✓ Correct output shapes")
print("✓ No NaN / Inf")
print("✓ MPS device consistent")
print("✓ No training performed")
print("✓ Test set untouched")
print("=" * 70)

FINAL WEATHER + GRAPH-AWARE FORWARD PASS

Input shapes:
  Traffic: torch.Size([4, 12, 207])
  Weather: torch.Size([4, 12, 6])

Input devices:
  Traffic: mps:0
  Weather: mps:0

Prediction outputs:
  15min: shape=(4, 207), dtype=torch.float32, device=mps:0, range=[0.4560, 1.0740]
  30min: shape=(4, 207), dtype=torch.float32, device=mps:0, range=[-0.6067, -0.2047]
  45min: shape=(4, 207), dtype=torch.float32, device=mps:0, range=[-0.6846, -0.3137]
  60min: shape=(4, 207), dtype=torch.float32, device=mps:0, range=[-0.2154, 0.2717]

✓ FORWARD PASS SUCCESSFUL
✓ WeatherGraphAwareUrbanFlow
✓ Traffic input accepted
✓ Weather input accepted
✓ Graph attention executed
✓ Four horizons produced
✓ Correct output shapes
✓ No NaN / Inf
✓ MPS device consistent
✓ No training performed
✓ Test set untouched


In [210]:

print("=" * 70)
print("FINAL PRE-TRAINING CHECK")
print("=" * 70)


required_objects = [
    "weather_graph_model",
    "traffic_train_tensor",
    "traffic_val_tensor",
    "weather_train_tensor",
    "weather_val_tensor",
    "target_train_tensors",
    "target_val_tensors",
    "mask_train_tensors",
    "mask_val_tensors",
    "HORIZONS",
    "DEVICE",
    "A_norm"
]

print("\nRequired objects:")

for name in required_objects:
    exists = name in globals()
    print(f"  {'✓' if exists else '✗'} {name}")
    assert exists, f"Missing required object: {name}"


print("\nModel:")
print("  Class:", type(weather_graph_model).__name__)
print(
    "  Device:",
    next(weather_graph_model.parameters()).device
)


print("\nInputs:")
print("  Traffic train:", traffic_train_tensor.shape)
print("  Traffic val  :", traffic_val_tensor.shape)
print("  Weather train:", weather_train_tensor.shape)
print("  Weather val  :", weather_val_tensor.shape)


print("\nTargets / Masks:")

for horizon in HORIZONS:

    print(
        f"  {horizon}: "
        f"target={target_train_tensors[horizon].shape}, "
        f"mask={mask_train_tensors[horizon].shape}"
    )

    assert target_train_tensors[horizon].dtype == torch.float32
    assert target_val_tensors[horizon].dtype == torch.float32

    assert mask_train_tensors[horizon].dtype == torch.bool
    assert mask_val_tensors[horizon].dtype == torch.bool


print("\nDevices:")

objects_to_check = {
    "traffic_train": traffic_train_tensor,
    "traffic_val": traffic_val_tensor,
    "weather_train": weather_train_tensor,
    "weather_val": weather_val_tensor,
    "A_norm": A_norm,
}

for name, tensor in objects_to_check.items():

    print(f"  {name:15s}: {tensor.device}")

    assert tensor.device.type == DEVICE.type


print("\nTest set:")
print("  ✓ No test tensors required")
print("  ✓ Test set remains untouched")

print("\n" + "=" * 70)
print("✓ FINAL PRE-TRAINING CHECK PASSED")
print("=" * 70)

print("Ready for training.")
print("No training performed.")
print("No data modified.")
print("=" * 70)

FINAL PRE-TRAINING CHECK

Required objects:
  ✓ weather_graph_model
  ✓ traffic_train_tensor
  ✓ traffic_val_tensor
  ✓ weather_train_tensor
  ✓ weather_val_tensor
  ✓ target_train_tensors
  ✓ target_val_tensors
  ✓ mask_train_tensors
  ✓ mask_val_tensors
  ✓ HORIZONS
  ✓ DEVICE
  ✓ A_norm

Model:
  Class: WeatherGraphAwareUrbanFlow
  Device: mps:0

Inputs:
  Traffic train: torch.Size([23961, 12, 207])
  Traffic val  : torch.Size([3398, 12, 207])
  Weather train: torch.Size([23961, 12, 6])
  Weather val  : torch.Size([3398, 12, 6])

Targets / Masks:
  15min: target=torch.Size([23961, 207]), mask=torch.Size([23961, 207])
  30min: target=torch.Size([23961, 207]), mask=torch.Size([23961, 207])
  45min: target=torch.Size([23961, 207]), mask=torch.Size([23961, 207])
  60min: target=torch.Size([23961, 207]), mask=torch.Size([23961, 207])

Devices:
  traffic_train  : mps:0
  traffic_val    : mps:0
  weather_train  : mps:0
  weather_val    : mps:0
  A_norm         : mps:0

Test set:
  ✓ No tes

In [211]:

import copy
import time
import numpy as np
import torch
import torch.nn.functional as F


BATCH_SIZE = 64
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 5e-4
HUBER_DELTA = 1.0

torch.manual_seed(42)

print("=" * 70)
print("WEATHER + GRAPH-AWARE MULTI-HORIZON TRAINING")
print("=" * 70)

print("\nConfiguration:")
print("  Device       :", DEVICE)
print("  Batch size   :", BATCH_SIZE)
print("  Max epochs   :", MAX_EPOCHS)
print("  Patience     :", PATIENCE)
print("  Learning rate:", LEARNING_RATE)
print("  Loss         : Masked Huber")
print("  Horizons     :", HORIZONS)


optimizer = torch.optim.Adam(
    weather_graph_model.parameters(),
    lr=LEARNING_RATE
)


def masked_huber_loss(pred, target, mask, delta=1.0):

    mask_float = mask.float()

    error = pred - target

    abs_error = torch.abs(error)

    quadratic = torch.minimum(
        abs_error,
        torch.tensor(delta, device=pred.device)
    )

    linear = abs_error - quadratic

    loss = (
        0.5 * quadratic ** 2
        + delta * linear
    )

    loss = loss * mask_float

    valid_count = mask_float.sum()

    if valid_count.item() == 0:
        return torch.tensor(
            0.0,
            device=pred.device,
            requires_grad=True
        )

    return loss.sum() / valid_count



def masked_metrics(pred, target, mask):

    mask_float = mask.float()

    error = pred - target

    abs_error = torch.abs(error)
    squared_error = error ** 2

    valid_count = mask_float.sum()

    mae = (
        (abs_error * mask_float).sum()
        / valid_count
    )

    rmse = torch.sqrt(
        (squared_error * mask_float).sum()
        / valid_count
    )

    return mae.item(), rmse.item()



history = []

best_val_mae_30 = float("inf")
best_state = None
best_epoch = None
epochs_without_improvement = 0

N_TRAIN = traffic_train_tensor.shape[0]
N_VAL = traffic_val_tensor.shape[0]

print("\nTrain samples:", N_TRAIN)
print("Val samples  :", N_VAL)


for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()


    weather_graph_model.train()

    permutation = torch.randperm(
        N_TRAIN,
        device=DEVICE
    )

    train_loss_sum = 0.0
    train_batches = 0

    for start in range(0, N_TRAIN, BATCH_SIZE):

        indices = permutation[start:start + BATCH_SIZE]

        traffic_batch = traffic_train_tensor[indices]
        weather_batch = weather_train_tensor[indices]

        optimizer.zero_grad(set_to_none=True)

        predictions = weather_graph_model(
            traffic_batch,
            weather_batch
        )

        total_loss = torch.tensor(
            0.0,
            device=DEVICE
        )

        for horizon in HORIZONS:

            loss_h = masked_huber_loss(
                predictions[horizon],
                target_train_tensors[horizon][indices],
                mask_train_tensors[horizon][indices],
                delta=HUBER_DELTA
            )

            total_loss = total_loss + loss_h

        total_loss = total_loss / len(HORIZONS)

        if not torch.isfinite(total_loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}"
            )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            weather_graph_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += total_loss.item()
        train_batches += 1

    avg_train_loss = train_loss_sum / train_batches


    weather_graph_model.eval()

    val_predictions = {
        horizon: []
        for horizon in HORIZONS
    }

    with torch.no_grad():

        for start in range(0, N_VAL, BATCH_SIZE):

            end = min(start + BATCH_SIZE, N_VAL)

            traffic_batch = traffic_val_tensor[start:end]
            weather_batch = weather_val_tensor[start:end]

            predictions = weather_graph_model(
                traffic_batch,
                weather_batch
            )

            for horizon in HORIZONS:
                val_predictions[horizon].append(
                    predictions[horizon]
                )


    epoch_metrics = {}

    for horizon in HORIZONS:

        pred = torch.cat(
            val_predictions[horizon],
            dim=0
        )

        mae, rmse = masked_metrics(
            pred,
            target_val_tensors[horizon],
            mask_val_tensors[horizon]
        )

        epoch_metrics[horizon] = {
            "mae": mae,
            "rmse": rmse
        }

    val_mae_30 = epoch_metrics["30min"]["mae"]

    elapsed = time.time() - epoch_start


    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,

        "mae_15": epoch_metrics["15min"]["mae"],
        "rmse_15": epoch_metrics["15min"]["rmse"],

        "mae_30": epoch_metrics["30min"]["mae"],
        "rmse_30": epoch_metrics["30min"]["rmse"],

        "mae_45": epoch_metrics["45min"]["mae"],
        "rmse_45": epoch_metrics["45min"]["rmse"],

        "mae_60": epoch_metrics["60min"]["mae"],
        "rmse_60": epoch_metrics["60min"]["rmse"],
    })


    print(
        f"\nEpoch {epoch:02d} | "
        f"Loss {avg_train_loss:.4f} | "
        f"Time {elapsed:.1f}s"
    )

    for horizon in HORIZONS:

        print(
            f"  {horizon}: "
            f"MAE={epoch_metrics[horizon]['mae']:.4f} | "
            f"RMSE={epoch_metrics[horizon]['rmse']:.4f}"
        )


    if val_mae_30 < best_val_mae_30:

        best_val_mae_30 = val_mae_30
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = copy.deepcopy(
            weather_graph_model.state_dict()
        )

        print(
            f"  ✓ New best model "
            f"(30-min MAE = {best_val_mae_30:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"  No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

        if epochs_without_improvement >= PATIENCE:

            print("\nEarly stopping triggered.")
            break



if best_state is None:
    raise RuntimeError("No valid model checkpoint was saved.")

weather_graph_model.load_state_dict(best_state)


print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Best epoch      :", best_epoch)
print("Best 30-min MAE :", f"{best_val_mae_30:.4f}")
print("Epochs completed:", len(history))

print("\n✓ Best model restored")
print("✓ Validation used for model selection")
print("✓ Test set untouched")
print("=" * 70)

WEATHER + GRAPH-AWARE MULTI-HORIZON TRAINING

Configuration:
  Device       : mps
  Batch size   : 64
  Max epochs   : 20
  Patience     : 4
  Learning rate: 0.0005
  Loss         : Masked Huber
  Horizons     : ['15min', '30min', '45min', '60min']

Train samples: 23961
Val samples  : 3398

Epoch 01 | Loss 0.1876 | Time 215.1s
  15min: MAE=0.3251 | RMSE=0.6097
  30min: MAE=0.3747 | RMSE=0.7072
  45min: MAE=0.4210 | RMSE=0.7725
  60min: MAE=0.4606 | RMSE=0.8226
  ✓ New best model (30-min MAE = 0.3747)

Epoch 02 | Loss 0.1793 | Time 262.0s
  15min: MAE=0.3216 | RMSE=0.6079
  30min: MAE=0.3765 | RMSE=0.7093
  45min: MAE=0.4158 | RMSE=0.7776
  60min: MAE=0.4506 | RMSE=0.8302
  No improvement (1/4)

Epoch 03 | Loss 0.1767 | Time 262.9s
  15min: MAE=0.3182 | RMSE=0.6046
  30min: MAE=0.3643 | RMSE=0.7031
  45min: MAE=0.4025 | RMSE=0.7712
  60min: MAE=0.4340 | RMSE=0.8206
  ✓ New best model (30-min MAE = 0.3643)

Epoch 04 | Loss 0.1750 | Time 267.9s
  15min: MAE=0.3203 | RMSE=0.6023
  30min: M

In [212]:

import pandas as pd

print("=" * 70)
print("COMPLETED TRAINING — FULL HISTORY")
print("=" * 70)

history_df = pd.DataFrame(history)

print("\nTraining history:")
print(history_df.to_string(index=False))

print("\n" + "=" * 70)
print("BEST EPOCH")
print("=" * 70)

best_row = history_df.loc[
    history_df["mae_30"].idxmin()
]

print(best_row.to_string())

print("\n" + "=" * 70)
print("TARGET SCALE CHECK")
print("=" * 70)

for horizon in HORIZONS:

    train_target = target_train_tensors[horizon]
    val_target = target_val_tensors[horizon]

    print(f"\n{horizon}")

    print(
        f"  Train range: "
        f"{train_target.min().item():.4f} → "
        f"{train_target.max().item():.4f}"
    )

    print(
        f"  Val range  : "
        f"{val_target.min().item():.4f} → "
        f"{val_target.max().item():.4f}"
    )

print("\n" + "=" * 70)
print("NO TRAINING")
print("NO MODEL MODIFICATION")
print("TEST SET NOT USED")
print("=" * 70)

COMPLETED TRAINING — FULL HISTORY

Training history:
 epoch  train_loss   mae_15  rmse_15   mae_30  rmse_30   mae_45  rmse_45   mae_60  rmse_60
     1    0.187627 0.325143 0.609747 0.374706 0.707189 0.421038 0.772495 0.460551 0.822607
     2    0.179260 0.321559 0.607943 0.376501 0.709254 0.415768 0.777599 0.450640 0.830201
     3    0.176695 0.318187 0.604631 0.364281 0.703115 0.402451 0.771200 0.433986 0.820576
     4    0.175027 0.320302 0.602280 0.373155 0.702573 0.416372 0.770318 0.449668 0.819992
     5    0.173984 0.317910 0.595468 0.366469 0.696527 0.407154 0.767854 0.434499 0.818476
     6    0.173070 0.318604 0.592915 0.373088 0.693373 0.417223 0.763458 0.453506 0.813928
     7    0.172511 0.318117 0.596278 0.364149 0.697690 0.407171 0.769523 0.441587 0.820499
     8    0.171923 0.320053 0.598157 0.367821 0.695800 0.407057 0.764798 0.441289 0.814727
     9    0.171324 0.316549 0.592443 0.368272 0.693864 0.412846 0.767011 0.449162 0.817622
    10    0.170681 0.313602 0.596249 

In [213]:

print("=" * 70)
print("TARGET SCALE — FINAL CHECK")
print("=" * 70)

for horizon in HORIZONS:

    train_target = target_train_tensors[horizon]
    val_target = target_val_tensors[horizon]

    print(f"\n{horizon}")

    print(
        f"  Train: "
        f"min={train_target.min().item():.6f}, "
        f"max={train_target.max().item():.6f}, "
        f"mean={train_target.mean().item():.6f}, "
        f"std={train_target.std().item():.6f}"
    )

    print(
        f"  Val  : "
        f"min={val_target.min().item():.6f}, "
        f"max={val_target.max().item():.6f}, "
        f"mean={val_target.mean().item():.6f}, "
        f"std={val_target.std().item():.6f}"
    )

print("\n" + "=" * 70)
print("NO TRAINING")
print("NO MODEL MODIFICATION")
print("TEST SET NOT USED")
print("=" * 70)

TARGET SCALE — FINAL CHECK

15min
  Train: min=-28.913452, max=7.060560, mean=-0.000138, std=1.000158
  Val  : min=-18.654598, max=7.060560, mean=0.037637, std=1.058797

30min
  Train: min=-28.913452, max=7.060560, mean=-0.000112, std=1.000136
  Val  : min=-18.654598, max=7.060560, mean=0.038993, std=1.055974

45min
  Train: min=-28.913452, max=7.060560, mean=-0.000118, std=1.000148
  Val  : min=-18.654598, max=7.060560, mean=0.040353, std=1.053449

60min
  Train: min=-28.913452, max=7.060560, mean=-0.000107, std=1.000151
  Val  : min=-18.654598, max=7.060560, mean=0.041587, std=1.051494

NO TRAINING
NO MODEL MODIFICATION
TEST SET NOT USED


In [214]:

print("=" * 70)
print("SEARCHING FOR TARGET NORMALIZATION / SCALER")
print("=" * 70)

keywords = [
    "scaler",
    "standard",
    "normalize",
    "normalization",
    "mean",
    "std",
    "scale",
    "speed"
]

found = []

for name, value in sorted(globals().items()):

    name_lower = name.lower()

    if any(keyword in name_lower for keyword in keywords):

        if name.startswith("_"):
            continue

        found.append((name, value))

        print(f"\n{name}")
        print(f"  type: {type(value)}")

        for attr in [
            "mean_",
            "scale_",
            "var_",
            "n_features_in_",
            "feature_names_in_"
        ]:
            if hasattr(value, attr):
                attr_value = getattr(value, attr)

                if hasattr(attr_value, "shape"):
                    print(
                        f"  {attr}: "
                        f"shape={attr_value.shape}"
                    )
                else:
                    print(
                        f"  {attr}: "
                        f"{attr_value}"
                    )

        if hasattr(value, "shape"):
            print(
                f"  shape={value.shape}, "
                f"dtype={getattr(value, 'dtype', 'N/A')}"
            )

print("\n" + "=" * 70)
print(f"FOUND {len(found)} POSSIBLE NORMALIZATION OBJECTS")
print("=" * 70)

print("NO TRAINING")
print("NO MODEL MODIFICATION")
print("TEST SET NOT USED")

SEARCHING FOR TARGET NORMALIZATION / SCALER

StandardScaler
  type: <class 'type'>

benchmark_speed_analysis
  type: <class 'pandas.DataFrame'>
  shape=(4, 4), dtype=N/A

benchmark_speed_categories
  type: <class 'pandas.Categorical'>
  shape=(664844,), dtype=category

huber_speed_analysis
  type: <class 'pandas.DataFrame'>
  shape=(4, 4), dtype=N/A

huber_speed_categories
  type: <class 'pandas.Categorical'>
  shape=(664844,), dtype=category

mean_absolute_error
  type: <class 'function'>

mean_squared_error
  type: <class 'function'>

persistence_scaled
  type: <class 'numpy.ndarray'>
  shape=(6826, 207), dtype=float32

pred_scaled
  type: <class 'numpy.ndarray'>
  shape=(3398, 207), dtype=float32

predictions_scaled
  type: <class 'numpy.ndarray'>
  shape=(3410, 207), dtype=float32

predictions_scaled_corrected
  type: <class 'numpy.ndarray'>
  shape=(3410, 207), dtype=float32

predictions_scaled_sensor
  type: <class 'numpy.ndarray'>
  shape=(3410, 207), dtype=float32

scale_target

In [ ]:

print("=" * 70)
print("SEARCHING EXECUTED CODE FOR TARGET CONSTRUCTION")
print("=" * 70)

matches = []

if "In" in globals():

    for execution_number, code in enumerate(In):

        if not code:
            continue

        code_lower = code.lower()

        if (
            "y_train" in code_lower
            or "y_val" in code_lower
            or "standardscaler" in code_lower
            or "scaler.fit" in code_lower
            or "scaler.transform" in code_lower
        ):

            matches.append(
                (execution_number, code)
            )

for execution_number, code in matches:

    print("\n" + "=" * 70)
    print(f"EXECUTION CELL {execution_number}")
    print("=" * 70)
    print(code)

print("\n" + "=" * 70)
print(f"FOUND {len(matches)} RELEVANT EXECUTED CELLS")
print("=" * 70)

print("NO TRAINING")
print("NO MODEL MODIFICATION")
print("TEST SET NOT USED")

In [217]:

print("=" * 70)
print("SAVED TARGET DICTIONARY STRUCTURE")
print("=" * 70)

print("\ny_train")
print("  type:", type(y_train))

if isinstance(y_train, dict):

    print("  keys:", list(y_train.keys()))

    for key, value in y_train.items():

        print(f"\n  [{key}]")
        print("    type :", type(value))
        print("    shape:", getattr(value, "shape", None))
        print("    dtype:", getattr(value, "dtype", None))

        if hasattr(value, "shape"):

            print(
                "    min  :",
                np.nanmin(value)
            )

            print(
                "    max  :",
                np.nanmax(value)
            )

            print(
                "    mean :",
                np.nanmean(value)
            )

            print(
                "    std  :",
                np.nanstd(value)
            )

print("\n" + "=" * 70)
print("y_val")
print("=" * 70)

print("  type:", type(y_val))

if isinstance(y_val, dict):

    print("  keys:", list(y_val.keys()))

    for key, value in y_val.items():

        print(f"\n  [{key}]")
        print("    type :", type(value))
        print("    shape:", getattr(value, "shape", None))
        print("    dtype:", getattr(value, "dtype", None))

        if hasattr(value, "shape"):

            print(
                "    min  :",
                np.nanmin(value)
            )

            print(
                "    max  :",
                np.nanmax(value)
            )

            print(
                "    mean :",
                np.nanmean(value)
            )

            print(
                "    std  :",
                np.nanstd(value)
            )

print("\n" + "=" * 70)
print("NO TRAINING")
print("NO MODEL MODIFICATION")
print("TEST SET NOT USED")
print("=" * 70)

SAVED TARGET DICTIONARY STRUCTURE

y_train
  type: <class 'dict'>
  keys: ['15min', '30min', '45min', '60min']

  in]
    type : <class 'numpy.ndarray'>
    shape: (23961, 207)
    dtype: float32
    min  : -28.913452
    max  : 7.06056
    mean : -0.00013767686
    std  : 1.000158

  in]
    type : <class 'numpy.ndarray'>
    shape: (23961, 207)
    dtype: float32
    min  : -28.913452
    max  : 7.06056
    mean : -0.00011212402
    std  : 1.0001361

  in]
    type : <class 'numpy.ndarray'>
    shape: (23961, 207)
    dtype: float32
    min  : -28.913452
    max  : 7.06056
    mean : -0.00011807937
    std  : 1.0001477

  in]
    type : <class 'numpy.ndarray'>
    shape: (23961, 207)
    dtype: float32
    min  : -28.913452
    max  : 7.06056
    mean : -0.00010667173
    std  : 1.0001508

y_val
  type: <class 'dict'>
  keys: ['15min', '30min', '45min', '60min']

  in]
    type : <class 'numpy.ndarray'>
    shape: (3398, 207)
    dtype: float32
    min  : -18.654598
    max  : 7.0605

In [2]:

from pathlib import Path
import torch

print("=" * 70)
print("URBANFLOW NOTEBOOK 10 — RECOVERY CHECK")
print("=" * 70)

print("\n[1] Model in memory:")
print("weather_graph_model exists:", "weather_graph_model" in globals())

print("\n[2] Validation objects:")
for name in [
    "traffic_val_tensor",
    "weather_val_tensor",
    "target_val_tensors",
    "mask_val_tensors"
]:
    print(f"{name}: {name in globals()}")

print("\n[3] Saved model checkpoints:")

search_roots = [
    Path("../../models"),
    Path("../../results")
]

checkpoint_files = []

for root in search_roots:
    if root.exists():
        checkpoint_files.extend(
            p for p in root.rglob("*")
            if p.is_file() and p.suffix.lower() in {".pt", ".pth", ".ckpt"}
        )

if checkpoint_files:
    for path in checkpoint_files:
        print("FOUND:", path)
else:
    print("No .pt / .pth / .ckpt checkpoint found.")

print("\n" + "=" * 70)
print("RECOVERY CHECK COMPLETE")
print("=" * 70)

URBANFLOW NOTEBOOK 10 — RECOVERY CHECK

[1] Model in memory:
weather_graph_model exists: False

[2] Validation objects:
traffic_val_tensor: False
weather_val_tensor: False
target_val_tensors: False
mask_val_tensors: False

[3] Saved model checkpoints:
FOUND: ../../models/transformer_dmodel_64_best.pth
FOUND: ../../models/transformer_layers_3_best.pth
FOUND: ../../models/improved_urbanflow_best.pth
FOUND: ../../models/transformer_lr_0005_best.pth
FOUND: ../../models/lstm_best.pth
FOUND: ../../models/spatial_temporal_fusion_best.pth
FOUND: ../../models/transformer_lr_002_best.pth
FOUND: ../../models/spatial_residual_correction_best.pth
FOUND: ../../models/corrected_spatial_temporal_fusion_best.pth
FOUND: ../../models/weather_multi_horizon_transformer_best.pth
FOUND: ../../models/urbanflow_best.pth
FOUND: ../../models/transformer_heads_2_best.pth
FOUND: ../../models/weather_aware_urbanflow_v1_best.pth
FOUND: ../../models/transformer_dropout_01_best.pth
FOUND: ../../models/transformer_dmod

In [11]:

import numpy as np
from pathlib import Path

DATA_DIR = Path("../../data/processed")

train_data = np.load(
    DATA_DIR / "train_sequences.npz",
    allow_pickle=True
)

val_data = np.load(
    DATA_DIR / "validation_sequences.npz",
    allow_pickle=True
)

print("=" * 70)
print("SAVED SEQUENCE FORMAT")
print("=" * 70)

print("\nTRAIN FILE")
print("Keys:", train_data.files)

for key in train_data.files:
    arr = train_data[key]
    print(
        f"{key}: "
        f"type={type(arr).__name__}, "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

print("\nVALIDATION FILE")
print("Keys:", val_data.files)

for key in val_data.files:
    arr = val_data[key]
    print(
        f"{key}: "
        f"type={type(arr).__name__}, "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

print("\n" + "=" * 70)
print("READ-ONLY CHECK COMPLETE")
print("=" * 70)

SAVED SEQUENCE FORMAT

TRAIN FILE
Keys: ['X', 'y']
X: type=ndarray, shape=(23973, 12, 207), dtype=float32
y: type=ndarray, shape=(23973, 207), dtype=float32

VALIDATION FILE
Keys: ['X', 'y']
X: type=ndarray, shape=(3410, 12, 207), dtype=float32
y: type=ndarray, shape=(3410, 207), dtype=float32

READ-ONLY CHECK COMPLETE


In [12]:

from pathlib import Path

DATA_DIR = Path("../../data/processed")

print("=" * 70)
print("EXISTING MULTI-HORIZON / GRAPH FILES")
print("=" * 70)

print("\nTarget-related files:")

for path in sorted(DATA_DIR.glob("*")):
    name = path.name.lower()

    if (
        "15min" in name
        or "30min" in name
        or "45min" in name
        or "60min" in name
        or "target" in name
        or "horizon" in name
    ):
        print(" ", path.name)


print("\nGraph-related files:")

for path in sorted(DATA_DIR.glob("*")):
    name = path.name.lower()

    if (
        "adj" in name
        or "graph" in name
        or "distance" in name
        or "sensor" in name
    ):
        print(" ", path.name)


print("\nWeather-related files:")

for path in sorted(DATA_DIR.glob("*")):
    name = path.name.lower()

    if "weather" in name:
        print(" ", path.name)

print("\n" + "=" * 70)
print("READ-ONLY FILE DISCOVERY COMPLETE")
print("=" * 70)

EXISTING MULTI-HORIZON / GRAPH FILES

Target-related files:
  test_target_mask.npy
  test_target_mask_15min.npy
  test_target_mask_30min.npy
  test_target_mask_45min.npy
  test_target_mask_60min.npy
  train_target_mask.npy
  train_target_mask_15min.npy
  train_target_mask_30min.npy
  train_target_mask_45min.npy
  train_target_mask_60min.npy
  val_target_mask_15min.npy
  val_target_mask_30min.npy
  val_target_mask_45min.npy
  val_target_mask_60min.npy
  validation_target_mask.npy
  y_test_15min.npy
  y_test_30min.npy
  y_test_45min.npy
  y_test_60min.npy
  y_train_15min.npy
  y_train_30min.npy
  y_train_45min.npy
  y_train_60min.npy
  y_val_15min.npy
  y_val_30min.npy
  y_val_45min.npy
  y_val_60min.npy

Graph-related files:
  adjacency_matrix.npy
  graph_sensor_ids.txt

Weather-related files:
  weather_5min_aligned.csv
  weather_metadata.json
  weather_scaler.pkl
  weather_test_sequences.npy
  weather_train_input_sequences.npy
  weather_train_scaled.npy
  weather_train_sequences.npy
  

In [13]:
from pathlib import Path

DATA_DIR = Path("../../data/processed")

print("=" * 70)
print("TARGET FILES")
print("=" * 70)

for p in sorted(DATA_DIR.glob("y_*")):
    print(p.name)

print("\n" + "=" * 70)
print("GRAPH FILES")
print("=" * 70)

for p in sorted(DATA_DIR.glob("*")):
    name = p.name.lower()
    if any(x in name for x in ["adj", "graph", "distance", "sensor"]):
        print(p.name)

print("\n" + "=" * 70)
print("WEATHER FILES")
print("=" * 70)

for p in sorted(DATA_DIR.glob("*weather*")):
    print(p.name)

TARGET FILES
y_test_15min.npy
y_test_30min.npy
y_test_45min.npy
y_test_60min.npy
y_train_15min.npy
y_train_30min.npy
y_train_45min.npy
y_train_60min.npy
y_val_15min.npy
y_val_30min.npy
y_val_45min.npy
y_val_60min.npy

GRAPH FILES
adjacency_matrix.npy
graph_sensor_ids.txt

WEATHER FILES
weather_5min_aligned.csv
weather_metadata.json
weather_scaler.pkl
weather_test_sequences.npy
weather_train_input_sequences.npy
weather_train_scaled.npy
weather_train_sequences.npy
weather_validation_input_sequences.npy
weather_validation_scaled.npy
weather_validation_sequences.npy


In [18]:

from pathlib import Path
import numpy as np

DATA_DIR = Path("../../data/processed")

HORIZONS = ["15min", "30min", "45min", "60min"]

print("=" * 70)
print("EXACT SAVED DATA SHAPES")
print("=" * 70)

print("\nMULTI-HORIZON TARGETS")
for h in HORIZONS:
    train = np.load(DATA_DIR / f"y_train_{h}.npy", mmap_mode="r")
    val = np.load(DATA_DIR / f"y_val_{h}.npy", mmap_mode="r")

    print(
        f"{h}: "
        f"train={train.shape}, "
        f"val={val.shape}, "
        f"dtype={train.dtype}"
    )

print("\nTRAFFIC")
train_x = np.load(
    DATA_DIR / "train_sequences.npz"
)["X"]

val_x = np.load(
    DATA_DIR / "validation_sequences.npz"
)["X"]

print("X_train:", train_x.shape)
print("X_val  :", val_x.shape)

print("\nWEATHER")
weather_train = np.load(
    DATA_DIR / "weather_train_sequences.npy",
    mmap_mode="r"
)

weather_val = np.load(
    DATA_DIR / "weather_validation_sequences.npy",
    mmap_mode="r"
)

print("Weather train:", weather_train.shape)
print("Weather val  :", weather_val.shape)

print("\nGRAPH")
adjacency = np.load(
    DATA_DIR / "adjacency_matrix.npy",
    mmap_mode="r"
)

print("Adjacency:", adjacency.shape)

print("\n" + "=" * 70)
print("READ-ONLY CHECK COMPLETE")
print("=" * 70)

EXACT SAVED DATA SHAPES

MULTI-HORIZON TARGETS
15min: train=(23970, 207), val=(3407, 207), dtype=float32
30min: train=(23973, 207), val=(3410, 207), dtype=float32
45min: train=(23964, 207), val=(3401, 207), dtype=float32
60min: train=(23961, 207), val=(3398, 207), dtype=float32

TRAFFIC
X_train: (23973, 12, 207)
X_val  : (3410, 12, 207)

WEATHER
Weather train: (23973, 12, 6)
Weather val  : (3410, 12, 6)

GRAPH
Adjacency: (207, 207)

READ-ONLY CHECK COMPLETE


In [22]:

objects = [
    "traffic_train_tensor",
    "traffic_val_tensor",
    "weather_train_tensor",
    "weather_val_tensor",
    "target_train_tensors",
    "target_val_tensors",
    "mask_train_tensors",
    "mask_val_tensors",
    "A_norm",
    "weather_graph_model"
]

print("=" * 70)
print("CURRENT URBANFLOW KERNEL STATE")
print("=" * 70)

for name in objects:
    print(f"{name:30s}: {name in globals()}")

print("=" * 70)

CURRENT URBANFLOW KERNEL STATE
traffic_train_tensor          : True
traffic_val_tensor            : True
weather_train_tensor          : True
weather_val_tensor            : True
target_train_tensors          : True
target_val_tensors            : True
mask_train_tensors            : True
mask_val_tensors              : True
A_norm                        : True
weather_graph_model           : True


In [24]:

print("=" * 70)
print("URBANFLOW — FINAL PRE-TRAINING VERIFICATION")
print("=" * 70)



print("\n[1] MODEL")

model_device = next(
    weather_graph_model.parameters()
).device

parameter_count = sum(
    p.numel()
    for p in weather_graph_model.parameters()
    if p.requires_grad
)

print(
    "Model:",
    type(weather_graph_model).__name__
)

print(
    "Parameters:",
    f"{parameter_count:,}"
)

assert parameter_count == 352868



print("\n[2] DEVICES")

print("Model  :", model_device)
print("Traffic:", traffic_train_tensor.device)
print("Weather:", weather_train_tensor.device)
print("Graph  :", A_norm.device)


assert model_device.type == DEVICE.type
assert traffic_train_tensor.device.type == DEVICE.type
assert weather_train_tensor.device.type == DEVICE.type
assert A_norm.device.type == DEVICE.type

print("✓ All components use:", DEVICE.type)



print("\n[3] SHAPES")

print(
    "Traffic train:",
    tuple(traffic_train_tensor.shape)
)

print(
    "Weather train:",
    tuple(weather_train_tensor.shape)
)

print(
    "Graph:",
    tuple(A_norm.shape)
)

for horizon in HORIZONS:

    print(
        f"{horizon}: "
        f"target={tuple(target_train_tensors[horizon].shape)}, "
        f"mask={tuple(mask_train_tensors[horizon].shape)}"
    )

assert traffic_train_tensor.shape == (
    23961,
    12,
    207
)

assert weather_train_tensor.shape == (
    23961,
    12,
    6
)

assert A_norm.shape == (
    207,
    207
)

for horizon in HORIZONS:

    assert target_train_tensors[horizon].shape == (
        23961,
        207
    )

    assert mask_train_tensors[horizon].shape == (
        23961,
        207
    )



print("\n[4] FORWARD PASS")

weather_graph_model.eval()

with torch.no_grad():

    sample_outputs = weather_graph_model(
        traffic_train_tensor[:4],
        weather_train_tensor[:4]
    )

for horizon in HORIZONS:

    output = sample_outputs[horizon]

    print(
        f"{horizon}: "
        f"shape={tuple(output.shape)}, "
        f"NaN={torch.isnan(output).sum().item()}, "
        f"Inf={torch.isinf(output).sum().item()}"
    )

    assert output.shape == (
        4,
        207
    )

    assert torch.isfinite(output).all()



print("\n[5] TARGET / MASK CHECK")

for horizon in HORIZONS:

    target = target_train_tensors[horizon]
    mask = mask_train_tensors[horizon]

    print(
        f"{horizon}: "
        f"target={target.dtype}, "
        f"mask={mask.dtype}, "
        f"valid={mask.sum().item():,}"
    )

    assert target.shape == mask.shape
    assert target.dtype == torch.float32
    assert mask.dtype == torch.bool



print("\n" + "=" * 70)
print("✓ MODEL STATE VALID")
print("✓ INPUTS VALID")
print("✓ GRAPH VALID")
print("✓ FOUR TARGETS VALID")
print("✓ MASKS VALID")
print("✓ FORWARD PASS VALID")
print("✓ TEST SET NOT USED")
print("✓ NO TRAINING PERFORMED")
print("=" * 70)

URBANFLOW — FINAL PRE-TRAINING VERIFICATION

[1] MODEL
Model: WeatherGraphAwareUrbanFlow
Parameters: 352,868

[2] DEVICES
Model  : mps:0
Traffic: mps:0
Weather: mps:0
Graph  : mps:0
✓ All components use: mps

[3] SHAPES
Traffic train: (23961, 12, 207)
Weather train: (23961, 12, 6)
Graph: (207, 207)
15min: target=(23961, 207), mask=(23961, 207)
30min: target=(23961, 207), mask=(23961, 207)
45min: target=(23961, 207), mask=(23961, 207)
60min: target=(23961, 207), mask=(23961, 207)

[4] FORWARD PASS
15min: shape=(4, 207), NaN=0, Inf=0
30min: shape=(4, 207), NaN=0, Inf=0
45min: shape=(4, 207), NaN=0, Inf=0
60min: shape=(4, 207), NaN=0, Inf=0

[5] TARGET / MASK CHECK
15min: target=torch.float32, mask=torch.bool, valid=4,959,927
30min: target=torch.float32, mask=torch.bool, valid=4,959,927
45min: target=torch.float32, mask=torch.bool, valid=4,959,927
60min: target=torch.float32, mask=torch.bool, valid=4,959,927

✓ MODEL STATE VALID
✓ INPUTS VALID
✓ GRAPH VALID
✓ FOUR TARGETS VALID
✓ MASKS VA

In [25]:

from pathlib import Path

ROOT = Path("../../")

print("=" * 70)
print("SEARCHING EXISTING SCALER / NORMALIZATION ARTIFACTS")
print("=" * 70)

keywords = [
    "scaler",
    "standard",
    "normal",
    "mean",
    "std",
    "statistics",
    "metadata"
]

found = []

for path in ROOT.rglob("*"):

    if any(
        part in {".venv", ".git", "__pycache__", ".ipynb_checkpoints"}
        for part in path.parts
    ):
        continue

    name = path.name.lower()

    if any(k in name for k in keywords):

        if path.is_file():
            found.append(path)

print("\nPotential existing artifacts:")

if found:
    for path in sorted(found):
        print("✓", path)
else:
    print("None found.")

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("NO FILES MODIFIED")
print("NO DATA LOADED")
print("NO TRAINING")
print("=" * 70)

SEARCHING EXISTING SCALER / NORMALIZATION ARTIFACTS

Potential existing artifacts:
✓ ../../data/processed/metadata.json
✓ ../../data/processed/traffic_scaler.pkl
✓ ../../data/processed/weather_metadata.json
✓ ../../data/processed/weather_scaler.pkl

SEARCH COMPLETE
NO FILES MODIFIED
NO DATA LOADED
NO TRAINING


In [35]:

import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error


HORIZONS = ["15min", "30min", "45min", "60min"]

RESULTS_DIR = Path("../../results")
MODELS_DIR = Path("../../models")
DATA_DIR = Path("../../data/processed")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SCALER_PATH = DATA_DIR / "traffic_scaler.pkl"

BATCH_SIZE_EVAL = 16

print("=" * 70)
print("URBANFLOW — FINAL MULTI-HORIZON VALIDATION")
print("ORIGINAL MPH SCALE")
print("=" * 70)


assert "weather_graph_model" in globals(), (
    "weather_graph_model is not available."
)

assert "traffic_val_tensor" in globals(), (
    "traffic_val_tensor is not available."
)

assert "weather_val_tensor" in globals(), (
    "weather_val_tensor is not available."
)

assert "target_val_tensors" in globals(), (
    "target_val_tensors are not available."
)

assert "mask_val_tensors" in globals(), (
    "mask_val_tensors are not available."
)

assert SCALER_PATH.exists(), (
    f"Traffic scaler not found: {SCALER_PATH}"
)


traffic_scaler = joblib.load(SCALER_PATH)

assert traffic_scaler.mean_.shape == (207,), (
    f"Unexpected scaler shape: {traffic_scaler.mean_.shape}"
)

assert traffic_scaler.scale_.shape == (207,), (
    f"Unexpected scaler scale shape: {traffic_scaler.scale_.shape}"
)

print("\nScaler:")
print("  Type:", type(traffic_scaler).__name__)
print("  Sensors:", traffic_scaler.mean_.shape[0])


weather_graph_model.eval()

model_device = next(
    weather_graph_model.parameters()
).device

print("\nModel:")
print("  Type:", type(weather_graph_model).__name__)
print(
    "  Parameters:",
    f"{sum(p.numel() for p in weather_graph_model.parameters()):,}"
)
print("  Device:", model_device)


validation_predictions = {
    horizon: []
    for horizon in HORIZONS
}

N_VAL = traffic_val_tensor.shape[0]

print("\nValidation samples:", N_VAL)
print("Batch size:", BATCH_SIZE_EVAL)

with torch.no_grad():

    for start in range(0, N_VAL, BATCH_SIZE_EVAL):

        end = min(start + BATCH_SIZE_EVAL, N_VAL)

        traffic_batch = traffic_val_tensor[start:end]
        weather_batch = weather_val_tensor[start:end]

        outputs = weather_graph_model(
            traffic_batch,
            weather_batch
        )

        for horizon in HORIZONS:

            pred = outputs[horizon]

            validation_predictions[horizon].append(
                pred.detach().cpu().numpy()
            )


for horizon in HORIZONS:

    validation_predictions[horizon] = np.concatenate(
        validation_predictions[horizon],
        axis=0
    )

    print(
        f"{horizon}: prediction shape = "
        f"{validation_predictions[horizon].shape}"
    )


validation_metrics = []

for horizon in HORIZONS:

    predictions_scaled = validation_predictions[horizon]

    targets_scaled = (
        target_val_tensors[horizon]
        .detach()
        .cpu()
        .numpy()
    )

    mask = (
        mask_val_tensors[horizon]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )

    predictions_mph = traffic_scaler.inverse_transform(
        predictions_scaled
    )

    targets_mph = traffic_scaler.inverse_transform(
        targets_scaled
    )

    y_pred = predictions_mph[mask]
    y_true = targets_mph[mask]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    validation_metrics.append({
        "horizon": horizon,
        "MAE_mph": mae,
        "RMSE_mph": rmse,
        "valid_values": int(mask.sum()),
        "prediction_min_mph": float(np.min(y_pred)),
        "prediction_max_mph": float(np.max(y_pred)),
        "target_min_mph": float(np.min(y_true)),
        "target_max_mph": float(np.max(y_true))
    })

    np.save(
        RESULTS_DIR / f"weather_graph_{horizon}_validation_predictions_mph.npy",
        predictions_mph
    )


validation_results = pd.DataFrame(validation_metrics)

metrics_path = (
    RESULTS_DIR /
    "weather_graph_multi_horizon_validation_mph.csv"
)

validation_results.to_csv(
    metrics_path,
    index=False
)


print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(
    validation_results[
        ["horizon", "MAE_mph", "RMSE_mph", "valid_values"]
    ].to_string(index=False)
)

assert np.isfinite(
    validation_results[["MAE_mph", "RMSE_mph"]].values
).all()

assert (
    validation_results["valid_values"] > 0
).all()


checkpoint_path = (
    MODELS_DIR /
    "weather_graph_multi_horizon_final_validation.pth"
)

torch.save(
    {
        "model_state_dict": weather_graph_model.state_dict(),
        "model_class": type(weather_graph_model).__name__,
        "horizons": HORIZONS,
        "validation_metrics_mph": validation_results.to_dict(
            orient="records"
        ),
        "traffic_scaler_path": str(SCALER_PATH),
        "num_parameters": sum(
            p.numel()
            for p in weather_graph_model.parameters()
        ),
    },
    checkpoint_path
)

print("\nSaved:")
print("  Metrics:")
print(f"    {metrics_path}")

print("\n  Predictions:")
for horizon in HORIZONS:
    print(
        f"    {RESULTS_DIR / f'weather_graph_{horizon}_validation_predictions_mph.npy'}"
    )

print("\n  Model checkpoint:")
print(f"    {checkpoint_path}")

print("\n" + "=" * 70)
print("FINAL VALIDATION COMPLETE")
print("TEST SET WAS NOT USED")
print("NO TRAINING PERFORMED")
print("=" * 70)

URBANFLOW — FINAL MULTI-HORIZON VALIDATION
ORIGINAL MPH SCALE

Scaler:
  Type: StandardScaler
  Sensors: 207

Model:
  Type: WeatherGraphAwareUrbanFlow
  Parameters: 352,868
  Device: mps:0

Validation samples: 3398
Batch size: 16
15min: prediction shape = (3398, 207)
30min: prediction shape = (3398, 207)
45min: prediction shape = (3398, 207)
60min: prediction shape = (3398, 207)

VALIDATION RESULTS
horizon   MAE_mph  RMSE_mph  valid_values
  15min  6.046566 12.019218        703386
  30min  6.031898 11.891264        703386
  45min  7.272453 11.251121        703386
  60min 14.152357 19.170214        703386

Saved:
  Metrics:
    ../../results/weather_graph_multi_horizon_validation_mph.csv

  Predictions:
    ../../results/weather_graph_15min_validation_predictions_mph.npy
    ../../results/weather_graph_30min_validation_predictions_mph.npy
    ../../results/weather_graph_45min_validation_predictions_mph.npy
    ../../results/weather_graph_60min_validation_predictions_mph.npy

  Model ch

In [43]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error


DATA_DIR = Path("../../data/processed")
RESULTS_DIR = Path("../../results")
MODELS_DIR = Path("../../models")

CHECKPOINT_PATH = (
    MODELS_DIR /
    "weather_multi_horizon_transformer_best.pth"
)

SCALER_PATH = (
    DATA_DIR /
    "traffic_scaler.pkl"
)

HORIZONS = [
    "15min",
    "30min",
    "45min",
    "60min"
]

BATCH_SIZE = 16

print("=" * 70)
print("URBANFLOW — FINAL UNTOUCHED TEST EVALUATION")
print("WEATHER-AWARE MULTI-HORIZON TRANSFORMER")
print("=" * 70)


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

print("\nCheckpoint loaded successfully.")


input_weight = state_dict["input_projection.weight"]

D_MODEL = input_weight.shape[0]
NUM_SENSORS = input_weight.shape[1]

weather_weight = state_dict[
    "weather_projection.weight"
]

WEATHER_DIM = weather_weight.shape[0]
WEATHER_FEATURES = weather_weight.shape[1]

fusion_weight = state_dict[
    "fusion_projection.weight"
]

FUSION_INPUT = fusion_weight.shape[1]

layer_numbers = []

for key in state_dict.keys():

    if key.startswith("transformer.layers."):

        parts = key.split(".")

        if len(parts) > 2:
            try:
                layer_numbers.append(
                    int(parts[2])
                )
            except ValueError:
                pass

NUM_LAYERS = max(layer_numbers) + 1

PE_SHAPE = state_dict[
    "position_encoding.pe"
].shape

MAX_LEN = PE_SHAPE[1]

ATTENTION_DIM = state_dict[
    "transformer.layers.0.self_attn.in_proj_weight"
].shape[1]

NHEAD = 4

FFN_DIM = state_dict[
    "transformer.layers.0.linear1.weight"
].shape[0]

print("\nArchitecture recovered from checkpoint:")
print("  Sensors:", NUM_SENSORS)
print("  d_model:", D_MODEL)
print("  Weather features:", WEATHER_FEATURES)
print("  Weather embedding:", WEATHER_DIM)
print("  Fusion input:", FUSION_INPUT)
print("  Transformer layers:", NUM_LAYERS)
print("  Attention heads:", NHEAD)
print("  FFN dimension:", FFN_DIM)
print("  Positional length:", MAX_LEN)


class ExactPositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len
    ):
        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (
                -np.log(10000.0)
                / d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        return (
            x +
            self.pe[:, :x.size(1), :]
        )



class ExactWeatherMultiHorizonTransformer(nn.Module):

    def __init__(self):

        super().__init__()


        self.input_projection = nn.Linear(
            NUM_SENSORS,
            D_MODEL
        )


        self.position_encoding = (
            ExactPositionalEncoding(
                D_MODEL,
                MAX_LEN
            )
        )


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=FFN_DIM,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=NUM_LAYERS
        )


        self.weather_projection = nn.Linear(
            WEATHER_FEATURES,
            WEATHER_DIM
        )


        self.fusion_projection = nn.Linear(
            FUSION_INPUT,
            D_MODEL
        )


        self.head_15 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_30 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_45 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_60 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

    def forward(
        self,
        traffic,
        weather
    ):


        x = self.input_projection(
            traffic
        )

        x = self.position_encoding(
            x
        )

        x = self.transformer(
            x
        )

        traffic_features = x[:, -1, :]

        weather_features = self.weather_projection(
            weather[:, -1, :]
        )

        fused = torch.cat(
            [
                traffic_features,
                weather_features
            ],
            dim=1
        )

        fused = self.fusion_projection(
            fused
        )

        return {
            "15min": self.head_15(fused),
            "30min": self.head_30(fused),
            "45min": self.head_45(fused),
            "60min": self.head_60(fused)
        }



final_model = (
    ExactWeatherMultiHorizonTransformer()
)

final_model.load_state_dict(
    state_dict,
    strict=True
)

final_model = final_model.to(DEVICE)
final_model.eval()

print("\n" + "=" * 70)
print("TRAINED MODEL RESTORED SUCCESSFULLY")
print("=" * 70)

print(
    "Parameters:",
    f"{sum(p.numel() for p in final_model.parameters()):,}"
)

print(
    "Device:",
    next(final_model.parameters()).device
)


traffic_scaler = joblib.load(
    SCALER_PATH
)

assert traffic_scaler.mean_.shape == (
    NUM_SENSORS,
)

assert traffic_scaler.scale_.shape == (
    NUM_SENSORS,
)

print("\nTraffic scaler loaded.")
print("Sensors:", NUM_SENSORS)


test_data = np.load(
    DATA_DIR /
    "test_sequences.npz"
)

X_test = np.asarray(
    test_data["X"],
    dtype=np.float32
)

weather_test = np.asarray(
    np.load(
        DATA_DIR /
        "weather_test_sequences.npy"
    ),
    dtype=np.float32
)

Y_test = {
    horizon: np.asarray(
        np.load(
            DATA_DIR /
            f"y_test_{horizon}.npy"
        ),
        dtype=np.float32
    )
    for horizon in HORIZONS
}

MASK_test = {
    horizon: np.isfinite(
        Y_test[horizon]
    )
    for horizon in HORIZONS
}

print("\nTest data:")
print("  Traffic:", X_test.shape)
print("  Weather:", weather_test.shape)


traffic_test_tensor = (
    torch.from_numpy(
        X_test
    ).to(DEVICE)
)

weather_test_tensor = (
    torch.from_numpy(
        weather_test
    ).to(DEVICE)
)

test_predictions = {
    horizon: []
    for horizon in HORIZONS
}

print("\nRunning final test inference...")

with torch.no_grad():

    for start in range(
        0,
        len(X_test),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(X_test)
        )

        outputs = final_model(
            traffic_test_tensor[start:end],
            weather_test_tensor[start:end]
        )

        for horizon in HORIZONS:

            test_predictions[
                horizon
            ].append(
                outputs[horizon]
                .detach()
                .cpu()
                .numpy()
            )


final_results = []

for horizon in HORIZONS:

    predictions_scaled = np.concatenate(
        test_predictions[horizon],
        axis=0
    )

    predictions_mph = (
        traffic_scaler.inverse_transform(
            predictions_scaled
        )
    )

    targets_mph = (
        traffic_scaler.inverse_transform(
            Y_test[horizon]
        )
    )

    mask = MASK_test[horizon]

    y_pred = predictions_mph[mask]
    y_true = targets_mph[mask]

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    final_results.append({
        "horizon": horizon,
        "MAE_mph": mae,
        "RMSE_mph": rmse,
        "valid_values": int(mask.sum())
    })

    np.save(
        RESULTS_DIR /
        f"weather_transformer_{horizon}_FINAL_TEST_predictions_mph.npy",
        predictions_mph
    )


final_test_df = pd.DataFrame(
    final_results
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR /
    "weather_transformer_FINAL_TEST_results.csv"
)

final_test_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)


print("\n" + "=" * 70)
print("FINAL TEST RESULTS — ORIGINAL MPH SCALE")
print("=" * 70)

print(
    final_test_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nSaved:")
print(
    " ",
    FINAL_RESULTS_PATH
)

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION COMPLETE")
print("NO TRAINING PERFORMED")
print("TEST SET EVALUATED")
print("=" * 70)

URBANFLOW — FINAL UNTOUCHED TEST EVALUATION
WEATHER-AWARE MULTI-HORIZON TRANSFORMER

Checkpoint loaded successfully.

Architecture recovered from checkpoint:
  Sensors: 207
  d_model: 128
  Weather features: 6
  Weather embedding: 32
  Fusion input: 160
  Transformer layers: 2
  Attention heads: 4
  FFN dimension: 256
  Positional length: 12

TRAINED MODEL RESTORED SUCCESSFULLY
Parameters: 419,228
Device: mps:0

Traffic scaler loaded.
Sensors: 207

Test data:
  Traffic: (6838, 12, 207)
  Weather: (6838, 12, 6)

Running final test inference...


IndexError: boolean index did not match indexed array along axis 0; size of axis is 6838 but size of corresponding boolean axis is 6835

In [45]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error


DATA_DIR = Path("../../data/processed")
RESULTS_DIR = Path("../../results")
MODELS_DIR = Path("../../models")

CHECKPOINT_PATH = (
    MODELS_DIR /
    "weather_multi_horizon_transformer_best.pth"
)

SCALER_PATH = (
    DATA_DIR /
    "traffic_scaler.pkl"
)

HORIZONS = [
    "15min",
    "30min",
    "45min",
    "60min"
]

BATCH_SIZE = 16

print("=" * 70)
print("URBANFLOW — FINAL UNTOUCHED TEST EVALUATION")
print("WEATHER-AWARE MULTI-HORIZON TRANSFORMER")
print("=" * 70)


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state_dict = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
    else checkpoint
)


D_MODEL = state_dict[
    "input_projection.weight"
].shape[0]

NUM_SENSORS = state_dict[
    "input_projection.weight"
].shape[1]

WEATHER_DIM = state_dict[
    "weather_projection.weight"
].shape[0]

WEATHER_FEATURES = state_dict[
    "weather_projection.weight"
].shape[1]

FFN_DIM = state_dict[
    "transformer.layers.0.linear1.weight"
].shape[0]

NUM_LAYERS = max(
    int(k.split(".")[2])
    for k in state_dict
    if k.startswith("transformer.layers.")
) + 1

MAX_LEN = state_dict[
    "position_encoding.pe"
].shape[1]

NHEAD = 4

print("\nCheckpoint architecture:")
print("  Sensors:", NUM_SENSORS)
print("  d_model:", D_MODEL)
print("  Weather features:", WEATHER_FEATURES)
print("  Weather embedding:", WEATHER_DIM)
print("  Transformer layers:", NUM_LAYERS)
print("  Attention heads:", NHEAD)
print("  FFN:", FFN_DIM)


class ExactPositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[
            :, :x.size(1), :
        ]



class ExactWeatherMultiHorizonTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.input_projection = nn.Linear(
            NUM_SENSORS,
            D_MODEL
        )

        self.position_encoding = (
            ExactPositionalEncoding(
                D_MODEL,
                MAX_LEN
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=FFN_DIM,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=NUM_LAYERS
        )

        self.weather_projection = nn.Linear(
            WEATHER_FEATURES,
            WEATHER_DIM
        )

        self.fusion_projection = nn.Linear(
            D_MODEL + WEATHER_DIM,
            D_MODEL
        )

        self.head_15 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_30 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_45 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

        self.head_60 = nn.Linear(
            D_MODEL,
            NUM_SENSORS
        )

    def forward(
        self,
        traffic,
        weather
    ):

        x = self.input_projection(
            traffic
        )

        x = self.position_encoding(
            x
        )

        x = self.transformer(
            x
        )

        traffic_features = x[:, -1, :]

        weather_features = self.weather_projection(
            weather[:, -1, :]
        )

        fused = torch.cat(
            [
                traffic_features,
                weather_features
            ],
            dim=1
        )

        fused = self.fusion_projection(
            fused
        )

        return {
            "15min": self.head_15(fused),
            "30min": self.head_30(fused),
            "45min": self.head_45(fused),
            "60min": self.head_60(fused)
        }



final_model = (
    ExactWeatherMultiHorizonTransformer()
)

final_model.load_state_dict(
    state_dict,
    strict=True
)

final_model = final_model.to(DEVICE)
final_model.eval()

print("\nTRAINED MODEL RESTORED SUCCESSFULLY")
print(
    "Parameters:",
    f"{sum(p.numel() for p in final_model.parameters()):,}"
)


traffic_scaler = joblib.load(
    SCALER_PATH
)


test_data = np.load(
    DATA_DIR / "test_sequences.npz"
)

X_test = np.asarray(
    test_data["X"],
    dtype=np.float32
)

weather_test = np.asarray(
    np.load(
        DATA_DIR /
        "weather_test_sequences.npy"
    ),
    dtype=np.float32
)

Y_test = {
    h: np.asarray(
        np.load(
            DATA_DIR /
            f"y_test_{h}.npy"
        ),
        dtype=np.float32
    )
    for h in HORIZONS
}

MASK_test = {
    h: np.isfinite(
        Y_test[h]
    )
    for h in HORIZONS
}

print("\nTest inputs:")
print("  Traffic:", X_test.shape)
print("  Weather:", weather_test.shape)


traffic_test_tensor = torch.from_numpy(
    X_test
).to(DEVICE)

weather_test_tensor = torch.from_numpy(
    weather_test
).to(DEVICE)

test_predictions = {
    h: []
    for h in HORIZONS
}

print("\nRunning final test inference...")

with torch.no_grad():

    for start in range(
        0,
        len(X_test),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(X_test)
        )

        outputs = final_model(
            traffic_test_tensor[start:end],
            weather_test_tensor[start:end]
        )

        for h in HORIZONS:

            test_predictions[h].append(
                outputs[h]
                .detach()
                .cpu()
                .numpy()
            )


final_results = []

for h in HORIZONS:

    predictions_scaled = np.concatenate(
        test_predictions[h],
        axis=0
    )

    targets_scaled = Y_test[h]

    n = min(
        predictions_scaled.shape[0],
        targets_scaled.shape[0]
    )

    predictions_scaled = (
        predictions_scaled[:n]
    )

    targets_scaled = (
        targets_scaled[:n]
    )

    mask = MASK_test[h][:n]

    predictions_mph = (
        traffic_scaler.inverse_transform(
            predictions_scaled
        )
    )

    targets_mph = (
        traffic_scaler.inverse_transform(
            targets_scaled
        )
    )

    y_pred = predictions_mph[mask]
    y_true = targets_mph[mask]

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    final_results.append({
        "horizon": h,
        "MAE_mph": mae,
        "RMSE_mph": rmse,
        "valid_values": int(mask.sum()),
        "samples_aligned": n
    })

    np.save(
        RESULTS_DIR /
        f"weather_transformer_{h}_FINAL_TEST_predictions_mph.npy",
        predictions_mph
    )


final_test_df = pd.DataFrame(
    final_results
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR /
    "weather_transformer_FINAL_TEST_results.csv"
)

final_test_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS — ORIGINAL MPH SCALE")
print("=" * 70)

print(
    final_test_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nSaved:")
print(FINAL_RESULTS_PATH)

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION COMPLETE")
print("NO TRAINING PERFORMED")
print("=" * 70)

URBANFLOW — FINAL UNTOUCHED TEST EVALUATION
WEATHER-AWARE MULTI-HORIZON TRANSFORMER

Checkpoint architecture:
  Sensors: 207
  d_model: 128
  Weather features: 6
  Weather embedding: 32
  Transformer layers: 2
  Attention heads: 4
  FFN: 256

TRAINED MODEL RESTORED SUCCESSFULLY
Parameters: 419,228

Test inputs:
  Traffic: (6838, 12, 207)
  Weather: (6838, 12, 6)

Running final test inference...

FINAL TEST RESULTS — ORIGINAL MPH SCALE
horizon  MAE_mph  RMSE_mph  valid_values  samples_aligned
  15min 317.3307  423.7142       1414845             6835
  30min 362.8594  474.1496       1415466             6838
  45min 402.9258  541.6904       1413603             6829
  60min 486.2226  665.9058       1412982             6826

Saved:
../../results/weather_transformer_FINAL_TEST_results.csv

FINAL TEST EVALUATION COMPLETE
NO TRAINING PERFORMED


In [59]:

import os
import pickle
import numpy as np
import pandas as pd
import torch
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Evaluation device:", device)

print("=" * 70)
print("URBANFLOW — FINAL UNTOUCHED TEST EVALUATION")
print("ORIGINAL WEATHER-AWARE MULTI-HORIZON TRANSFORMER")
print("=" * 70)


MODEL_PATH = "../../models/weather_multi_horizon_transformer_best.pth"
SCALER_PATH = "../../data/processed/traffic_scaler.pkl"

TEST_X_PATH = "../../data/processed/test_sequences.npz"
TEST_WEATHER_PATH = "../../data/processed/weather_test_sequences.npy"

TARGET_DIR = "../../data/processed"
RESULT_DIR = "../../results"

os.makedirs(RESULT_DIR, exist_ok=True)


checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint


import joblib

traffic_scaler = joblib.load(SCALER_PATH)

print("Traffic scaler loaded successfully.")


test_data = np.load(TEST_X_PATH)

X_test = np.asarray(
    test_data["X"],
    dtype=np.float32
)

weather_test = np.asarray(
    np.load(TEST_WEATHER_PATH),
    dtype=np.float32
)

print("Test traffic :", X_test.shape)
print("Test weather :", weather_test.shape)




model = model.to(device)

if "position_encoding.pe" in state_dict:
    model.position_encoding.pe = state_dict[
        "position_encoding.pe"
    ].to(device)

model.load_state_dict(
    state_dict,
    strict=True
)

model.eval()

print()
print("TRAINED MODEL RESTORED SUCCESSFULLY")
print("Model class:", type(model).__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)
print("Positional encoding:", tuple(model.position_encoding.pe.shape))

print()
print("TRAINED MODEL RESTORED SUCCESSFULLY")
print("Model class:", type(model).__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)


traffic_tensor = torch.from_numpy(X_test).to(device)
weather_tensor = torch.from_numpy(weather_test).to(device)


print()
print("Running final untouched test inference...")

predictions = {
    "15min": [],
    "30min": [],
    "45min": [],
    "60min": []
}

BATCH_SIZE = 64

with torch.no_grad():

    for start in range(0, len(X_test), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(X_test))

        traffic_batch = traffic_tensor[start:end]
        weather_batch = weather_tensor[start:end]

        output = model(
            traffic_batch,
            weather_batch
        )

        for horizon in predictions:

            predictions[horizon].append(
                output[horizon]
                .detach()
                .cpu()
                .numpy()
            )

for horizon in predictions:
    predictions[horizon] = np.concatenate(
        predictions[horizon],
        axis=0
    )

print("Inference completed.")


target_arrays = {}

for horizon in ["15min", "30min", "45min", "60min"]:

    path = os.path.join(
        TARGET_DIR,
        f"y_test_{horizon}.npy"
    )

    target_arrays[horizon] = np.asarray(
        np.load(path),
        dtype=np.float32
    )


results = []

for horizon in ["15min", "30min", "45min", "60min"]:

    pred = predictions[horizon]
    true = target_arrays[horizon]

    n_samples = min(
        pred.shape[0],
        true.shape[0]
    )

    pred = pred[:n_samples]
    true = true[:n_samples]

    mask = np.isfinite(true)

    pred_valid = pred[mask]
    true_valid = true[mask]


    pred_mph_matrix = traffic_scaler.inverse_transform(
    pred
    )

    true_mph_matrix = traffic_scaler.inverse_transform(
    true
    )   

    pred_mph = pred_mph_matrix[mask]
    true_mph = true_mph_matrix[mask]

    mae = np.mean(
        np.abs(pred_mph - true_mph)
    )

    rmse = np.sqrt(
        np.mean(
            (pred_mph - true_mph) ** 2
        )
    )

    results.append({
        "horizon": horizon,
        "MAE_mph": mae,
        "RMSE_mph": rmse,
        "valid_values": len(pred_mph),
        "samples_aligned": n_samples
    })

    np.save(
        os.path.join(
            RESULT_DIR,
            f"weather_transformer_FINAL_TEST_{horizon}_predictions_mph.npy"
        ),
        pred_mph
    )


results_df = pd.DataFrame(results)

RESULT_PATH = os.path.join(
    RESULT_DIR,
    "weather_transformer_FINAL_TEST_results.csv"
)

results_df.to_csv(
    RESULT_PATH,
    index=False
)


print()
print("=" * 70)
print("FINAL TEST RESULTS — ORIGINAL MPH SCALE")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print()
print("Saved:")
print(RESULT_PATH)

print()
print("=" * 70)
print("FINAL TEST EVALUATION COMPLETE")
print("NO TRAINING PERFORMED")
print("=" * 70)

Evaluation device: mps
URBANFLOW — FINAL UNTOUCHED TEST EVALUATION
ORIGINAL WEATHER-AWARE MULTI-HORIZON TRANSFORMER
Traffic scaler loaded successfully.
Test traffic : (6838, 12, 207)
Test weather : (6838, 12, 6)

TRAINED MODEL RESTORED SUCCESSFULLY
Model class: WeatherMultiHorizonTransformer
Parameters: 419228
Device: mps
Positional encoding: (1, 12, 128)

TRAINED MODEL RESTORED SUCCESSFULLY
Model class: WeatherMultiHorizonTransformer
Parameters: 419228
Device: mps

Running final untouched test inference...
Inference completed.

FINAL TEST RESULTS — ORIGINAL MPH SCALE
horizon  MAE_mph  RMSE_mph  valid_values  samples_aligned
  15min  11.9106   15.8627       1414845             6835
  30min  12.5384   16.7166       1415466             6838
  45min  12.7671   17.1296       1413603             6829
  60min  12.9665   17.5311       1412982             6826

Saved:
../../results/weather_transformer_FINAL_TEST_results.csv

FINAL TEST EVALUATION COMPLETE
NO TRAINING PERFORMED
